In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 1 (MODIFICADA): COMPILACIÓN CUDA — sd-cli + qwen-tts
# CAMBIOS vs original:
#   - Eliminado constraint de NumPy (Wan2GP requiere numpy==2.1.2)
#   - Eliminado transformers==4.49.0 (Wan2GP requiere 4.54.0)
#   - Ref: https://github.com/deepbeepmeep/Wan2GP/blob/main/requirements.txt
#   - Ref: https://www.kaggle.com/code/jigs0907/ltx-2-5-wan2gp-kaggle
# ═══════════════════════════════════════════════════════════════
import os, sys, subprocess, shutil, json, time

FORCE_REBUILD_SDCPP   = False
FORCE_REBUILD_QWENTTS = False

def sh(cmd, fatal=True, cwd=None):
    print(f"→ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    out = (r.stderr or "") + (r.stdout or "")
    if r.returncode != 0:
        print(out[-6000:])
        if fatal:
            raise RuntimeError(f"Fallo: {cmd}")
    return r

IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/working")
if not IS_KAGGLE:
    raise RuntimeError("Esta celda está pensada para Kaggle.")

SDCPP_DIR    = "/kaggle/working/stable-diffusion.cpp"
QWENTTS_DIR  = "/kaggle/working/qwentts.cpp"
KAGGLE_TMP   = "/kaggle/tmp"
os.makedirs(KAGGLE_TMP, exist_ok=True)

# ── SIN CONSTRAINT DE NUMPY ──
# Wan2GP/requirements.txt especifica numpy==2.1.2 y transformers==4.54.0
# sd-cli y qwen-tts son binarios C++ independientes de NumPy/transformers Python
# Ref: https://github.com/deepbeepmeep/Wan2GP/blob/main/requirements.txt
# --- SIN CONSTRAINT DE NUMPY (Wan2GP requiere numpy==2.1.2) ---
# Ref: https://github.com/deepbeepmeep/Wan2GP/blob/main/requirements.txt
C = ""

sh("apt-get update -y > /dev/null 2>&1")
sh("apt-get install -y git cmake ninja-build ffmpeg libsndfile1 sox > /dev/null 2>&1")
sh(f"pip install -q {C} openai edge-tts gradio psutil kaggle")
sh(f"pip install -q {C} faster-whisper ctranslate2>=4.5.0 \'av==18.1.0\'")
sh("pip install -q 'pyannote.audio>=4.0.1' torchcodec")
sh(f"pip install -q {C} pandas>=2.2.3 nltk omegaconf>=2.3.0")
sh("pip install -q --no-deps git+https://github.com/m-bain/whisperx.git")
# NOTA: NO instalar transformers/diffusers/huggingface_hub aquí.
# El backend TTS usa su propio pin estable de Transformers 5.9.0.
# No se instala aquí ninguna versión 4.x de Transformers.
# Qwen3-TTS Python NO se instala en el entorno principal.
# Su backend Python se ejecuta en un virtualenv aislado creado en Celda 6.
# Esto evita que Qwen cambie transformers / huggingface_hub del resto del proyecto.
sh("pip install -q 'numba>=0.61.0' 'librosa' 'soundfile' 'sox'")
sh("pip install -q mmgp")
sh("pip install -q 'numba>=0.61.0'")
sh(f"pip install -q {C} langdetect")
# comfy-cli YA NO ES NECESARIO (Wan2GP reemplaza ComfyUI para video)

# --- GPU / CUDA ---
r = sh("nvidia-smi -L", fatal=False)
if r.returncode != 0 or not (r.stdout or "").strip():
    raise RuntimeError("No hay GPU (nvidia-smi). Celda CUDA-only.")

r = sh("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", fatal=False)
CUDA_ARCH = ((r.stdout or "").strip().splitlines() or ["7.5"])[0].replace(".", "") or "75"
print(f"CUDA_ARCH: {CUDA_ARCH}")

CUDA_HOME = "/usr/local/cuda"
NVCC = shutil.which("nvcc") or f"{CUDA_HOME}/bin/nvcc"
if not os.path.exists(NVCC):
    raise RuntimeError(f"nvcc no encontrado: {NVCC}")
sh(f"'{NVCC}' --version")

def find_libcuda_so1() -> str:
    r = subprocess.run("ldconfig -p 2>/dev/null | grep 'libcuda.so.1' || true",
                       shell=True, capture_output=True, text=True)
    for line in (r.stdout or "").splitlines():
        if "=>" in line:
            p = line.split("=>", 1)[1].strip()
            if os.path.exists(p): return p
    for p in ["/usr/local/nvidia/lib64/libcuda.so.1",
              "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
              "/lib/x86_64-linux-gnu/libcuda.so.1"]:
        if os.path.exists(p): return p
    r = subprocess.run(
        "find /usr/local/nvidia /usr/lib /lib -name 'libcuda.so.1' 2>/dev/null | head -n 1",
        shell=True, capture_output=True, text=True)
    p = (r.stdout or "").strip()
    return p if p and os.path.exists(p) else ""

LIBCUDA_SO1 = find_libcuda_so1()
print(f"libcuda.so.1: {LIBCUDA_SO1}")
if not LIBCUDA_SO1:
    raise RuntimeError("No se encontró libcuda.so.1")

CUDA_LINK_DIR = "/kaggle/working/cuda-link"
os.makedirs(CUDA_LINK_DIR, exist_ok=True)
LIBCUDA_SO = os.path.join(CUDA_LINK_DIR, "libcuda.so")
if os.path.lexists(LIBCUDA_SO): os.remove(LIBCUDA_SO)
os.symlink(LIBCUDA_SO1, LIBCUDA_SO)

os.environ["LIBRARY_PATH"] = f"{CUDA_LINK_DIR}:/usr/local/nvidia/lib64:" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{CUDA_LINK_DIR}:/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["CPATH"] = f"{CUDA_HOME}/include:" + os.environ.get("CPATH", "")
os.environ["CUDA_PATH"] = CUDA_HOME
os.environ["CUDAToolkit_ROOT"] = CUDA_HOME

def git_head(repo_dir):
    r = subprocess.run(f"git -C '{repo_dir}' rev-parse HEAD", shell=True, capture_output=True, text=True)
    return (r.stdout or "").strip()

# ── sd-cli ──
sdcli_bin = f"{SDCPP_DIR}/build/bin/sd-cli"
fp_path_sd = f"{SDCPP_DIR}/build/BUILD_FINGERPRINT.json"

if FORCE_REBUILD_SDCPP and os.path.exists(SDCPP_DIR):
    sh(f"rm -rf '{SDCPP_DIR}'")

if not os.path.exists(SDCPP_DIR):
    sh(f"git clone --recursive https://github.com/leejet/stable-diffusion.cpp '{SDCPP_DIR}'")
else:
    sh(f"git -C '{SDCPP_DIR}' reset --hard HEAD")
    sh(f"git -C '{SDCPP_DIR}' submodule foreach --recursive 'git reset --hard || true'", fatal=False)
    sh(f"git -C '{SDCPP_DIR}' submodule foreach --recursive 'git clean -fdx || true'", fatal=False)
    _need = FORCE_REBUILD_SDCPP or (not os.path.exists(sdcli_bin))
    if _need:
        sh(f"git -C '{SDCPP_DIR}' pull --recurse-submodules || true", fatal=False)
        sh(f"git -C '{SDCPP_DIR}' submodule update --init --recursive --force")
    else:
        sh(f"git -C '{SDCPP_DIR}' submodule update --init --recursive || true", fatal=False)
        print("✓ Fuentes sd.cpp sin pull (binario existente).")

def sd_up_to_date():
    if not (os.path.exists(sdcli_bin) and os.path.exists(fp_path_sd)): return False
    try:
        fp = json.load(open(fp_path_sd))
        return fp.get("commit") == git_head(SDCPP_DIR) and fp.get("backend") == "cuda" and fp.get("cuda_arch") == CUDA_ARCH
    except: return False

if sd_up_to_date():
    print("✓ sd-cli CUDA ya compilado.")
else:
    sh(f"rm -rf '{SDCPP_DIR}/build'")
    cmake_cmd = (
        f"cd '{SDCPP_DIR}' && cmake -S . -B build -G Ninja "
        f"-DCMAKE_BUILD_TYPE=Release -DSD_CUDA=ON -DGGML_CUDA=ON "
        f"-DCMAKE_CUDA_COMPILER='{NVCC}' -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} "
        f"-DCUDAToolkit_ROOT='{CUDA_HOME}' -DCUDA_cuda_LIBRARY='{LIBCUDA_SO}' "
        f"-DCMAKE_LIBRARY_PATH='{CUDA_LINK_DIR};/usr/local/nvidia/lib64' "
        f"-DCMAKE_EXE_LINKER_FLAGS='-L{CUDA_LINK_DIR} -L/usr/local/nvidia/lib64'"
    )
    sh(cmake_cmd)
    sh(f"cd '{SDCPP_DIR}' && cmake --build build --config Release --target sd-cli")
    if not os.path.exists(sdcli_bin):
        raise RuntimeError("No se generó build/bin/sd-cli")
    json.dump({"commit": git_head(SDCPP_DIR), "backend": "cuda", "cuda_arch": CUDA_ARCH, "libcuda": LIBCUDA_SO1},
              open(fp_path_sd, "w"), indent=2)
    print("¡Build sd-cli CUDA exitoso!")

open(f"{SDCPP_DIR}/build/SDCLI_BACKEND.txt", "w").write("cuda")

# ── qwen-tts ──
qwentts_bin = f"{QWENTTS_DIR}/build/bin/qwen-tts"
fp_path_tts = f"{QWENTTS_DIR}/build/BUILD_FINGERPRINT.json"

if FORCE_REBUILD_QWENTTS and os.path.exists(QWENTTS_DIR):
    sh(f"rm -rf '{QWENTTS_DIR}'")

if not os.path.exists(QWENTTS_DIR):
    sh(f"git clone --recurse-submodules https://github.com/ServeurpersoCom/qwentts.cpp.git '{QWENTTS_DIR}'")

def tts_up_to_date():
    if not (os.path.exists(qwentts_bin) and os.path.exists(fp_path_tts)): return False
    try:
        fp = json.load(open(fp_path_tts))
        return fp.get("commit") == git_head(QWENTTS_DIR) and fp.get("backend") == "cuda" and fp.get("cuda_arch") == CUDA_ARCH
    except: return False

if tts_up_to_date():
    print("✓ qwen-tts CUDA ya compilado.")
else:
    print("→ Compilando qwen-tts...")
    sh(f"rm -rf '{QWENTTS_DIR}/build'")
    cmake_tts = (
        f"cd '{QWENTTS_DIR}' && cmake -S . -B build -G Ninja "
        f"-DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON "
        f"-DCMAKE_CUDA_COMPILER='{NVCC}' -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} "
        f"-DCUDAToolkit_ROOT='{CUDA_HOME}' "
        f"-DCMAKE_LIBRARY_PATH='{CUDA_LINK_DIR};/usr/local/nvidia/lib64' "
        f"-DCUDA_cuda_LIBRARY='{LIBCUDA_SO}' "
        f"-DCMAKE_EXE_LINKER_FLAGS='-L{CUDA_LINK_DIR} -L/usr/local/nvidia/lib64 -lcuda' "
        f"-DCMAKE_SHARED_LINKER_FLAGS='-L{CUDA_LINK_DIR} -L/usr/local/nvidia/lib64 -lcuda'"
    )
    r_cmake = sh(cmake_tts, fatal=False)
    if r_cmake.returncode != 0:
        print("⚠ CMake falló. Fallback CPU...")
        sh(f"cd '{QWENTTS_DIR}' && cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=OFF", fatal=False)
    sh(f"cd '{QWENTTS_DIR}' && cmake --build build --config Release", fatal=False)
    if not os.path.exists(qwentts_bin):
        r_find = subprocess.run(f"find '{QWENTTS_DIR}' -maxdepth 5 -type f -name 'qwen-tts' 2>/dev/null",
                                shell=True, capture_output=True, text=True)
        found = (r_find.stdout or "").strip().splitlines()
        if found:
            os.makedirs(os.path.dirname(qwentts_bin), exist_ok=True)
            shutil.copy2(found[0], qwentts_bin)
    if os.path.exists(qwentts_bin):
        os.chmod(qwentts_bin, 0o755)
        json.dump({"commit": git_head(QWENTTS_DIR), "backend": "cuda", "cuda_arch": CUDA_ARCH},
                  open(fp_path_tts, "w"), indent=2)
        print("¡Build qwen-tts exitoso!")
    else:
        print("⚠ qwen-tts no compiló. Edge-tts como fallback.")
"""
# ── ComfyUI (se mantiene para compatibilidad, pero NO se usa para LTX-2.5) ──
COMFYUI_DIR = "/kaggle/working/ComfyUI"
if not os.path.isdir(COMFYUI_DIR):
    print("Instalando ComfyUI...")
    sh(f"yes | comfy --workspace='{COMFYUI_DIR}' install --nvidia", fatal=False)
else:
    print("✓ ComfyUI ya instalado.")

COMFYUI_GGUF_DIR = f"{COMFYUI_DIR}/custom_nodes/ComfyUI-GGUF"
if not os.path.isdir(COMFYUI_GGUF_DIR):
    sh(f"git clone https://github.com/city96/ComfyUI-GGUF '{COMFYUI_GGUF_DIR}'", fatal=False)
    sh(f"cd '{COMFYUI_GGUF_DIR}' && pip install -q -r requirements.txt", fatal=False)
"""

# ── Wan2GP (código fuente para LTX-2.5) ──────────────────────────────
# Sin esto, /kaggle/working/Wan2GP/models/ltx2/ no existe en entornos nuevos.
WAN2GP_DIR    = "/kaggle/working/Wan2GP"
WAN2GP_REPO   = "https://github.com/deepbeepmeep/Wan2GP"
WAN2GP_COMMIT = ""   # ← pega aquí el hash de tu entorno que funciona (ver abajo). Vacío = main
_WAN2GP_MARK  = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx2_handler.py")

if os.path.isfile(_WAN2GP_MARK):
    print(f"✓ Wan2GP ya presente ({git_head(WAN2GP_DIR)[:10] or 'sin .git'}).")
else:
    print("Clonando Wan2GP...")
    _src = "/kaggle/tmp/Wan2GP_src"
    sh(f"rm -rf '{_src}'")
    sh(f"git clone {WAN2GP_REPO} '{_src}'")
    if WAN2GP_COMMIT:
        sh(f"git -C '{_src}' checkout {WAN2GP_COMMIT}")
    os.makedirs(WAN2GP_DIR, exist_ok=True)
    # Fusiona sin pisar lo que ya haya (symlinks de models/ckpts, script, defaults)
    sh(f"cp -a '{_src}/.' '{WAN2GP_DIR}/'")
    sh(f"rm -rf '{_src}'")
    if not os.path.isfile(_WAN2GP_MARK):
        raise RuntimeError("Wan2GP clonado pero falta models/ltx2/ltx2_handler.py "
                           "(¿commit incorrecto o cambió la estructura del repo?)")
    print(f"✅ Wan2GP listo ({git_head(WAN2GP_DIR)[:10]}).")

# Permisos
for _bin in [sdcli_bin, qwentts_bin]:
    if os.path.exists(_bin): os.chmod(_bin, 0o755)
for _bin_dir in [f"{SDCPP_DIR}/build/bin", f"{QWENTTS_DIR}/build/bin"]:
    if os.path.isdir(_bin_dir):
        for fname in os.listdir(_bin_dir):
            fpath = os.path.join(_bin_dir, fname)
            if os.path.isfile(fpath): os.chmod(fpath, 0o755)

print("\nVerificando sd-cli...")
r = subprocess.run([sdcli_bin, "--help"], capture_output=True, text=True)
help_txt = (r.stdout or "") + (r.stderr or "")
print(f"OK: {sdcli_bin} | --llm_vision: {'SÍ' if '--llm_vision' in help_txt else 'NO'}")

if os.path.exists(qwentts_bin):
    print(f"OK: {qwentts_bin}")
else:
    print("qwen-tts no disponible.")

# ── FIX CRÍTICO: Restaurar Gradio tras qwen-tts-compat ──
sh("pip install -q --force-reinstall 'gradio>=6.0.0,<7.0.0'")
print("✅ Gradio reinstalado correctamente")

print("\n✅ CELDA 1 COMPLETADA (stack TTS fijado y sin mezclar Transformers 4.x/5.x).")
print("   Qwen3-TTS Python quedará aislado en /tmp/qwen3_tts_env")
print("   Reinicia el kernel después de ejecutar todas las celdas.")
time.sleep(2)
os.kill(os.getpid(), 9)


→ apt-get update -y > /dev/null 2>&1
→ apt-get install -y git cmake ninja-build ffmpeg libsndfile1 sox > /dev/null 2>&1
→ pip install -q  openai edge-tts gradio psutil kaggle
→ pip install -q  faster-whisper ctranslate2>=4.5.0 'av==18.1.0'
→ pip install -q 'pyannote.audio>=4.0.1' torchcodec
→ pip install -q  pandas>=2.2.3 nltk omegaconf>=2.3.0
→ pip install -q --no-deps git+https://github.com/m-bain/whisperx.git
→ pip install -q 'numba>=0.61.0' 'librosa' 'soundfile' 'sox'
→ pip install -q mmgp
→ pip install -q 'numba>=0.61.0'
→ pip install -q  langdetect
→ nvidia-smi -L
→ nvidia-smi --query-gpu=compute_cap --format=csv,noheader
CUDA_ARCH: 75
→ '/usr/local/cuda/bin/nvcc' --version
libcuda.so.1: /usr/local/nvidia/lib64/libcuda.so.1
→ git -C '/kaggle/working/stable-diffusion.cpp' reset --hard HEAD
→ git -C '/kaggle/working/stable-diffusion.cpp' submodule foreach --recursive 'git reset --hard || true'
→ git -C '/kaggle/working/stable-diffusion.cpp' submodule foreach --recursive 'git clean 

In [1]:
# ═══════════════════════════════════════════════════════════════
# CELDA 2: CONFIGURACIÓN GENERAL — Sistema de Audiciones (Got Talent / La Voz)
# Plantillas · Selectores globales · Prompts oficiales · Voces · Persistencia
# ═══════════════════════════════════════════════════════════════

import os, gc, json, subprocess, shutil, time, random
from IPython.display import display, Javascript

IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/working")
KAGGLE_INPUT_DIR = "/kaggle/input"
KAGGLE_TMP_DIR   = "/kaggle/tmp"
os.makedirs(KAGGLE_TMP_DIR, exist_ok=True)

if not IS_KAGGLE:
    raise RuntimeError("Notebook diseñado para Kaggle.")

PLATFORM   = "kaggle"
OUTPUT_DIR = "/kaggle/working/VideoAutomationProject"
SDCPP_DIR  = "/kaggle/working/stable-diffusion.cpp"
QWENTTS_DIR = "/kaggle/working/qwentts.cpp"
COMFYUI_DIR = "/kaggle/working/ComfyUI"

os.makedirs(OUTPUT_DIR, exist_ok=True)
PLANTILLAS_DIR = f"{OUTPUT_DIR}/plantillas"
VIDEOS_DIR     = f"{OUTPUT_DIR}/videos"
LOCAL_MODELS_DIR = f"{OUTPUT_DIR}/local_models"          # caché persistente 20GB
LOCAL_MODELS_TMP_DIR = f"{KAGGLE_TMP_DIR}/local_models_tmp"  # scratch ~60GB no persistente (LTX pesado)
for d in [PLANTILLAS_DIR, VIDEOS_DIR, LOCAL_MODELS_DIR, LOCAL_MODELS_TMP_DIR]:
    os.makedirs(d, exist_ok=True)

LOG_FILE = f"{OUTPUT_DIR}/run.log"
os.environ["PROJECT_LOG_FILE"] = LOG_FILE
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        f.write("=== SISTEMA DE AUDICIONES — LOG ===\n")

def _append_log(line: str):
    try:
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(line.rstrip() + "\n")
    except Exception:
        pass

def _log_cell2(msg: str):
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    _append_log(line)

_log_cell2(f"OUTPUT_DIR: {OUTPUT_DIR}")
_log_cell2(f"PLANTILLAS_DIR: {PLANTILLAS_DIR}")
_log_cell2(f"VIDEOS_DIR: {VIDEOS_DIR}")
_log_cell2(f"LOCAL_MODELS_DIR (persistente, 20GB compartidos): {LOCAL_MODELS_DIR}")
_log_cell2(f"LOCAL_MODELS_TMP_DIR (scratch /kaggle/tmp, ~60GB, NO persistente): {LOCAL_MODELS_TMP_DIR}")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
CPU_THREADS = max(1, (os.cpu_count() or 2) // 2)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
SDCLI_THREADS = max(1, os.cpu_count() or 2)

SDCLI_GPU_AVAILABLE = shutil.which("nvidia-smi") is not None
if SDCLI_GPU_AVAILABLE:
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
        SDCLI_GPU_AVAILABLE = r.returncode == 0 and bool((r.stdout or "").strip())
    except Exception:
        SDCLI_GPU_AVAILABLE = False

# ---------------------------------------------------------------
# API KEY xKIRO
# ---------------------------------------------------------------
XKIRO_API_KEY = "sk-xt-4a5373fc29fc7292c082ad27e10b5333952cacf3fbf30c1c"
XKIRO_BASE_URL = "https://api.xkiro.com/v1"
os.environ["XKIRO_API_KEY"] = XKIRO_API_KEY

XKIRO_MODEL_OPTIONS = {
    "🇫🇷 Mistral Large 3 (MoE, el más capaz)":            "mistralai/mistral-large-2512",
    "🇫🇷 Mistral Medium 3.5 (agentic, texto+imagen)":     "mistralai/mistral-medium-3.5",
    "🇫🇷 Mistral Small 4 (instruct+razonamiento+código)": "mistralai/mistral-small-2603",
    "🇫🇷 Codestral (código)":                             "mistralai/codestral-2508",
    "🇫🇷 Devstral 2 Medium (agentic, código)":            "mistralai/devstral-medium",
    "🇫🇷 Ministral 3 14B (texto+visión)":                 "mistralai/ministral-14b",
    "🇫🇷 Ministral 3 8B (eficiente)":                     "mistralai/ministral-8b",
    "🇫🇷 Ministral 3 3B (ultraligero)":                   "mistralai/ministral-3b",
    "🌀 MiniMax M3 (1M contexto, frontier)":               "minimax/minimax-m3:free",
    "🌀 MiniMax M2.7":                                      "minimax/minimax-m2.7:free",
    "🌀 MiniMax M2.7 Highspeed":                            "minimax/minimax-m2.7-highspeed:free",
    "🌀 MiniMax M2.5 (código)":                             "minimax/minimax-m2.5:free",
    "🌀 MiniMax M2.5 Highspeed":                            "minimax/minimax-m2.5-highspeed:free",
    "🌀 MiniMax M2.1 (razonamiento)":                       "minimax/minimax-m2.1:free",
    "🌀 MiniMax M2.1 Highspeed":                            "minimax/minimax-m2.1-highspeed:free",
    "🌀 MiniMax M2 (agentic, function calling)":            "minimax/minimax-m2:free",
    "🟣 Qwen3.8 Omni Flash (multimodal, 1M)":               "qwen/qwen3.8-omni-flash:free",
    "🟣 Qwen3.8 Max (flagship, 1M) [RECOMENDADO]":          "qwen/qwen3.8-max:free",
    "🟣 Qwen3.7 Max":                                       "qwen/qwen3.7-max:free",
    "🟣 Qwen3.7 Plus (económico)":                          "qwen/qwen3.7-plus:free",
    "🟣 Qwen3.7 Flash (vision-language)":                   "qwen/qwen3.7-flash:free",
    "🟣 Qwen3.6 Plus (MoE híbrido)":                        "qwen/qwen3.6-plus:free",
    "🟣 Qwen3.6 Max Preview (~1T params)":                  "qwen/qwen3.6-max-preview:free",
    "🟣 Qwen3.6 27B (denso)":                               "qwen/qwen3.6-27b:free",
    "🟣 Qwen3.5 Plus":                                      "qwen/qwen3.5-plus:free",
    "🟣 Qwen3.5 Omni Plus (multimodal)":                    "qwen/qwen3.5-omni-plus:free",
    "🟣 Qwen3.6 35B A3B (multimodal)":                      "qwen/qwen3.6-35b-a3b:free",
    "🟣 Qwen3.5 Flash":                                     "qwen/qwen3.5-flash:free",
    "🟣 Qwen3.5 397B A17B (vision-language)":               "qwen/qwen3.5-397b-a17b:free",
    "🟣 Qwen3.5 Omni Flash":                                "qwen/qwen3.5-omni-flash:free",
    "🟣 Qwen3 Max":                                         "qwen/qwen3-max:free",
    "🟣 Qwen Plus 0728 (razonamiento híbrido)":             "qwen/qwen-plus-2025-07-28:free",
    "🟣 Qwen3 Coder Plus (código)":                         "qwen/qwen3-coder-plus:free",
    "🟣 Qwen3 VL Plus (visión, OCR)":                       "qwen/qwen3-vl-plus:free",
    "🟣 Qwen3 Omni Flash":                                  "qwen/qwen3-omni-flash:free",
    "🔵 SenseNova 6.8 Flash-Lite (multimodal)":             "sensenova/sensenova-6.8-flash-lite",
    "🔵 SenseNova 6.7 Flash-Lite (multimodal)":             "sensenova/sensenova-6.7-flash-lite",
}
DEFAULT_XKIRO_TEXT_MODEL_ID = XKIRO_MODEL_OPTIONS["🟣 Qwen3.5 Flash"]
XKIRO_FALLBACK_POOL = [
    "qwen/qwen3.8-max:free",
    "minimax/minimax-m2:free",
    "mistralai/mistral-large-2512",
]

SDCLI_PATH   = f"{SDCPP_DIR}/build/bin/sd-cli"
QWENTTS_PATH = f"{QWENTTS_DIR}/build/bin/qwen-tts"
try:
    with open(f"{SDCPP_DIR}/build/SDCLI_BACKEND.txt") as _f:
        SDCLI_BACKEND = _f.read().strip() or "cpu"
except Exception:
    SDCLI_BACKEND = "cpu"

QWEN_TTS_AVAILABLE = os.path.exists(QWENTTS_PATH)

# ═══════════════════════════════════════════════════════════════
# 4.1 / 4.2 — SELECTORES GLOBALES DE PERSONAJE (afectan TODOS los prompts)
# ═══════════════════════════════════════════════════════════════
PERSONAJE_OPCIONES = {
    "Niña":   {"sujeto": "niña",   "el_ella": "ella", "articulo": "la",  "concord_a": "a", "genero": "f", "edad_tipo": "infantil"},
    "Niño":   {"sujeto": "niño",   "el_ella": "él",   "articulo": "el",  "concord_a": "o", "genero": "m", "edad_tipo": "infantil"},
    "Chica":  {"sujeto": "chica",  "el_ella": "ella", "articulo": "la",  "concord_a": "a", "genero": "f", "edad_tipo": "adolescente"},
    "Chico":  {"sujeto": "chico",  "el_ella": "él",   "articulo": "el",  "concord_a": "o", "genero": "m", "edad_tipo": "adolescente"},
    "Mujer":  {"sujeto": "mujer",  "el_ella": "ella", "articulo": "la",  "concord_a": "a", "genero": "f", "edad_tipo": "adulto"},
    "Hombre": {"sujeto": "hombre", "el_ella": "él",   "articulo": "el",  "concord_a": "o", "genero": "m", "edad_tipo": "adulto"},
}

def concordar(base: str, sufijo_f: str, sufijo_m: str, genero: str) -> str:
    """Aplica concordancia de género a una palabra base (p.ej. 'atractiv' -> 'atractiva'/'atractivo')."""
    return f"{base}{sufijo_f if genero == 'f' else sufijo_m}"

def frase_el_ella(genero: str, mayus_inicial: bool = True) -> str:
    palabra = "Ella" if genero == "f" else "Él"
    return palabra if mayus_inicial else palabra.lower()

def concursante_articulado(personaje_key: str) -> str:
    """'La niña' / 'El niño' / 'La mujer' ... para insertar directamente en prompts."""
    p = PERSONAJE_OPCIONES[personaje_key]
    return f"{p['articulo']} {p['sujeto']}"

# ═══════════════════════════════════════════════════════════════
# 3. SISTEMA DE PLANTILLAS — estructura persistente en disco
# ═══════════════════════════════════════════════════════════════
PLANTILLA_SUBDIRS = {
    "jueces_imagenes":        "imagenes/jueces",            # J1..J4 (nunca se editan)
    "escenario_lejano":       "imagenes/escenario_lejano",  # plantilla_plano_lejano_1..3
    "escenario_cercano":      "imagenes/escenario_cercano",  # plantilla_plano_cercano
    "voces_jueces":           "voces/jueces",               # clips fuente para clonar (3-10s)
    "reacciones_primera_mitad": "video/reacciones_primera_mitad",
    "reacciones_segunda_mitad": "video/reacciones_segunda_mitad",
}

def crear_plantilla(nombre: str) -> str:
    """Crea la estructura de carpetas de una plantilla nueva. Infinitas plantillas, siempre editables."""
    slug = "".join(c if c.isalnum() or c in "_-" else "_" for c in nombre.strip().lower())
    base = f"{PLANTILLAS_DIR}/{slug}"
    for sub in PLANTILLA_SUBDIRS.values():
        os.makedirs(f"{base}/{sub}", exist_ok=True)
    meta_path = f"{base}/plantilla.json"
    if not os.path.exists(meta_path):
        meta = {
            "nombre": nombre,
            "slug": slug,
            "creada_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "jueces": [{"id": i, "nombre": f"Juez {i}", "voz_clonada": False} for i in range(1, 5)],
            "planos_lejanos_disponibles": 0,
            "reacciones_primera_mitad": [],
            "reacciones_segunda_mitad": [],
        }
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
    _log_cell2(f"[PLANTILLA] Creada/verificada: {slug} → {base}")
    return base

def listar_plantillas() -> list:
    if not os.path.isdir(PLANTILLAS_DIR):
        return []
    out = []
    for slug in sorted(os.listdir(PLANTILLAS_DIR)):
        meta_path = f"{PLANTILLAS_DIR}/{slug}/plantilla.json"
        if os.path.exists(meta_path):
            try:
                meta = json.load(open(meta_path, encoding="utf-8"))
                out.append((meta.get("nombre", slug), slug))
            except Exception:
                out.append((slug, slug))
    return out

def cargar_plantilla(slug: str) -> dict:
    meta_path = f"{PLANTILLAS_DIR}/{slug}/plantilla.json"
    if not os.path.exists(meta_path):
        raise RuntimeError(f"Plantilla '{slug}' no encontrada")
    meta = json.load(open(meta_path, encoding="utf-8"))
    meta["_base_dir"] = f"{PLANTILLAS_DIR}/{slug}"
    return meta

def guardar_plantilla(meta: dict) -> None:
    base = meta.pop("_base_dir", None) or f"{PLANTILLAS_DIR}/{meta['slug']}"
    meta_path = f"{base}/plantilla.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    meta["_base_dir"] = base

# ═══════════════════════════════════════════════════════════════
# 5.2 — PROMPTS OFICIALES DE ETAPA 1 (editables, con placeholders)
# ═══════════════════════════════════════════════════════════════
PROMPT_1_CREACION_BASE = (
    "Créame una imagen de {ART_UNA} {SUJETO} {ATRACTIV} de unos {EDAD} años vestid{CONCORD} "
    "para un concurso de talentos de canto {VESTUARIO}, plano cercano de su torso y rostro, "
    "que esté de frente a la cámara, micrófono en su mano derecha y cerca de su boca, exprecion del rostro tranquilo y relajado, "
    "azul plano, no puede tener microfonos en sus orejas ni audifonos,no puede tener ningun audicular en sus orejas o cualquier elemento tecnologico en sus orejas, máximo realismo, resolución 9:16."
)

PROMPT_2_CUERPO_COMPLETO = (
    "Plano donde se pueda ver todo su cuerpo y zapatos, resolución 9:16."
)

PROMPT_3_INTERCAMBIO_PLANTILLA = (
    "Coloca a {NUEVO_PERSONAJE} de la imagen 1 en el escenario de la imagen 2, elimina a "
    "{PERSONAJE_PLANTILLA} de la imagen 2 y sustitúyela por {NUEVO_PERSONAJE}, no cambies nada "
    "más en la imagen 2, la escala de {NUEVO_PERSONAJE} tiene que ser la misma de el de la de la imagen 2 "
    "para que se vea coherente y realista con la lejanía del plano, mantén el vestuario de "
    "{NUEVO_PERSONAJE} original y el micrófono en la mano derecha. Que se vea muy realista y alta "
    "resolución de detalles."
)

PROMPT_4_CLOSEUP = "Plano más cercano, máximo realismo y detalles."

def render_prompt_1(personaje_key: str, edad: int, vestuario: str) -> str:
    p = PERSONAJE_OPCIONES[personaje_key]
    art_una = "una" if p["genero"] == "f" else "un"
    atractiv = concordar("atractiv", "a", "o", p["genero"])
    concord = concordar("", "a", "o", p["genero"])
    return PROMPT_1_CREACION_BASE.format(
        ART_UNA=art_una, SUJETO=p["sujeto"], ATRACTIV=atractiv,
        EDAD=int(edad), CONCORD=concord, VESTUARIO=vestuario,
    )

def render_prompt_3(personaje_key: str, personaje_plantilla_key: str) -> str:
    nuevo = concursante_articulado(personaje_key)
    plantilla_p = concursante_articulado(personaje_plantilla_key)
    return PROMPT_3_INTERCAMBIO_PLANTILLA.format(
        NUEVO_PERSONAJE=nuevo, PERSONAJE_PLANTILLA=plantilla_p,
    )

# ═══════════════════════════════════════════════════════════════
# 7.5 — PROMPT LIPSYNC DE DIÁLOGOS (Etapas 2 y 4)
# ═══════════════════════════════════════════════════════════════
PROMPT_LIPSYNC_DIALOGO = (
    "{EL_ELLA} hablando, sincronización labial natural y muy realista, no modificar los rasgos "
    "y la apariencia de su rostro. Nunca mira a la cámara y mantiene la mirada al frente siempre, "
    "nunca girar la cabeza ni cambia la mirada, la camara se mantiene estática sin desplasamientos."
)

def render_prompt_lipsync_dialogo(personaje_key: str) -> str:
    p = PERSONAJE_OPCIONES[personaje_key]
    return PROMPT_LIPSYNC_DIALOGO.format(EL_ELLA=frase_el_ella(p["genero"]))

# ═══════════════════════════════════════════════════════════════
# 8.2 — PROMPT DE 5s (plano lejano, doblemente dinámico)
# ═══════════════════════════════════════════════════════════════
PROMPT_5S_PLANO_LEJANO = (
    "{CONCURSANTE_ART} del escenario cantando muy relajada, sus movimientos son muy sutiles y leves, micrófono en su mano derecha cerca de su "
    "boca todo el tiempo, todos en silencio sin aplaudir, prohibido añadir más personas que no estén "
    "en la imagen o personas con escalas grandes que no respeten el plano lejano, la cámara "
    "manteniendo el plano lejano todo el tiempo y se desplaza a {DIRECCION} muy lentamente, no "
    "realices cambios en los letreros de la imagen, que se vea muy real ."
   
)

def render_prompt_5s(personaje_key: str, direccion: str = None) -> str:
    direccion = direccion or random.choice(["izquierda", "derecha"])
    return PROMPT_5S_PLANO_LEJANO.format(
        CONCURSANTE_ART=concursante_articulado(personaje_key).capitalize(),
        DIRECCION=direccion,
    ), direccion

# ═══════════════════════════════════════════════════════════════
# 8.4 — PROMPT LIPSYNC DE CANCIÓN (15s)
# ═══════════════════════════════════════════════════════════════
PROMPT_LIPSYNC_CANCION = (
    "Sincronización labial natural, que el fondo esté animado, máximo realismo, emociones "
    "naturales al cantar."
)

# ═══════════════════════════════════════════════════════════════
# 7.2 / 9 — PROMPTS DE CONTEXTO PARA GUIONES DE DIÁLOGO (xKiro)
#   Entregables creados desde cero según especificación (secciones 7.2 y 9)
# ═══════════════════════════════════════════════════════════════

def build_dialogo_inicial_system_prompt(personaje_key: str, historia_base: str = "") -> str:
    p = PERSONAJE_OPCIONES[personaje_key]
    el_ella = frase_el_ella(p["genero"], mayus_inicial=False)
    return f"""Eres un guionista experto en programas de talento tipo Got Talent / La Voz, especializado en
generar EMPATÍA PROFUNDA y EMOCIÓN para un público mayor de 40 años.

CONTEXTO: Estás escribiendo los DIÁLOGOS INICIALES de una audición, antes de que el concursante
cante. El concursante es {concursante_articulado(personaje_key)}.
{f"Historia personal a incorporar: {historia_base}" if historia_base else ""}

REGLAS OBLIGATORIAS E INVIOLABLES:
1. Idioma: ESPAÑOL, natural y emotivo.
2. DISTRIBUCIÓN DE VOZ: el 65-70% del tiempo total de habla debe ser del CONCURSANTE. Los jueces
   solo hacen preguntas breves (máximo 1-2 frases cada intervención).
3. El concursante se presenta, cuenta su historia personal (dificultades superadas, por qué canta,
   qué significa este momento para {el_ella}), y genera conexión emocional real con el espectador.
4. Los jueces intervienen con preguntas cortas y naturales: "¿Cómo te llamas?", "¿Por qué has venido
   hoy?", "¿Qué vas a cantar?", reacciones breves de sorpresa o ternura.
5. FRAGMENTOS: cada fragmento de texto (hablado por una sola persona) debe poder decirse en MÁXIMO
   15 SEGUNDOS (LTX-Video no puede sincronizar labios en clips más largos). Estima ~2.5 palabras
   por segundo en español hablado pausado y emotivo.
6. DURACIÓN TOTAL: la suma de duraciones estimadas de todos los fragmentos debe rondar los 3.5
   MINUTOS (210 segundos), nunca menos de 120 segundos.
7. TONO: cálido, humano, que genere empatía y conmueva. Nada de humor forzado ni cinismo.
8. Concordancia de género: usa "{el_ella}" y las concordancias correctas para {concursante_articulado(personaje_key)}
   en TODO el guion.
9. FORMATO DE SALIDA: SOLO un JSON válido, sin texto adicional, con esta estructura EXACTA:
{{
  "fragmentos": [
    {{
      "hablante": "concursante" | "juez_1" | "juez_2" | "juez_3" | "juez_4",
      "texto": "texto exacto que dirá, en español",
      "duracion_estimada_s": 12.5,
      "emocion": "nervioso" | "emocionado" | "triste" | "esperanzado" | "agradecido" | "tímido" | "orgulloso"
    }}
  ],
  "duracion_total_estimada_min": 3.5
}}
10. Ningún fragmento puede superar 15.0 segundos estimados. Si una idea es larga, divídela en
    varios fragmentos consecutivos del mismo hablante.
"""

def build_dialogo_final_system_prompt(personaje_key: str) -> str:
    p = PERSONAJE_OPCIONES[personaje_key]
    el_ella = frase_el_ella(p["genero"], mayus_inicial=False)
    return f"""Eres un guionista experto en programas de talento tipo Got Talent / La Voz, especializado en
generar CIERRES EMOTIVOS que conmuevan a un público mayor de 40 años.

CONTEXTO: Estás escribiendo los DIÁLOGOS FINALES de una audición, justo después de que el
concursante ({concursante_articulado(personaje_key)}) ha terminado de cantar. Los jueces van a dar
su veredicto y opinión.

REGLAS OBLIGATORIAS E INVIOLABLES:
1. Idioma: ESPAÑOL, natural y emotivo.
2. DISTRIBUCIÓN DE VOZ: el 70% O MÁS del tiempo total de habla debe ser de LOS JUECES (repartido
   entre juez_1, juez_2, juez_3, juez_4). El concursante solo responde brevemente si es necesario
   (agradecimientos cortos, reacciones de emoción).
3. Los jueces dan su opinión sincera y detallada: valoran la técnica vocal, la emoción transmitida,
   comentan la historia personal del concursante, y generan un CIERRE EMOTIVO que conecte con el
   público (ej. reflexiones sobre la superación, el talento, la segunda oportunidad).
4. Al menos un juez debe mencionar explícitamente la historia personal del concursante para reforzar
   la empatía.
5. FRAGMENTOS: cada fragmento debe poder decirse en MÁXIMO 15 SEGUNDOS. Estima ~2.5 palabras por
   segundo en español hablado pausado y emotivo.
6. DURACIÓN TOTAL: la suma debe rondar los 3.5 MINUTOS (210 segundos), nunca menos de 120 segundos.
7. TONO: cálido, sincero, emotivo, con un cierre que resuene y conmueva, apto para mayores de 40 años.
8. Concordancia de género: usa "{el_ella}" y las concordancias correctas para
   {concursante_articulado(personaje_key)} en TODO el guion.
9. FORMATO DE SALIDA: SOLO un JSON válido, sin texto adicional, MISMA estructura que el guion inicial:
{{
  "fragmentos": [
    {{
      "hablante": "concursante" | "juez_1" | "juez_2" | "juez_3" | "juez_4",
      "texto": "texto exacto que dirá, en español",
      "duracion_estimada_s": 12.5,
      "emocion": "emocionado" | "orgulloso" | "conmovido" | "sorprendido" | "agradecido"
    }}
  ],
  "duracion_total_estimada_min": 3.5
}}
10. Ningún fragmento puede superar 15.0 segundos estimados.
"""

# ═══════════════════════════════════════════════════════════════
# 7.1 — LISTA EXTENSA DE TIPOS DE VOZ PARA QWEN3-TTS (VoiceDesign)
#   Cada entrada genera el string exacto para "--instruct" de qwen-tts
#   (documentado: --instruct "male, young adult, moderate pitch" --lang Spanish)
# ═══════════════════════════════════════════════════════════════
VOICE_DESIGN_CATALOG = {
    # ---------------- NIÑA ----------------
    "Niña — Dulce y tierna":         {"personaje": "Niña", "instruct": "female child, sweet and tender voice, soft pitch, gentle warmth"},
    "Niña — Tímida e inocente":      {"personaje": "Niña", "instruct": "female child, shy and innocent voice, soft and hesitant tone"},
    "Niña — Energética y traviesa":  {"personaje": "Niña", "instruct": "female child, energetic and mischievous voice, playful high pitch"},
    "Niña — Soñadora y poética":     {"personaje": "Niña", "instruct": "female child, dreamy and poetic voice, slow gentle cadence"},
    "Niña — Valiente y decidida":    {"personaje": "Niña", "instruct": "female child, brave and determined voice, firm clear tone"},
    "Niña — Alegre y juguetona":     {"personaje": "Niña", "instruct": "female child, cheerful and playful voice, bright happy tone"},

    # ---------------- NIÑO ----------------
    "Niño — Valiente y aventurero":  {"personaje": "Niño", "instruct": "male child, brave and adventurous voice, confident bright tone"},
    "Niño — Curioso y preguntón":    {"personaje": "Niño", "instruct": "male child, curious and inquisitive voice, quick excited pitch"},
    "Niño — Juguetón y bromista":    {"personaje": "Niño", "instruct": "male child, playful and joking voice, mischievous tone"},
    "Niño — Serio y maduro":         {"personaje": "Niño", "instruct": "male child, serious and mature-sounding voice, calm steady tone"},
    "Niño — Tímido pero expresivo":  {"personaje": "Niño", "instruct": "male child, shy but expressive voice, soft with emotional bursts"},
    "Niño — Enérgico y competitivo": {"personaje": "Niño", "instruct": "male child, energetic and competitive voice, fast excited pitch"},

    # ---------------- CHICA ADOLESCENTE ----------------
    "Chica — Rebelde y segura":      {"personaje": "Chica", "instruct": "female teenager, rebellious and self-assured voice, confident edge"},
    "Chica — Dulce e insegura":      {"personaje": "Chica", "instruct": "female teenager, sweet and insecure voice, soft hesitant tone"},
    "Chica — Apasionada y soñadora": {"personaje": "Chica", "instruct": "female teenager, passionate and dreamy voice, warm emotional tone"},
    "Chica — Sarcástica pero cálida":{"personaje": "Chica", "instruct": "female teenager, sarcastic but warm voice, dry humor undertone"},
    "Chica — Introvertida y profunda":{"personaje": "Chica", "instruct": "female teenager, introverted and thoughtful voice, low soft pitch"},

    # ---------------- CHICO ADOLESCENTE ----------------
    "Chico — Confiado y carismático":{"personaje": "Chico", "instruct": "male teenager, confident and charismatic voice, warm bright tone"},
    "Chico — Torpe pero encantador": {"personaje": "Chico", "instruct": "male teenager, clumsy but charming voice, awkward warm tone"},
    "Chico — Apasionado y romántico":{"personaje": "Chico", "instruct": "male teenager, passionate and romantic voice, soft emotional tone"},
    "Chico — Rebelde y enérgico":    {"personaje": "Chico", "instruct": "male teenager, rebellious and energetic voice, edgy fast pitch"},
    "Chico — Sensible y reflexivo":  {"personaje": "Chico", "instruct": "male teenager, sensitive and reflective voice, slow thoughtful tone"},

    # ---------------- MUJER ----------------
    "Mujer — Cálida y maternal":     {"personaje": "Mujer", "instruct": "adult female, warm and maternal voice, soft nurturing tone, moderate pitch"},
    "Mujer — Profesional y elegante":{"personaje": "Mujer", "instruct": "adult female, professional and elegant voice, clear articulate tone"},
    "Mujer — Sensual y suave":       {"personaje": "Mujer", "instruct": "adult female, sensual and soft voice, low breathy tone"},
    "Mujer — Fuerte y decidida":     {"personaje": "Mujer", "instruct": "adult female, strong and determined voice, firm confident tone"},
    "Mujer — Joven y vibrante":      {"personaje": "Mujer", "instruct": "young adult female, vibrant and lively voice, bright energetic pitch"},
    "Mujer — Madura y sabia":        {"personaje": "Mujer", "instruct": "mature adult female, wise and calm voice, low steady pitch"},
    "Mujer — Nerviosa y emotiva":    {"personaje": "Mujer", "instruct": "adult female, nervous and emotional voice, trembling soft tone"},

    # ---------------- HOMBRE ----------------
    "Hombre — Grave y profundo":     {"personaje": "Hombre", "instruct": "adult male, deep and profound voice, very low pitch, resonant tone"},
    "Hombre — Juvenil y ligero":     {"personaje": "Hombre", "instruct": "young adult male, light and youthful voice, moderate bright pitch"},
    "Hombre — Ronco y curtido":      {"personaje": "Hombre", "instruct": "adult male, rough and weathered voice, gravelly low tone"},
    "Hombre — Elegante y pausado":   {"personaje": "Hombre", "instruct": "adult male, elegant and measured voice, calm articulate tone"},
    "Hombre — Enérgico y firme":     {"personaje": "Hombre", "instruct": "adult male, energetic and firm voice, confident moderate pitch"},
    "Hombre — Emotivo y tembloroso": {"personaje": "Hombre", "instruct": "adult male, emotional and trembling voice, soft unsteady tone"},
    "Hombre — Maduro y sabio":       {"personaje": "Hombre", "instruct": "mature adult male, wise and calm voice, low steady resonant tone"},
}

# Añadir idioma español a todas las entradas (parámetro --lang de qwen-tts)
for _k in VOICE_DESIGN_CATALOG:
    VOICE_DESIGN_CATALOG[_k]["lang"] = "Spanish"

_log_cell2(f"Catálogo de voces (VoiceDesign): {len(VOICE_DESIGN_CATALOG)} variantes en 6 categorías.")

# ═══════════════════════════════════════════════════════════════
# ESTILO VISUAL DEL PROGRAMA (para prompts de escenario/plantilla si se regeneran)
# ═══════════════════════════════════════════════════════════════
AUDICION_STYLE_GUIDE = {
    "resolucion_9_16": ["A", "B"],
    "resolucion_16_9": ["C", "D", "E", "F", "G", "J1", "J2", "J3", "J4"],
    "dim_multiplo": 32,   # OBLIGATORIO por documentación sd.cpp Qwen-Image-2.1
    "cfg_scale_recomendado": 6.0,  # documentado en qwen_image_2.1.md
    "sampling_method": "euler",
}

_log_cell2("\nCELDA 2 COMPLETADA — Sistema de Audiciones configurado.")
_log_cell2(f"sd-cli: {SDCLI_PATH} (existe={os.path.exists(SDCLI_PATH)}) | backend: {SDCLI_BACKEND}")
_log_cell2(f"qwen-tts: {QWENTTS_PATH} (disponible={QWEN_TTS_AVAILABLE})")
_log_cell2(f"ComfyUI: {COMFYUI_DIR} (existe={os.path.isdir(COMFYUI_DIR)})")
_log_cell2(f"Plantillas existentes: {listar_plantillas()}")

[18:03:46] OUTPUT_DIR: /kaggle/working/VideoAutomationProject
[18:03:46] PLANTILLAS_DIR: /kaggle/working/VideoAutomationProject/plantillas
[18:03:46] VIDEOS_DIR: /kaggle/working/VideoAutomationProject/videos
[18:03:46] LOCAL_MODELS_DIR (persistente, 20GB compartidos): /kaggle/working/VideoAutomationProject/local_models
[18:03:46] LOCAL_MODELS_TMP_DIR (scratch /kaggle/tmp, ~60GB, NO persistente): /kaggle/tmp/local_models_tmp
[18:03:46] Catálogo de voces (VoiceDesign): 36 variantes en 6 categorías.
[18:03:46] 
CELDA 2 COMPLETADA — Sistema de Audiciones configurado.
[18:03:46] sd-cli: /kaggle/working/stable-diffusion.cpp/build/bin/sd-cli (existe=True) | backend: cuda
[18:03:46] qwen-tts: /kaggle/working/qwentts.cpp/build/bin/qwen-tts (disponible=True)
[18:03:46] ComfyUI: /kaggle/working/ComfyUI (existe=False)
[18:03:46] Plantillas existentes: [('Mi Plantilla 1', 'mi_plantilla_1')]


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3: UTILIDADES BASE + DISCO + PERSISTENCIA + DESCARGA MODELOS
# (Kaggle-friendly, sin reventar /kaggle/working)
# ═══════════════════════════════════════════════════════════════
import os, re, json, time, shutil, subprocess, math, glob
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

from huggingface_hub import hf_hub_download

# ---------------- LOGS (a archivo + consola) ----------------
PROJECT_LOG_FILE = os.environ.get("PROJECT_LOG_FILE", f"{OUTPUT_DIR}/run.log")

def _append_log(line: str):
    try:
        with open(PROJECT_LOG_FILE, "a", encoding="utf-8") as f:
            f.write(line.rstrip() + "\n")
    except Exception:
        pass

def log(msg: str):
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    _append_log(line)

def tail_log(max_chars: int = 12000) -> str:
    try:
        if not os.path.exists(PROJECT_LOG_FILE):
            return ""
        with open(PROJECT_LOG_FILE, "rb") as f:
            data = f.read()
        text = data.decode("utf-8", errors="ignore")
        return text[-max_chars:]
    except Exception:
        return ""

# ---------------- DISCO (working vs tmp) ----------------
def free_gb(path: str) -> float:
    st = os.statvfs(path)
    return (st.f_bavail * st.f_frsize) / 1024**3

log(f"Disco libre /kaggle/working: {free_gb('/kaggle/working'):.2f} GB")
log(f"Disco libre /kaggle/tmp:     {free_gb('/kaggle/tmp'):.2f} GB")

# ---------------- Kaggle INPUT find (dataset adjunto) ----------------
def find_in_kaggle_input(filename: str) -> str:
    if not os.path.isdir(KAGGLE_INPUT_DIR):
        return ""
    for root, _, files in os.walk(KAGGLE_INPUT_DIR):
        if filename in files:
            p = os.path.join(root, filename)
            if os.path.getsize(p) > 1024 * 1024:
                return p
    return ""

# ---------------- HF download (xet-safe) ----------------
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # acelera si hf_transfer está disponible

def hf_download_if_missing(repo_id: str, filename: str, dst_dir: str) -> str:
    """
    Descarga vía huggingface_hub (maneja xet / LFS). Si el archivo ya existe en dst_dir, lo reutiliza.
    """
    os.makedirs(dst_dir, exist_ok=True)
    out_path = os.path.join(dst_dir, filename)
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1024 * 1024:
        return out_path

    # Si está en /kaggle/input, usarlo sin descargar:
    k_in = find_in_kaggle_input(filename)
    if k_in:
        return k_in

    log(f"[HF] Descargando {repo_id}/{filename} → {out_path}")
    tmp = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=dst_dir, local_dir_use_symlinks=False)
    # hf_hub_download devuelve path real; asegurar nombre esperado:
    if tmp != out_path and os.path.exists(tmp):
        shutil.copyfile(tmp, out_path)
    return out_path

# ---------------- PUBLICAR MODELOS COMO DATASET (opcional) ----------------
# (Reusa tu función original, pero generalizada. Kaggle dataset = persistencia gratuita fuera de /working.)
def publish_dir_as_kaggle_dataset(dataset_slug: str, src_dir: str, title: str = None, update_existing: bool = True) -> str:
    if PLATFORM != "kaggle":
        return "Solo Kaggle."

    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
    except Exception as e:
        return (f"No se pudieron leer credenciales desde Kaggle Secrets ({e}). "
                f"Add-ons → Secrets: KAGGLE_USERNAME y KAGGLE_KEY.")

    staging = f"{OUTPUT_DIR}/_dataset_staging/{dataset_slug}"
    os.makedirs(staging, exist_ok=True)

    # Copiar todo (solo archivos) para versionar
    for root, _, files in os.walk(src_dir):
        rel = os.path.relpath(root, src_dir)
        for fn in files:
            src = os.path.join(root, fn)
            dst = os.path.join(staging, rel, fn)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
                shutil.copyfile(src, dst)

    meta = {
        "title": title or dataset_slug,
        "id": f"{os.environ.get('KAGGLE_USERNAME', 'usuario')}/{dataset_slug}",
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(os.path.join(staging, "dataset-metadata.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    if update_existing:
        cmd = f"kaggle datasets version -p '{staging}' -m 'Actualización modelos/pipeline' -r zip"
    else:
        cmd = f"kaggle datasets create -p '{staging}' -r zip"

    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout + "\n" + r.stderr

# ---------------- MODELOS: Qwen-Image 2.1 (sd-cli) ----------------
QWEN_IMAGE_REPO = "leejet/Qwen-Image-2.1-GGUF"
QWEN_VL_REPO    = "Qwen/Qwen3-VL-8B-Instruct-GGUF"
QWEN_VAE_REPO   = "Comfy-Org/Qwen-Image-2.1"
QWEN_VAE_FILE   = "qwen_image_2.1_vae_bf16.safetensors"

QWEN_IMAGE_DIFFUSION_QUANTS = {
    "Q2_K":  "qwen_image_2.1-Q2_K.gguf",
    "Q3_K":  "qwen_image_2.1-Q3_K.gguf",
    "Q4_0":  "qwen_image_2.1-Q4_0.gguf",
    "Q4_K":  "qwen_image_2.1-Q4_K.gguf",
    "Q5_0":  "qwen_image_2.1-Q5_0.gguf",
    "Q6_K":  "qwen_image_2.1-Q6_K.gguf",
    "Q8_0":  "qwen_image_2.1-Q8_0.gguf",
}

QWEN_VL_ENCODERS = {
    "Q4_K_M": "Qwen3VL-8B-Instruct-Q4_K_M.gguf",
    "Q8_0":   "Qwen3VL-8B-Instruct-Q8_0.gguf",
    "F16":    "Qwen3VL-8B-Instruct-F16.gguf",
}

QWEN_VL_MMPROJ = {
    "F16":  "mmproj-Qwen3VL-8B-Instruct-F16.gguf",
    "Q8_0": "mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf",
}

def ensure_qwen_image21_models(diffusion_quant="Q8_0", encoder_quant="Q4_K_M", mmproj_quant="Q8_0") -> Dict[str, str]:
    """
    Descarga/ubica diffusion GGUF + encoder GGUF + mmproj + VAE.
    Para edición se requiere mmproj con --llm_vision (documentado). <!--citation:1-->
    """
    diff_file = QWEN_IMAGE_DIFFUSION_QUANTS[diffusion_quant]
    enc_file  = QWEN_VL_ENCODERS[encoder_quant]
    mm_file   = QWEN_VL_MMPROJ[mmproj_quant]

    diff_path = hf_download_if_missing(QWEN_IMAGE_REPO, diff_file, LOCAL_MODELS_DIR)
    enc_path  = hf_download_if_missing(QWEN_VL_REPO, enc_file, LOCAL_MODELS_DIR)
    mm_path   = hf_download_if_missing(QWEN_VL_REPO, mm_file, LOCAL_MODELS_DIR)
    vae_path  = hf_download_if_missing(QWEN_VAE_REPO, f"vae/{QWEN_VAE_FILE}", LOCAL_MODELS_DIR)

    # hf_hub_download con subcarpeta 'vae/' queda en LOCAL_MODELS_DIR/vae/...
    if "/vae/" in vae_path or vae_path.endswith(QWEN_VAE_FILE):
        pass
    else:
        # fallback
        pass

    return {"diffusion": diff_path, "encoder": enc_path, "mmproj": mm_path, "vae": vae_path}

# ---------------- MODELOS: Qwen3-TTS (GGUF) ----------------
QWEN_TTS_REPO = "Serveurperso/Qwen3-TTS-GGUF"  # listado de talkers + tokenizers en HF <!--citation:7-->

QWEN_TTS_PRESETS = {
    "BASE_Q8": {
        "talker": "qwen-talker-1.7b-base-Q8_0.gguf",
        "codec":  "qwen-tokenizer-12hz-Q8_0.gguf",
        "mode": "base"
    },
    "BASE_Q4": {
        "talker": "qwen-talker-1.7b-base-Q4_K_M.gguf",
        "codec":  "qwen-tokenizer-12hz-Q4_K_M.gguf",
        "mode": "base"
    },
    "VOICEDESIGN_Q8": {
        "talker": "qwen-talker-1.7b-voicedesign-Q8_0.gguf",
        "codec":  "qwen-tokenizer-12hz-Q8_0.gguf",
        "mode": "voicedesign"
    },
    "CUSTOMVOICE_Q8": {
        "talker": "qwen-talker-1.7b-customvoice-Q8_0.gguf",
        "codec":  "qwen-tokenizer-12hz-Q8_0.gguf",
        "mode": "customvoice"
    },
}

def ensure_qwen_tts_models(preset_key="VOICEDESIGN_Q8") -> Dict[str, str]:
    p = QWEN_TTS_PRESETS[preset_key]
    talker = hf_download_if_missing(QWEN_TTS_REPO, p["talker"], LOCAL_MODELS_DIR)
    codec  = hf_download_if_missing(QWEN_TTS_REPO, p["codec"],  LOCAL_MODELS_DIR)
    return {"talker": talker, "codec": codec, "mode": p["mode"]}

# ---------------- Workaround PR #483 (ComfyUI-GGUF) ----------------
def patch_comfyui_gguf_for_qwen_image21(comfy_custom_nodes_dir: str):
    """
    Workaround documentado en PR #483: añade detección de arquitectura qwen_image cuando
    sd.cpp exporta GGUF sin general.architecture. <!--citation:6-->
    Solo se aplica si tienes ComfyUI-GGUF instalado (opcional).
    """
    gguf_dir = os.path.join(comfy_custom_nodes_dir, "ComfyUI-GGUF")
    convert_py = os.path.join(gguf_dir, "tools", "convert.py")
    if not os.path.exists(convert_py):
        return "ComfyUI-GGUF no encontrado; patch omitido."

    txt = open(convert_py, "r", encoding="utf-8", errors="ignore").read()
    if "class ModelQwenImage" in txt or "ModelQwenImage" in txt:
        return "Patch PR #483 ya presente (ModelQwenImage)."

    # Inserción mínima: agregar clase cerca de otros ModelTemplate detectors
    # (No cambia nada más; solo habilita fingerprint por keys).
    inject = """
class ModelQwenImage(ModelTemplate):
    arch = "qwen_image"
    keys_detect = [
        ("img_in.weight", "modulation.1.weight"),
        ("img_in.weight", "img_mlp.gate_layer.weight"),
    ]
"""
    # Heurística: insertar antes de la lista/registro final si existe, sino al final.
    marker = "class Model"
    idx = txt.rfind("class Model")
    if idx == -1:
        txt2 = txt + "\n" + inject + "\n"
    else:
        # insertar antes del último bloque de modelos para minimizar riesgos
        txt2 = txt[:idx] + inject + "\n" + txt[idx:]

    open(convert_py, "w", encoding="utf-8").write(txt2)
    return "Patch PR #483 aplicado en tools/convert.py (ModelQwenImage)."

log("CELDA 3 COMPLETADA.")

[18:03:52] Disco libre /kaggle/working: 17.72 GB
[18:03:52] Disco libre /kaggle/tmp:     1068.97 GB
[18:03:52] CELDA 3 COMPLETADA.


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELDA 4: ESTRUCTURAS DE PROYECTO (VIDEO) + TIMELINE JSON + RENDER HTML
# ═══════════════════════════════════════════════════════════════
import datetime
import random

def _slug(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"[^a-z0-9áéíóúüñ_-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:80] or "video"

def now_slug() -> str:
    return datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")

@dataclass
class TimelineClip:
    clip_id: str
    tipo: str                 # "dialogo" | "plano_lejano_5s" | "lipsync_15s" | "reaccion"
    hablante: str             # concursante/juez_1..4/""
    dur_s: float
    path_video: str
    path_audio_ref: str = ""  # si aplica (p.ej. lipsync con audio de canción recortado)
    estado: str = "pendiente" # pendiente/generando/completado/error
    meta: dict = None

def timeline_to_json(clips: List[TimelineClip]) -> list:
    out = []
    t = 0.0
    for c in clips:
        d = float(c.dur_s)
        out.append({
            "clip_id": c.clip_id,
            "tipo": c.tipo,
            "hablante": c.hablante,
            "start": t,
            "end": t + d,
            "dur_s": d,
            "path_video": c.path_video,
            "path_audio_ref": c.path_audio_ref,
            "estado": c.estado,
            "meta": c.meta or {},
        })
        t += d
    return out

def render_timeline_html(timeline_items: list, width_px: int = 1100) -> str:
    """
    Timeline estilo editor: una sola pista (simplificado, pero muy claro),
    coloreado por tipo.
    """
    if not timeline_items:
        return "<div style='font-family:monospace'>Timeline vacío.</div>"

    total = float(timeline_items[-1]["end"]) if timeline_items else 1.0
    if total <= 0:
        total = 1.0

    colors = {
        "dialogo": "#4e79a7",
        "plano_lejano_5s": "#59a14f",
        "lipsync_15s": "#f28e2b",
        "reaccion": "#af7aa1",
    }

    bars = []
    for it in timeline_items:
        start = float(it["start"])
        end   = float(it["end"])
        left  = int((start / total) * width_px)
        w     = max(2, int(((end - start) / total) * width_px))
        tipo  = it["tipo"]
        c     = colors.get(tipo, "#999")
        label = f"{tipo} | {it.get('hablante','')} | {it['dur_s']:.2f}s | {it.get('estado','')}"
        bars.append(
            f"<div title='{label}' "
            f"style='position:absolute;left:{left}px;top:0px;width:{w}px;height:34px;"
            f"background:{c};border:1px solid #111;border-radius:6px;overflow:hidden;"
            f"color:#111;font-family:monospace;font-size:11px;line-height:14px;padding:3px;'>"
            f"{tipo}<br>{it.get('hablante','')}"
            f"</div>"
        )
    html = (
        "<div style='font-family:monospace;margin:4px 0;'>"
        f"<div style='position:relative;width:{width_px}px;height:38px;background:#222;border-radius:8px;border:1px solid #444;'>"
        + "".join(bars) +
        "</div>"
        f"<div style='color:#333;font-size:12px;margin-top:4px;'>Duración total: {total:.2f}s</div>"
        "</div>"
    )
    return html

# ---------------- VideoProject (persistente) ----------------
class VideoProject:
    def __init__(self, video_dir: str):
        self.video_dir = video_dir
        os.makedirs(video_dir, exist_ok=True)

        self.paths = {
            "estado":  os.path.join(video_dir, "estado.json"),
            "prompts": os.path.join(video_dir, "prompts.json"),
            "params":  os.path.join(video_dir, "params.json"),
            "timeline": os.path.join(video_dir, "timeline.json"),
        }
        self.estado = {}
        self.prompts = {}
        self.params = {}
        self.timeline_items = []

        self._load()

    def _load(self):
        for k, p in self.paths.items():
            if os.path.exists(p):
                try:
                    data = json.load(open(p, "r", encoding="utf-8"))
                    if k == "estado": self.estado = data
                    if k == "prompts": self.prompts = data
                    if k == "params": self.params = data
                    if k == "timeline": self.timeline_items = data
                except Exception:
                    pass

    def save(self):
        json.dump(self.estado, open(self.paths["estado"], "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        json.dump(self.prompts, open(self.paths["prompts"], "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        json.dump(self.params, open(self.paths["params"], "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        json.dump(self.timeline_items, open(self.paths["timeline"], "w", encoding="utf-8"), ensure_ascii=False, indent=2)

def create_new_video_project(template_slug: str, personaje_key: str) -> VideoProject:
    vid_slug = f"{now_slug()}_{_slug(template_slug)}_{_slug(personaje_key)}"
    vd = os.path.join(VIDEOS_DIR, vid_slug)
    for sub in ["etapa1_imagenes","etapa2_dialogos","etapa3_cancion","etapa4_final","tmp","comfy_input","comfy_output"]:
        os.makedirs(os.path.join(vd, sub), exist_ok=True)
    vp = VideoProject(vd)
    vp.estado.setdefault("aprobados", {})
    vp.save()
    return vp

log("CELDA 4 COMPLETADA.")

[18:03:57] CELDA 4 COMPLETADA.


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELDA 5: ETAPA 1 — GENERACIÓN DE IMÁGENES (A→G) CON sd-cli (Qwen-Image-2.1)
# Implementa EXACTAMENTE la lógica de la especificación:
#  - A: text2img (9:16)
#  - B: edición desde A usando -r y prompt 2
#  - C/E/F/G: edición de plantilla usando SOLO B (como identidad) + plantilla como base (2 refs: -r B -r plantilla)
#  - D: edición desde C (close-up)
# Documentación sd.cpp Qwen-Image-2.1 (flags -r / --llm_vision / cfg 6 / euler / dims /32). <!--citation:1-->
# ═══════════════════════════════════════════════════════════════
import tempfile
import subprocess

def _ffprobe_duration(path: str) -> Optional[float]:
    try:
        cmd = ["ffprobe","-v","error","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1", path]
        r = subprocess.run(cmd, capture_output=True, text=True)
        return float((r.stdout or "").strip())
    except Exception:
        return None

def sdcli_help_flags(sdcli_path: str) -> set:
    try:
        r = subprocess.run([sdcli_path, "--help"], capture_output=True, text=True, timeout=30)
        txt = (r.stdout or "") + (r.stderr or "")
        return set(re.findall(r'(--[a-zA-Z0-9][a-zA-Z0-9_-]*)', txt))
    except Exception:
        return set()

class SDQwenImage21:
    def __init__(self, sdcli_path: str, model_paths: Dict[str,str]):
        self.sdcli = sdcli_path
        self.mp = model_paths
        self.flags = sdcli_help_flags(sdcli_path)

    def _supports(self, flag: str) -> bool:
        return (not self.flags) or (flag in self.flags)

    def _run(self, args: List[str]):
        log("sd-cli CMD: " + " ".join(shlex_quote(a) for a in args))
        t0 = time.time()
        r = subprocess.run(args, capture_output=True, text=True)
        dt = time.time() - t0
        if r.returncode != 0:
            tail = ((r.stderr or "") + "\n" + (r.stdout or ""))[-4000:]
            raise RuntimeError(f"sd-cli falló (rc={r.returncode}) en {dt:.1f}s:\n{tail}")
        log(f"sd-cli OK en {dt:.1f}s")
        return r

    def generate_text2img(self, prompt: str, out_png: str,
                          w: int, h: int, steps: int, cfg: float, seed: int = -1,
                          offload_to_cpu: bool = True, use_fa: bool = True):
        os.makedirs(os.path.dirname(out_png), exist_ok=True)

        # Base según doc qwen_image_2.1.md: diffusion+vae+llm, cfg=6.0, euler, --offload-to-cpu, y --fa si existe. <!--citation:1-->
        cmd = [
            self.sdcli,
            "--diffusion-model", self.mp["diffusion"],
            "--vae", self.mp["vae"],
            "--llm", self.mp["encoder"],
            "-p", prompt,
            "--cfg-scale", str(cfg),
            "--sampling-method", "euler",
            "--steps", str(int(steps)),
            "--seed", str(int(seed)),
            "-o", out_png,
            "-W", str(int(w)),
            "-H", str(int(h)),
        ]

        if offload_to_cpu and self._supports("--offload-to-cpu"):
            cmd.append("--offload-to-cpu")

        # FA: según build puede ser --fa o --diffusion-fa; priorizar --fa por doc. <!--citation:1-->
        if use_fa:
            if self._supports("--fa"): cmd.append("--fa")
            elif self._supports("--diffusion-fa"): cmd.append("--diffusion-fa")

        self._run(cmd)
        if not os.path.exists(out_png) or os.path.getsize(out_png) < 5000:
            raise RuntimeError("No se generó PNG válido.")
        return out_png

    def edit_with_refs(self, prompt: str, refs: List[str], out_png: str,
                       w: int, h: int, steps: int, cfg: float, seed: int = -1,
                       offload_to_cpu: bool = True, use_fa: bool = True):
        os.makedirs(os.path.dirname(out_png), exist_ok=True)

        cmd = [
            self.sdcli,
            "--diffusion-model", self.mp["diffusion"],
            "--vae", self.mp["vae"],
            "--llm", self.mp["encoder"],
            "--llm_vision", self.mp["mmproj"],   # requerido para edición con encoder GGUF <!--citation:1-->
        ]
        for rimg in refs:
            cmd += ["-r", rimg]  # múltiples refs repitiendo -r <!--citation:1-->

        cmd += [
            "-p", prompt,
            "--cfg-scale", str(cfg),
            "--sampling-method", "euler",
            "--steps", str(int(steps)),
            "--seed", str(int(seed)),
            "-o", out_png,
            "-W", str(int(w)),
            "-H", str(int(h)),
        ]

        if offload_to_cpu and self._supports("--offload-to-cpu"):
            cmd.append("--offload-to-cpu")
        if use_fa:
            if self._supports("--fa"): cmd.append("--fa")
            elif self._supports("--diffusion-fa"): cmd.append("--diffusion-fa")

        self._run(cmd)
        if not os.path.exists(out_png) or os.path.getsize(out_png) < 5000:
            raise RuntimeError("No se generó PNG válido.")
        return out_png

def shlex_quote(s: str) -> str:
    import shlex
    return shlex.quote(str(s))

def etapa1_generar_imagenes(
    vp: VideoProject,
    plantilla_slug: str,
    personaje_key: str,
    personaje_plantilla_key: str,
    edad: int,
    vestuario: str,
    planos_lejanos_a_usar: int,
    diffusion_quant="Q8_0",
    encoder_quant="Q4_K_M",
    mmproj_quant="Q8_0",
    steps_A=30,
    steps_edit=30,
    cfg=2.0,
    seed: int = -1,
) -> Dict[str, str]:
    """
    Genera A,B,C,D y E/F/G opcionales.
    - A y B: 9:16 (múltiplos de 32)
    - C,D,E,F,G: 16:9 (múltiplos de 32)
    """
    tpl = cargar_plantilla(plantilla_slug)
    base = tpl["_base_dir"]

    # resoluciones (múltiplos de 32): 9:16 → 832x1472; 16:9 → 1376x768
    W_9_16, H_9_16 = 832, 1472
    W_16_9, H_16_9 = 1376, 768

    img_dir = os.path.join(vp.video_dir, "etapa1_imagenes")
    os.makedirs(img_dir, exist_ok=True)

    # Prompts (editables: guardamos en prompts.json)
    p1 = render_prompt_1(personaje_key, edad, vestuario)
    p2 = PROMPT_2_CUERPO_COMPLETO
    p3 = render_prompt_3(personaje_key, personaje_plantilla_key)
    p4 = PROMPT_4_CLOSEUP

    vp.prompts.setdefault("etapa1", {})
    vp.prompts["etapa1"].update({
        "prompt_1": p1,
        "prompt_2": p2,
        "prompt_3": p3,
        "prompt_4": p4,
    })

    vp.params.setdefault("etapa1", {})
    vp.params["etapa1"].update({
        "diffusion_quant": diffusion_quant,
        "encoder_quant": encoder_quant,
        "mmproj_quant": mmproj_quant,
        "steps_A": steps_A,
        "steps_edit": steps_edit,
        "cfg": cfg,
        "seed": seed,
        "res_A_B": [W_9_16, H_9_16],
        "res_rest": [W_16_9, H_16_9],
    })
    vp.save()

    # Modelos
    mp = ensure_qwen_image21_models(diffusion_quant, encoder_quant, mmproj_quant)
    sd = SDQwenImage21(SDCLI_PATH, mp)

    # Paths objetivo (A..G)
    paths = {
        "A": os.path.join(img_dir, "concursante_base.png"),
        "B": os.path.join(img_dir, "concursante_cuerpo_completo.png"),
        "C": os.path.join(img_dir, "concursante_plano_cercano.png"),
        "D": os.path.join(img_dir, "concursante_closeup.png"),
        "E": os.path.join(img_dir, "concursante_plano_lejano_1.png"),
        "F": os.path.join(img_dir, "concursante_plano_lejano_2.png"),
        "G": os.path.join(img_dir, "concursante_plano_lejano_3.png"),
    }

    # [PASO 1] A (text2img)
    log("[ETAPA 1] PASO 1/7: Generando A (text2img)...")
    sd.generate_text2img(p1, paths["A"], W_9_16, H_9_16, steps_A, cfg, seed=seed)

    # [PASO 2] B (edit desde A)
    log("[ETAPA 1] PASO 2/7: Generando B (edit desde A)...")
    sd.edit_with_refs(p2, refs=[paths["A"]], out_png=paths["B"], w=W_9_16, h=H_9_16, steps=steps_edit, cfg=cfg, seed=seed)

    # Plantilla plano cercano
    tpl_close = os.path.join(base, PLANTILLA_SUBDIRS["escenario_cercano"], "plantilla_plano_cercano.png")
    if not os.path.exists(tpl_close):
        raise RuntimeError(f"Falta plantilla_plano_cercano.png en {tpl_close}")

    # [PASO 3] C (edit: refs=[B, plantilla_cercano])
    log("[ETAPA 1] PASO 3/7: Generando C (edit plantilla plano cercano)...")
    sd.edit_with_refs(p3, refs=[paths["B"], tpl_close], out_png=paths["C"], w=W_16_9, h=H_16_9, steps=steps_edit, cfg=cfg, seed=seed)

    # [PASO 4] D (close-up desde C)
    log("[ETAPA 1] PASO 4/7: Generando D (close-up desde C)...")
    sd.edit_with_refs(p4, refs=[paths["C"]], out_png=paths["D"], w=W_16_9, h=H_16_9, steps=steps_edit, cfg=cfg, seed=seed)

    # [PASO 5-7] E/F/G opcionales (hasta planos_lejanos_a_usar)
    for i, key in enumerate(["E","F","G"], start=1):
        if i > int(planos_lejanos_a_usar):
            continue
        tpl_far = os.path.join(base, PLANTILLA_SUBDIRS["escenario_lejano"], f"plantilla_plano_lejano_{i}.png")
        if not os.path.exists(tpl_far):
            raise RuntimeError(f"Falta {os.path.basename(tpl_far)} en {tpl_far}")
        log(f"[ETAPA 1] PASO {4+i}/7: Generando {key} (edit plantilla plano lejano #{i})...")
        sd.edit_with_refs(p3, refs=[paths["B"], tpl_far], out_png=paths[key], w=W_16_9, h=H_16_9, steps=steps_edit, cfg=cfg, seed=seed)

    # Estado + persistencia
    for k,v in paths.items():
        vp.estado["aprobados"][f"etapa1_{k}"] = bool(os.path.exists(v))
    vp.save()

    log("[ETAPA 1] COMPLETADA.")
    return paths

log("CELDA 5 COMPLETADA.")

[18:04:05] CELDA 5 COMPLETADA.


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELDA 6 (v7): QWEN3-TTS AISLADO EN SUBENTORNO
#
# OBJETIVO CRÍTICO:
#   Qwen3-TTS NO instala ni modifica dependencias del entorno principal.
#
# El proyecto utiliza varios backends con requisitos incompatibles
# (Wan2GP/LTX, Whisper, Qwen, Gradio, etc.). Por eso Qwen se ejecuta
# en un proceso Python independiente con un virtualenv propio.
#
# STACK INTERNO QWEN — basado en el notebook AIQUEST:
#   qwen-tts==0.0.5
#   transformers==4.57.3
#   huggingface-hub==1.3.3
#   accelerate==1.12.0
#
# La interfaz pública de qwen3_tts se conserva para no cambiar la
# lógica del pipeline existente.
# ═══════════════════════════════════════════════════════════════
import os
import gc
import sys
import time
import shutil
import subprocess
import tempfile
import json
import threading
from typing import Optional, Dict, Tuple

import torch
import soundfile as sf


# ---------------------------------------------------------------
# CACHE EXCLUSIVA DEL WORKER QWEN
# ---------------------------------------------------------------
TTS_CACHE_DIR = "/tmp/hf_cache_tts"
os.makedirs(TTS_CACHE_DIR, exist_ok=True)

st_tmp = os.statvfs("/tmp")
_tmp_free_gb = (st_tmp.f_bavail * st_tmp.f_frsize) / 1024**3
log(f"[TTS] Cache Qwen aislada → {TTS_CACHE_DIR}")
log(f"[TTS] /tmp libre: {_tmp_free_gb:.1f} GB")


# ---------------------------------------------------------------
# PYAV / SOUNDFILE — SOLO para el entorno principal de Whisper.
# No cambiamos versiones globales de Hugging Face.
# ---------------------------------------------------------------
def _patch_audio_readers() -> None:
    try:
        import soundfile as _sf
        if not getattr(_sf.read, "_patched_metadata_errors", False):
            _orig_sf_read = _sf.read

            def _patched_sf_read(file, *args, **kwargs):
                kwargs.pop("metadata_errors", None)
                kwargs.pop("metadata_encoding", None)
                return _orig_sf_read(file, *args, **kwargs)

            _patched_sf_read._patched_metadata_errors = True
            _sf.read = _patched_sf_read
            log("[TTS] ✅ soundfile.read protegido contra metadata_errors.")
    except Exception as e:
        log(f"[TTS] ⚠ Patch soundfile omitido: {e}")

    try:
        import av as _av
        if not getattr(_av.open, "_patched_metadata_errors", False):
            _orig_av_open = _av.open

            def _patched_av_open(file, mode="r", **kwargs):
                kwargs.pop("metadata_errors", None)
                kwargs.pop("metadata_encoding", None)
                return _orig_av_open(file, mode=mode, **kwargs)

            _patched_av_open._patched_metadata_errors = True
            _av.open = _patched_av_open
            log(f"[TTS] ✅ av.open protegido | PyAV={getattr(_av, '__version__', '?')}")
    except Exception as e:
        log(f"[TTS] ⚠ Patch PyAV omitido: {e}")


_patch_audio_readers()


# ---------------------------------------------------------------
# WORKER AISLADO DE QWEN3-TTS
# ---------------------------------------------------------------
# Qwen ejecuta su backend Python en un virtualenv separado para no tocar
# las dependencias del resto del notebook.
#
# Stack interno basado en el notebook AIQUEST:
#   qwen-tts==0.0.5
#   transformers==4.57.3
#   huggingface-hub==0.36.0
#   accelerate==1.12.0
#
# Transformers 4.57.3 requiere huggingface-hub <1.0, por lo que 0.36.0
# es la versión correcta para este entorno aislado.
# ---------------------------------------------------------------

QWEN_TTS_VENV = "/tmp/qwen3_tts_env"
QWEN_TTS_WORKER = "/tmp/qwen3_tts_worker.py"
QWEN_TTS_PYTHON = os.path.join(QWEN_TTS_VENV, "bin", "python")
QWEN_TTS_READY_FILE = os.path.join(QWEN_TTS_VENV, ".qwen_ready.json")
QWEN_TTS_LOG_FILE = os.path.join(TTS_CACHE_DIR, "qwen_worker.log")
CONTESTANT_VOICE_STATE_FILE = os.path.join(OUTPUT_DIR, "contestant_voice_state.json")

QWEN_TTS_WORKER_SOURCE = r'''# -*- coding: utf-8 -*-
import gc
import json
import os
import sys
import time
import traceback

# --- Aislar stdout: SOLO el protocolo JSON puede salir por el stdout real.
# Cualquier print() de librerias (qwen_tts, sox, transformers...) se desvia a stderr.
_REAL_STDOUT = os.fdopen(os.dup(1), "w", encoding="utf-8", buffering=1)
os.dup2(2, 1)
sys.stdout = sys.stderr

import torch
import soundfile as sf

TTS_CACHE_DIR = "/tmp/hf_cache_tts"
os.makedirs(TTS_CACHE_DIR, exist_ok=True)

MODEL_IDS = {
    "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    "custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
}

current_model = None
current_model_type = None


def out_json(data):
    _REAL_STDOUT.write(json.dumps(data, ensure_ascii=False) + "\n")
    _REAL_STDOUT.flush()


def err_log(message):
    sys.stderr.write(str(message).rstrip() + "\n")
    sys.stderr.flush()


def unload_model():
    global current_model, current_model_type
    if current_model is not None:
        try:
            del current_model
        except Exception:
            pass
    current_model = None
    current_model_type = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def ensure_model(model_type):
    global current_model, current_model_type

    if model_type not in MODEL_IDS:
        raise ValueError(f"Tipo de modelo desconocido: {model_type}")

    if current_model is not None and current_model_type == model_type:
        return current_model

    unload_model()

    from huggingface_hub import snapshot_download
    from qwen_tts import Qwen3TTSModel

    model_name = MODEL_IDS[model_type]
    err_log(f"[QWEN-WORKER] Cargando '{model_type}': {model_name}")

    local_path = snapshot_download(
        repo_id=model_name,
        cache_dir=TTS_CACHE_DIR,
        local_files_only=False,
    )

    speech_tokenizer_dir = os.path.join(local_path, "speech_tokenizer")
    os.makedirs(speech_tokenizer_dir, exist_ok=True)

    for cfg_name in ("preprocessor_config.json", "config.json"):
        src = os.path.join(local_path, cfg_name)
        dst = os.path.join(speech_tokenizer_dir, cfg_name)
        if os.path.exists(src) and not os.path.lexists(dst):
            try:
                os.symlink(src, dst)
            except Exception:
                pass

    if torch.cuda.is_available():
        device = "cuda:0"
        dtype = torch.float16
    else:
        device = "cpu"
        dtype = torch.float32

    current_model = Qwen3TTSModel.from_pretrained(
        local_path,
        torch_dtype=dtype,
        device_map=device,
        attn_implementation="sdpa",
    )
    current_model_type = model_type

    vram = 0.0
    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated(0) / 1024**3
    err_log(f"[QWEN-WORKER] Modelo '{model_type}' listo | VRAM={vram:.2f} GB")
    return current_model


def save_wav(wavs, sr, out_path):
    if not wavs:
        raise RuntimeError("Qwen no devolvió audio.")
    out_path = os.path.abspath(out_path)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    sf.write(out_path, wavs[0], sr)
    if not os.path.exists(out_path) or os.path.getsize(out_path) < 1000:
        raise RuntimeError(f"No se pudo crear WAV válido: {out_path}")
    return out_path, float(len(wavs[0]) / sr), int(sr)


def handle(req):
    command = req.get("command", "")

    if command == "ping":
        import importlib.metadata as md
        return {
            "ok": True,
            "command": "ping",
            "qwen_tts": md.version("qwen-tts"),
            "transformers": md.version("transformers"),
            "huggingface_hub": md.version("huggingface-hub"),
            "accelerate": md.version("accelerate"),
            "torch": torch.__version__,
            "cuda": bool(torch.cuda.is_available()),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        }

    if command == "unload":
        unload_model()
        return {"ok": True, "command": "unload"}

    if command == "clone":
        text = str(req.get("text", "")).strip()
        ref_wav = str(req.get("ref_wav", ""))
        ref_text = str(req.get("ref_text", "")).strip()
        out_wav = str(req.get("out_wav", ""))
        fast_mode = bool(req.get("fast_mode", False))

        if not text:
            raise ValueError("Texto vacío.")
        if not ref_wav or not os.path.exists(ref_wav):
            raise FileNotFoundError(f"Referencia inexistente: {ref_wav}")
        if not out_wav:
            raise ValueError("Falta out_wav.")

        model = ensure_model("base")
        t0 = time.time()

        if fast_mode or not ref_text:
            prompt_items = model.create_voice_clone_prompt(
                ref_audio=ref_wav,
                x_vector_only_mode=True,
            )
        else:
            prompt_items = model.create_voice_clone_prompt(
                ref_audio=ref_wav,
                ref_text=ref_text,
                x_vector_only_mode=False,
            )

        with torch.inference_mode():
            wavs, sr = model.generate_voice_clone(
                text=text,
                voice_clone_prompt=prompt_items,
            )

        out_path, duration, sample_rate = save_wav(wavs, sr, out_wav)
        elapsed = time.time() - t0
        return {
            "ok": True,
            "command": "clone",
            "out_wav": out_path,
            "duration": duration,
            "sample_rate": sample_rate,
            "elapsed": elapsed,
        }

    if command == "design":
        text = str(req.get("text", "")).strip()
        instruct = str(req.get("instruct", "")).strip()
        out_wav = str(req.get("out_wav", ""))

        if not text:
            raise ValueError("Texto vacío.")
        if not instruct:
            raise ValueError("Instrucción de VoiceDesign vacía.")
        if not out_wav:
            raise ValueError("Falta out_wav.")

        model = ensure_model("design")
        t0 = time.time()
        with torch.inference_mode():
            wavs, sr = model.generate_voice_design(
                text=text,
                instruct=instruct,
            )

        out_path, duration, sample_rate = save_wav(wavs, sr, out_wav)
        elapsed = time.time() - t0
        return {
            "ok": True,
            "command": "design",
            "out_wav": out_path,
            "duration": duration,
            "sample_rate": sample_rate,
            "elapsed": elapsed,
        }

    if command == "custom":
        text = str(req.get("text", "")).strip()
        speaker = str(req.get("speaker", "serena")).strip().lower()
        instruct = str(req.get("instruct", "")).strip()
        out_wav = str(req.get("out_wav", ""))

        if not text:
            raise ValueError("Texto vacío.")
        if not out_wav:
            raise ValueError("Falta out_wav.")

        model = ensure_model("custom")
        t0 = time.time()
        kwargs = {"text": text, "speaker": speaker}
        if instruct:
            kwargs["instruct"] = instruct

        with torch.inference_mode():
            wavs, sr = model.generate_custom_voice(**kwargs)

        out_path, duration, sample_rate = save_wav(wavs, sr, out_wav)
        elapsed = time.time() - t0
        return {
            "ok": True,
            "command": "custom",
            "out_wav": out_path,
            "duration": duration,
            "sample_rate": sample_rate,
            "elapsed": elapsed,
        }

    raise ValueError(f"Comando desconocido: {command}")


def main():
    err_log(
        "[QWEN-WORKER] Proceso iniciado | "
        f"GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}"
    )

    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            req = json.loads(line)
            result = handle(req)
            out_json(result)
        except Exception as exc:
            err_log(traceback.format_exc())
            out_json({"ok": False, "error": str(exc), "traceback": traceback.format_exc()})


if __name__ == "__main__":
    main()
'''

# ---------------------------------------------------------------
# CREACIÓN DEL ENTORNO AISLADO
# ---------------------------------------------------------------
def _run_qwen_venv_command(args, check=True, extra_env=None):
    env = os.environ.copy()
    env["PYTHONNOUSERSITE"] = "1"
    env.pop("PYTHONPATH", None)
    if extra_env:
        env.update(extra_env)
    r = subprocess.run(args, capture_output=True, text=True, env=env)
    if check and r.returncode != 0:
        tail = ((r.stderr or "") + "\n" + (r.stdout or "")).strip()[-1500:]
        raise RuntimeError(f"rc={r.returncode} en: {' '.join(str(a) for a in args)}\n{tail}")
    return r


def _create_qwen_venv() -> str:
    """Crea el venv aislado probando varias estrategias (Kaggle suele no traer
    python3-venv/ensurepip). NO modifica el entorno principal."""
    py = sys.executable
    errors = []

    def _clean():
        shutil.rmtree(QWEN_TTS_VENV, ignore_errors=True)

    def _venv_has_pip() -> bool:
        if not os.path.exists(QWEN_TTS_PYTHON):
            return False
        return _run_qwen_venv_command([QWEN_TTS_PYTHON, "-m", "pip", "--version"], check=False).returncode == 0

    # 1) venv estándar
    # 2) venv sin ensurepip (usa el pip del sistema visible via system-site-packages)
    for label, cmd in [
        ("venv", [py, "-m", "venv", "--system-site-packages", QWEN_TTS_VENV]),
        ("venv --without-pip", [py, "-m", "venv", "--system-site-packages", "--without-pip", QWEN_TTS_VENV]),
    ]:
        _clean()
        try:
            _run_qwen_venv_command(cmd)
            if _venv_has_pip():
                return label
            errors.append(f"[{label}] creado pero sin pip utilizable")
        except Exception as e:
            errors.append(f"[{label}] {e}")

    # 3) uv (si existe en el sistema)
    uv_bin = shutil.which("uv")
    if uv_bin:
        _clean()
        try:
            _run_qwen_venv_command([uv_bin, "venv", "--system-site-packages", "--seed", "--python", py, QWEN_TTS_VENV])
            if _venv_has_pip():
                return "uv venv"
            errors.append("[uv] creado pero sin pip utilizable")
        except Exception as e:
            errors.append(f"[uv] {e}")

    # 4) virtualenv instalado en carpeta temporal (--target) → NO toca site-packages principal
    boot = "/tmp/qwen_bootstrap_virtualenv"
    _clean()
    try:
        _run_qwen_venv_command([py, "-m", "pip", "install", "-q", "--target", boot, "virtualenv"])
        _run_qwen_venv_command(
            [py, "-m", "virtualenv", "--system-site-packages", QWEN_TTS_VENV],
            extra_env={"PYTHONPATH": boot},
        )
        if _venv_has_pip():
            return "virtualenv"
        errors.append("[virtualenv] creado pero sin pip utilizable")
    except Exception as e:
        errors.append(f"[virtualenv] {e}")

    raise RuntimeError("No se pudo crear el venv de Qwen con ningún método:\n" + "\n".join(errors))


def _qwen_versions_in_venv():
    if not os.path.exists(QWEN_TTS_PYTHON):
        return None
    code = ("import importlib.metadata as m, json; "
            "print(json.dumps({"
            "'qwen_tts':m.version('qwen-tts'),"
            "'transformers':m.version('transformers'),"
            "'huggingface_hub':m.version('huggingface-hub'),"
            "'accelerate':m.version('accelerate')}))")
    try:
        r = _run_qwen_venv_command([QWEN_TTS_PYTHON, "-c", code], check=False)
        if r.returncode != 0:
            return None
        return json.loads((r.stdout or "").strip())
    except Exception:
        return None


def _ensure_qwen_isolated_environment() -> bool:
    expected = {
        "qwen_tts": "0.0.5",
        "transformers": "4.57.3",
        "huggingface_hub": "0.36.0",
        "accelerate": "1.12.0",
    }
    current = _qwen_versions_in_venv()
    if current != expected:
        log("[TTS] Preparando entorno Python AISLADO de Qwen3-TTS...")
        try:
            if os.path.isdir(QWEN_TTS_VENV):
                shutil.rmtree(QWEN_TTS_VENV, ignore_errors=True)
            _how = _create_qwen_venv()
            log(f"[TTS] Venv aislado creado con: {_how}")
            _run_qwen_venv_command([
                QWEN_TTS_PYTHON, "-m", "pip", "install", "-q", "--upgrade", "pip"
            ])
            _run_qwen_venv_command([
                QWEN_TTS_PYTHON, "-m", "pip", "install", "-q", "--no-cache-dir",
                "transformers==4.57.3", "huggingface-hub==0.36.0", "accelerate==1.12.0",
                "gradio==6.4.0", "librosa", "soundfile", "sox", "onnxruntime", "einops"
            ])
            _run_qwen_venv_command([
                QWEN_TTS_PYTHON, "-m", "pip", "install", "-q", "--no-cache-dir", "--no-deps", "qwen-tts==0.0.5"
            ])
            with open(QWEN_TTS_WORKER, "w", encoding="utf-8") as f:
                f.write(QWEN_TTS_WORKER_SOURCE)
            verified = _qwen_versions_in_venv()
            if verified != expected:
                raise RuntimeError(f"Versiones inesperadas: {verified}")
            with open(QWEN_TTS_READY_FILE, "w", encoding="utf-8") as f:
                json.dump(expected, f, ensure_ascii=False, indent=2)
        except Exception as e:
            log(f"[TTS] ❌ No se pudo preparar el entorno aislado: {e}")
            return False
    try:
        with open(QWEN_TTS_WORKER, "w", encoding="utf-8") as f:
            f.write(QWEN_TTS_WORKER_SOURCE)
    except Exception as e:
        log(f"[TTS] ❌ No se pudo escribir worker Qwen: {e}")
        return False
    return True


class _QwenWorkerClient:
    def __init__(self):
        self.process = None
        self.lock = threading.Lock()
        self.stderr_thread = None

    def _drain_stderr(self):
        if self.process is None or self.process.stderr is None:
            return
        try:
            for line in self.process.stderr:
                line = line.rstrip()
                if line:
                    log(line)
        except Exception:
            pass

    def start(self):
        if self.process is not None and self.process.poll() is None:
            return
        if not _ensure_qwen_isolated_environment():
            raise RuntimeError("No se pudo preparar el entorno aislado de Qwen3-TTS.")
        env = os.environ.copy()
        env["PYTHONNOUSERSITE"] = "1"
        env.pop("PYTHONPATH", None)
        env["HF_HOME"] = TTS_CACHE_DIR
        env["HUGGINGFACE_HUB_CACHE"] = TTS_CACHE_DIR
        env["HF_DATASETS_CACHE"] = TTS_CACHE_DIR
        env["PYTHONUNBUFFERED"] = "1"
        log("[TTS] Iniciando worker Qwen aislado...")
        self.process = subprocess.Popen(
            [QWEN_TTS_PYTHON, "-u", QWEN_TTS_WORKER],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True, bufsize=1, env=env
        )
        self.stderr_thread = threading.Thread(target=self._drain_stderr, daemon=True, name="qwen-tts-worker-stderr")
        self.stderr_thread.start()

    def request(self, payload: dict) -> dict:
        with self.lock:
            self.start()
            if self.process is None or self.process.stdin is None or self.process.stdout is None:
                raise RuntimeError("Worker Qwen no está operativo.")
            if self.process.poll() is not None:
                code = self.process.returncode
                self.process = None
                raise RuntimeError(f"Worker Qwen terminó antes de responder (rc={code}).")
            self.process.stdin.write(json.dumps(payload, ensure_ascii=False) + "\n")
            self.process.stdin.flush()
            data = None
            while data is None:
                line = self.process.stdout.readline()
                if not line:
                    code = self.process.poll()
                    self.process = None
                    raise RuntimeError(f"Worker Qwen cerró stdout sin respuesta (rc={code}).")
                line = line.strip()
                if not line:
                    continue                      # línea en blanco → ruido, ignorar
                if not line.startswith("{"):
                    log(f"[QWEN-WORKER stdout] {line[:300]}")
                    continue                      # print() ajeno → ignorar
                try:
                    data = json.loads(line)
                except json.JSONDecodeError:
                    log(f"[QWEN-WORKER stdout no-JSON] {line[:300]}")
            if not data.get("ok", False):
                raise RuntimeError(data.get("error", "Error desconocido del worker Qwen."))
            return data

    def close(self):
        with self.lock:
            proc = self.process
            self.process = None
            if proc is None:
                return
            try:
                if proc.stdin:
                    proc.stdin.write(json.dumps({"command": "unload"}) + "\n")
                    proc.stdin.flush()
            except Exception:
                pass
            try:
                proc.terminate()
                proc.wait(timeout=5)
            except Exception:
                try:
                    proc.kill()
                except Exception:
                    pass


class Qwen3TTSEngine:
    MODEL_IDS = {
        "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        "custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
        "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    }

    def __init__(self):
        self._worker = _QwenWorkerClient()
        self._available = None

    @property
    def available(self):
        if self._available is not None:
            return self._available
        try:
            r = self._worker.request({"command": "ping"})
            log(
                "[TTS] ✅ Worker Qwen disponible | "
                f"qwen-tts={r['qwen_tts']} | transformers={r['transformers']} | "
                f"huggingface-hub={r['huggingface_hub']} | accelerate={r['accelerate']} | GPU={r['gpu']}"
            )
            self._available = True
        except Exception as e:
            log(f"[TTS] ❌ Worker Qwen no disponible: {e}")
            self._available = False
        return self._available

    def _request(self, payload):
        if not self.available:
            raise RuntimeError("qwen_tts no disponible.")
        return self._worker.request(payload)

    def synth_clone(self, text, out_wav, ref_wav, ref_text="", fast_mode=False, seed=-1):
        r = self._request({"command":"clone","text":text,"out_wav":out_wav,"ref_wav":ref_wav,"ref_text":ref_text,"fast_mode":bool(fast_mode),"seed":int(seed)})
        log(f"[TTS Clone] {r['duration']:.1f}s audio | {r['elapsed']:.1f}s generación | RTF={r['elapsed']/max(r['duration'],0.1):.2f}x")
        return out_wav

    def synth_voice_design(self, text, out_wav, instruct, seed=-1):
        r = self._request({"command":"design","text":text,"out_wav":out_wav,"instruct":instruct,"seed":int(seed)})
        log(f"[TTS Design] {r['duration']:.1f}s audio | {r['elapsed']:.1f}s generación")
        return out_wav

    def synth_custom_voice(self, text, out_wav, speaker="Serena", instruction="", seed=-1):
        r = self._request({"command":"custom","text":text,"out_wav":out_wav,"speaker":speaker,"instruct":instruction,"seed":int(seed)})
        log(f"[TTS Custom] speaker={speaker} | {r['duration']:.1f}s audio | {r['elapsed']:.1f}s generación")
        return out_wav

    def unload(self):
        try:
            if self._available:
                self._worker.request({"command":"unload"})
        except Exception:
            pass
        self._worker.close()
        self._available = None

    def report_cache_usage(self):
        total=0
        if os.path.isdir(TTS_CACHE_DIR):
            for root,_,files in os.walk(TTS_CACHE_DIR):
                for fname in files:
                    try: total += os.path.getsize(os.path.join(root,fname))
                    except Exception: pass
        log(f"[TTS Cache] {TTS_CACHE_DIR}: {total/1024**3:.2f} GB usados en /tmp")


def load_contestant_voice_state():
    if os.path.exists(CONTESTANT_VOICE_STATE_FILE):
        try:
            with open(CONTESTANT_VOICE_STATE_FILE,"r",encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {"mode":None,"ref_wav_path":"","ref_text":"","design_instruct":"","preview_wav":"","validated":False}


def save_contestant_voice_state(state):
    os.makedirs(os.path.dirname(CONTESTANT_VOICE_STATE_FILE),exist_ok=True)
    with open(CONTESTANT_VOICE_STATE_FILE,"w",encoding="utf-8") as f:
        json.dump(state,f,ensure_ascii=False,indent=2)


_patch_audio_readers()
qwen3_tts = Qwen3TTSEngine()
_QWEN_TTS_PY_AVAILABLE = bool(qwen3_tts.available)
log(f"[TTS] Engine disponible: {_QWEN_TTS_PY_AVAILABLE}")
if _QWEN_TTS_PY_AVAILABLE:
    qwen3_tts.report_cache_usage()

CONTESTANT_VOICE_STATE = load_contestant_voice_state()


# ═══════════════════════════════════════════════════════════════
# SECCIÓN 6.4 — DIRECTORIOS DE REFERENCIA
# AUDIO de referencia → /kaggle/working/voice_refs (pequeños WAVs, persistentes)
# MODELOS HF         → /tmp/hf_cache_tts           (grandes, no persistentes)
# ═══════════════════════════════════════════════════════════════
VOICE_REFS_DIR         = os.path.join(OUTPUT_DIR, "voice_refs")
JUECES_REF_DIR         = os.path.join(VOICE_REFS_DIR, "jueces")
CONTESTANT_REF_WAV     = os.path.join(VOICE_REFS_DIR, "contestant_ref.wav")
CONTESTANT_REF_TXT     = os.path.join(VOICE_REFS_DIR, "contestant_ref.txt")
CONTESTANT_PREVIEW_WAV = os.path.join(VOICE_REFS_DIR, "contestant_preview.wav")
DESIGN_REF_WAV         = os.path.join(VOICE_REFS_DIR, "design_as_ref.wav")
DESIGN_REF_TXT         = os.path.join(VOICE_REFS_DIR, "design_as_ref.txt")

for _d in [VOICE_REFS_DIR, JUECES_REF_DIR]:
    os.makedirs(_d, exist_ok=True)
log(f"[TTS] WAVs de referencia → {VOICE_REFS_DIR} (/working, persistentes, <50MB)")


def get_juez_ref_wav(plantilla_slug: str, juez_id: int) -> str:
    if not plantilla_slug:
        return ""
    try:
        tpl = cargar_plantilla(plantilla_slug)
        voz_dir = os.path.join(tpl["_base_dir"], PLANTILLA_SUBDIRS["voces_jueces"])
        wav_path = os.path.join(voz_dir, f"juez_{juez_id}.wav")
        return wav_path if os.path.exists(wav_path) else ""
    except Exception:
        return ""

def get_juez_ref_text(plantilla_slug: str, juez_id: int) -> str:
    txt_path = os.path.join(JUECES_REF_DIR, f"juez_{juez_id}_ref.txt")
    if os.path.exists(txt_path):
        try: return open(txt_path, "r", encoding="utf-8").read().strip()
        except Exception: pass
    return ""

# ---------------------------------------------------------------
# TRANSCRIPTOR (faster-whisper, entorno principal)
# Contrato usado por el resto del pipeline:
#   transcribe_audio_file(wav, out_txt_path=None, language=None) -> (texto, idioma)
# Carga el modelo bajo demanda y lo libera al terminar para no robar VRAM
# a LTX / Qwen. Nunca lanza excepción: si falla devuelve ("", "").
# ---------------------------------------------------------------
WHISPER_MODEL_SIZE = "small"

def transcribe_audio_file(wav_path: str, out_txt_path: Optional[str] = None,
                          language: Optional[str] = None) -> Tuple[str, str]:
    if not wav_path or not os.path.exists(wav_path):
        log(f"[Transcriber] ❌ Audio inexistente: {wav_path}")
        return "", ""

    tmp16 = None
    model = None
    try:
        from faster_whisper import WhisperModel

        # Decodificar a 16 kHz mono con ffmpeg → evita depender de PyAV.
        tmp16 = tempfile.mktemp(suffix="_whisper16k.wav", dir="/tmp")
        r = subprocess.run(
            ["ffmpeg", "-y", "-i", wav_path, "-ac", "1", "-ar", "16000",
             "-acodec", "pcm_s16le", tmp16, "-loglevel", "error"],
            capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"ffmpeg: {(r.stderr or '')[-500:]}")
        audio, _sr = sf.read(tmp16, dtype="float32")

        try:
            if torch.cuda.is_available():
                model = WhisperModel(WHISPER_MODEL_SIZE, device="cuda", compute_type="float16")
            else:
                raise RuntimeError("sin CUDA")
        except Exception as e_gpu:
            log(f"[Transcriber] Whisper en CPU ({e_gpu})")
            model = WhisperModel(WHISPER_MODEL_SIZE, device="cpu", compute_type="int8")

        segments, info = model.transcribe(
            audio, language=language, beam_size=5, vad_filter=True)
        texto = " ".join(seg.text.strip() for seg in segments).strip()
        idioma = (getattr(info, "language", None) or language or "").strip()

        if out_txt_path and texto:
            os.makedirs(os.path.dirname(out_txt_path), exist_ok=True)
            with open(out_txt_path, "w", encoding="utf-8") as f:
                f.write(texto)
        log(f"[Transcriber] ✅ idioma={idioma or '?'} | {len(texto)} chars")
        return texto, idioma
    except Exception as e:
        log(f"[Transcriber] ❌ Falló la transcripción: {e}")
        return "", ""
    finally:
        try:
            del model
        except Exception:
            pass
        gc.collect()
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
        if tmp16 and os.path.exists(tmp16):
            try: os.remove(tmp16)
            except Exception: pass


def auto_transcribe_juez(plantilla_slug: str, juez_id: int) -> str:
    wav = get_juez_ref_wav(plantilla_slug, juez_id)
    if not wav:
        return ""
    txt_path = os.path.join(JUECES_REF_DIR, f"juez_{juez_id}_ref.txt")
    existing = get_juez_ref_text(plantilla_slug, juez_id)
    if existing:
        log(f"[Transcriber] Juez {juez_id}: transcripción existente reutilizada.")
        return existing
    texto, _ = transcribe_audio_file(wav, out_txt_path=txt_path)
    log(f"[Transcriber] Juez {juez_id}: '{texto[:80]}...'")
    return texto

def auto_transcribe_all_judges(plantilla_slug: str) -> Dict[str, str]:
    result = {}
    for i in range(1, 5):
        wav = get_juez_ref_wav(plantilla_slug, i)
        if wav:
            txt = auto_transcribe_juez(plantilla_slug, i)
            result[f"juez_{i}"] = txt
    return result


# ═══════════════════════════════════════════════════════════════
# SECCIÓN 6.5 — VALIDACIÓN DE VOZ DEL CONCURSANTE
# ═══════════════════════════════════════════════════════════════
PREVIEW_TEXT_ES = (
    "Hola, muchas gracias por estar aquí. "
    "Llevo años soñando con este momento y hoy estoy lista para dar todo."
)

def process_contestant_upload(audio_path: str) -> Tuple[str, str, str]:
    global CONTESTANT_VOICE_STATE
    if not audio_path or not os.path.exists(audio_path):
        return "", "", "❌ No se recibió audio válido."
    try:
        ffmpeg_to_wav_mono_24k(audio_path, CONTESTANT_REF_WAV)
        dur = _ffprobe_duration(CONTESTANT_REF_WAV) or 0
        if dur < 2.0:
            return "", "", f"⚠ Audio muy corto ({dur:.1f}s). Mínimo 3 segundos."

        log("[Contestant] Auto-transcribiendo audio subido...")
        texto, idioma = transcribe_audio_file(
            CONTESTANT_REF_WAV, out_txt_path=CONTESTANT_REF_TXT)

        # FIX: Si la transcripción falla, usar placeholder para NO bloquear el preview
        if not texto.strip():
            texto = PREVIEW_TEXT_ES
            idioma = idioma or "es"
            log("[Contestant] ⚠ Transcripción vacía, usando texto placeholder para preview.")

        CONTESTANT_VOICE_STATE.update({
            "mode": "clone_upload",
            "ref_wav_path": CONTESTANT_REF_WAV,
            "ref_text": texto,
            "validated": False,
            "preview_wav": "",
        })
        save_contestant_voice_state(CONTESTANT_VOICE_STATE)

        msg = (f"✅ Audio cargado ({dur:.1f}s) | Idioma: {idioma}\n"
               f"📝 «{texto[:200]}{'...' if len(texto)>200 else ''}»\n"
               f"→ Genera un preview para validar.")
        return texto, idioma, msg
    except Exception as e:
        log(f"[Contestant] Error: {e}")
        return "", "", f"❌ Error: {e}"

def generate_contestant_clone_preview(
    ref_wav: str = "", ref_text: str = "",
    preview_text: str = PREVIEW_TEXT_ES) -> Tuple[str, str]:
    global CONTESTANT_VOICE_STATE
    wav = ref_wav or CONTESTANT_REF_WAV
    txt = ref_text or CONTESTANT_VOICE_STATE.get("ref_text", "")
    if not wav or not os.path.exists(wav):
        return "", "❌ Primero sube el audio del concursante."
    if not preview_text or not preview_text.strip():
        preview_text = PREVIEW_TEXT_ES
    try:
        log("[Contestant Preview] Generando preview por clonación...")
        qwen3_tts.synth_clone(
            text=preview_text.strip(), out_wav=CONTESTANT_PREVIEW_WAV,
            ref_wav=wav, ref_text=txt,
            fast_mode=(not bool(txt.strip())))
        dur = _ffprobe_duration(CONTESTANT_PREVIEW_WAV) or 0
        CONTESTANT_VOICE_STATE["preview_wav"] = CONTESTANT_PREVIEW_WAV
        save_contestant_voice_state(CONTESTANT_VOICE_STATE)
        return CONTESTANT_PREVIEW_WAV, f"✅ Preview generado ({dur:.1f}s). Escucha y valida."
    except Exception as e:
        return "", f"❌ Error generando preview: {e}"

def generate_voice_design_preview(
    design_instruct: str,
    preview_text: str = PREVIEW_TEXT_ES) -> Tuple[str, str]:
    global CONTESTANT_VOICE_STATE
    if not design_instruct or not design_instruct.strip():
        return "", "❌ Escribe una descripción de voz."
    if not preview_text or not preview_text.strip():
        preview_text = PREVIEW_TEXT_ES
    preview_path = os.path.join(VOICE_REFS_DIR, "design_preview.wav")
    try:
        log("[VoiceDesign Preview] Generando preview...")
        qwen3_tts.synth_voice_design(
            text=preview_text.strip(), out_wav=preview_path,
            instruct=design_instruct.strip())
        dur = _ffprobe_duration(preview_path) or 0
        CONTESTANT_VOICE_STATE["preview_wav"] = preview_path
        CONTESTANT_VOICE_STATE["design_instruct"] = design_instruct.strip()
        save_contestant_voice_state(CONTESTANT_VOICE_STATE)
        return preview_path, (
            f"✅ Preview generado ({dur:.1f}s).\n"
            f"Descripción: «{design_instruct[:100]}»\n"
            f"Si te gusta, pulsa '✅ Usar esta voz como referencia'.")
    except Exception as e:
        return "", f"❌ Error: {e}"

def adopt_design_as_reference(design_instruct: str = "") -> Tuple[str, str]:
    global CONTESTANT_VOICE_STATE
    preview_src = CONTESTANT_VOICE_STATE.get("preview_wav", "")
    if not preview_src or not os.path.exists(preview_src):
        return "", "❌ Primero genera un preview de diseño de voz."
    try:
        shutil.copyfile(preview_src, DESIGN_REF_WAV)
        with open(DESIGN_REF_TXT, "w", encoding="utf-8") as f:
            f.write(PREVIEW_TEXT_ES)
        CONTESTANT_VOICE_STATE.update({
            "mode": "clone_design",
            "ref_wav_path": DESIGN_REF_WAV,
            "ref_text": PREVIEW_TEXT_ES,
            "design_instruct": design_instruct or CONTESTANT_VOICE_STATE.get("design_instruct", ""),
            "validated": True,
            "preview_wav": preview_src,
        })
        save_contestant_voice_state(CONTESTANT_VOICE_STATE)
        dur = _ffprobe_duration(DESIGN_REF_WAV) or 0
        msg = (f"✅ Voz de diseño adoptada ({dur:.1f}s).\n"
               f"Todo el guion usará clonación de esta voz.\n"
               f"Descripción: «{CONTESTANT_VOICE_STATE['design_instruct'][:100]}»")
        return DESIGN_REF_WAV, msg
    except Exception as e:
        return "", f"❌ Error: {e}"

def validate_contestant_voice() -> Tuple[bool, str]:
    global CONTESTANT_VOICE_STATE
    ref = CONTESTANT_VOICE_STATE.get("ref_wav_path", "")
    if not ref or not os.path.exists(ref):
        return False, "❌ No hay referencia de voz. Sube audio o diseña una voz primero."
    CONTESTANT_VOICE_STATE["validated"] = True
    save_contestant_voice_state(CONTESTANT_VOICE_STATE)
    txt = CONTESTANT_VOICE_STATE.get("ref_text", "")
    dur = _ffprobe_duration(ref) or 0
    msg = (f"✅ Voz VALIDADA | Modo: {CONTESTANT_VOICE_STATE.get('mode','?')}\n"
           f"Ref: {os.path.basename(ref)} ({dur:.1f}s)\n"
           f"«{txt[:120]}{'...' if len(txt)>120 else ''}»\n"
           f"→ Ya puedes ejecutar Etapas 2 y 4.")
    return True, msg

def get_contestant_ref_for_tts() -> Dict[str, str]:
    state = CONTESTANT_VOICE_STATE
    if not state.get("validated"):
        raise RuntimeError(
            "Voz del concursante no validada. "
            "Ve a '🎙️ VOZ DEL CONCURSANTE' y valídala primero.")
    ref_wav = state.get("ref_wav_path", "")
    ref_text = state.get("ref_text", "")
    if not ref_wav or not os.path.exists(ref_wav):
        raise RuntimeError("El WAV de referencia no existe.")
    return {"ref_wav": ref_wav, "ref_text": ref_text}


# ═══════════════════════════════════════════════════════════════
# SECCIÓN 6.6 — SÍNTESIS UNIFICADA DE FRAGMENTO
# ═══════════════════════════════════════════════════════════════
def synth_fragment_for_dialog(texto: str, hablante: str, out_wav: str,
                               plantilla_slug: str, fast_mode: bool = False) -> str:
    os.makedirs(os.path.dirname(os.path.abspath(out_wav)), exist_ok=True)

    if qwen3_tts.available:
        try:
            if hablante == "concursante":
                ref_info = get_contestant_ref_for_tts()
                return qwen3_tts.synth_clone(
                    text=texto, out_wav=out_wav,
                    ref_wav=ref_info["ref_wav"], ref_text=ref_info["ref_text"],
                    fast_mode=fast_mode or not bool(ref_info["ref_text"].strip()))
            elif hablante.startswith("juez_"):
                juez_id  = int(hablante.split("_")[1])
                ref_wav  = get_juez_ref_wav(plantilla_slug, juez_id)
                ref_text = get_juez_ref_text(plantilla_slug, juez_id)
                if not ref_wav or not os.path.exists(ref_wav):
                    log(f"[TTS] ⚠ Sin audio ref para {hablante}. VoiceDesign fallback.")
                    instruct_fb = (
                        "adult male, warm, calm voice, Spanish speaking"
                        if juez_id % 2 == 1 else
                        "adult female, warm, professional voice, Spanish speaking")
                    return qwen3_tts.synth_voice_design(
                        text=texto, out_wav=out_wav, instruct=instruct_fb)
                return qwen3_tts.synth_clone(
                    text=texto, out_wav=out_wav, ref_wav=ref_wav,
                    ref_text=ref_text,
                    fast_mode=fast_mode or not bool(ref_text.strip()))
            else:
                raise ValueError(f"Hablante desconocido: {hablante}")
        except Exception as e:
            log(f"[TTS] Error qwen3_tts '{hablante}': {e}. Fallback edge-tts.")

    # ── Fallback: edge-tts ────────────────────────────────────
    import asyncio
    import edge_tts as _edge_tts

    voice_map = {
        "concursante": "es-ES-AbrilNeural",
        "juez_1": "es-ES-AlvaroNeural",
        "juez_2": "es-ES-ElviraNeural",
        "juez_3": "es-MX-JorgeNeural",
        "juez_4": "es-MX-DaliaNeural",
    }
    voice = voice_map.get(hablante, "es-ES-AbrilNeural")

    async def _edge_synth():
        communicate = _edge_tts.Communicate(texto, voice)
        await communicate.save(out_wav)

    asyncio.run(_edge_synth())
    log(f"[TTS Fallback edge] {hablante} → {os.path.basename(out_wav)}")
    return out_wav


# ═══════════════════════════════════════════════════════════════
# SECCIÓN 6.7 — FFMPEG HELPERS
# ═══════════════════════════════════════════════════════════════
def _ffprobe_duration(path: str) -> Optional[float]:
    if not path or not os.path.exists(path):
        return None
    try:
        r = subprocess.run(
            ["ffprobe", "-v", "quiet", "-print_format", "json",
             "-show_format", path],
            capture_output=True, text=True)
        data = json.loads(r.stdout or "{}")
        d = float(data.get("format", {}).get("duration", 0))
        return d if d > 0 else None
    except Exception:
        return None

def ffmpeg_to_wav_mono_24k(src: str, dst: str) -> str:
    os.makedirs(os.path.dirname(os.path.abspath(dst)), exist_ok=True)
    cmd = (f"ffmpeg -y -i '{src}' -ac 1 -ar 24000 "
           f"-acodec pcm_s16le '{dst}' -loglevel error")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"FFmpeg falló:\n{(r.stderr or '')[-2000:]}")
    return dst

def fit_audio_exact_duration(in_wav: str, out_wav: str, target_s: float) -> str:
    d = _ffprobe_duration(in_wav) or 0.0
    os.makedirs(os.path.dirname(os.path.abspath(out_wav)), exist_ok=True)
    if d <= 0:
        raise RuntimeError(f"Duración WAV inválida: {in_wav}")
    if abs(d - target_s) < 0.02:
        cmd = (f"ffmpeg -y -i '{in_wav}' "
               f"-af 'atrim=0:{target_s:.6f},asetpts=N/SR/TB' "
               f"-acodec pcm_s16le -ar 24000 '{out_wav}' -loglevel error")
        subprocess.run(cmd, shell=True, capture_output=True)
        return out_wav
    if d > target_s:
        ratio  = d / target_s
        parts  = []
        r      = ratio
        while r > 2.0:
            parts.append("atempo=2.0")
            r /= 2.0
        parts.append(f"atempo={r:.6f}")
        chain = ",".join(parts)
        cmd = (f"ffmpeg -y -i '{in_wav}' "
               f"-af '{chain},atrim=0:{target_s:.6f},asetpts=N/SR/TB' "
               f"-acodec pcm_s16le -ar 24000 '{out_wav}' -loglevel error")
    else:
        cmd = (f"ffmpeg -y -i '{in_wav}' "
               f"-af 'apad=pad_dur=10,atrim=0:{target_s:.6f},asetpts=N/SR/TB' "
               f"-acodec pcm_s16le -ar 24000 '{out_wav}' -loglevel error")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"FFmpeg falló:\n{(r.stderr or '')[-2000:]}")
    return out_wav


# Alias legacy
qwen_tts = qwen3_tts

"""
# NOTA HISTÓRICA: este bloque quedó deshabilitado deliberadamente.
# NO volver a forzar Transformers 4.x aquí: rompería el backend
# Qwen3-TTS aislado utilizado por la Celda 6.
"""



log(f"\n✅ CELDA 6 (v7) COMPLETADA")
log(f"   Package: {'Qwen3-TTS aislado | qwen-tts 0.0.5 + transformers 4.57.3 + huggingface-hub 0.36.0' if _QWEN_TTS_PY_AVAILABLE else 'NO disponible → edge-tts fallback'}")
log(f"   TTS Engine disponible: {qwen3_tts.available}")
log(f"   Modelos cache: {TTS_CACHE_DIR} (en /tmp, ~15GB max, no toca /working)")
log(f"   WAVs referencia: {VOICE_REFS_DIR} (en /working, persistentes, <50MB)")
log(f"   Transcriptor: faster-whisper")
log(f"   Voz concursante validada: {CONTESTANT_VOICE_STATE.get('validated', False)}")

[18:04:20] [TTS] Cache Qwen aislada → /tmp/hf_cache_tts
[18:04:20] [TTS] /tmp libre: 1069.0 GB
[18:04:20] [TTS] ✅ soundfile.read protegido contra metadata_errors.
[18:04:20] [TTS] ✅ av.open protegido | PyAV=18.1.0
[18:04:20] [TTS] Preparando entorno Python AISLADO de Qwen3-TTS...
[18:04:21] [TTS] Venv aislado creado con: venv --without-pip
[18:04:50] [TTS] Iniciando worker Qwen aislado...
[18:04:52] [QWEN-WORKER] Proceso iniciado | GPU=Tesla T4
[18:04:52] [TTS] ✅ Worker Qwen disponible | qwen-tts=0.0.5 | transformers=4.57.3 | huggingface-hub=0.36.0 | accelerate=1.12.0 | GPU=Tesla T4
[18:04:52] [TTS] Engine disponible: True
[18:04:52] [TTS Cache] /tmp/hf_cache_tts: 0.00 GB usados en /tmp
[18:04:52] [TTS] WAVs de referencia → /kaggle/working/VideoAutomationProject/voice_refs (/working, persistentes, <50MB)
[18:04:52] 
✅ CELDA 6 (v7) COMPLETADA
[18:04:52]    Package: Qwen3-TTS aislado | qwen-tts 0.0.5 + transformers 4.57.3 + huggingface-hub 0.36.0
[18:04:52]    TTS Engine disponible: True

In [6]:
# ═══════════════════════════════════════════════════════════════
# CELDA 8 (v2): ETAPAS 2 y 4 — DIÁLOGOS (xKiro) + AUDIO CLONADO
#
# CAMBIOS vs v1:
#   - TTS → qwen3_tts.synth_clone() para TODO el guion (concursante + jueces)
#   - Auto-transcripción de jueces integrada al inicio de cada etapa
#   - Validación de voz del concursante ANTES de procesar
#   - fit_audio_exact_duration aplicado a cada fragmento
#   - Generación de video lipsync por fragmento (sin cambios)
#   - regenerar_fragmento_dialogo corregido y extendido
#   - publicar_modelos_como_dataset conservado
# ═══════════════════════════════════════════════════════════════
import asyncio
import shlex

from openai import OpenAI

# ── xKiro client (OpenAI compatible) ─────────────────────────

def xkiro_client():
    api_key = os.environ.get("XKIRO_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError(
            "Falta XKIRO_API_KEY en variables de entorno. "
            "Ponla en Kaggle Secrets → XKIRO_API_KEY."
        )
    return OpenAI(api_key=api_key, base_url=XKIRO_BASE_URL)


def call_xkiro_json(
    system_prompt: str,
    user_prompt: str,
    model_id: str = DEFAULT_XKIRO_TEXT_MODEL_ID,
    fallback_pool: List[str] = None,
    temperature: float = 0.7,
    max_tokens: int = 2500,
) -> dict:
    """
    Llama a xKiro y retorna un dict JSON.
    Prueba model_id + fallback_pool ante errores.
    """
    fallback_pool = fallback_pool or XKIRO_FALLBACK_POOL
    client = xkiro_client()
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    last_err = None
    models_to_try = [model_id] + [m for m in fallback_pool if m != model_id]

    for mid in models_to_try:
        for attempt in range(1, 4):
            try:
                resp = client.chat.completions.create(
                    model=mid,
                    messages=messages,
                    temperature=float(temperature),
                    max_tokens=int(max_tokens),
                )
                txt = (resp.choices[0].message.content or "").strip()
                # Extraer JSON aunque venga con markdown ```json ... ```
                txt_clean = re.sub(r"^```(?:json)?", "", txt).rstrip("`").strip()
                data = json.loads(txt_clean)
                log(f"[xKiro] ✅ {mid} (intento {attempt})")
                return data
            except Exception as e:
                last_err = e
                log(f"[xKiro] ⚠ {mid} intento {attempt}: {e}")
                time.sleep(2 * attempt)

    raise RuntimeError(
        f"xKiro falló en todos los modelos. Último error: {last_err}"
    )


# ═══════════════════════════════════════════════════════════════
# SECCIÓN 8.1 — GENERACIÓN DE GUION JSON (Etapas 2 y 4)
# ═══════════════════════════════════════════════════════════════

def generar_dialogos_json(
    vp: VideoProject,
    personaje_key: str,
    etapa: str,
    historia_base: str = "",
    model_id: str = DEFAULT_XKIRO_TEXT_MODEL_ID,
) -> dict:
    """
    Genera el guion JSON para etapa2 o etapa4 vía xKiro.
    Persiste en vp.estado y en disco.
    """
    assert etapa in ("etapa2", "etapa4"), f"Etapa inválida: {etapa}"

    if etapa == "etapa2":
        system = build_dialogo_inicial_system_prompt(
            personaje_key, historia_base=historia_base
        )
        user = (
            "Genera los diálogos iniciales según TODAS las reglas del system prompt. "
            "Devuelve SOLO JSON válido, sin texto adicional, sin markdown."
        )
    else:
        system = build_dialogo_final_system_prompt(personaje_key)
        user = (
            "Genera los diálogos finales según TODAS las reglas del system prompt. "
            "Devuelve SOLO JSON válido, sin texto adicional, sin markdown."
        )

    log(f"[xKiro] Generando guion {etapa} — modelo: {model_id}")
    data = call_xkiro_json(system, user, model_id=model_id)

    vp.estado.setdefault(etapa, {})["guion_json"] = data
    vp.save()
    n_frags = len(data.get("fragmentos", []))
    try:
        dur_total = float(data.get("duracion_total_estimada_min", 0) or 0)
    except Exception:
        dur_total = 0.0
    log(f"[xKiro] Guion {etapa}: {n_frags} fragmentos | ~{dur_total:.1f} min")
    return data


# ═══════════════════════════════════════════════════════════════
# SECCIÓN 8.2 — SÍNTESIS DE AUDIO POR FRAGMENTO
# ═══════════════════════════════════════════════════════════════

# ── Audio de diálogos: duración NATURAL (sin estirar/comprimir) ─────────────
DIALOGO_MAX_AUDIO_S = 15.0      # límite del lipsync LTX
DIALOGO_W, DIALOGO_H = 1152, 704  # = defaults de generar_clip_lipsync  768, 432        848, 480

def _fit_dialog_audio(in_wav: str, out_wav: str, max_s: float = DIALOGO_MAX_AUDIO_S) -> float:
    # Mantiene el audio TTS a su velocidad natural. Solo si supera max_s lo
    # acelera lo mínimo necesario. Seguro con in_wav == out_wav (ffmpeg no
    # puede editar in-place: se escribe a un temporal y se reemplaza).
    d = _ffprobe_duration(in_wav) or 0.0
    if d <= 0:
        raise RuntimeError(f"Duración WAV inválida: {in_wav}")
    if d <= max_s:
        if os.path.abspath(in_wav) != os.path.abspath(out_wav):
            shutil.copyfile(in_wav, out_wav)
        return d
    ratio = d / max_s
    log(f"[TTS] ⚠ Fragmento de {d:.1f}s supera {max_s:.0f}s → acelerado x{ratio:.2f} "
        f"(considera dividir ese fragmento del guion)")
    parts, r = [], ratio
    while r > 2.0:
        parts.append("atempo=2.0")
        r /= 2.0
    parts.append(f"atempo={r:.6f}")
    tmp = out_wav + ".fit.wav"
    cmd = ["ffmpeg", "-y", "-i", in_wav, "-af",
           ",".join(parts) + f",atrim=0:{max_s:.6f},asetpts=N/SR/TB",
           "-acodec", "pcm_s16le", "-ar", "24000", tmp, "-loglevel", "error"]
    rr = subprocess.run(cmd, capture_output=True, text=True)
    if rr.returncode != 0:
        raise RuntimeError(f"FFmpeg falló ajustando audio:\n{(rr.stderr or '')[-1500:]}")
    os.replace(tmp, out_wav)
    return max_s


def tts_fragmento(
    vp: VideoProject,
    etapa: str,
    idx: int,
    hablante: str,
    texto: str,
    dur_target_s: float,
    plantilla_slug: str,
    fast_mode: bool = False,
) -> str:
    """
    Genera audio para UN fragmento de diálogo:
    - Usa synth_fragment_for_dialog() → clonación siempre
    - Normaliza a 24k mono
    - Ajusta a dur_target_s exactos
    Retorna la ruta del WAV final.
    """
    assert etapa in ("etapa2", "etapa4"), f"Etapa inválida: {etapa}"

    out_dir = os.path.join(vp.video_dir, f"{etapa}_dialogos", "audio")
    os.makedirs(out_dir, exist_ok=True)

    safe_hablante = re.sub(r"[^a-z0-9_]", "", hablante)
    raw_wav = os.path.join(out_dir, f"frag_{idx:03d}_{safe_hablante}_raw.wav")
    out_wav = os.path.join(out_dir, f"frag_{idx:03d}_{safe_hablante}.wav")

    # ── Síntesis ──────────────────────────────────────────────
    synth_fragment_for_dialog(
        texto=texto,
        hablante=hablante,
        out_wav=raw_wav,
        plantilla_slug=plantilla_slug,
        fast_mode=fast_mode,
    )

    # ── Normalizar → 24k mono ─────────────────────────────────
    ffmpeg_to_wav_mono_24k(raw_wav, out_wav)

    # ── Duración natural (máx. 15s). dur_target_s es solo orientativo ──
    _fit_dialog_audio(out_wav, out_wav, DIALOGO_MAX_AUDIO_S)

    # ── Verificación ──────────────────────────────────────────
    dur_real = _ffprobe_duration(out_wav) or 0
    log(f"[TTS] Frag {idx:03d} {hablante} | target={dur_target_s:.1f}s | real={dur_real:.1f}s")

    return out_wav


# ═══════════════════════════════════════════════════════════════
# SECCIÓN 8.3 — PIPELINE COMPLETO DE DIÁLOGOS (Etapas 2 y 4)
# ═══════════════════════════════════════════════════════════════

def ejecutar_etapa_dialogos(
    vp: VideoProject,
    plantilla_slug: str,
    personaje_key: str,
    etapa: str,
    ltx_paths: dict,
    historia_base: str = "",
    model_id: str = DEFAULT_XKIRO_TEXT_MODEL_ID,
    fast_mode_tts: bool = False,      # ← AÑADIDO (era el error)
    voz_concursante_label: str = "Mujer — Cálida y maternal",
    regenerar_guion: bool = False,
) -> str:
    """
    Pipeline completo para una etapa de diálogos:
    1. Auto-transcribir jueces (si no está hecho)
    2. Validar voz del concursante
    3. Generar o reutilizar guion JSON
    4. Sintetizar audio por fragmento (clonación)
    5. Generar clips de video lipsync por fragmento
    6. Concatenar video+audio final de la etapa
    Retorna la ruta del video final de la etapa.
    """
    assert etapa in ("etapa2", "etapa4"), f"Etapa inválida: {etapa}"
    log(f"\n[{etapa.upper()}] Iniciando pipeline de diálogos...")

    # ── PASO 1: Auto-transcribir jueces ──────────────────────
    log(f"[{etapa}] Transcribiendo audios de jueces...")
    jueces_transcritos = auto_transcribe_all_judges(plantilla_slug)
    for jid, txt in jueces_transcritos.items():
        if txt:
            log(f"  {jid}: '{txt[:60]}...'")
        else:
            log(f"  {jid}: ⚠ sin audio o transcripción vacía")

    # ── PASO 2: Validar voz del concursante ──────────────────
    ref_info = get_contestant_ref_for_tts()  # Raises si no está validado
    log(f"[{etapa}] Voz concursante OK: {os.path.basename(ref_info['ref_wav'])}")

    # ── PASO 3: Generar o reutilizar guion ───────────────────
    guion_existente = vp.estado.get(etapa, {}).get("guion_json", {})
    if guion_existente and not regenerar_guion:
        log(f"[{etapa}] Reutilizando guion existente ({len(guion_existente.get('fragmentos',[]))} frags).")
        data = guion_existente
    else:
        data = generar_dialogos_json(
            vp, personaje_key, etapa,
            historia_base=historia_base,
            model_id=model_id,
        )

    fragmentos = data.get("fragmentos", [])
    if not fragmentos:
        raise RuntimeError(f"El guion de {etapa} está vacío o mal formado.")

    # ── PASO 4: Sintetizar audio por fragmento ────────────────
    audio_dir = os.path.join(vp.video_dir, f"{etapa}_dialogos", "audio")
    os.makedirs(audio_dir, exist_ok=True)
    audio_paths = []

    log(f"[{etapa}] Sintetizando {len(fragmentos)} fragmentos de audio...")
    for i, frag in enumerate(fragmentos):
        hablante  = frag.get("hablante", "concursante")
        texto     = frag.get("texto", "").strip()
        dur_est   = float(frag.get("duracion_estimada_s", 5.0))

        if not texto:
            log(f"  Frag {i:03d} VACÍO — saltando.")
            continue

        out_wav = tts_fragmento(
            vp=vp,
            etapa=etapa,
            idx=i,
            hablante=hablante,
            texto=texto,
            dur_target_s=dur_est,
            plantilla_slug=plantilla_slug,
            fast_mode=fast_mode_tts,
        )
        audio_paths.append({"idx": i, "hablante": hablante, "wav": out_wav, "dur": dur_est})

    log(f"[{etapa}] ✅ {len(audio_paths)}/{len(fragmentos)} fragmentos sintetizados.")

    # ── PASO 5: Generar video lipsync por fragmento ───────────
    video_dir_etapa = os.path.join(vp.video_dir, f"{etapa}_dialogos", "video")
    os.makedirs(video_dir_etapa, exist_ok=True)

    img_dir = os.path.join(vp.video_dir, "etapa1_imagenes")
    img_concursante_c = os.path.join(img_dir, "concursante_plano_cercano.png")
    img_concursante_d = os.path.join(img_dir, "concursante_closeup.png")
    juez_imgs_dir = os.path.join(
        cargar_plantilla(plantilla_slug)["_base_dir"],
        PLANTILLA_SUBDIRS["jueces_imagenes"]
    )

    video_paths = []
    for item in audio_paths:
        i        = item["idx"]
        hablante = item["hablante"]
        wav      = item["wav"]
        dur      = item["dur"]
        frag     = fragmentos[i]
        emocion  = frag.get("emocion", "neutral")

        # Elegir imagen de referencia según hablante
        if hablante == "concursante":
            # Alternar entre plano cercano y closeup para variedad
            img_ref = img_concursante_d if i % 3 == 0 else img_concursante_c
        else:
            juez_id = hablante.replace("juez_", "")
            img_ref = os.path.join(juez_imgs_dir, f"juez_{juez_id}.png")
            if not os.path.exists(img_ref):
                img_ref = img_concursante_c  # fallback

        out_clip = os.path.join(video_dir_etapa, f"frag_{i:03d}_{hablante}.mp4")

        if os.path.exists(out_clip) and os.path.getsize(out_clip) > 10000:
            log(f"  Frag {i:03d}: video ya existe, reutilizando.")
            video_paths.append(out_clip)
            continue

        try:
            if not os.path.exists(img_ref):
                raise FileNotFoundError(f"Imagen ref no encontrada: {img_ref}")

            # Prompt de lipsync según hablante
            if hablante == "concursante":
                prompt_ls = render_prompt_lipsync_dialogo(personaje_key)
            else:
                juez_num  = int(hablante.split("_")[1]) if "_" in hablante else 1
                prompt_ls = (
                    f"El juez hablando con emoción natural, "
                    f"sincronización labial realista, "
                    f"expresión {emocion}, mirada al concursante."
                )

            clip_path = generar_clip_lipsync_dialogo(
                vp=vp,
                img_ref=img_ref,
                audio_wav=wav,
                prompt=prompt_ls,
                out_mp4=out_clip,
                ltx_paths=ltx_paths,
                dur_s=dur,
            )
            video_paths.append(clip_path)
            log(f"  Frag {i:03d} {hablante}: ✅ {os.path.basename(clip_path)}")

        except Exception as e:
            log(f"  Frag {i:03d} {hablante}: ❌ {e} — generando clip imagen estática.")
            static_clip = _make_static_clip_with_audio(img_ref, wav, out_clip, dur)
            if static_clip:
                video_paths.append(static_clip)

    # ── PASO 6: Concatenar video final de la etapa ───────────
    if not video_paths:
        raise RuntimeError(f"No se generaron clips de video para {etapa}.")

    video_final = os.path.join(vp.video_dir, f"{etapa}_dialogos", f"{etapa}_final.mp4")
    _concatenar_clips(video_paths, video_final)

    vp.estado.setdefault(etapa, {})["video_final"] = video_final
    vp.estado[etapa]["audio_paths"] = [a["wav"] for a in audio_paths]
    vp.estado[etapa]["n_fragmentos"] = len(fragmentos)
    vp.save()

    dur_final = _ffprobe_duration(video_final) or 0
    log(f"[{etapa.upper()}] ✅ Video final: {video_final} ({dur_final:.1f}s)")
    return video_final


def _make_static_clip_with_audio(img_path: str, audio_wav: str, out_mp4: str, dur_s: float) -> str:
    """
    Fallback: crea un clip de video estático (imagen fija + audio) cuando el lipsync falla.
    """
    try:
        os.makedirs(os.path.dirname(os.path.abspath(out_mp4)), exist_ok=True)
        cmd = (
            f"ffmpeg -y "
            f"-loop 1 -i '{img_path}' "
            f"-i '{audio_wav}' "
            f"-t {dur_s:.3f} "
            f"-vf 'scale=trunc(iw/2)*2:trunc(ih/2)*2' "
            f"-c:v libx264 -pix_fmt yuv420p "
            f"-c:a aac -shortest "
            f"'{out_mp4}' -loglevel error"
        )
        r = subprocess.run(cmd, shell=True, capture_output=True)
        if r.returncode == 0 and os.path.exists(out_mp4):
            return out_mp4
    except Exception as e:
        log(f"[StaticClip] Error: {e}")
    return ""


def _concatenar_clips(clip_paths: List[str], out_mp4: str) -> str:
    """Concatena una lista de clips MP4 en un solo archivo."""
    os.makedirs(os.path.dirname(os.path.abspath(out_mp4)), exist_ok=True)
    list_file = out_mp4 + ".concat_list.txt"
    with open(list_file, "w", encoding="utf-8") as f:
        for p in clip_paths:
            f.write(f"file '{os.path.abspath(p)}'\n")
    cmd = (
        f"ffmpeg -y -f concat -safe 0 -i '{list_file}' "
        f"-c:v libx264 -pix_fmt yuv420p -c:a aac "
        f"'{out_mp4}' -loglevel error"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    try:
        os.remove(list_file)
    except Exception:
        pass
    if r.returncode != 0:
        raise RuntimeError(f"FFmpeg concat falló:\n{(r.stderr or '')[-2000:]}")
    return out_mp4


def generar_clip_lipsync_dialogo(
    vp: VideoProject,
    img_ref: str,
    audio_wav: str,
    prompt: str,
    out_mp4: str,
    ltx_paths: dict,
    dur_s: float,
    fps: int = 24,
) -> str:
    """
    Genera un clip de video con lipsync usando LTX-2.5 (Wan2GP).
    Si Wan2GP no está disponible, usa clip estático como fallback.
    """
    try:
        backend = get_ltx_backend()
        if not backend.is_available():
            raise RuntimeError("Backend LTX-2.5 no disponible.")

        result = backend.generate_lipsync(
            image_path=img_ref,
            audio_path=audio_wav,
            prompt=prompt,
            output_path=out_mp4,
            duration_s=dur_s,
            fps=fps,
            ltx_paths=ltx_paths,
        )
        return result
    except Exception as e:
        log(f"[Lipsync] ⚠ LTX falló ({e}). Usando clip estático.")
        return _make_static_clip_with_audio(img_ref, audio_wav, out_mp4, dur_s)


# ═══════════════════════════════════════════════════════════════
# SECCIÓN 8.4 — REGENERAR FRAGMENTO INDIVIDUAL
# ═══════════════════════════════════════════════════════════════

def regenerar_fragmento_dialogo(
    vp: VideoProject,
    plantilla_slug: str,
    personaje_key: str,
    etapa: str,
    frag_idx: int,
    nuevo_texto: str = "",
    ltx_paths: dict = None,
    fast_mode_tts: bool = False,
) -> str:
    """
    Regenera UN fragmento específico del guion (audio + video).
    Si nuevo_texto está vacío, usa el texto original del guion.
    Actualiza el guion JSON y persiste.
    Retorna status_msg.
    """
    guion = (vp.estado.get(etapa, {}) or {}).get("guion_json", {})
    fragmentos = guion.get("fragmentos", [])
    if not (0 <= frag_idx < len(fragmentos)):
        return f"❌ Índice {frag_idx} fuera de rango (0..{len(fragmentos)-1})."

    frag = fragmentos[frag_idx]
    hablante   = frag.get("hablante", "concursante")
    texto_orig = frag.get("texto", "")
    dur_est    = float(frag.get("duracion_estimada_s", 5.0))

    texto_final = nuevo_texto.strip() if nuevo_texto and nuevo_texto.strip() else texto_orig
    if not texto_final:
        return "❌ El fragmento no tiene texto."

    log(f"[Regen] {etapa} frag {frag_idx} ({hablante}): '{texto_final[:60]}...'")

    # Actualizar texto en guion si se proporcionó uno nuevo
    if nuevo_texto and nuevo_texto.strip():
        fragmentos[frag_idx]["texto"] = texto_final
        # Recalcular duración estimada
        dur_nueva = max(3.0, len(texto_final.split()) / 2.5)
        fragmentos[frag_idx]["duracion_estimada_s"] = dur_nueva
        dur_est = dur_nueva
        vp.estado[etapa]["guion_json"]["fragmentos"] = fragmentos
        vp.save()

    try:
        # Regenerar audio
        out_wav = tts_fragmento(
            vp=vp,
            etapa=etapa,
            idx=frag_idx,
            hablante=hablante,
            texto=texto_final,
            dur_target_s=dur_est,
            plantilla_slug=plantilla_slug,
            fast_mode=fast_mode_tts,
        )

        # Regenerar video lipsync
        img_dir = os.path.join(vp.video_dir, "etapa1_imagenes")
        img_concursante_c = os.path.join(img_dir, "concursante_plano_cercano.png")
        img_concursante_d = os.path.join(img_dir, "concursante_closeup.png")
        juez_imgs_dir = os.path.join(
            cargar_plantilla(plantilla_slug)["_base_dir"],
            PLANTILLA_SUBDIRS["jueces_imagenes"]
        )

        if hablante == "concursante":
            img_ref = img_concursante_d if frag_idx % 3 == 0 else img_concursante_c
            prompt_ls = render_prompt_lipsync_dialogo(personaje_key)
        else:
            juez_id = hablante.replace("juez_", "")
            img_ref = os.path.join(juez_imgs_dir, f"juez_{juez_id}.png")
            if not os.path.exists(img_ref):
                img_ref = img_concursante_c
            prompt_ls = (
                f"El juez hablando con emoción natural, "
                f"sincronización labial realista, expresión {frag.get('emocion','neutral')}."
            )

        video_dir_etapa = os.path.join(vp.video_dir, f"{etapa}_dialogos", "video")
        safe_hablante   = re.sub(r"[^a-z0-9_]", "", hablante)
        out_clip        = os.path.join(video_dir_etapa, f"frag_{frag_idx:03d}_{safe_hablante}.mp4")

        generar_clip_lipsync_dialogo(
            vp=vp,
            img_ref=img_ref,
            audio_wav=out_wav,
            prompt=prompt_ls,
            out_mp4=out_clip,
            ltx_paths=ltx_paths or {},
            dur_s=dur_est,
        )

        dur_real = _ffprobe_duration(out_wav) or 0
        return (f"✅ Fragmento {frag_idx} regenerado.\n"
                f"Hablante: {hablante} | Duración: {dur_real:.1f}s\n"
                f"Texto: «{texto_final[:100]}»")

    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        log(f"[Regen] ❌ {e}\n{tb[-1000:]}")
        return f"❌ Error regenerando fragmento {frag_idx}: {e}"


# ═══════════════════════════════════════════════════════════════
# SECCIÓN 8.5 — RENDER FINAL (Etapas 2+3+4 → video completo)
# ═══════════════════════════════════════════════════════════════

def render_video_final(
    vp: VideoProject,
    include: dict = None,
) -> str:
    """
    Concatena los videos de las etapas en el orden:
    E2 (diálogos) → E3 (canción) → E4 (diálogos finales)
    Retorna la ruta del video final.
    """
    if include is None:
        include = {"etapa2": True, "etapa3": True, "etapa4": True}

    clips = []
    if include.get("etapa2"):
        v2 = vp.estado.get("etapa2", {}).get("video_final", "")
        if v2 and os.path.exists(v2):
            clips.append(v2)
        else:
            log("[Render Final] ⚠ Etapa 2 no disponible.")

    if include.get("etapa3"):
        v3 = vp.estado.get("etapa3", {}).get("video_final", "")
        if v3 and os.path.exists(v3):
            clips.append(v3)
        else:
            log("[Render Final] ⚠ Etapa 3 no disponible.")

    if include.get("etapa4"):
        v4 = vp.estado.get("etapa4", {}).get("video_final", "")
        if v4 and os.path.exists(v4):
            clips.append(v4)
        else:
            log("[Render Final] ⚠ Etapa 4 no disponible.")

    if not clips:
        raise RuntimeError("No hay clips disponibles para el render final.")

    out_final = os.path.join(vp.video_dir, "VIDEO_FINAL.mp4")
    _concatenar_clips(clips, out_final)

    vp.estado["video_final"] = out_final
    vp.save()

    dur = _ffprobe_duration(out_final) or 0
    log(f"[Render Final] ✅ {out_final} ({dur:.1f}s, {len(clips)} etapas)")
    return out_final


# ═══════════════════════════════════════════════════════════════
# SECCIÓN 8.6 — PUBLICAR MODELOS COMO DATASET KAGGLE
# ═══════════════════════════════════════════════════════════════

def publicar_modelos_como_dataset() -> str:
    """Publica LOCAL_MODELS_DIR como Kaggle Dataset para persistencia."""
    return publish_dir_as_kaggle_dataset(
        dataset_slug="audiciones-modelos",
        src_dir=LOCAL_MODELS_DIR,
        title="Audiciones — Modelos persistentes",
        update_existing=True,
    )
# ═══════════════════════════════════════════════════════════════
# SECCIÓN 8.7 — EDITAR GUION MANUALMENTE (NUEVA FUNCIONALIDAD)
# ═══════════════════════════════════════════════════════════════
def guardar_guion_editado(vp: VideoProject, etapa: str, guion_json_str: str) -> str:
    """
    Guarda un guion JSON editado manualmente por el usuario.
    Valida estructura antes de guardar.
    Retorna mensaje de estado.
    """
    try:
        data = json.loads(guion_json_str)
        fragmentos = data.get("fragmentos", [])
        if not fragmentos:
            return "❌ El JSON no contiene 'fragmentos' o está vacío."

        # Validar estructura mínima de cada fragmento
        for i, frag in enumerate(fragmentos):
            if "hablante" not in frag or "texto" not in frag:
                return f"❌ Fragmento {i}: falta 'hablante' o 'texto'."
            if "duracion_estimada_s" not in frag:
                # Auto-calcular duración si falta
                frag["duracion_estimada_s"] = max(3.0, len(frag["texto"].split()) / 2.5)
            if "emocion" not in frag:
                frag["emocion"] = "neutral"

        # Recalcular duración total
        total_s = sum(f.get("duracion_estimada_s", 5.0) for f in fragmentos)
        data["duracion_total_estimada_min"] = round(total_s / 60.0, 2)

        # Guardar en estado del proyecto
        vp.estado.setdefault(etapa, {})["guion_json"] = data
        vp.save()

        n_frags = len(fragmentos)
        return (f"✅ Guion {etapa} guardado correctamente.\n"
                f"   {n_frags} fragmentos | ~{data['duracion_total_estimada_min']:.1f} min\n"
                f"   → Pulsa 'Ejecutar Etapa' para sintetizar con este guion.")

    except json.JSONDecodeError as e:
        return f"❌ JSON inválido: {e}"
    except Exception as e:
        return f"❌ Error guardando guion: {e}"




log("\n✅ CELDA 8 (v2) COMPLETADA")
log("   Pipeline diálogos: clonación consistente para concursante y jueces")
log("   Auto-transcripción: faster-whisper (jueces + concursante)")
log("   Fallback lipsync: clip estático con audio si LTX no disponible")

[18:05:26] 
✅ CELDA 8 (v2) COMPLETADA
[18:05:26]    Pipeline diálogos: clonación consistente para concursante y jueces
[18:05:26]    Auto-transcripción: faster-whisper (jueces + concursante)
[18:05:26]    Fallback lipsync: clip estático con audio si LTX no disponible


In [7]:
# ═══════════════════════════════════════════════════════════════
# CELDA 9 (v2): ETAPA 3 — CANCIÓN CON DISTRIBUCIÓN RÍTMICA
# Reglas de distribución:
#   - NUNCA dos lipsync consecutivos
#   - Máximo 2 reacciones consecutivas
#   - Patrón preferido: Reacción → PlanoLejano5s → Lipsync → PlanoLejano → Reacción
#   - Intro (~12%): solo reacciones + planos lejanos (SIN lipsync)
#   - Audio de canción = pista maestra del render final
#   - Frame-perfect: cada clip ajustado a duración exacta
# ═══════════════════════════════════════════════════════════════
def cut_audio_segment(in_audio: str, out_wav: str, start_s: float, dur_s: float, sr: int = 24000) -> str:
    os.makedirs(os.path.dirname(out_wav), exist_ok=True)
    cmd = f"ffmpeg -y -i '{in_audio}' -ss {start_s:.6f} -t {dur_s:.6f} -ac 1 -ar {sr} -acodec pcm_s16le '{out_wav}'"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("FFmpeg falló:\n" + (r.stderr or "")[-2000:])
    return out_wav

def mute_video_audio(in_video: str, out_video: str) -> str:
    os.makedirs(os.path.dirname(out_video), exist_ok=True)
    cmd = f"ffmpeg -y -i '{in_video}' -c:v copy -an '{out_video}'"
    subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return out_video

def trim_or_pad_video_to_duration(in_video: str, out_video: str, target_s: float, fps: int = 30) -> str:
    """Hace que cada fragmento dure EXACTAMENTE target_s para evitar desfaces."""
    d = _ffprobe_duration(in_video) or 0.0
    os.makedirs(os.path.dirname(out_video), exist_ok=True)
    if d <= 0:
        raise RuntimeError("Duración de video inválida.")
    if d >= target_s:
        cmd = f"ffmpeg -y -i '{in_video}' -t {target_s:.6f} -r {fps} -c:v libx264 -pix_fmt yuv420p -an '{out_video}'"
    else:
        pad = target_s - d
        cmd = (f"ffmpeg -y -i '{in_video}' "
               f"-vf 'tpad=stop_mode=clone:stop_duration={pad:.6f}' "
               f"-t {target_s:.6f} -r {fps} -c:v libx264 -pix_fmt yuv420p -an '{out_video}'")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("FFmpeg falló:\n" + (r.stderr or "")[-2000:])
    return out_video

def build_song_timeline(vp: VideoProject, plantilla_slug: str, personaje_key: str,
                        song_audio_path: str, use_far_planes: List[str],
                        target_fps: int = 30) -> list:
    """
    Construye timeline con distribución rítmica profesional:
      - Intro (~12%): reacciones + planos lejanos ONLY (sin lipsync)
      - Main: ciclo Reacción → PlanoLejano5s → Lipsync → PlanoLejano → Reacción
      - NUNCA dos lipsync consecutivos
      - Máximo 2 reacciones consecutivas (solo cuando no queda tiempo para lipsync)
    """
    tpl = cargar_plantilla(plantilla_slug)
    base = tpl["_base_dir"]

    song_dur = _ffprobe_duration(song_audio_path) or 0.0
    if song_dur <= 5:
        raise RuntimeError("Audio de canción demasiado corto.")

    # ── Cargar assets ──────────────────────────────────────────
    react1_dir = os.path.join(base, PLANTILLA_SUBDIRS["reacciones_primera_mitad"])
    react2_dir = os.path.join(base, PLANTILLA_SUBDIRS["reacciones_segunda_mitad"])
    reacts1 = sorted([os.path.join(react1_dir, f) for f in os.listdir(react1_dir)
                      if f.lower().endswith(".mp4")]) if os.path.isdir(react1_dir) else []
    reacts2 = sorted([os.path.join(react2_dir, f) for f in os.listdir(react2_dir)
                      if f.lower().endswith(".mp4")]) if os.path.isdir(react2_dir) else []

    far5_dir = os.path.join(vp.video_dir, "etapa3_cancion", "planos_5s")
    far5 = sorted([os.path.join(far5_dir, f) for f in os.listdir(far5_dir)
                   if f.lower().endswith(".mp4")]) if os.path.isdir(far5_dir) else []
    if not far5:
        raise RuntimeError("No hay clips 5s generados. Ejecuta PASO 3.1 primero.")

    img_dir = os.path.join(vp.video_dir, "etapa1_imagenes")
    img_C = os.path.join(img_dir, "concursante_plano_cercano.png")
    img_D = os.path.join(img_dir, "concursante_closeup.png")
    if not os.path.exists(img_C) or not os.path.exists(img_D):
        raise RuntimeError("Faltan imágenes C/D para lipsync.")

    # ── Helpers internos ───────────────────────────────────────
    clips = []
    t = 0.0
    clip_idx = 0
    consecutive_reactions = 0   # contador para limitar a max 2
    last_was_lipsync = False    # flag para NUNCA poner 2 lipsync seguidos

    def add(tipo, hablante, dur, path):
        nonlocal clip_idx, clips, t, consecutive_reactions, last_was_lipsync
        clips.append(TimelineClip(
            clip_id=f"c{clip_idx:04d}",
            tipo=tipo,
            hablante=hablante,
            dur_s=float(dur),
            path_video=path,
            estado="pendiente",
            meta={}
        ))
        clip_idx += 1
        t += float(dur)
        # Actualizar contadores de ritmo
        if tipo == "reaccion":
            consecutive_reactions += 1
            last_was_lipsync = False
        elif tipo == "lipsync_15s":
            consecutive_reactions = 0
            last_was_lipsync = True
        else:  # plano_lejano_5s
            consecutive_reactions = 0
            last_was_lipsync = False

    def pick_reaction():
        """Selecciona reacción según mitad de la canción."""
        if reacts2 and (t > song_dur / 2):
            return random.choice(reacts2)
        elif reacts1:
            return random.choice(reacts1)
        return None

    def remaining():
        return song_dur - t

    # ══════════════════════════════════════════════════════════
    # FASE 1: INTRO (~12% de la canción, máx 30s)
    # Solo reacciones + planos lejanos. NINGÚN lipsync.
    # ══════════════════════════════════════════════════════════
    intro_target = min(30.0, song_dur * 0.12)
    while t < intro_target and remaining() > 1.0:
        rv = pick_reaction()
        if rv:
            d = _ffprobe_duration(rv) or 3.0
            if t + d <= intro_target + 2.0:  # tolerancia pequeña
                add("reaccion", f"juez_{random.randint(1,4)}", d, rv)
        add("plano_lejano_5s", "concursante", 5.0, random.choice(far5))

    # ══════════════════════════════════════════════════════════
    # FASE 2: MAIN LOOP — Distribución rítmica profesional
    # Patrón: Reacción → PlanoLejano5s → Lipsync → PlanoLejano → (repeat)
    # Reglas:
    #   - Nunca 2 lipsync seguidos
    #   - Max 2 reacciones seguidas
    #   - Si no cabe lipsync (15s), rellenar con lejano+reacción
    # ══════════════════════════════════════════════════════════
    while t + 1.0 < song_dur:
        rem = remaining()

        # ── Caso A: Cabe un lipsync completo (≥15s) y NO viene de otro lipsync ──
        if rem >= 10.0 and not last_was_lipsync:
            # Paso 1: Reacción (si no llevamos ya 2 seguidas)
            if consecutive_reactions < 2:
                rv = pick_reaction()
                if rv:
                    d = _ffprobe_duration(rv) or 3.0
                    if t + d + 5.0 + 10.0 <= song_dur + 2.0:  # cabe reacción+lejano+lipsync
                        add("reaccion", f"juez_{random.randint(1,4)}", d, rv)

            # Paso 2: Plano lejano 5s (separador visual antes del lipsync)
            if t + 5.0 + 10.0 <= song_dur + 2.0:
                add("plano_lejano_5s", "concursante", 5.0, random.choice(far5))

            # Paso 3: Lipsync 15s
            if remaining() >= 10.0:
                add("lipsync_15s", "concursante", 10.0,
                    img_C if random.random() < 0.5 else img_D)

            # Paso 4: Plano lejano 5s (separador visual después del lipsync)
            if remaining() >= 5.0:
                add("plano_lejano_5s", "concursante", 5.0, random.choice(far5))

        # ── Caso B: No cabe lipsync pero cabe plano lejano + reacción ──
        elif rem >= 5.0:
            add("plano_lejano_5s", "concursante", 5.0, random.choice(far5))
            # Reacción solo si no llevamos 2 seguidas
            if consecutive_reactions < 2:
                rv = pick_reaction()
                if rv:
                    d = _ffprobe_duration(rv) or 3.0
                    if t + d <= song_dur + 1.0:
                        add("reaccion", f"juez_{random.randint(1,4)}", d, rv)

        # ── Caso C: Solo queda relleno fino (<5s) ──
        else:
            # Permitir hasta 2 reacciones seguidas como cierre
            if consecutive_reactions < 2:
                rv = pick_reaction()
                if rv:
                    d = min(_ffprobe_duration(rv) or 3.0, rem)
                    if d >= 0.5:
                        add("reaccion", f"juez_{random.randint(1,4)}", d, rv)
            # Si aún sobra tiempo, un último plano lejano recortado
            if remaining() >= 1.0:
                add("plano_lejano_5s", "concursante", min(5.0, remaining()), random.choice(far5))
            break

        # Seguridad anti-loop infinito
        if len(clips) > 9999:
            log("[TIMELINE] ⚠ Límite de seguridad alcanzado (9999 clips).")
            break

    # ── Ajuste final: recortar último clip para cuadrar con song_dur ──
    total = sum(c.dur_s for c in clips)
    if total > song_dur and clips:
        excess = total - song_dur
        clips[-1].dur_s = max(0.5, clips[-1].dur_s - excess)

    # ── Persistir ──────────────────────────────────────────────
    vp.timeline_items = timeline_to_json(clips)
    vp.save()

    # Log resumen de distribución
    tipo_counts = {}
    for c in clips:
        tipo_counts[c.tipo] = tipo_counts.get(c.tipo, 0) + 1
    log(f"[TIMELINE] Construido: {len(clips)} clips | {song_dur:.1f}s total")
    log(f"[TIMELINE] Distribución: {tipo_counts}")

    # Verificar regla: nunca 2 lipsync seguidos
    for i in range(len(clips) - 1):
        if clips[i].tipo == "lipsync_15s" and clips[i+1].tipo == "lipsync_15s":
            log(f"[TIMELINE] ⚠ VIOLACIÓN: 2 lipsync seguidos en {clips[i].clip_id}/{clips[i+1].clip_id}")

    return vp.timeline_items

log("CELDA 9 (v2) COMPLETADA — Distribución rítmica profesional.")

[18:05:34] CELDA 9 (v2) COMPLETADA — Distribución rítmica profesional.


In [8]:
# ═══════════════════════════════════════════════════════════════
# CELDA 10: FUNCIONES AUXILIARES DE UI (SIN LAUNCH — UI real en Celda 18)
# ═══════════════════════════════════════════════════════════════

# --- Estado global UI (persistente por working) ---
GLOBAL_STATE_PATH = os.path.join(OUTPUT_DIR, "ui_state.json")

def load_ui_state():
    if os.path.exists(GLOBAL_STATE_PATH):
        try:
            return json.load(open(GLOBAL_STATE_PATH, "r", encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_ui_state(st: dict):
    json.dump(
        st,
        open(GLOBAL_STATE_PATH, "w", encoding="utf-8"),
        ensure_ascii=False,
        indent=2
    )

UI_STATE         = load_ui_state()
CURRENT_VIDEO_DIR = UI_STATE.get("current_video_dir", "")
CURRENT_TEMPLATE  = UI_STATE.get("current_template_slug", "")

def ui_set_current(video_dir: str, template_slug: str):
    global CURRENT_VIDEO_DIR, CURRENT_TEMPLATE
    CURRENT_VIDEO_DIR = video_dir
    CURRENT_TEMPLATE  = template_slug
    UI_STATE["current_video_dir"]     = video_dir
    UI_STATE["current_template_slug"] = template_slug
    save_ui_state(UI_STATE)

def get_current_vp() -> VideoProject:
    if not CURRENT_VIDEO_DIR or not os.path.isdir(CURRENT_VIDEO_DIR):
        raise RuntimeError(
            "No hay video activo. Crea un proyecto de video primero."
        )
    return VideoProject(CURRENT_VIDEO_DIR)

# ---------------------------------------------------------------
# Funciones auxiliares de UI reutilizadas por Celda 18
# ---------------------------------------------------------------
def ui_refresh_templates():
    opts = listar_plantillas()
    return gr.update(
        choices=[(n, slug) for (n, slug) in opts],
        value=(
            CURRENT_TEMPLATE if CURRENT_TEMPLATE
            else (opts[0][1] if opts else None)
        ),
    )

def ui_upload_template_assets(
    template_slug, juez_imgs, escenario_cercano,
    escenarios_lejanos, juez_voice_wavs, reacciones_1, reacciones_2
):
    tpl  = cargar_plantilla(template_slug)
    base = tpl["_base_dir"]

    def _copy_files(files, dst_dir, rename_map=None):
        os.makedirs(dst_dir, exist_ok=True)
        out = []
        if not files:
            return out
        if not isinstance(files, list):
            files = [files]
        for i, f in enumerate(files, start=1):
            if not f:
                continue
            src = f if isinstance(f, str) else getattr(f, "name", "")
            if not src or not os.path.exists(src):
                continue
            fn = os.path.basename(src)
            if rename_map and i in rename_map:
                fn = rename_map[i]
            dst = os.path.join(dst_dir, fn)
            shutil.copyfile(src, dst)
            out.append(dst)
        return out

    # Jueces
    jdir = os.path.join(base, PLANTILLA_SUBDIRS["jueces_imagenes"])
    _copy_files(
        juez_imgs, jdir,
        rename_map={1:"juez_1.png", 2:"juez_2.png",
                    3:"juez_3.png", 4:"juez_4.png"}
    )

    # Escenario cercano
    cdir = os.path.join(base, PLANTILLA_SUBDIRS["escenario_cercano"])
    if escenario_cercano:
        src = (escenario_cercano if isinstance(escenario_cercano, str)
               else escenario_cercano.name)
        shutil.copyfile(src, os.path.join(cdir, "plantilla_plano_cercano.png"))

    # Escenarios lejanos
    fdir = os.path.join(base, PLANTILLA_SUBDIRS["escenario_lejano"])
    _copy_files(
        escenarios_lejanos, fdir,
        rename_map={1:"plantilla_plano_lejano_1.png",
                    2:"plantilla_plano_lejano_2.png",
                    3:"plantilla_plano_lejano_3.png"}
    )

    # Voces jueces
    vdir  = os.path.join(base, PLANTILLA_SUBDIRS["voces_jueces"])
    vpaths = _copy_files(
        juez_voice_wavs, vdir,
        rename_map={1:"juez_1.wav", 2:"juez_2.wav",
                    3:"juez_3.wav", 4:"juez_4.wav"}
    )
    for p in vpaths:
        ffmpeg_to_wav_mono_24k(p, p)

    # Reacciones
    r1dir = os.path.join(base, PLANTILLA_SUBDIRS["reacciones_primera_mitad"])
    r2dir = os.path.join(base, PLANTILLA_SUBDIRS["reacciones_segunda_mitad"])
    _copy_files(reacciones_1, r1dir)
    _copy_files(reacciones_2, r2dir)

    # Actualizar meta
    meta = cargar_plantilla(template_slug)
    meta["planos_lejanos_disponibles"] = sum(
        1 for i in [1,2,3]
        if os.path.exists(
            os.path.join(
                base,
                PLANTILLA_SUBDIRS["escenario_lejano"],
                f"plantilla_plano_lejano_{i}.png"
            )
        )
    )
    guardar_plantilla(meta)
    return "Assets cargados/actualizados OK."

def ui_set_song_audio(song_file):
    vp = get_current_vp()
    if not song_file:
        return "No se subió audio."
    src = song_file if isinstance(song_file, str) else song_file.name
    out = os.path.join(vp.video_dir, "etapa3_cancion", "song.wav")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    ffmpeg_to_wav_mono_24k(src, out)
    vp.estado.setdefault("etapa3", {})["song_wav"] = out
    vp.save()
    return f"Canción lista: {out} ({_ffprobe_duration(out):.2f}s)"

def ui_build_timeline(tpl, personaje):
    vp   = get_current_vp()
    song = vp.estado.get("etapa3", {}).get("song_wav", "")
    if not song or not os.path.exists(song):
        raise RuntimeError("Sube primero el audio de la canción.")
    tl   = build_song_timeline(
        vp, tpl, personaje,
        song_audio_path=song,
        use_far_planes=["E","F","G"]
    )
    html = render_timeline_html(tl)
    return "Timeline construido.", html

# ---------------------------------------------------------------
# SIN demo.launch() — La UI real está en CELDA 18
# ---------------------------------------------------------------
log("CELDA 10 COMPLETADA (funciones auxiliares, sin UI propia).")

[18:05:39] CELDA 10 COMPLETADA (funciones auxiliares, sin UI propia).


In [9]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print("✅ hf_transfer desactivado")

✅ hf_transfer desactivado


In [10]:
# ═══════════════════════════════════════════════════════════════
# CELDA 12 (v3 — DISK-SAFE): LTX-2.5 MODELOS
# CAMBIO CRÍTICO vs v2:
#   - TODO va a /tmp/models (NO toca /kaggle/working)
#   - Wan2GP/models/ y Wan2GP/ckpts/ son SOLO symlinks → /tmp/models/
#   - Companions (~7GB) + Transformer (19.5GB) + Gemma4 → /tmp/
#   - /kaggle/working permanece libre para el pipeline
# ═══════════════════════════════════════════════════════════════
import os, sys, time, json, shutil

for mod in list(sys.modules.keys()):
    if mod.startswith('huggingface_hub'):
        del sys.modules[mod]

from huggingface_hub import hf_hub_download

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# ── Rutas: TODO en /tmp (scratch ilimitado, NO /kaggle/working) ──
TMP_MODELS    = '/tmp/models'          # único directorio real de modelos
WAN2GP_DIR    = '/kaggle/working/Wan2GP'
MODEL_DIR     = os.path.join(WAN2GP_DIR, 'models')   # SOLO symlinks
CKPTS_DIR     = os.path.join(WAN2GP_DIR, 'ckpts')    # SOLO symlinks
OUTPUTS_DIR   = os.path.join(WAN2GP_DIR, 'outputs')

for d in [TMP_MODELS, MODEL_DIR, CKPTS_DIR, OUTPUTS_DIR,
          '/kaggle/working/outputs', '/tmp/ltx25_outputs']:
    os.makedirs(d, exist_ok=True)

LTX25_REPO    = 'DeepBeepMeep/LTX-2'
GEMMA4_SUB    = 'gemma4-12b-ltx-v1'
GEMMA4_TMP    = os.path.join(TMP_MODELS, GEMMA4_SUB)
os.makedirs(GEMMA4_TMP, exist_ok=True)

def safe_symlink(src, dest):
    """Crea symlink solo si el destino no existe."""
    abs_src = os.path.abspath(src)
    if not os.path.exists(abs_src):
        return
    if os.path.lexists(dest):
        return
    try:
        os.symlink(abs_src, dest)
    except Exception as e:
        log(f"  [symlink] {os.path.basename(dest)}: {e}")

def hf_to_tmp(filename, subfolder=None):
    """
    Descarga a /tmp/models (siempre).
    Si subfolder, descarga a /tmp/models/<subfolder>/<filename>.
    Retorna la ruta real del archivo descargado.
    """
    if subfolder:
        dst_dir = os.path.join(TMP_MODELS, subfolder)
        dst_path = os.path.join(dst_dir, filename)
    else:
        dst_dir = TMP_MODELS
        dst_path = os.path.join(dst_dir, filename)

    os.makedirs(dst_dir, exist_ok=True)

    if os.path.exists(dst_path) and os.path.getsize(dst_path) > 1024 * 1024:
        return dst_path
    # v8: archivos pequeños (config/tokenizer): reutilizar solo si son válidos (no vacíos / JSON correcto)
    if os.path.exists(dst_path) and os.path.getsize(dst_path) > 0:
        try:
            if filename.endswith('.json'):
                json.load(open(dst_path, 'r', encoding='utf-8'))
            return dst_path
        except Exception:
            log(f"  [fix] {filename} inválido → se vuelve a descargar")
            os.remove(dst_path)
    elif os.path.lexists(dst_path):
        os.remove(dst_path)

    # Buscar en /kaggle/input primero
    for root, _, files in os.walk('/kaggle/input'):
        if filename in files:
            p = os.path.join(root, filename)
            if os.path.getsize(p) > 1024 * 1024:
                log(f"  [cache] {filename} ← /kaggle/input")
                return p

    log(f"  [HF↓] {filename} → /tmp/models/{subfolder or ''}")
    sys.stdout.flush()
    t0 = time.time()

    if subfolder:
        got = hf_hub_download(
            repo_id=LTX25_REPO, subfolder=subfolder,
            filename=filename, local_dir=dst_dir
        )
        # hf_hub_download con subfolder puede crear un nivel extra
        nested = os.path.join(dst_dir, subfolder, filename)
        if os.path.exists(nested) and not os.path.exists(dst_path):
            shutil.move(nested, dst_path)
        # v8: usar la ruta que DEVUELVE hf_hub_download (cambia entre versiones de huggingface_hub)
        elif (not os.path.exists(dst_path) and got and os.path.exists(got)
              and os.path.abspath(got) != os.path.abspath(dst_path)):
            shutil.copyfile(got, dst_path)
    else:
        hf_hub_download(repo_id=LTX25_REPO, filename=filename, local_dir=dst_dir)

    # Limpiar cache HF para no desperdiciar espacio en /tmp
    for cache_dir in [os.path.join(dst_dir, '.cache'), os.path.join(TMP_MODELS, '.cache')]:
        shutil.rmtree(cache_dir, ignore_errors=True)

    elapsed = time.time() - t0
    size_gb = os.path.getsize(dst_path) / 1024**3 if os.path.exists(dst_path) else 0
    log(f"    ✓ {filename} ({size_gb:.2f} GB, {elapsed:.0f}s)")
    return dst_path

def link_to_wan2gp(real_path, filename, subfolder=None):
    """
    Crea symlinks en Wan2GP/models/ y Wan2GP/ckpts/ apuntando al archivo real en /tmp/.
    NO copia archivos, solo symlinks.
    """
    if subfolder:
        m_dir = os.path.join(MODEL_DIR, subfolder)
        c_dir = os.path.join(CKPTS_DIR, subfolder)
        os.makedirs(m_dir, exist_ok=True)
        os.makedirs(c_dir, exist_ok=True)
        safe_symlink(real_path, os.path.join(m_dir, filename))
        safe_symlink(real_path, os.path.join(c_dir, filename))
    else:
        safe_symlink(real_path, os.path.join(MODEL_DIR, filename))
        safe_symlink(real_path, os.path.join(CKPTS_DIR, filename))

# ═══ 1. TRANSFORMER (19.5 GB) → /tmp/models/ ═══
TRANSFORMER_FILE = 'ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors'
log('Verificando Transformer LTX-2.5...')
tf_real = hf_to_tmp(TRANSFORMER_FILE)
link_to_wan2gp(tf_real, TRANSFORMER_FILE)

# ═══ 2. COMPANION FILES (~7 GB) → /tmp/models/ ═══
COMPANION_FILES = [
    'ltx-2.5-spatial-upscaler-x2-1.0_bf16.safetensors',
    'ltx-2.5-temporal-upscaler-x2-1.0_bf16.safetensors',
    'ltx-2.5-22b_video_embeddings_connector_int8_convrot.safetensors',
    'ltx-2.5-22b_audio_embeddings_connector_int8_convrot.safetensors',
    'ltx-2.5-22b_text_embedding_projection_bf16.safetensors',
    'ltx-2.5-22b_video_vae_bf16.safetensors',
    'ltx-2.5-22b_audio_vae_bf16.safetensors',
    'ltx-2.5-22b_vocoder_bf16.safetensors',
]
log('Verificando companion files...')
for f in COMPANION_FILES:
    real = hf_to_tmp(f)
    link_to_wan2gp(real, f)

# ═══ 3. GEMMA4 TEXT ENCODER → /tmp/models/gemma4-12b-ltx-v1/ ═══
GEMMA4_FILES = [
    'gemma4-12b-ltx-v1_int8_convrot.safetensors',
    'config.json', 'chat_template.jinja',
    'tokenizer.json', 'tokenizer_config.json',
]
log('Verificando Gemma4 text encoder...')
for gf in GEMMA4_FILES:
    real = hf_to_tmp(gf, subfolder=GEMMA4_SUB)
    link_to_wan2gp(real, gf, subfolder=GEMMA4_SUB)

# ═══ 4. CONFIGURAR DEFAULTS WAN2GP ═══
DEFAULTS_DIR = os.path.join(WAN2GP_DIR, 'defaults')
os.makedirs(DEFAULTS_DIR, exist_ok=True)
cfg = {
    "model": {"name": "LTX-2 2.5 Distilled 22B", "architecture": "ltx2_25_22B",
              "ltx2_pipeline": "distilled"},
    "guidance_phases": 1, "num_inference_steps": 8,
    "video_length": 73, "resolution": "640x384"
}
for cn in ['ltx2_25_22B.json', 'ltx2_25_22B_distilled.json', 'ltx2_25_22B_distilled_nvfp4.json']:
    try:
        json.dump(cfg, open(os.path.join(DEFAULTS_DIR, cn), 'w'), indent=4)
    except Exception:
        pass

# ═══ 5. VERIFICACIÓN FINAL ═══
log('\n=== Verificación Celda 12 ===')
critical = [
    (TMP_MODELS,  TRANSFORMER_FILE),
    (MODEL_DIR,   TRANSFORMER_FILE),       # symlink
    (CKPTS_DIR,   TRANSFORMER_FILE),       # symlink
    (TMP_MODELS,  'ltx-2.5-22b_video_vae_bf16.safetensors'),
    (GEMMA4_TMP,  'gemma4-12b-ltx-v1_int8_convrot.safetensors'),
    (GEMMA4_TMP,  'config.json'),
    (GEMMA4_TMP,  'tokenizer.json'),
]
all_ok = True
for dir_, fname in critical:
    path = os.path.join(dir_, fname)
    exists = os.path.exists(path)   # os.path.exists sigue symlinks
    size   = os.path.getsize(path) if exists else 0
    ok     = exists and size > 1000
    if not ok: all_ok = False
    status = '✅' if ok else '❌'
    rel    = os.path.relpath(path, '/kaggle/working')
    log(f'  {status} {rel} ({size/1024**2:.0f} MB)' if exists else f'  {status} MISSING: {rel}')

st_w = os.statvfs('/kaggle/working')
st_t = os.statvfs('/tmp')
log(f'\n📦 /kaggle/working libre: {(st_w.f_bavail*st_w.f_frsize)/1024**3:.1f} GB  ← NO debe bajar')
log(f'📦 /tmp libre:           {(st_t.f_bavail*st_t.f_frsize)/1024**3:.1f} GB')

# ── Alias para Celda 18 (compatibilidad) ──
LTX23_PRESETS = {
    "⚡ RÁPIDO — LTX-2.5 Distilled INT8 (~25 GB /tmp, 8 steps)": {
        "transformer_key": "INT8_CONVROT", "encoder_key": "GEMMA4_INT8",
        "desc": "LTX-2.5 22B Distilled. 8 pasos. Todo en /tmp, cero impacto en /working.",
    },
}
DEFAULT_LTX23_PRESET = list(LTX23_PRESETS.keys())[0]

def ensure_ltx23_models(preset_label=None):
    """Retorna dict de rutas. Todos los archivos en /tmp/models/."""
    paths = {
        'transformer':           os.path.join(TMP_MODELS, TRANSFORMER_FILE),
        'encoder':               os.path.join(GEMMA4_TMP, 'gemma4-12b-ltx-v1_int8_convrot.safetensors'),
        'video_vae':             os.path.join(TMP_MODELS, 'ltx-2.5-22b_video_vae_bf16.safetensors'),
        'audio_vae':             os.path.join(TMP_MODELS, 'ltx-2.5-22b_audio_vae_bf16.safetensors'),
        'embeddings_connector':  os.path.join(TMP_MODELS, 'ltx-2.5-22b_video_embeddings_connector_int8_convrot.safetensors'),
        'gemma4_dir':            GEMMA4_TMP,
        '_preset':               preset_label or DEFAULT_LTX23_PRESET,
        '_steps': 8, '_cfg': 1.0, '_vram_estimado': 6, '_tipo': 'distilled',
    }
    for k, v in paths.items():
        if k.startswith('_'): continue
        if not os.path.exists(v):
            log(f'[LTX25] ❌ Faltante: {k} → {v}')
            return None
    return paths

ensure_ltx25_models = ensure_ltx23_models

if all_ok:
    log('\n✅ CELDA 12 (v3 DISK-SAFE) COMPLETADA')
    log('   ► Todos los modelos en /tmp/models/ (cero impacto en /kaggle/working)')
    log('   ► Wan2GP/models/ y Wan2GP/ckpts/ = solo symlinks')
else:
    log('\n⚠ Algunos archivos faltan. Revisa la descarga.')

[18:05:47] Verificando Transformer LTX-2.5...
[18:05:47]   [HF↓] ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors → /tmp/models/


ltx-2.5-22b-distilled_diffusion_model_in(…): reconstructing file:   0%|          |  0.00B / 19.5GB            

ltx-2.5-22b-distilled_diffusion_model_in(…): downloading bytes:           |  0.00B            

[18:07:06]     ✓ ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors (18.15 GB, 79s)
[18:07:06] Verificando companion files...
[18:07:06]   [HF↓] ltx-2.5-spatial-upscaler-x2-1.0_bf16.safetensors → /tmp/models/


ltx-2.5-spatial-upscaler-x2-1.0_bf16.saf(…): reconstructing file:   0%|          |  0.00B /  996MB            

...

gemma4-12b-ltx-v1/tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

gemma4-12b-ltx-v1/tokenizer.json: downloading bytes:           |  0.00B            

[18:09:04]     ✓ tokenizer.json (0.03 GB, 1s)
[18:09:04]   [HF↓] tokenizer_config.json → /tmp/models/gemma4-12b-ltx-v1


tokenizer_config.json:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

[18:09:04]     ✓ tokenizer_config.json (0.00 GB, 0s)
[18:09:04] 
=== Verificación Celda 12 ===
[18:09:04]   ✅ ../../tmp/models/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors (18581 MB)
[18:09:04]   ✅ Wan2GP/models/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors (18581 MB)
[18:09:04]   ✅ Wan2GP/ckpts/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors (18581 MB)
[18:09:04]   ✅ ../../tmp/models/ltx-2.5-22b_video_vae_bf16.safetensors (1385 MB)
[18:09:04]   ✅ ../../tmp/models/gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors (12325 MB)
[18:09:04]   ✅ ../../tmp/models/gemma4-12b-ltx-v1/config.json (0 MB)
[18:09:04]   ✅ ../../tmp/models/gemma4-12b-ltx-v1/tokenizer.json (31 MB)
[18:09:04] 
📦 /kaggle/working libre: 17.7 GB  ← NO debe bajar
[18:09:04] 📦 /tmp libre:           1031.5 GB
[18:09:04] 
✅ CELDA 12 (v3 DISK-SAFE) COMPLETADA
[18:09:04]    ► Todos los modelos en /tmp/models/ (cero impacto en /kaggle/working)
[18:09:04]    ► Wan2GP/mode

In [11]:
# ═══════════════════════════════════════════════════════════════
# CELDA 12b (UNIFICADA DEFINITIVA): PARCHES WAN2GP + run_ltx25_generate.py
#
# CONSOLIDA EN UNA SOLA CELDA:
#   - Cell 12b original (parches disco + script v4)
#   - REESCRITURA COMPLETA (VERSIÓN DEFINITIVA)
#   - CELDA FIX I2V RAÍZ (video_encoder fp32, resize exacto, strength=1.0)
#
# REGLA: NO tocar nada de lo que funciona. Escritura única del script final.
# ═══════════════════════════════════════════════════════════════
import subprocess as _sp, sys, os
from pathlib import Path

WAN2GP_DIR   = '/kaggle/working/Wan2GP'
LTX25_SCRIPT = os.path.join(WAN2GP_DIR, 'run_ltx25_generate.py')
TMP_MODELS   = '/tmp/models'
GEMMA4_TMP   = os.path.join(TMP_MODELS, 'gemma4-12b-ltx-v1')

# ══════════════════════════════════════════════════════════════
# BLOQUE 1: PARCHES EN DISCO A ARCHIVOS WAN2GP
# Idénticos al proyecto de referencia AIQUEST + fixes adicionales validados.
# Todos son idempotentes (verifican antes de aplicar).
# ══════════════════════════════════════════════════════════════
print("Aplicando parches a Wan2GP en disco...")
patches_applied = 0

def patch_file(fpath, old, new, label):
    global patches_applied
    if not os.path.isfile(fpath): return
    code = open(fpath, encoding='utf-8').read()
    if new in code or old not in code: return
    open(fpath, 'w', encoding='utf-8').write(code.replace(old, new))
    patches_applied += 1
    print(f"  ✅ {label}")

for base in [WAN2GP_DIR, '.']:
    # P1: FP16 en lugar de BFloat16 (T4 Tensor Cores ~24s/paso)
    patch_file(f"{base}/models/ltx2/ltx_pipelines/distilled.py",
        "self.dtype = torch.bfloat16",
        "self.dtype = getattr(models, 'dtype', torch.float16)",
        'distilled.py → FP16 (a)')
    patch_file(f"{base}/models/ltx2/ltx_pipelines/distilled.py",
        "dtype = torch.bfloat16",
        "dtype = getattr(self, 'dtype', torch.float16)",
        'distilled.py → FP16 (b)')

    # P2: PixelNorm en FP32 (evita overflow FP16 → fondo gris/negro)
    patch_file(f"{base}/models/ltx2/ltx_core/model/common/normalization.py",
        "        mean_sq = torch.mean(x**2, dim=self.dim, keepdim=True)\n"
        "        # Normalize by the root-mean-square (RMS).\n"
        "        rms = torch.sqrt(mean_sq + self.eps)\n"
        "        return x / rms",
        "        orig_dtype = x.dtype\n"
        "        x_float = x.float()\n"
        "        mean_sq = torch.mean(x_float**2, dim=self.dim, keepdim=True)\n"
        "        rms = torch.sqrt(mean_sq + self.eps)\n"
        "        return (x_float / rms).to(orig_dtype)",
        'normalization.py → PixelNorm FP32')

    # P3: feature_extractor RMS en FP32
    patch_file(f"{base}/models/ltx2/ltx_core/text_encoders/gemma/feature_extractor.py",
        "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, "
        "attention_mask: torch.Tensor) -> torch.Tensor:\n"
        "    b, t, d, l = encoded_text.shape\n"
        "    variance = torch.mean(encoded_text**2, dim=2, keepdim=True)\n"
        "    normed = encoded_text * torch.rsqrt(variance + 1e-6)\n"
        "    normed = normed.reshape(b, t, d * l)\n"
        "    mask_3d = attention_mask.bool().unsqueeze(-1)\n"
        "    return torch.where(mask_3d, normed, torch.zeros_like(normed))",
        "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, "
        "attention_mask: torch.Tensor) -> torch.Tensor:\n"
        "    orig_dtype = encoded_text.dtype\n"
        "    enc_f = encoded_text.float()\n"
        "    b, t, d, l = enc_f.shape\n"
        "    variance = torch.mean(enc_f**2, dim=2, keepdim=True)\n"
        "    normed = enc_f * torch.rsqrt(variance + 1e-6)\n"
        "    normed = normed.reshape(b, t, d * l).to(orig_dtype)\n"
        "    mask_3d = attention_mask.bool().unsqueeze(-1)\n"
        "    return torch.where(mask_3d, normed, torch.zeros_like(normed))",
        'feature_extractor.py → RMS FP32')

    # P4: CausalConv3d auto-cast (dtype mismatch VAE)
    patch_file(f"{base}/models/ltx2/ltx_core/model/video_vae/convolution.py",
        "        x = self.conv(x)",
        "        if hasattr(self, 'conv') and hasattr(self.conv, 'weight') "
        "and x.dtype != self.conv.weight.dtype:\n"
        "            x = x.to(dtype=self.conv.weight.dtype)\n"
        "        x = self.conv(x)",
        'convolution.py → CausalConv3d auto-cast')

    # P5: audio_vae latent auto-cast
    patch_file(f"{base}/models/ltx2/ltx_core/model/audio_vae/audio_vae.py",
        "    decoded_audio = audio_decoder(latent)",
        "    if hasattr(audio_decoder, 'conv_in') and hasattr(audio_decoder.conv_in, 'weight'):\n"
        "        latent = latent.to(dtype=audio_decoder.conv_in.weight.dtype)\n"
        "    decoded_audio = audio_decoder(latent)\n"
        "    if hasattr(vocoder, 'vocoder') and hasattr(vocoder.vocoder, 'conv_pre'):\n"
        "        decoded_audio = decoded_audio.to(dtype=vocoder.vocoder.conv_pre.weight.dtype)",
        'audio_vae.py → latent auto-cast')

    # P6: vocoder buffers auto-cast (LowPassFilter1d)
    patch_file(f"{base}/models/ltx2/ltx_core/model/audio_vae/vocoder.py",
        "return F.conv1d(x, self.filter.expand(n_channels, -1, -1), "
        "stride=self.stride, groups=n_channels)",
        "filt = self.filter.to(dtype=x.dtype, device=x.device).expand(n_channels, -1, -1)\n"
        "        return F.conv1d(x, filt, stride=self.stride, groups=n_channels)",
        'vocoder.py → LowPassFilter buffer')

    # P7: vocoder STFT buffer
    patch_file(f"{base}/models/ltx2/ltx_core/model/audio_vae/vocoder.py",
        "spec = F.conv1d(y, self.forward_basis, stride=self.hop_length, padding=0)",
        "basis = self.forward_basis.to(dtype=y.dtype, device=y.device)\n"
        "        spec = F.conv1d(y, basis, stride=self.hop_length, padding=0)",
        'vocoder.py → STFT buffer')

    # P8: ltx2.py requires_grad_(False) para carga limpia de pesos INT8
    patch_file(f"{base}/models/ltx2/ltx2.py",
        'def _load_component(model, path, sd_ops=None, postprocess=None, '
        'ignore_unused_weights=False):\n            if postprocess is None and sd_ops is not None:',
        'def _load_component(model, path, sd_ops=None, postprocess=None, '
        'ignore_unused_weights=False):\n            model.requires_grad_(False)\n'
        '            if postprocess is None and sd_ops is not None:',
        'ltx2.py → requires_grad_(False)')

print(f"  → {patches_applied} parches en disco aplicados.")

# ══════════════════════════════════════════════════════════════
# BLOQUE 2: ESCRIBIR run_ltx25_generate.py — VERSIÓN DEFINITIVA FINAL
#
# Esta es la versión que fue VALIDADA funcionando:
#   ✅ video_encoder → float32 (fix frames negros I2V)
#   ✅ PNG frames → ffmpeg (guardado robusto, reemplaza save_video)
#   ✅ _patch_video_encoder_float32 hook (seguridad adicional)
#   ✅ image_start resize EXACTO a width×height
#   ✅ input_video_strength = 1.0 SIEMPRE (sin ruido sobre latente)
#   ✅ Progreso correcto (sin 5000%, anti-loop)
#   ✅ nan_to_num antes de guardar
#   ✅ convertWeightsFloatTo=torch.bfloat16 (validado en perfil 4)
#   ✅ SDPA: flash=False, mem_efficient=True (T4/Turing)
# ══════════════════════════════════════════════════════════════
print("\nEscribiendo run_ltx25_generate.py (VERSIÓN DEFINITIVA FINAL)...")

GENERATE_SCRIPT = r'''#!/usr/bin/env python3
"""
run_ltx25_generate.py — VERSIÓN DEFINITIVA FINAL
Consolidación de todos los fixes validados:
  - video_encoder float32 (fix I2V frames negros)
  - PNG frames → ffmpeg pipe (guardado robusto)
  - _patch_video_encoder_float32 hook (seguridad extra)
  - image_start resize EXACTO a width×height
  - input_video_strength = 1.0 SIEMPRE
  - Progreso correcto (sin 5000%)
  - nan_to_num antes de guardar
"""
import sys, os, json, gc, time, random, traceback, tempfile, shutil
import torch
import numpy as np
import io as _io

# stdout sin buffer (crítico para que Popen lea línea a línea en tiempo real)
sys.stdout = _io.TextIOWrapper(sys.stdout.buffer, line_buffering=True)

def _p(msg):
    print(msg, flush=True)

# ── Bootstrap ──────────────────────────────────────────────────
WAN2GP_DIR = os.environ.get("WAN2GP_DIR", "/kaggle/working/Wan2GP")
TMP_MODELS = "/tmp/models"
GEMMA4_TMP = os.path.join(TMP_MODELS, "gemma4-12b-ltx-v1")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

# ── Parches en disco ANTES de importar (seguridad en caso de reinicio) ─
def _patch(fpath, old, new):
    if not os.path.isfile(fpath): return False
    c = open(fpath, encoding="utf-8").read()
    if new in c or old not in c: return False
    open(fpath, "w", encoding="utf-8").write(c.replace(old, new))
    return True

_r = _patch(
    f"{WAN2GP_DIR}/models/ltx2/ltx_core/model/common/normalization.py",
    "        mean_sq = torch.mean(x**2, dim=self.dim, keepdim=True)\n"
    "        # Normalize by the root-mean-square (RMS).\n"
    "        rms = torch.sqrt(mean_sq + self.eps)\n"
    "        return x / rms",
    "        orig_dtype = x.dtype\n"
    "        x_float = x.float()\n"
    "        mean_sq = torch.mean(x_float**2, dim=self.dim, keepdim=True)\n"
    "        rms = torch.sqrt(mean_sq + self.eps)\n"
    "        return (x_float / rms).to(orig_dtype)"
)
_p(f"[PATCH] PixelNorm FP32: {'aplicado' if _r else 'ya activo'}")

_patch(f"{WAN2GP_DIR}/models/ltx2/ltx_pipelines/distilled.py",
       "self.dtype = torch.bfloat16",
       "self.dtype = getattr(models, 'dtype', torch.float16)")

_patch(f"{WAN2GP_DIR}/models/ltx2/ltx_core/model/video_vae/convolution.py",
       "        x = self.conv(x)",
       "        if hasattr(self,'conv') and hasattr(self.conv,'weight') "
       "and x.dtype != self.conv.weight.dtype:\n"
       "            x = x.to(dtype=self.conv.weight.dtype)\n"
       "        x = self.conv(x)")

_patch(f"{WAN2GP_DIR}/models/ltx2/ltx_core/text_encoders/gemma/feature_extractor.py",
       "    b, t, d, l = encoded_text.shape\n"
       "    variance = torch.mean(encoded_text**2, dim=2, keepdim=True)\n"
       "    normed = encoded_text * torch.rsqrt(variance + 1e-6)\n"
       "    normed = normed.reshape(b, t, d * l)\n"
       "    mask_3d = attention_mask.bool().unsqueeze(-1)\n"
       "    return torch.where(mask_3d, normed, torch.zeros_like(normed))",
       "    orig_dtype = encoded_text.dtype\n"
       "    enc_f = encoded_text.float()\n"
       "    b, t, d, l = enc_f.shape\n"
       "    variance = torch.mean(enc_f**2, dim=2, keepdim=True)\n"
       "    normed = enc_f * torch.rsqrt(variance + 1e-6)\n"
       "    normed = normed.reshape(b, t, d * l).to(orig_dtype)\n"
       "    mask_3d = attention_mask.bool().unsqueeze(-1)\n"
       "    return torch.where(mask_3d, normed, torch.zeros_like(normed))")

_patch(f"{WAN2GP_DIR}/models/ltx2/ltx_core/model/audio_vae/audio_vae.py",
       "    decoded_audio = audio_decoder(latent)",
       "    if hasattr(audio_decoder,'conv_in') and hasattr(audio_decoder.conv_in,'weight'):\n"
       "        latent = latent.to(dtype=audio_decoder.conv_in.weight.dtype)\n"
       "    decoded_audio = audio_decoder(latent)\n"
       "    if hasattr(vocoder,'vocoder') and hasattr(vocoder.vocoder,'conv_pre'):\n"
       "        decoded_audio = decoded_audio.to(dtype=vocoder.vocoder.conv_pre.weight.dtype)")

_patch(f"{WAN2GP_DIR}/models/ltx2/ltx_core/model/audio_vae/vocoder.py",
       "return F.conv1d(x, self.filter.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)",
       "filt = self.filter.to(dtype=x.dtype, device=x.device).expand(n_channels, -1, -1)\n"
       "        return F.conv1d(x, filt, stride=self.stride, groups=n_channels)")
_patch(f"{WAN2GP_DIR}/models/ltx2/ltx_core/model/audio_vae/vocoder.py",
       "spec = F.conv1d(y, self.forward_basis, stride=self.hop_length, padding=0)",
       "basis = self.forward_basis.to(dtype=y.dtype, device=y.device)\n"
       "        spec = F.conv1d(y, basis, stride=self.hop_length, padding=0)")

# ── HfFolder backward-compat ───────────────────────────────────
try:
    import huggingface_hub.utils as _hfu
    if not hasattr(_hfu, "HfFolder"):
        class HfFolder:
            @staticmethod
            def get_token():
                try:
                    from huggingface_hub import get_token
                    return get_token() or os.environ.get("HF_TOKEN")
                except: return os.environ.get("HF_TOKEN")
            @staticmethod
            def save_token(t): pass
        setattr(_hfu, "HfFolder", HfFolder)
        if "huggingface_hub.utils" in sys.modules:
            setattr(sys.modules["huggingface_hub.utils"], "HfFolder", HfFolder)
except: pass

# ── PyTorch optimizaciones ─────────────────────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.backends.cuda.enable_flash_sdp(False)       # T4/Turing: usar mem_efficient
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# ── Imports Wan2GP ─────────────────────────────────────────────
from mmgp import offload, quant_router
from models.ltx2.ltx2_handler import family_handler

# Registrar todos los handlers de cuantización
for h in [
    "shared.qtypes.scaled_fp8", "shared.qtypes.nvfp4",
    "shared.qtypes.bnb_nf4",   "shared.qtypes.nunchaku_int4",
    "shared.qtypes.nunchaku_fp4", "shared.qtypes.asym_w4a8_int8",
    "shared.qtypes.int8_convrot", "shared.qtypes.gguf",
]:
    try: quant_router.register_handler(h)
    except: pass
quant_router.unregister_handler(".fp8_quanto_bridge")
try:
    from shared.qtypes import gguf as _gguf
    quant_router.register_file_extension("gguf", _gguf)
except: pass

from PIL import Image
import subprocess as _sp

# Limpiar lock files de ejecuciones anteriores
for lf in ["startup.lock", "wgp.lock"]:
    lp = os.path.join(WAN2GP_DIR, lf)
    if os.path.exists(lp):
        try: os.remove(lp)
        except: pass

# ── Resolver rutas de modelos ──────────────────────────────────
TF = "ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"
GF = "gemma4-12b-ltx-v1_int8_convrot.safetensors"

def _find(*candidates):
    for c in candidates:
        if os.path.exists(c) and os.path.getsize(c) > 1024*1024: return c
    return ""

transformer_path = _find(
    os.path.join(TMP_MODELS, TF),
    os.path.join(WAN2GP_DIR, "models", TF),
    os.path.join(WAN2GP_DIR, "ckpts",  TF))
text_encoder_file = _find(
    os.path.join(GEMMA4_TMP, GF),
    os.path.join(WAN2GP_DIR, "models", "gemma4-12b-ltx-v1", GF),
    os.path.join(WAN2GP_DIR, "ckpts",  "gemma4-12b-ltx-v1", GF))

if not transformer_path:  _p("ERROR: Transformer no encontrado."); sys.exit(1)
if not text_encoder_file: _p("ERROR: Text encoder no encontrado."); sys.exit(1)

_p(f"[SUBPROCESS] Transformer : {os.path.basename(transformer_path)}")
_p(f"[SUBPROCESS] Text Encoder: {os.path.basename(text_encoder_file)}")
_p("[SUBPROCESS] Cargando modelo LTX-2.5 22B Distilled...")

# ── Cargar modelo ──────────────────────────────────────────────
base_model_type = "ltx2_25_22B"
model_def = {
    "ltx2_pipeline": "distilled",
    "name":          "LTX-2 2.5 Distilled 22B",
    "architecture":  "ltx2_25_22B",
}
try:
    model_def.update(family_handler.query_model_def(base_model_type, model_def))
except: pass

with torch.inference_mode():
    with torch.set_grad_enabled(False):
        wan_model, pipe = family_handler.load_model(
            model_filename        = transformer_path,
            model_type            = "ltx2_25_22B_distilled",
            base_model_type       = base_model_type,
            model_def             = model_def,
            dtype                 = torch.float16,
            VAE_dtype             = torch.float16,
            text_encoder_filename = text_encoder_file,
        )

kwargs = {}
if isinstance(pipe, dict) and "pipe" in pipe:
    kwargs = pipe; pipe = kwargs.pop("pipe")
loras = kwargs.pop("loras", [])
if "transformer" in pipe and "transformer" not in loras:
    loras.append("transformer")

# ── FIX CRÍTICO: video_encoder → float32 ANTES del offload ────
# Sin este fix: overflow FP16 → NaN → frames completamente negros en I2V
def _patch_video_encoder_float32(pipe_obj):
    """Fuerza video_encoder a float32 para evitar overflow en I2V."""
    # Caso 1: pipe es dict
    if isinstance(pipe_obj, dict):
        ve = pipe_obj.get("video_encoder")
        if ve is not None:
            try:
                ve.to(torch.float32)
                _p("[SUBPROCESS] ✅ video_encoder (dict) → float32 (fix I2V negro)")
                # Parchear el método encode si existe
                if hasattr(ve, 'encode'):
                    _orig = ve.encode
                    def _enc_fp32(*a, **kw):
                        with torch.autocast('cuda', dtype=torch.float32):
                            return _orig(*a, **kw)
                    ve.encode = _enc_fp32
                    _p("[SUBPROCESS] ✅ video_encoder.encode → autocast float32")
                return
            except Exception as e:
                _p(f"[SUBPROCESS] ⚠ video_encoder dict float32: {e}")
    # Caso 2: pipe es objeto, buscar atributos
    for attr in ["video_encoder", "vae", "encoder"]:
        ve = getattr(pipe_obj, attr, None) if not isinstance(pipe_obj, dict) else pipe_obj.get(attr)
        if ve is not None:
            try:
                ve.to(torch.float32)
                _p(f"[SUBPROCESS] ✅ {attr} → float32")
                return
            except: pass
    _p("[SUBPROCESS] ⚠ video_encoder no encontrado en pipe (se continuará sin fix)")

_patch_video_encoder_float32(pipe)

_p("[SUBPROCESS] Aplicando offload profile 4 (Partial Pinning, Async DMA)...")
offload.profile(
    pipe,
    profile_no            = 4,
    pinnedMemory          = "transformer",
    partialPinning        = True,
    perc_reserved_mem_max = 0.35,
    asyncTransfers        = True,
    quantizeTransformer   = False,
    convertWeightsFloatTo = torch.bfloat16,   # validado: bfloat16 para perfil 4
    loras                 = loras,
    budgets = {
        "transformer":       6000,
        "text_encoder":      1500,
        "video_encoder":     2000,
        "video_decoder":     3000,
        "audio_encoder":     1000,
        "audio_decoder":     1000,
        "vocoder":           500,
        "spatial_upsampler": 1500,
        "vae":               1000,
        "":                  1000,
    },
    **kwargs
)
offload.shared_state["_attention"] = "sdpa"
offload.shared_state["_radial"]    = False
_p("[SUBPROCESS] ✅ Modelo listo para generar.")

# ══════════════════════════════════════════════════════════════
# GUARDADO ROBUSTO: PNG frames → ffmpeg
# Reemplaza save_video() que fallaba silenciosamente (6 KB).
# Método validado: T frames PNG a disco → ffmpeg libx264.
# ══════════════════════════════════════════════════════════════
def save_robust(tensor, out_path, fps=24):
    """Guarda tensor de video de forma robusta vía PNG frames + ffmpeg."""
    t = tensor.detach().cpu().float()
    _p(f"[SUBPROCESS] Tensor recibido: shape={list(t.shape)}, dtype={tensor.dtype}, "
       f"min={t.min():.3f}, max={t.max():.3f}, "
       f"nan={torch.isnan(t).any().item()}, inf={torch.isinf(t).any().item()}")

    # Limpiar NaN/Inf antes de procesar
    if torch.isnan(t).any() or torch.isinf(t).any():
        _p("[SUBPROCESS] ⚠ NaN/Inf detectado — aplicando nan_to_num")
        t = torch.nan_to_num(t, nan=0.0, posinf=255.0, neginf=0.0)

    # Quitar batch dim si existe
    if t.ndim == 5: t = t[0]

    # Normalizar a (T, H, W, C)
    if t.ndim == 4:
        s = t.shape
        # (C, T, H, W) → (T, H, W, C) si C<=4 y T>4
        if s[0] <= 4 and s[1] > 4 and s[2] > 4 and s[3] > 4:
            t = t.permute(1, 2, 3, 0)
    if t.ndim == 3: t = t.unsqueeze(0)

    T, H, W, C = t.shape
    if C == 1:   t = t.repeat(1, 1, 1, 3); C = 3
    elif C == 4: t = t[..., :3];            C = 3
    elif C != 3: raise RuntimeError(f"Canales inesperados: C={C}, shape={t.shape}")

    # Convertir a uint8 [0, 255]
    vmin, vmax = t.min().item(), t.max().item()
    if vmax <= 1.0 + 1e-4:
        # Rango [0,1] o [-1,1]
        t = ((t + 1) / 2 * 255 if vmin < -0.5 else t * 255).clamp(0, 255)
    else:
        t = t.clamp(0, 255)
    frames = t.numpy().astype(np.uint8)

    _p(f"[SUBPROCESS] Frames numpy: T={T} H={H} W={W} C={C} | "
       f"uint8 min={frames.min()} max={frames.max()}")

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

    # ── Guardar PNGs a disco → ffmpeg ─────────────────────────
    frames_dir = tempfile.mkdtemp(prefix="ltx_fr_")
    try:
        _p(f"[SUBPROCESS] Guardando {T} frames PNG...")
        from PIL import Image as _PIL
        for i, fr in enumerate(frames):
            _PIL.fromarray(fr).save(os.path.join(frames_dir, f"f{i:06d}.png"))

        _p(f"[SUBPROCESS] Codificando con ffmpeg → {out_path}")
        r = _sp.run([
            "ffmpeg", "-y",
            "-framerate", str(int(fps)),
            "-i", os.path.join(frames_dir, "f%06d.png"),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-crf", "18",
            "-preset", "fast",
            "-movflags", "+faststart",
            out_path
        ], capture_output=True, timeout=300)

        if r.returncode == 0 and os.path.exists(out_path):
            sz = os.path.getsize(out_path)
            _p(f"[SUBPROCESS] ✅ ffmpeg PNG→MP4 OK: {sz // 1024} KB")
            return out_path, sz
        _p(f"[SUBPROCESS] ⚠ ffmpeg rc={r.returncode}: "
           f"{r.stderr[-300:].decode(errors='ignore')}")
    except Exception as e:
        _p(f"[SUBPROCESS] ⚠ PNG frames falló: {e}")
    finally:
        shutil.rmtree(frames_dir, ignore_errors=True)

    # ── Fallback: ffmpeg rawvideo pipe ─────────────────────────
    try:
        _p(f"[SUBPROCESS] Fallback: ffmpeg rawvideo pipe...")
        proc = _sp.Popen([
            "ffmpeg", "-y",
            "-f", "rawvideo", "-vcodec", "rawvideo",
            "-s", f"{W}x{H}", "-pix_fmt", "rgb24",
            "-r", str(int(fps)), "-i", "pipe:0",
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-crf", "18", "-preset", "fast",
            out_path
        ], stdin=_sp.PIPE, stdout=_sp.PIPE, stderr=_sp.PIPE)
        for i in range(T):
            proc.stdin.write(frames[i].tobytes())
        proc.stdin.close()
        _, stderr = proc.communicate(timeout=120)
        if proc.returncode == 0 and os.path.exists(out_path):
            sz = os.path.getsize(out_path)
            if sz > 10000:
                _p(f"[SUBPROCESS] ✅ ffmpeg pipe OK: {sz // 1024} KB")
                return out_path, sz
        _p(f"[SUBPROCESS] ⚠ pipe rc={proc.returncode}: "
           f"{stderr[-200:].decode(errors='ignore')}")
    except Exception as e:
        _p(f"[SUBPROCESS] ⚠ ffmpeg pipe falló: {e}")

    # ── Último recurso: devolver lo que haya si es > 1 KB ─────
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        sz = os.path.getsize(out_path)
        _p(f"[SUBPROCESS] ⚠ Usando archivo parcial existente: {sz} bytes")
        return out_path, sz

    raise RuntimeError(f"Todos los métodos de guardado fallaron para {out_path}")


# ── Callback de progreso ───────────────────────────────────────
_pstate = {
    "phase": "", "total": 8, "done": 0,
    "t0": time.monotonic(), "lt": time.monotonic()
}

def progress_cb(step, latent=None, is_start=False,
                override_num_inference_steps=None,
                pass_no=None, **kw):
    idx   = int(step)
    phase = str(kw.get("progress_title") or "Denoising")
    if pass_no is not None: phase += f"/p{pass_no}"
    phase = phase.replace(":", "")

    supplied = None
    if override_num_inference_steps is not None and not isinstance(override_num_inference_steps, bool):
        supplied = max(1, int(override_num_inference_steps))

    now = time.monotonic()
    if is_start or _pstate["phase"] != phase:
        _pstate.update({
            "phase": phase, "total": supplied or 8,
            "done": 0, "t0": now, "lt": now
        })
        print(f"PROGRESS:stage:{phase}", flush=True)
        print(f"PROGRESS:start:{_pstate['total']}", flush=True)
        return

    if idx < 0: return   # consulta de interrupción interna de mmgp

    if supplied: _pstate["total"] = supplied
    total   = _pstate["total"]
    current = min(total, max(1, idx + 1))
    if current <= _pstate["done"]: return

    dt  = max(0.0, now - _pstate["lt"]) / max(1, current - _pstate["done"])
    avg = max(0.0, now - _pstate["t0"]) / current
    rem = max(0, total - current) * avg
    free = torch.cuda.mem_get_info()[0] / 1024**3 if torch.cuda.is_available() else 0
    pct  = min(100, int(current * 100 / total))

    _pstate["done"] = current
    _pstate["lt"]   = now
    print(f"PROGRESS:step:{current}:{total}:{dt:.1f}:{rem:.0f}:{free:.1f}:{phase}:{pct}",
          flush=True)

def set_stage(s):
    print(f"PROGRESS:stage:{s}", flush=True)
    if "Decod" in s or "Upsamp" in s:
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

def _audio_inputs_for_ltx(params, frame_num, fps):
    import wave

    audio_path = params.get("audio_path", "")
    if not audio_path:
        # Los clips de 5s siguen usando exactamente la llamada anterior.
        return {}

    if not os.path.isfile(audio_path):
        raise FileNotFoundError(f"Audio no encontrado: {audio_path}")

    with wave.open(audio_path, "rb") as wav:
        if wav.getsampwidth() != 2:
            raise RuntimeError(
                "La entrada A2V debe ser WAV PCM de 16 bits. "
                "Normalízala con ffmpeg_to_wav_mono_24k()."
            )

        channels = wav.getnchannels()
        sample_rate = wav.getframerate()
        raw_audio = wav.readframes(wav.getnframes())

    waveform = (
        np.frombuffer(raw_audio, dtype="<i2")
        .reshape(-1, channels)
        .astype(np.float32)
        / 32768.0
    )

    if waveform.shape[0] == 0:
        raise RuntimeError("El WAV está vacío.")

    # Cubrir los frames extra de la cuadrícula temporal de LTX.
    # No modifica el archivo original que después se adjunta al vídeo.
    needed_samples = int(
        np.ceil(float(frame_num) / float(fps) * sample_rate)
    )

    if waveform.shape[0] < needed_samples:
        waveform = np.pad(
            waveform,
            ((0, needed_samples - waveform.shape[0]), (0, 0)),
            mode="constant",
        )

    waveform = np.ascontiguousarray(waveform, dtype=np.float32)
    peak = float(np.abs(waveform).max())

    _p(
        f"[SUBPROCESS] A2V: audio={os.path.basename(audio_path)} | "
        f"sr={sample_rate} | shape={waveform.shape} | "
        f"peak={peak:.4f} | audio_scale=1.0"
    )

    if peak < 1e-5:
        _p(
            "[SUBPROCESS] AVISO: audio prácticamente silencioso; "
            "no sirve para validar sincronización de habla."
        )

    return {
        "input_waveform": waveform,
        "input_waveform_sample_rate": int(sample_rate),
        "audio_prompt_type": "A",
        "audio_scale": 1.0,
    }



# ── Función principal de generación ───────────────────────────
def generate(params):
    prompt  = params.get("prompt", "")
    imgpath = params.get("image_path", "")
    outpath = params.get("output_path", "")
    W       = int(params.get("width",     768))
    H       = int(params.get("height",    432))
    NF      = int(params.get("num_frames", 121))
    fps     = float(params.get("fps",      24))
    seed    = int(params.get("seed",       -1))
    steps   = int(params.get("steps",       8))
    cfg     = float(params.get("cfg",      1.0))
    acfg    = float(params.get("audio_cfg", 1.0))

    if seed < 0: seed = random.randint(0, 2**32 - 1)
    if not outpath:
        os.makedirs("/tmp/ltx25_outputs", exist_ok=True)
        outpath = f"/tmp/ltx25_outputs/ltx25_{int(time.time())}_{seed}.mp4"
    os.makedirs(os.path.dirname(outpath) or ".", exist_ok=True)

    # Resetear estado de progreso
    _pstate.update({
        "phase": "", "done": 0,
        "t0": time.monotonic(), "lt": time.monotonic()
    })

    # ── Pre-procesar imagen I2V ────────────────────────────────
    image_start = None
    if imgpath and os.path.exists(imgpath):
        img = Image.open(imgpath).convert("RGB")
        # Resize EXACTO a W×H (un pixel de diferencia causa mismatch en VAE → frames negros)
        image_start = img.resize((W, H), Image.LANCZOS)
        _p(f"[SUBPROCESS] image_start: {image_start.size} (exacto {W}×{H})")

    mode = "I2V" if image_start else "T2V"
    _p(f"[SUBPROCESS] {mode} | {W}x{H} | {NF}f@{fps}fps | steps={steps} | cfg={cfg} | seed={seed}")

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    t0 = time.time()
    result = wan_model.generate(
        input_prompt         = prompt,
        image_start          = image_start,
        height               = H,
        width                = W,
        frame_num            = NF,
        fps                  = fps,
        seed                 = seed,
        callback             = progress_cb,
        set_progress_status  = set_stage,
        VAE_tile_size        = 0,          # sin seams de tiling
        input_video_strength = 1.0,        # SIEMPRE 1.0: sin ruido sobre latente de imagen
        denoising_strength   = 1.0,
        guide_scale          = cfg,
        audio_cfg_scale      = acfg,
        sampling_steps       = steps,
        guide_phases         = 1,
        sample_solver        = "distilled_8_steps",
        self_refiner_setting = 0,
        n_prompt             = "",
        **_audio_inputs_for_ltx(params, NF, fps),  # NUEVO
    )
    elapsed = time.time() - t0
    _p(f"[SUBPROCESS] Pipeline completado en {elapsed:.1f}s")

    # ── Extraer tensor de video ────────────────────────────────
    vt = None
    if isinstance(result, dict):
        # Elegir tensor con mayor rango dinámico (el más "real")
        best = -1.0
        for k, v in result.items():
            if not torch.is_tensor(v): continue
            rng = (v.float().max() - v.float().min()).item()
            _p(f"[SUBPROCESS] result['{k}']: shape={list(v.shape)} "
               f"dtype={v.dtype} range={rng:.3f}")
            if rng > best: best = rng; vt = v
    elif isinstance(result, (list, tuple)):
        vt = next((x for x in result if torch.is_tensor(x)), None)
    elif torch.is_tensor(result):
        vt = result

    if vt is None:
        raise RuntimeError(f"Sin tensor de video. Tipo resultado: {type(result)}")

    _p("[SUBPROCESS] Guardando video...")
    op, sz = save_robust(vt, outpath, fps=fps)

    # Copia a almacenamiento persistente
    try:
        bn = os.path.basename(op)
        for d in ["/kaggle/working/outputs",
                  os.path.join(WAN2GP_DIR, "outputs")]:
            os.makedirs(d, exist_ok=True)
            shutil.copy2(op, os.path.join(d, bn))
    except Exception as e:
        _p(f"[SUBPROCESS] copia persistente: {e}")

    del vt
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    _p(f"[SUBPROCESS] ✅ Generado en {elapsed:.1f}s → {op} ({sz // 1024} KB)")
    return op


# ── Entry point ───────────────────────────────────────────────
if __name__ == "__main__":
    if len(sys.argv) < 2:
        _p("Uso: python run_ltx25_generate.py '<json_params>'")
        sys.exit(1)
    try:
        op = generate(json.loads(sys.argv[1]))
        print(f"OUTPUT_PATH:{op}", flush=True)
    except Exception as e:
        traceback.print_exc()
        sys.exit(1)
'''

SCRIPT = Path(LTX25_SCRIPT)
SCRIPT.write_text(GENERATE_SCRIPT, encoding="utf-8")
SCRIPT.chmod(0o755)

# ══════════════════════════════════════════════════════════════
# BLOQUE 3: VERIFICACIÓN FINAL
# ══════════════════════════════════════════════════════════════
content = SCRIPT.read_text(encoding="utf-8")

checks = {
    "PNG frames (guardado robusto)":        "f%06d.png"                          in content,
    "video_encoder float32 (hook)":         "_patch_video_encoder_float32"       in content,
    "video_encoder.to(float32)":            "ve.to(torch.float32)"               in content,
    "image_start resize exacto W×H":        "img.resize((W, H)"                  in content,
    "input_video_strength = 1.0 SIEMPRE":   "input_video_strength = 1.0"         in content,
    "PixelNorm FP32 (patch runtime)":       "x_float = x.float()"                in content,
    "Progreso anti-loop (idx<0)":           "if idx < 0: return"                 in content,
    "nan_to_num antes de guardar":          "nan_to_num"                          in content,
    "convertWeightsFloatTo=bfloat16":       "convertWeightsFloatTo = torch.bfloat16" in content,
    "SDPA mem_efficient (T4)":              "enable_mem_efficient_sdp(True)"      in content,
    "flash_sdp=False (T4/Turing)":          "enable_flash_sdp(False)"             in content,
    "stdout line_buffering":                "line_buffering=True"                 in content,
    "OUTPUT_PATH: en stdout":               "OUTPUT_PATH:"                        in content,
    "PROGRESS:step: en stdout":             "PROGRESS:step:"                      in content,
}

print(f"\n✅ Script escrito: {SCRIPT.stat().st_size:,} bytes\n")
print("── Verificación de checks críticos ──")
all_ok = True
for nombre, ok in checks.items():
    symbol = "✅" if ok else "❌"
    print(f"  {symbol} {nombre}")
    if not ok: all_ok = False

print(f"\n{'✅ CELDA 12b UNIFICADA LISTA — Sin cambios pendientes.' if all_ok else '❌ Revisar los checks fallidos.'}")
print("\n── Resumen ──────────────────────────────────────────────")
print(f"  Parches en disco Wan2GP: {patches_applied}")
print(f"  Script:  {LTX25_SCRIPT}")
print(f"  Tamaño:  {SCRIPT.stat().st_size // 1024} KB")
print("\nFlujo de ejecución correcto:")
print("  Celda 12  → descarga modelos a /tmp/models")
print("  Celda 12b → parches Wan2GP + escribe script definitivo  ← ESTA CELDA")
print("  Celda 13  → SubprocessLTXBackend (Popen + progreso)")
print("  Celda 14  → wrappers generar_clip_5s / lipsync")
print("  Celda 15  → Etapa 3 completa")
print("  Celda 16  → Etapas 2 y 4 (diálogos)")
print("  Celda 17  → Render final")
print("  Celda 18  → UI Gradio")

Aplicando parches a Wan2GP en disco...
  ✅ convolution.py → CausalConv3d auto-cast
  ✅ audio_vae.py → latent auto-cast
  → 2 parches en disco aplicados.

Escribiendo run_ltx25_generate.py (VERSIÓN DEFINITIVA FINAL)...

✅ Script escrito: 25,562 bytes

── Verificación de checks críticos ──
  ✅ PNG frames (guardado robusto)
  ✅ video_encoder float32 (hook)
  ✅ video_encoder.to(float32)
  ✅ image_start resize exacto W×H
  ✅ input_video_strength = 1.0 SIEMPRE
  ✅ PixelNorm FP32 (patch runtime)
  ✅ Progreso anti-loop (idx<0)
  ✅ nan_to_num antes de guardar
  ✅ convertWeightsFloatTo=bfloat16
  ✅ SDPA mem_efficient (T4)
  ✅ flash_sdp=False (T4/Turing)
  ✅ stdout line_buffering
  ✅ OUTPUT_PATH: en stdout
  ✅ PROGRESS:step: en stdout

✅ CELDA 12b UNIFICADA LISTA — Sin cambios pendientes.

── Resumen ──────────────────────────────────────────────
  Parches en disco Wan2GP: 2
  Script:  /kaggle/working/Wan2GP/run_ltx25_generate.py
  Tamaño:  24 KB

Flujo de ejecución correcto:
  Celda 12  → descar

In [12]:
# ═══════════════════════════════════════════════════════════════
# CELDA 13 (v4): MOTOR LTX-2.5 CON PROGRESO EN TIEMPO REAL
# CAMBIOS vs v3:
#   - Usa subprocess.Popen en vez de run() → lee stdout línea a línea
#   - Parsea líneas PROGRESS: del script → actualiza archivo de progreso
#   - Expone get_ltx_progress() para que la UI la lea vía timer
#   - Mantiene 100% de compatibilidad con Celdas 14-18
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, os, json, random, time, threading

WAN2GP_DIR   = "/kaggle/working/Wan2GP"
LTX25_SCRIPT = os.path.join(WAN2GP_DIR, "run_ltx25_generate.py")
TMP_MODELS   = "/tmp/models"
GEMMA4_TMP   = os.path.join(TMP_MODELS, "gemma4-12b-ltx-v1")

# ── Archivo de progreso (compartido entre subprocess y UI timer) ──
LTX_PROGRESS_FILE = "/tmp/ltx25_progress.json"

def _write_progress(data: dict):
    try:
        import json as _j
        with open(LTX_PROGRESS_FILE, "w") as f:
            _j.dump(data, f)
    except: pass

def get_ltx_progress() -> dict:
    """Llamada por el timer de Gradio para actualizar la UI."""
    try:
        import json as _j
        if os.path.exists(LTX_PROGRESS_FILE):
            return _j.load(open(LTX_PROGRESS_FILE))
    except: pass
    return {}

def calc_num_frames(target_s, fps=24):
    raw = max(9, int(round(target_s * fps)))
    n   = max(1, (raw - 1) // 8)
    return max(9, min([8*n+1, 8*(n+1)+1], key=lambda x: abs(x - raw)))



# ── FIX LIPSYNC (v7): compatibilidad de huggingface_hub SOLO para el subproceso LTX ──
# Síntoma en el log:  ImportError: cannot import name 'is_offline_mode' from 'huggingface_hub'
# Causa: la Celda 17 deja huggingface_hub<1.0 (lo necesita gradio) pero el transformers
#        instalado exige huggingface_hub>=1.0 → el subproceso LTX moría en 7 s y el
#        pipeline caía EN SILENCIO al clip estático.
# Solución: NO se toca el kernel. Se instala huggingface_hub>=1.0 en una carpeta aparte
#           que solo ve el subproceso LTX (vía PYTHONPATH).
LTX_HF_TARGET = "/kaggle/tmp/ltx_hf_compat"
_LTX_ENV = {"ok": False, "pythonpath": ""}

def _ltx_import_check(pythonpath: str = ""):
    env = os.environ.copy()
    if pythonpath:
        env["PYTHONPATH"] = pythonpath + os.pathsep + env.get("PYTHONPATH", "")
    r = subprocess.run(
        [sys.executable, "-c", "import transformers; from mmgp import offload, quant_router"],
        capture_output=True, text=True, env=env, timeout=900)
    return r.returncode == 0, ((r.stderr or "") + (r.stdout or ""))[-2500:]

# v8: rango de huggingface_hub que exige el transformers instalado (5.x → >=1.5,<2.0).
#     Si el kernel tiene hub 0.x (is_offline_mode) o 2.x (version check) el subproceso LTX muere;
#     la solución es darle SOLO al subproceso un hub 1.x compatible, sin tocar kernel ni Qwen.
LTX_HF_SPEC = "huggingface_hub>=1.5.0,<2.0.0"

def _ltx_err_is_hf(err: str) -> bool:
    e = (err or "").lower().replace("-", "_")
    return ("huggingface_hub" in e) or ("is_offline_mode" in e)

def ensure_ltx_env(force: bool = False) -> str:
    """Verifica que el subproceso LTX pueda importar transformers+mmgp.
    Devuelve el PYTHONPATH extra a usar ('' si no hace falta). Lanza RuntimeError si no hay arreglo."""
    import shutil as _sh
    if _LTX_ENV["ok"] and not force:
        return _LTX_ENV["pythonpath"]
    # Si ya existe un target válido de una ejecución previa, probarlo primero (evita reinstalar).
    if os.path.isdir(LTX_HF_TARGET) and os.listdir(LTX_HF_TARGET):
        ok, err = _ltx_import_check(LTX_HF_TARGET)
        if ok:
            _LTX_ENV.update(ok=True, pythonpath=LTX_HF_TARGET)
            return LTX_HF_TARGET
    ok, err = _ltx_import_check("")
    if ok:
        _LTX_ENV.update(ok=True, pythonpath="")
        return ""
    if _ltx_err_is_hf(err):
        log(f"[LTX25] ⚠ Conflicto huggingface_hub/transformers → instalando '{LTX_HF_SPEC}' "
            f"aislado en {LTX_HF_TARGET} (solo para el subproceso LTX)...")
        # Intento 1: solo huggingface_hub (--no-deps) → no pisa ninguna otra librería del subproceso.
        # Intento 2: con dependencias, por si falta algún módulo auxiliar de hub 1.x.
        for extra in (["--no-deps"], []):
            _sh.rmtree(LTX_HF_TARGET, ignore_errors=True)
            os.makedirs(LTX_HF_TARGET, exist_ok=True)
            r = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-cache-dir",
                 "--target", LTX_HF_TARGET] + extra + [LTX_HF_SPEC],
                capture_output=True, text=True)
            if r.returncode != 0:
                log(f"[LTX25] pip falló ({' '.join(extra) or 'con deps'}): "
                    f"{((r.stderr or '') + (r.stdout or ''))[-600:]}")
                continue
            ok, err = _ltx_import_check(LTX_HF_TARGET)
            if ok:
                _LTX_ENV.update(ok=True, pythonpath=LTX_HF_TARGET)
                log(f"[LTX25] ✅ Entorno LTX corregido (huggingface_hub aislado, {' '.join(extra) or 'con deps'}).")
                return LTX_HF_TARGET
    raise RuntimeError("El entorno de LTX-2.5 no puede importar transformers/mmgp:\n" + err[-1500:])


# ── FIX LIPSYNC (v8): archivos pequeños del Gemma4 (config/tokenizer) corruptos o vacíos ──
# Síntoma:  json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
#           en build_gemma_text_encoder → json.load(config.json)  (el subproceso moría a los ~30 s)
# Causa:    uno de los json de /tmp/models/gemma4-12b-ltx-v1 (o una copia en Wan2GP/ckpts|models) está
#           vacío/no es JSON (descarga hecha con otra versión de huggingface_hub, o interrumpida), y
#           ensure_ltx23_models() solo comprueba que el archivo EXISTA.
# Solución: validar esos archivos y, si están mal, volver a bajarlos por URL directa (independiente de
#           la versión de huggingface_hub) y re-enlazarlos. No toca el .safetensors (19+ GB).
GEMMA_SUB    = "gemma4-12b-ltx-v1"
GEMMA_SMALL  = ["config.json", "tokenizer_config.json", "tokenizer.json", "chat_template.jinja"]
GEMMA_REPO   = "DeepBeepMeep/LTX-2"
_GEMMA_OK    = {"ok": False}

def _gemma_valid(path: str, name: str) -> bool:
    try:
        if not os.path.exists(path) or os.path.getsize(path) <= 0:
            return False
        if name.endswith(".json"):
            with open(path, "r", encoding="utf-8") as f:
                json.load(f)
        return True
    except Exception:
        return False

def _gemma_diag(path: str) -> str:
    try:
        if not os.path.lexists(path):
            return "no existe"
        real = os.path.realpath(path)
        if not os.path.exists(real):
            return f"symlink roto → {real}"
        with open(real, "rb") as f:
            head = f.read(80)
        return f"{os.path.getsize(real)} bytes, inicio={head!r}"
    except Exception as e:
        return f"error leyendo: {e}"

def _gemma_download(name: str, dst: str):
    import urllib.request
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    url = f"https://huggingface.co/{GEMMA_REPO}/resolve/main/{GEMMA_SUB}/{name}"
    err = None
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=180) as r, open(tmp, "wb") as f:
            shutil.copyfileobj(r, f)
        if _gemma_valid(tmp, name):
            os.replace(tmp, dst)
            return
        err = f"descarga directa inválida ({_gemma_diag(tmp)})"
    except Exception as e:
        err = f"descarga directa falló: {e}"
    finally:
        if os.path.exists(tmp):
            try: os.remove(tmp)
            except Exception: pass
    # Respaldo: huggingface_hub a una carpeta temporal y copia del archivo devuelto
    log(f"[LTX25]   ⚠ {name}: {err} → probando huggingface_hub...")
    from huggingface_hub import hf_hub_download
    td = "/tmp/_gemma_dl"
    shutil.rmtree(td, ignore_errors=True)
    got = hf_hub_download(repo_id=GEMMA_REPO, subfolder=GEMMA_SUB, filename=name, local_dir=td)
    shutil.copyfile(got, dst + ".part")
    shutil.rmtree(td, ignore_errors=True)
    if not _gemma_valid(dst + ".part", name):
        bad = _gemma_diag(dst + ".part")
        try: os.remove(dst + ".part")
        except Exception: pass
        raise RuntimeError(f"No se pudo obtener un {name} válido ({bad})")
    os.replace(dst + ".part", dst)

def ensure_gemma_files(force: bool = False):
    """Garantiza que config/tokenizer del Gemma4 sean válidos en /tmp/models y en Wan2GP/ckpts|models."""
    if _GEMMA_OK["ok"] and not force:
        return
    for name in GEMMA_SMALL:
        real = os.path.join(GEMMA4_TMP, name)
        if not _gemma_valid(real, name):
            log(f"[LTX25] ⚠ Gemma4 {name} inválido ({_gemma_diag(real)}) → re-descargando...")
            try:
                if os.path.lexists(real): os.remove(real)
            except Exception: pass
            _gemma_download(name, real)
            log(f"[LTX25] ✅ Gemma4 {name} restaurado ({os.path.getsize(real)} bytes)")
        # Copias/symlinks que Wan2GP puede leer en lugar del de /tmp
        for base in (os.path.join(WAN2GP_DIR, "ckpts"), os.path.join(WAN2GP_DIR, "models")):
            dst = os.path.join(base, GEMMA_SUB, name)
            try:
                if os.path.lexists(dst) and not _gemma_valid(dst, name):
                    log(f"[LTX25] ⚠ {os.path.relpath(dst, WAN2GP_DIR)} inválido ({_gemma_diag(dst)}) → re-enlazado.")
                    os.remove(dst)
                if not os.path.lexists(dst):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    os.symlink(os.path.abspath(real), dst)
            except Exception as e:
                log(f"[LTX25] ⚠ no se pudo re-enlazar {dst}: {e}")
    _GEMMA_OK["ok"] = True


class SubprocessLTXBackend:
    """Motor LTX-2.5 con progreso en tiempo real vía Popen."""

    def is_available(self):
        return (os.path.isfile(LTX25_SCRIPT) and
                os.path.isdir(WAN2GP_DIR)    and
                os.path.isdir(TMP_MODELS))

    def _resolve_output_path(self, suggested=None):
        os.makedirs("/tmp/ltx25_outputs", exist_ok=True)
        if suggested:
            return os.path.join("/tmp/ltx25_outputs", os.path.basename(suggested))
        return f"/tmp/ltx25_outputs/ltx25_{int(time.time())}_{random.randint(0,99999)}.mp4"

    def _parse_progress_line(self, line: str) -> str:
        """
        Parsea líneas PROGRESS: del script y actualiza el archivo de progreso.
        Retorna un string legible para mostrar en UI.
        """
        if not line.startswith("PROGRESS:"): return ""
        parts = line.split(":")
        kind  = parts[1] if len(parts) > 1 else ""

        if kind == "start":
            total = parts[2] if len(parts) > 2 else "?"
            msg   = f"⚡ Iniciando denoising ({total} pasos)..."
            _write_progress({"status": msg, "step": 0,
                             "total": int(total) if total.isdigit() else 8,
                             "pct": 0, "active": True})
            return msg

        elif kind == "step":
            # PROGRESS:step:current:total:dt:rem:free_vram
            try:
                cur   = int(parts[2])
                total = int(parts[3])
                dt    = float(parts[4])
                rem   = float(parts[5])
                free  = float(parts[6])
                pct   = int(cur / total * 100) if total > 0 else 0
                bar   = "█" * (pct // 10) + "░" * (10 - pct // 10)
                msg   = (f"⚡ [{bar}] {pct}% | Paso {cur}/{total} | "
                         f"{dt:.1f}s/paso | ~{rem:.0f}s restantes | "
                         f"VRAM libre: {free:.1f}GB")
                _write_progress({"status": msg, "step": cur, "total": total,
                                 "pct": pct, "dt": dt, "rem": rem,
                                 "free_vram": free, "active": True})
                return msg
            except: return line

        elif kind == "stage":
            stage = ":".join(parts[2:])
            stage_icons = {
                "VAE Encoding": "🎞️ Codificando frames entrada...",
                "VAE Decoding": "🎬 Decodificando latents → frames...",
                "Upsampling":   "🔭 Upsampling espacial...",
            }
            msg = stage_icons.get(stage, f"⏳ {stage}...")
            _write_progress({"status": msg, "active": True})
            return msg

        return ""

    def generate(self,
                 prompt="", image_path=None, audio_path=None,
                 num_frames=121, width=1152, height=704,       #768×432   848×480 
                 seed=-1, steps=8, cfg=1.0, fps=24,
                 audio_cfg=1.0,
                 input_video_strength=0.8, denoise_strength=1.0,
                 output_path=None):
        """
        Genera video con Popen (lectura línea a línea de stdout).
        Actualiza LTX_PROGRESS_FILE en tiempo real.
        Retorna ruta del MP4 generado.
        """
        if not self.is_available():
            raise RuntimeError(
                f"Backend no disponible.\n"
                f"  Script: {os.path.isfile(LTX25_SCRIPT)}\n"
                f"  /tmp/models: {os.path.isdir(TMP_MODELS)}\n"
                "Ejecuta Celdas 12 y 12b."
            )

        if seed < 0:
            seed = random.randint(0, 2**32 - 1)
        out_path = self._resolve_output_path(output_path)

        params = {
            "prompt": prompt,
            "image_path": image_path or "",
            "audio_path": audio_path or "",  # NUEVO
            "output_path":          out_path,
            "width":                width,
            "height":               height,
            "num_frames":           num_frames,
            "fps":                  fps,
            "seed":                 seed,
            "steps":                steps,
            "cfg":                  cfg,
            "audio_cfg":            audio_cfg,
            "input_video_strength": input_video_strength,
        }

        mode = "I2V" if (image_path and os.path.exists(image_path)) else "T2V"
        log(f"[LTX25] {mode} | {width}x{height} | {num_frames}f@{fps}fps | "
            f"steps={steps} | cfg={cfg} | seed={seed}")

        _write_progress({"status": "⏳ Iniciando subproceso LTX-2.5...",
                         "active": True, "step": 0, "total": steps, "pct": 0})

        ensure_gemma_files()                  # v8: config/tokenizer de Gemma4 válidos antes de lanzar el subproceso
        _extra_pp = ensure_ltx_env()          # v7: arregla el import de huggingface_hub (ver arriba)
        env = os.environ.copy()
        if _extra_pp:
            env["PYTHONPATH"] = _extra_pp + os.pathsep + env.get("PYTHONPATH", "")
        env["WAN2GP_DIR"]       = WAN2GP_DIR
        env["PYTHONUNBUFFERED"] = "1"

        t0           = time.time()
        final_path   = out_path
        stderr_lines = []

        # ── Popen: leer stdout línea a línea en tiempo real ────
        proc = subprocess.Popen(
            [sys.executable, "-u", LTX25_SCRIPT, json.dumps(params)],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,          # line-buffered
            env=env,
        )

        # Hilo separado para capturar stderr sin bloquear
        def _read_stderr():
            for line in proc.stderr:
                line = line.rstrip()
                if line:
                    stderr_lines.append(line)
        stderr_thread = threading.Thread(target=_read_stderr, daemon=True)
        stderr_thread.start()

        # Leer stdout línea a línea
        for raw_line in proc.stdout:
            line = raw_line.rstrip()
            if not line: continue

            if line.startswith("OUTPUT_PATH:"):
                candidate = line.split("OUTPUT_PATH:", 1)[1].strip()
                if os.path.exists(candidate):
                    final_path = candidate
                log(f"[LTX25] OUTPUT_PATH detectado: {final_path}")
                continue

            if line.startswith("PROGRESS:"):
                ui_msg = self._parse_progress_line(line)
                if ui_msg:
                    log(f"[LTX25] {ui_msg}")
                continue

            # Líneas normales de log del subprocess
            if line.startswith("[SUBPROCESS]"):
                log(f"  {line}")
            else:
                # mmgp y otros mensajes informativos
                log(f"  {line}")

        proc.wait()
        stderr_thread.join(timeout=5)
        elapsed = time.time() - t0

        # ── Resultado ──────────────────────────────────────────
        if proc.returncode != 0:
            err_tail = "\n".join(stderr_lines[-30:])
            _write_progress({"status": f"❌ Error (rc={proc.returncode})",
                             "active": False})
            log(f"[LTX25] ❌ Subprocess falló (rc={proc.returncode}, {elapsed:.1f}s)")
            log(f"  STDERR:\n{err_tail[-2000:]}")
            raise RuntimeError(
                f"LTX-2.5 subprocess falló (rc={proc.returncode}):\n{err_tail[-2000:]}"
            )

        # Verificar archivo generado
        if not os.path.exists(final_path):
            raise RuntimeError(f"Video no encontrado: {final_path}")
        sz = os.path.getsize(final_path)
        if sz < 50_000:
            raise RuntimeError(
                f"Video demasiado pequeño ({sz} bytes): {final_path}\n"
                f"El guardado falló. Revisa stderr en los logs."
            )

        _write_progress({
            "status": f"✅ Completado en {elapsed:.1f}s | {sz/1024/1024:.1f} MB",
            "active": False, "pct": 100, "path": final_path
        })
        log(f"[LTX25] ✅ {elapsed:.1f}s → {final_path} ({sz/1024:.0f} KB)")
        return final_path


# ── Singleton ──────────────────────────────────────────────────
_LTX_BACKEND = None

def get_ltx_backend():
    global _LTX_BACKEND
    if _LTX_BACKEND is None:
        _LTX_BACKEND = SubprocessLTXBackend()
    return _LTX_BACKEND

def get_ltx_engine():
    return get_ltx_backend()

_b = get_ltx_backend()
log("CELDA 13 (v4 TIEMPO REAL) COMPLETADA.")
log(f"  ► Backend: {_b.is_available()}")
log(f"  ► Progreso en tiempo real: {LTX_PROGRESS_FILE}")
log(f"  ► /tmp libre: {free_gb('/tmp'):.1f} GB | /working libre: {free_gb('/kaggle/working'):.1f} GB")

[18:33:28] CELDA 13 (v4 TIEMPO REAL) COMPLETADA.
[18:33:28]   ► Backend: True
[18:33:28]   ► Progreso en tiempo real: /tmp/ltx25_progress.json
[18:33:28]   ► /tmp libre: 1031.5 GB | /working libre: 17.7 GB


In [13]:
# ═══════════════════════════════════════════════════════════════
# CELDA 14 (NUEVA): CLIPS LTX-2.5 — WRAPPERS COMPATIBLES
# Mantiene MISMA interfaz que Celda 18 espera:
#   generar_clip_5s(vp, image_path, prompt, out_mp4, ltx_paths, ...)
#   generar_clip_lipsync(vp, image_path, audio_path, prompt, out_mp4, ...)
#   generar_clip_lipsync_15s(...)
#   generar_clip_lipsync_dialogo(...)
# Internamente usa Wan2GPLTXBackend en vez de ComfyUI   
# ═══════════════════════════════════════════════════════════════
import os, subprocess, shutil, random, time

def _ffprobe_duration(path):
    try:
        cmd = ["ffprobe","-v","error","-show_entries","format=duration",
               "-of","default=noprint_wrappers=1:nokey=1", path]
        r = subprocess.run(cmd, capture_output=True, text=True)
        return float((r.stdout or "").strip())
    except: return None

def _to_mp4_exact(in_path, out_path, target_s, fps=24, mute=False):
    d = _ffprobe_duration(in_path) or 0.0
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    af = "-an" if mute else "-c:a aac -b:a 192k"
    if d >= target_s:
        cmd = f"ffmpeg -y -i '{in_path}' -t {target_s:.6f} -r {fps} -c:v libx264 -pix_fmt yuv420p {af} '{out_path}'"
    else:
        pad = target_s - d
        cmd = (f"ffmpeg -y -i '{in_path}' -vf 'tpad=stop_mode=clone:stop_duration={pad:.6f}' "
               f"-t {target_s:.6f} -r {fps} -c:v libx264 -pix_fmt yuv420p {af} '{out_path}'")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("FFmpeg _to_mp4_exact falló:\n" + (r.stderr or "")[-2000:])
    if in_path != out_path and os.path.exists(in_path):
        try: os.remove(in_path)
        except: pass
    return out_path

def generar_clip_5s(vp, image_path, prompt, out_mp4, ltx_paths=None,
                    seed=-1, fps=24, width=1152, height=704):
    """Clip 5s I2V usando Wan2GP LTX-2.5. Misma firma que Celda 18 espera."""
    os.makedirs(os.path.dirname(out_mp4) or ".", exist_ok=True)
    engine = get_ltx_backend()
    if not engine.is_available():
        raise RuntimeError("Wan2GP no disponible. Verifica Celda 13.")

    num_frames = calc_num_frames(5.0, fps)
    steps = int(ltx_paths.get("_steps", 8)) if ltx_paths else 8
    cfg = float(ltx_paths.get("_cfg", 1.0)) if ltx_paths else 1.0

    log(f"[CLIP 5s] I2V | {width}x{height} | {num_frames}f@{fps}fps | steps={steps}")

    raw = engine.generate(
        prompt=prompt, image_path=image_path,
        num_frames=num_frames, width=width, height=height,
        seed=seed, steps=steps, cfg=cfg, fps=fps,
        input_video_strength=0.8, denoise_strength=1.0,
    )
    _to_mp4_exact(raw, out_mp4, target_s=5.0, fps=fps, mute=True)
    log(f"[CLIP 5s] ✅ → {out_mp4}")
    return out_mp4

def generar_clip_lipsync(vp, image_path, audio_path, prompt, out_mp4,
                         ltx_paths=None, target_s=15.0, seed=-1, fps=24,
                         width=1152, height=704, audio_strength=0.9):  # 864 x 480 
    """Clip lipsync I2V usando Wan2GP LTX-2.5. Audio se muxa post-generación."""
    os.makedirs(os.path.dirname(out_mp4) or ".", exist_ok=True)
    engine = get_ltx_backend()
    if not engine.is_available():
        raise RuntimeError("Wan2GP no disponible. Verifica Celda 13.")

    num_frames = calc_num_frames(target_s, fps)
    steps = int(ltx_paths.get("_steps", 8)) if ltx_paths else 8
    cfg = float(ltx_paths.get("_cfg", 1.0)) if ltx_paths else 1.0

    log(f"[LIPSYNC {target_s:.1f}s] I2V | {width}x{height} | {num_frames}f | steps={steps}")

    raw = engine.generate(
    prompt=prompt,
    image_path=image_path,
    audio_path=audio_path,  # NUEVO: entregar audio al backend
    num_frames=num_frames,
    width=width,
    height=height,
    seed=seed,
    steps=steps,
    cfg=cfg,
    fps=fps,
    input_video_strength=0.8,
    denoise_strength=1.0,
)

    tmp_video = out_mp4 + ".tmp_noaudio.mp4"
    _to_mp4_exact(raw, tmp_video, target_s=target_s, fps=fps, mute=True)

    if audio_path and os.path.exists(audio_path):
        cmd_mux = (f"ffmpeg -y -i '{tmp_video}' -i '{audio_path}' "
                   f"-c:v copy -c:a aac -b:a 192k -shortest '{out_mp4}'")
        r_mux = subprocess.run(cmd_mux, shell=True, capture_output=True, text=True)
        if r_mux.returncode != 0:
            log(f"[LIPSYNC] ⚠ Mux falló: {(r_mux.stderr or '')[-500:]}")
            shutil.move(tmp_video, out_mp4)
        else:
            try: os.remove(tmp_video)
            except: pass
    else:
        shutil.move(tmp_video, out_mp4)

    if not os.path.exists(out_mp4) or os.path.getsize(out_mp4) < 10000:
        raise RuntimeError(f"No se generó vídeo lipsync válido: {out_mp4}")
    log(f"[LIPSYNC] ✅ → {out_mp4} ({target_s:.2f}s)")
    return out_mp4

# ═══ ALIAS OBLIGATORIOS — usados por Celdas 15, 16 y 18 ═══
def generar_clip_lipsync_15s(vp, image_path, audio_path, prompt, out_mp4,
                             ltx_paths=None, seed=-1, fps=24, **kwargs):
    """Alias para generar_clip_lipsync con target_s=10.0."""
    return generar_clip_lipsync(vp, image_path, audio_path, prompt, out_mp4,
                                ltx_paths=ltx_paths, target_s=10.0, seed=seed, fps=fps, **kwargs)

def generar_clip_lipsync_dialogo(vp, image_path, audio_path, prompt, out_mp4,
                                 ltx_paths=None, dur_target_s=8.0, seed=-1, fps=24, **kwargs):
    """Alias para generar_clip_lipsync con duración variable."""
    return generar_clip_lipsync(vp, image_path, audio_path, prompt, out_mp4,
                                ltx_paths=ltx_paths, target_s=dur_target_s, seed=seed, fps=fps, **kwargs)

log("CELDA 14 (NUEVA LTX-2.5 WRAPPERS) COMPLETADA.")
log("  ✅ Misma interfaz que Celda 18 espera")
log("  ✅ generar_clip_5s / generar_clip_lipsync / _15s / _dialogo definidos")
log("  ✅ Backend: Wan2GP LTX-2.5 (NO ComfyUI)")

[18:33:36] CELDA 14 (NUEVA LTX-2.5 WRAPPERS) COMPLETADA.
[18:33:36]   ✅ Misma interfaz que Celda 18 espera
[18:33:36]   ✅ generar_clip_5s / generar_clip_lipsync / _15s / _dialogo definidos
[18:33:36]   ✅ Backend: Wan2GP LTX-2.5 (NO ComfyUI)


In [14]:
# ═══════════════════════════════════════════════════════════════
# CELDA 15 (v2 CORREGIDA): ETAPA 3 COMPLETA — CANCIÓN CON LIPSYNC
# FIX: usa generar_clip_lipsync_15s (ahora definido en Celda 14)
# FIX: ltx_paths propagado correctamente
# ═══════════════════════════════════════════════════════════════

def etapa3_paso1_generar_clips_5s(
    vp: VideoProject,
    plantilla_slug: str,
    personaje_key: str,
    ltx_paths: dict,
    fps: int = 24,
    seed: int = -1,
) -> List[str]:
    """
    PASO 3.1: Genera 2 clips de 5s por cada plano lejano (E, F, G).
    """
    img_dir   = os.path.join(vp.video_dir, "etapa1_imagenes")
    out_5s_dir = os.path.join(vp.video_dir, "etapa3_cancion", "planos_5s")
    os.makedirs(out_5s_dir, exist_ok=True)

    keys_far = {
        "E": os.path.join(img_dir, "concursante_plano_lejano_1.png"),
        "F": os.path.join(img_dir, "concursante_plano_lejano_2.png"),
        "G": os.path.join(img_dir, "concursante_plano_lejano_3.png"),
    }

    generated = []
    for key, img_path in keys_far.items():
        if not os.path.exists(img_path):
            log(f"[ETAPA 3.1] Imagen {key} no existe, se omite.")
            continue
        for variant in [1, 2]:
            prompt_str, direccion = render_prompt_5s(personaje_key)
            out_mp4 = os.path.join(out_5s_dir, f"far_{key}_v{variant}.mp4")
            if os.path.exists(out_mp4) and os.path.getsize(out_mp4) > 10000:
                log(f"[ETAPA 3.1] {os.path.basename(out_mp4)} ya existe, se omite.")
                generated.append(out_mp4)
                continue
            log(f"[ETAPA 3.1] Generando {os.path.basename(out_mp4)} | dir={direccion}")
            vp.prompts.setdefault("etapa3_5s", {})[f"{key}_v{variant}"] = {
                "prompt": prompt_str,
                "direccion": direccion,
                "imagen": key,
            }
            vp.save()
            try:
                generar_clip_5s(
                    vp, img_path, prompt_str, out_mp4, ltx_paths,
                    seed=seed, fps=fps, width=1152, height=704,
                )
                generated.append(out_mp4)
            except Exception as e:
                log(f"[ETAPA 3.1] ❌ Error generando {key}_v{variant}: {e}")

    vp.estado.setdefault("etapa3", {})["clips_5s"] = generated
    vp.save()
    log(f"[ETAPA 3.1] Clips 5s generados: {len(generated)}")
    return generated


def etapa3_regenerar_clip_5s(
    vp: VideoProject,
    clip_path: str,
    personaje_key: str,
    ltx_paths: dict,
    fps: int = 24,
    seed: int = -1,
) -> str:
    """Regenera UN clip de 5s específico."""
    img_key = None
    for k in ["E", "F", "G"]:
        if k.lower() in os.path.basename(clip_path).lower():
            img_key = k
            break
    if not img_key:
        raise RuntimeError(f"No se pudo determinar la imagen base para {clip_path}")

    idx_map = {"E": 1, "F": 2, "G": 3}
    img_dir = os.path.join(vp.video_dir, "etapa1_imagenes")
    img_path = os.path.join(img_dir, f"concursante_plano_lejano_{idx_map[img_key]}.png")
    if not os.path.exists(img_path):
        raise RuntimeError(f"Imagen base no existe: {img_path}")

    prompt_str, direccion = render_prompt_5s(personaje_key)
    if os.path.exists(clip_path):
        os.remove(clip_path)

    generar_clip_5s(vp, img_path, prompt_str, clip_path, ltx_paths, seed=seed, fps=fps)
    log(f"[ETAPA 3.1 REGEN] Clip regenerado: {clip_path}")
    return clip_path



def _sig_lipsync_cancion(start_s, dur_s, img_path, song_wav, prompt):
    """Firma de un clip lipsync de la canción: si no cambia, el clip se REUTILIZA."""
    import hashlib
    def _mt(p):
        try: return os.path.getmtime(p)
        except Exception: return 0
    raw = f"{start_s:.3f}|{dur_s:.3f}|{img_path}|{_mt(img_path):.0f}|{_mt(song_wav):.0f}|{prompt}"
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]


def etapa3_paso3_renderizar_secuencia(
    vp: VideoProject,
    plantilla_slug: str,
    personaje_key: str,
    ltx_paths: dict,
    fps: int = 24,
    forzar_lipsync: bool = False,
) -> str:
    """
    PASO 3.3: Renderiza todos los clips del timeline y ensambla frame-perfect.
    v7: los clips lipsync ya generados y vigentes se REUTILIZAN (no se rehacen en cada render);
        forzar_lipsync=True los rehace todos.
    FIX: usa generar_clip_lipsync_15s (definido en Celda 14 corregida).
    """
    song_wav = vp.estado.get("etapa3", {}).get("song_wav", "")
    if not song_wav or not os.path.exists(song_wav):
        raise RuntimeError("No hay song.wav. Sube el audio de la canción primero.")

    if not vp.timeline_items:
        raise RuntimeError("El timeline está vacío. Ejecuta PASO 3.2 primero.")

    clips_dir = os.path.join(vp.video_dir, "etapa3_cancion", "clips_renderizados")
    os.makedirs(clips_dir, exist_ok=True)

    img_dir  = os.path.join(vp.video_dir, "etapa1_imagenes")
    song_dur = _ffprobe_duration(song_wav) or 0.0

    video_parts = []
    audio_t = 0.0

    prompt_lipsync = vp.prompts.get("etapa3_lipsync", PROMPT_LIPSYNC_CANCION)

    for item in vp.timeline_items:
        cid   = item["clip_id"]
        tipo  = item["tipo"]
        dur   = float(item["dur_s"])
        src   = item.get("path_video", "")
        out   = os.path.join(clips_dir, f"{cid}.mp4")

        if tipo == "plano_lejano_5s":
            if os.path.exists(src) and src.endswith(".mp4"):
                trim_or_pad_video_to_duration(src, out, target_s=dur, fps=fps)
                muted = out + "_m.mp4"
                mute_video_audio(out, muted)
                shutil.move(muted, out)
                item["estado"] = "completado"
            else:
                log(f"[PASO 3.3] AVISO: clip 5s no encontrado: {src}. Usando imagen estática.")
                img_path = os.path.join(img_dir, "concursante_plano_lejano_1.png")
                if os.path.exists(img_path):
                    cmd = (f"ffmpeg -y -loop 1 -i '{img_path}' "
                           f"-t {dur:.6f} -r {fps} -c:v libx264 -pix_fmt yuv420p -an '{out}'")
                    subprocess.run(cmd, shell=True, capture_output=True)
                    item["estado"] = "completado_fallback"

        elif tipo == "reaccion":
            if os.path.exists(src):
                trim_or_pad_video_to_duration(src, out, target_s=dur, fps=fps)
                muted = out + "_m.mp4"
                mute_video_audio(out, muted)
                shutil.move(muted, out)
                item["estado"] = "completado"
            else:
                log(f"[PASO 3.3] Reacción no encontrada: {src}")

        elif tipo == "lipsync_15s":
            seg_wav = os.path.join(clips_dir, f"{cid}_audio.wav")
            safe_dur = min(dur, max(0.1, song_dur - audio_t))
            if safe_dur < 0.5:
                log(f"[PASO 3.3] Audio insuficiente para lipsync en {cid}. Saltando.")
                audio_t += dur
                continue

            img_C = os.path.join(img_dir, "concursante_plano_cercano.png")
            img_D = os.path.join(img_dir, "concursante_closeup.png")
            use_img = (src if src.endswith(".png") and os.path.exists(src)
                      else (img_C if random.random() < 0.5 else img_D))

            sig = _sig_lipsync_cancion(audio_t, safe_dur, use_img, song_wav, prompt_lipsync)
            if (not forzar_lipsync and item.get("estado") == "completado"
                    and item.get("sig") == sig
                    and os.path.exists(out) and os.path.getsize(out) > 10000):
                log(f"[PASO 3.3] {cid} lipsync vigente → reutilizado (no se regenera).")
            else:
                cut_audio_segment(song_wav, seg_wav, start_s=audio_t, dur_s=safe_dur)
                try:
                    # FIX: usa generar_clip_lipsync_15s (ahora definido en Celda 14)
                    generar_clip_lipsync_15s(
                        vp, use_img, seg_wav, prompt_lipsync, out,
                        ltx_paths=ltx_paths,
                        seed=random.randint(0, 2**31 - 1), fps=fps,
                    )
                    item["estado"] = "completado"
                    item["sig"] = sig
                except Exception as e:
                    log(f"[PASO 3.3] ❌ Lipsync {cid} falló: {e}. Fallback imagen estática.")
                    cmd_fb = (f"ffmpeg -y -loop 1 -i '{use_img}' -i '{seg_wav}' "
                              f"-c:v libx264 -r {fps} -pix_fmt yuv420p -c:a aac -b:a 192k "
                              f"-t {safe_dur:.6f} '{out}'")
                    subprocess.run(cmd_fb, shell=True, capture_output=True)
                    item["estado"] = "completado_fallback"
                    item.pop("sig", None)     # v7: un fallback NUNCA se reutiliza (se reintenta)

        audio_t += dur
        if os.path.exists(out):
            video_parts.append(out)

        vp.save()

    if not video_parts:
        raise RuntimeError("No se generaron partes de video para ensamblar.")

    # ── Ensamblaje final ─────────────────────────────────────────
    concat_list = os.path.join(clips_dir, "concat_list.txt")
    with open(concat_list, "w", encoding="utf-8") as f:
        for p in video_parts:
            f.write(f"file '{os.path.abspath(p)}'\n")

    out_video_noaudio = os.path.join(vp.video_dir, "etapa3_cancion", "cancion_noaudio.mp4")
    cmd_concat = (
        f"ffmpeg -y -f concat -safe 0 -i '{concat_list}' "
        f"-c:v libx264 -r {fps} -pix_fmt yuv420p -an '{out_video_noaudio}'"
    )
    r = subprocess.run(cmd_concat, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("FFmpeg concat falló:\n" + (r.stderr or "")[-3000:])

    v_dur = _ffprobe_duration(out_video_noaudio) or 0.0
    a_dur = _ffprobe_duration(song_wav) or 0.0
    log(f"[PASO 3.3] Video: {v_dur:.3f}s | Audio: {a_dur:.3f}s | diff: {abs(v_dur-a_dur)*1000:.1f}ms")

    out_final = os.path.join(vp.video_dir, "etapa3_cancion", "cancion_final.mp4")
    cmd_mix = (
        f"ffmpeg -y -i '{out_video_noaudio}' -i '{song_wav}' "
        f"-c:v copy -c:a aac -b:a 192k -shortest '{out_final}'"
    )
    subprocess.run(cmd_mix, shell=True, capture_output=True, text=True)

    vp.estado["etapa3"]["video_final"] = out_final
    vp.save()
    log(f"[ETAPA 3] ✅ COMPLETADA → {out_final}")
    return out_final



# ═══════════════════════════════════════════════════════════════
# v7 — REGENERAR UN CLIP LIPSYNC DE LA CANCIÓN + FILAS PARA EL TIMELINE DE LA UI
# ═══════════════════════════════════════════════════════════════
def _etapa3_lipsync_pendientes(vp: VideoProject) -> list:
    """clip_id de los lipsync que aún no tienen un clip LTX real vigente."""
    clips_dir = os.path.join(vp.video_dir, "etapa3_cancion", "clips_renderizados")
    pend = []
    for it in (vp.timeline_items or []):
        if it.get("tipo") != "lipsync_15s":
            continue
        out = os.path.join(clips_dir, f"{it['clip_id']}.mp4")
        if not (it.get("estado") == "completado" and it.get("sig")
                and os.path.exists(out) and os.path.getsize(out) > 10000):
            pend.append(it["clip_id"])
    return pend


def etapa3_regenerar_clip_lipsync(vp: VideoProject, plantilla_slug: str, personaje_key: str,
                                  clip_id: str, ltx_paths: dict, fps: int = 24,
                                  reensamblar: bool = True):
    """Regenera UN clip lipsync (LTX) del timeline de la canción.
    - Corta de nuevo su tramo de audio y genera con una seed nueva.
    - Si LTX falla, se conserva el clip anterior (no se sustituye por nada).
    - Reensambla la canción solo si TODOS los lipsync ya están renderizados
      (si no, evita lanzar de golpe los que faltan).
    Devuelve (ruta_clip, reensamblado: bool)."""
    song_wav = vp.estado.get("etapa3", {}).get("song_wav", "")
    if not song_wav or not os.path.exists(song_wav):
        raise RuntimeError("No hay song.wav. Sube el audio de la canción primero.")
    item = next((i for i in (vp.timeline_items or []) if i.get("clip_id") == clip_id), None)
    if item is None:
        raise RuntimeError(f"El clip {clip_id} no está en el timeline actual "
                           f"(¿reconstruiste el timeline? Pulsa 'Actualizar fragmentos').")
    if item.get("tipo") != "lipsync_15s":
        raise RuntimeError(f"{clip_id} no es un clip lipsync ({item.get('tipo')}).")

    clips_dir = os.path.join(vp.video_dir, "etapa3_cancion", "clips_renderizados")
    os.makedirs(clips_dir, exist_ok=True)
    img_dir = os.path.join(vp.video_dir, "etapa1_imagenes")
    song_dur = _ffprobe_duration(song_wav) or 0.0
    start = float(item["start"]); dur = float(item["dur_s"])
    safe_dur = min(dur, max(0.1, song_dur - start))
    if safe_dur < 0.5:
        raise RuntimeError(f"Audio insuficiente para el lipsync {clip_id}.")

    src = item.get("path_video", "") or ""
    use_img = (src if src.endswith(".png") and os.path.exists(src)
               else os.path.join(img_dir, "concursante_plano_cercano.png"))
    if not os.path.exists(use_img):
        raise RuntimeError(f"Falta la imagen base del lipsync: {use_img}")
    prompt_lipsync = vp.prompts.get("etapa3_lipsync", PROMPT_LIPSYNC_CANCION)

    seg_wav = os.path.join(clips_dir, f"{clip_id}_audio.wav")
    out = os.path.join(clips_dir, f"{clip_id}.mp4")
    backup = out + ".prev"
    cut_audio_segment(song_wav, seg_wav, start_s=start, dur_s=safe_dur)
    if os.path.exists(out):
        shutil.copyfile(out, backup)
    log(f"[ETAPA 3 REGEN] {clip_id}: lipsync {start:.1f}s→{start+safe_dur:.1f}s con seed nueva")
    try:
        generar_clip_lipsync_15s(vp, use_img, seg_wav, prompt_lipsync, out,
                                 ltx_paths=ltx_paths,
                                 seed=random.randint(0, 2**31 - 1), fps=fps)
    except Exception:
        if os.path.exists(backup):
            shutil.move(backup, out)          # conserva el clip anterior
        elif os.path.exists(out):
            try: os.remove(out)
            except Exception: pass
        raise
    if os.path.exists(backup):
        try: os.remove(backup)
        except Exception: pass

    item["estado"] = "completado"
    item["sig"] = _sig_lipsync_cancion(start, safe_dur, use_img, song_wav, prompt_lipsync)
    vp.save()

    reens = False
    if reensamblar and not _etapa3_lipsync_pendientes(vp):
        etapa3_paso3_renderizar_secuencia(vp, plantilla_slug, personaje_key, ltx_paths, fps)
        reens = True
    return out, reens


def etapa3_filas_timeline(vp: VideoProject) -> list:
    """Filas (una por clip del timeline) para pintar el timeline por fragmentos en la UI."""
    clips_dir = os.path.join(vp.video_dir, "etapa3_cancion", "clips_renderizados")
    rows = []
    for it in (vp.timeline_items or []):
        out = os.path.join(clips_dir, f"{it['clip_id']}.mp4")
        ok = os.path.exists(out) and os.path.getsize(out) > 10000
        rows.append({
            "clip_id": it["clip_id"], "tipo": it.get("tipo", ""),
            "hablante": it.get("hablante", ""),
            "start": float(it.get("start", 0)), "end": float(it.get("end", 0)),
            "dur": float(it.get("dur_s", 0)),
            "estado": it.get("estado", "pendiente"),
            "video": out if ok else "",
        })
    return rows


log("CELDA 15 (v2 CORREGIDA) COMPLETADA.")

[18:33:51] CELDA 15 (v2 CORREGIDA) COMPLETADA.


In [15]:
# ═══════════════════════════════════════════════════════════════
# CELDA 16 (v2) — PIPELINE DE DIÁLOGOS: guion → TTS → lipsync → ensamblaje
#
# Modos de guion (ejecutar_etapa_dialogos):
#   regenerar_guion=True   → xKiro genera un guion NUEVO (el anterior queda
#                            respaldado en estado[etapa]["guion_json_prev"]).
#   usar_guion_actual=True → NUNCA llama a xKiro; usa el guion guardado/editado.
#   ninguno                → reutiliza el guion si existe, si no lo genera.
#
# Garantías:
#   - Audio y clip se reutilizan SOLO si su firma (hablante+texto+voz) coincide:
#     si editas el guion, solo se rehacen los fragmentos que cambiaron.
#   - Todo el TTS se hace primero y Qwen se descarga antes del lipsync (VRAM).
#   - NO se escribe nada en vp.timeline_items (es el timeline de la canción,
#     Etapa 3). El timeline de diálogos va en estado[etapa]["timeline_dialogo"].
# ═══════════════════════════════════════════════════════════════
import hashlib

def _fnum(x, default=0.0):
    try:
        return float(x)
    except Exception:
        return float(default)

def _norm_hablante(h) -> str:
    h = str(h or "concursante").strip().lower().replace(" ", "_").replace("-", "_")
    h = re.sub(r"[^a-z0-9_]", "", h)
    m = re.match(r"^juez_?(\d+)$", h)
    if m:
        return f"juez_{int(m.group(1))}"
    if h == "concursante":
        return h
    log(f"[Diálogos] ⚠ Hablante desconocido '{h}' → se trata como 'concursante'.")
    return "concursante"

def _read_txt(p: str) -> str:
    try:
        return open(p, "r", encoding="utf-8").read().strip()
    except Exception:
        return ""

def _write_txt(p: str, txt: str) -> None:
    with open(p, "w", encoding="utf-8") as f:
        f.write(txt)

def _rm(*paths) -> None:
    for p in paths:
        try:
            if os.path.exists(p):
                os.remove(p)
        except Exception:
            pass

def _voice_sig(hablante: str, plantilla_slug: str) -> str:
    try:
        if hablante == "concursante":
            p = CONTESTANT_VOICE_STATE.get("ref_wav_path", "")
            t = CONTESTANT_VOICE_STATE.get("ref_text", "")
        else:
            jid = int(hablante.split("_")[1])
            p = get_juez_ref_wav(plantilla_slug, jid)
            t = get_juez_ref_text(plantilla_slug, jid)
        mt = os.path.getmtime(p) if p and os.path.exists(p) else 0
        return f"{p}|{mt:.0f}|{t}"
    except Exception:
        return ""

def _frag_sig(hablante: str, texto: str, fast_mode: bool, plantilla_slug: str) -> str:
    raw = f"{hablante}|{texto}|{int(bool(fast_mode))}|{_voice_sig(hablante, plantilla_slug)}"
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]

def _frag_paths(vp: VideoProject, etapa: str, idx: int, hablante: str) -> dict:
    d = os.path.join(vp.video_dir, f"{etapa}_dialogos", f"frag_{idx:03d}_{hablante}")
    os.makedirs(d, exist_ok=True)
    return {"dir": d,
            "wav": os.path.join(d, "audio.wav"), "wav_sig": os.path.join(d, "audio.sig"),
            "mp4": os.path.join(d, "clip.mp4"),  "mp4_sig": os.path.join(d, "clip.sig"),
            # v7: preview solo-imágenes y marca de "lipsync falló → clip estático"
            "prev": os.path.join(d, "preview.mp4"), "prev_sig": os.path.join(d, "preview.sig"),
            "fb": os.path.join(d, "clip.fallback")}

def _imagen_hablante_segura(vp: VideoProject, plantilla_slug: str, hablante: str, idx: int) -> str:
    img_dir = os.path.join(vp.video_dir, "etapa1_imagenes")
    cerca = os.path.join(img_dir, "concursante_plano_cercano.png")
    close = os.path.join(img_dir, "concursante_closeup.png")
    cand = []
    if hablante == "concursante":
        cand = [close if idx % 3 == 0 else cerca, cerca, close]
    else:
        try:
            tpl = cargar_plantilla(plantilla_slug)
            cand.append(os.path.join(tpl["_base_dir"], PLANTILLA_SUBDIRS["jueces_imagenes"],
                                     f"juez_{hablante.split('_')[1]}.png"))
        except Exception:
            pass
        cand += [cerca, close]
    for p in cand:
        if p and os.path.exists(p):
            return p
    return ""

def _clip_estatico_dialogo(img_path: str, wav: str, out_mp4: str, dur_s: float, fps: int) -> bool:
    # Fallback si el lipsync falla: imagen fija + audio, MISMA resolución que LTX.
    try:
        _rm(out_mp4)
        cmd = ["ffmpeg", "-y", "-loop", "1", "-framerate", str(fps), "-i", img_path, "-i", wav,
               "-t", f"{dur_s:.3f}",
               "-vf", (f"scale={DIALOGO_W}:{DIALOGO_H}:force_original_aspect_ratio=increase,"
                       f"crop={DIALOGO_W}:{DIALOGO_H},fps={fps},format=yuv420p"),
               "-c:v", "libx264", "-c:a", "aac", "-b:a", "192k", "-shortest",
               out_mp4, "-loglevel", "error"]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            log(f"[StaticClip] ffmpeg: {(r.stderr or '')[-400:]}")
        return r.returncode == 0 and os.path.exists(out_mp4) and os.path.getsize(out_mp4) > 1000
    except Exception as e:
        log(f"[StaticClip] Error: {e}")
        return False

def _prompt_lipsync_hablante(personaje_key: str, hablante: str) -> str:
    if hablante == "concursante":
        return render_prompt_lipsync_dialogo(personaje_key)
    return ("La persona hablando, sincronización labial natural y muy realista, no modificar los "
            "rasgos y la apariencia de su rostro. Nunca mira a la cámara y mantiene la mirada al "
            "frente siempre, nunca girar la cabeza ni cambia la mirada.")

def generar_guion_con_respaldo(vp: VideoProject, personaje_key: str, etapa: str,
                               historia_base: str = "",
                               model_id: str = DEFAULT_XKIRO_TEXT_MODEL_ID) -> dict:
    # Genera un guion NUEVO con xKiro. Respalda el anterior y, si el nuevo
    # viene vacío/mal formado, RESTAURA el anterior (nunca pierdes un guion bueno).
    prev = (vp.estado.get(etapa, {}) or {}).get("guion_json")
    try:
        data = generar_dialogos_json(vp, personaje_key, etapa,
                                     historia_base=historia_base, model_id=model_id)
        ok = isinstance(data, dict) and bool(data.get("fragmentos"))
    except Exception:
        if prev:
            vp.estado.setdefault(etapa, {})["guion_json"] = prev
            vp.save()
        raise
    if not ok:
        if prev:
            vp.estado.setdefault(etapa, {})["guion_json"] = prev
            vp.save()
        raise RuntimeError(f"xKiro devolvió un guion vacío o mal formado para {etapa}. "
                           f"{'Se conservó el guion anterior.' if prev else ''}")
    if prev:
        vp.estado[etapa]["guion_json_prev"] = prev
        vp.save()
    return data

def _dialogo_audio_fragmento(vp, etapa, idx, hablante, texto, dur_est,
                             plantilla_slug, fast_mode_tts, force=False):
    fp = _frag_paths(vp, etapa, idx, hablante)
    sig = _frag_sig(hablante, texto, fast_mode_tts, plantilla_slug)
    if (not force and os.path.exists(fp["wav"]) and os.path.getsize(fp["wav"]) > 8000
            and _read_txt(fp["wav_sig"]) == sig):
        log(f"  [{idx:03d}] audio vigente (sin cambios) → reutilizado.")
        return fp["wav"], sig
    _rm(fp["wav"], fp["wav_sig"], fp["mp4"], fp["mp4_sig"],
        fp["prev"], fp["prev_sig"], fp["fb"])                    # invalida audio, clip y preview
    wav_tts = tts_fragmento(vp=vp, etapa=etapa, idx=idx, hablante=hablante, texto=texto,
                            dur_target_s=dur_est, plantilla_slug=plantilla_slug,
                            fast_mode=fast_mode_tts)
    shutil.copyfile(wav_tts, fp["wav"])
    _write_txt(fp["wav_sig"], sig)
    return fp["wav"], sig

def _dialogo_clip_fragmento(vp, plantilla_slug, personaje_key, etapa, idx, hablante,
                            ltx_paths, fps, sig, permitir_respaldo=True) -> str:
    # v7: un clip estático de RESPALDO ya no se da por "vigente": lleva la marca clip.fallback
    #     y NO guarda clip.sig, así que la próxima ejecución reintenta el lipsync real.
    #     permitir_respaldo=False (regeneración manual): si LTX falla devuelve "" y no crea nada.
    fp = _frag_paths(vp, etapa, idx, hablante)
    if (os.path.exists(fp["mp4"]) and os.path.getsize(fp["mp4"]) > 10000
            and not os.path.exists(fp["fb"])
            and _read_txt(fp["mp4_sig"]) == sig):
        log(f"  [{idx:03d}] clip vigente (sin cambios) → reutilizado.")
        return fp["mp4"]
    _rm(fp["mp4"], fp["mp4_sig"], fp["fb"])
    real_dur = min(_ffprobe_duration(fp["wav"]) or 5.0, DIALOGO_MAX_AUDIO_S)
    img = _imagen_hablante_segura(vp, plantilla_slug, hablante, idx)
    prompt = _prompt_lipsync_hablante(personaje_key, hablante)
    ok = False
    try:
        if not img:
            raise FileNotFoundError(f"Sin imagen de referencia para {hablante}")
        generar_clip_lipsync_dialogo(vp, img, fp["wav"], prompt, fp["mp4"], ltx_paths,
                                     dur_target_s=real_dur,
                                     seed=random.randint(0, 2**31 - 1), fps=fps)
        ok = os.path.exists(fp["mp4"]) and os.path.getsize(fp["mp4"]) > 10000
    except Exception as e:
        log(f"  [{idx:03d}] ❌ lipsync {hablante}: {e}"
            + (" → clip estático de respaldo." if permitir_respaldo else ""))
    if not ok:
        if not permitir_respaldo:
            _rm(fp["mp4"])
            return ""
        if not img or not _clip_estatico_dialogo(img, fp["wav"], fp["mp4"], real_dur, fps):
            log(f"  [{idx:03d}] ❌ no se pudo crear clip para {hablante}.")
            return ""
        _write_txt(fp["fb"], "estatico")      # marca: NO es lipsync real → se reintentará
        return fp["mp4"]
    _write_txt(fp["mp4_sig"], sig)
    return fp["mp4"]

def _items_validos_guion(guion: dict):
    out = []
    for idx, frag in enumerate((guion or {}).get("fragmentos", []) or []):
        texto = str(frag.get("texto", "")).strip()
        if not texto:
            continue
        out.append({"idx": idx, "hablante": _norm_hablante(frag.get("hablante")),
                    "texto": texto, "emocion": frag.get("emocion", "neutral"),
                    "dur_est": _fnum(frag.get("duracion_estimada_s"), 5.0)})
    return out

def _ensamblar_etapa_dialogos(vp: VideoProject, etapa: str, clips: list, fps: int = 24) -> str:
    etapa_dir = os.path.join(vp.video_dir, f"{etapa}_dialogos")
    concat_list = os.path.join(etapa_dir, "concat_list.txt")
    with open(concat_list, "w", encoding="utf-8") as f:
        for c in clips:
            f.write(f"file '{os.path.abspath(c)}'\n")
    out_final = os.path.join(etapa_dir, f"{etapa}_final.mp4")
    cmd = (f"ffmpeg -y -f concat -safe 0 -i '{concat_list}' "
           f"-c:v libx264 -r {fps} -pix_fmt yuv420p -c:a aac -b:a 192k '{out_final}' -loglevel error")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("FFmpeg concat falló:\n" + (r.stderr or "")[-3000:])
    return out_final

def _registrar_etapa_dialogos(vp, etapa, items, clips_por_idx, out_final):
    # Timeline PROPIO de diálogos (no toca vp.timeline_items de la canción).
    t, tl = 0.0, []
    for it in items:
        c = clips_por_idx.get(it["idx"])
        if not c:
            continue
        d = _ffprobe_duration(c) or it["dur_est"]
        tl.append({"clip_id": f"{etapa}_frag_{it['idx']:03d}_{it['hablante']}", "tipo": "dialogo",
                   "hablante": it["hablante"], "start": round(t, 3), "end": round(t + d, 3),
                   "dur_s": round(d, 3), "path_video": c,
                   "estado": ("completado_fallback"
                              if os.path.exists(_frag_paths(vp, etapa, it["idx"], it["hablante"])["fb"])
                              else "completado"),
                   "meta": {"emocion": it["emocion"], "texto": it["texto"][:80]}})
        t += d
    st = vp.estado.setdefault(etapa, {})
    st["video_final"] = out_final
    st["n_fragmentos"] = len(items)
    st["audio_paths"] = [_frag_paths(vp, etapa, it["idx"], it["hablante"])["wav"] for it in items]
    st["timeline_dialogo"] = tl
    vp.save()

def ejecutar_etapa_dialogos(
    vp: VideoProject,
    plantilla_slug: str,
    personaje_key: str,
    etapa: str,
    ltx_paths: dict,
    historia_base: str = "",
    model_id: str = DEFAULT_XKIRO_TEXT_MODEL_ID,
    fast_mode_tts: bool = False,
    voz_concursante_label: str = "Mujer — Cálida y maternal",   # compat (no usado)
    fps: int = 24,
    seed: int = -1,                                              # compat (no usado)
    regenerar_guion: bool = False,
    usar_guion_actual: bool = False,
    solo_preview: bool = False,      # v7: True → TTS + video SOLO con imágenes (sin LTX ni lipsync)
) -> str:
    assert etapa in ("etapa2", "etapa4"), "etapa debe ser 'etapa2' o 'etapa4'"
    os.makedirs(os.path.join(vp.video_dir, f"{etapa}_dialogos"), exist_ok=True)
    log(f"\n[{etapa.upper()}] Iniciando pipeline de diálogos...")

    # PASO 0 — falla rápido y claro si la voz no está validada
    ref_info = get_contestant_ref_for_tts()
    log(f"[{etapa.upper()}] Voz concursante OK: {os.path.basename(ref_info['ref_wav'])}")

    # PASO 1 — guion
    guion = (vp.estado.get(etapa, {}) or {}).get("guion_json")
    tiene = bool(guion and guion.get("fragmentos"))
    if usar_guion_actual:
        if not tiene:
            raise RuntimeError(f"No hay guion guardado para {etapa}. Pulsa '📝 Generar solo guion' "
                               f"(o 'Ejecutar completa') primero.")
        log(f"[{etapa.upper()}] Usando el guion ACTUAL ({len(guion['fragmentos'])} frags). xKiro NO se llama.")
    elif regenerar_guion or not tiene:
        log(f"[{etapa.upper()}] Generando guion NUEVO con xKiro...")
        guion = generar_guion_con_respaldo(vp, personaje_key, etapa,
                                           historia_base=historia_base, model_id=model_id)
    else:
        log(f"[{etapa.upper()}] Reutilizando guion existente ({len(guion['fragmentos'])} frags).")

    items = _items_validos_guion(guion)
    if not items:
        raise RuntimeError(f"El guion de {etapa} no tiene fragmentos con texto.")
    log(f"[{etapa.upper()}] {len(items)} fragmentos | "
        f"dur. estimada: {_fnum(guion.get('duracion_total_estimada_min')):.1f} min")

    # PASO 1b — transcripción de jueces (no fatal)
    try:
        auto_transcribe_all_judges(plantilla_slug)
    except Exception as e:
        log(f"[{etapa.upper()}] ⚠ Transcripción de jueces omitida: {e}")

    # PASO 2 — TODO el audio primero (Qwen)
    log(f"[{etapa.upper()}] PASO 2: audio de {len(items)} fragmentos...")
    sigs = {}
    for k, it in enumerate(items):
        log(f"[{etapa.upper()}] audio [{k+1}/{len(items)}] {it['hablante']} | «{it['texto'][:70]}»")
        _wav, sigs[it["idx"]] = _dialogo_audio_fragmento(
            vp, etapa, it["idx"], it["hablante"], it["texto"], it["dur_est"],
            plantilla_slug, fast_mode_tts)
    try:                       # liberar VRAM de Qwen antes de LTX
        qwen3_tts.unload()
    except Exception:
        pass

    # v7 — MODO PREVIEW: timeline solo con imágenes + audio para validar el guion hablado.
    #      No usa LTX. Para pasar al lipsync después: «Continuar con lipsync» (reutiliza los audios).
    if solo_preview:
        return _preview_etapa_dialogos(vp, plantilla_slug, etapa, items, sigs, fps)

    # PASO 3 — lipsync por fragmento (LTX)
    log(f"[{etapa.upper()}] PASO 3: lipsync de {len(items)} fragmentos...")
    clips_por_idx = {}
    for k, it in enumerate(items):
        log(f"[{etapa.upper()}] video [{k+1}/{len(items)}] {it['hablante']}")
        c = _dialogo_clip_fragmento(vp, plantilla_slug, personaje_key, etapa,
                                    it["idx"], it["hablante"], ltx_paths, fps, sigs[it["idx"]])
        if c:
            clips_por_idx[it["idx"]] = c
    if not clips_por_idx:
        raise RuntimeError(f"No se generaron clips para {etapa}.")
    if len(clips_por_idx) < len(items):
        log(f"[{etapa.upper()}] ⚠ {len(items)-len(clips_por_idx)} fragmentos sin clip (se omiten).")

    n_fb = sum(1 for it in items
               if os.path.exists(_frag_paths(vp, etapa, it["idx"], it["hablante"])["fb"]))
    if n_fb:
        log(f"[{etapa.upper()}] ⚠ {n_fb} fragmento(s) quedaron con clip ESTÁTICO porque el lipsync LTX "
            f"falló (revisa el log). Usa «🔄 Regenerar lipsync» en el timeline o vuelve a ejecutar la etapa.")

    # PASO 4 — ensamblaje
    clips = [clips_por_idx[it["idx"]] for it in items if it["idx"] in clips_por_idx]
    out_final = _ensamblar_etapa_dialogos(vp, etapa, clips, fps)
    _registrar_etapa_dialogos(vp, etapa, items, clips_por_idx, out_final)
    log(f"[{etapa.upper()}] ✅ COMPLETADA → {out_final} ({_ffprobe_duration(out_final) or 0:.2f}s)")
    return out_final

def regenerar_fragmento_dialogo(
    vp: VideoProject,
    plantilla_slug: str,
    personaje_key: str,
    etapa: str,
    frag_idx: int,
    nuevo_texto: str = "",
    ltx_paths: dict = None,
    voz_concursante_label: str = "Mujer — Cálida y maternal",   # compat (no usado)
    fps: int = 24,
    fast_mode_tts: bool = False,
) -> str:
    # Regenera UN fragmento (audio + lipsync). nuevo_texto vacío = texto actual.
    try:
        guion = (vp.estado.get(etapa, {}) or {}).get("guion_json") or {}
        frags = guion.get("fragmentos", []) or []
        if not frags:
            return f"❌ No hay guion guardado para {etapa}."
        frag_idx = int(frag_idx)
        if not (0 <= frag_idx < len(frags)):
            return f"❌ Índice {frag_idx} fuera de rango (0..{len(frags)-1})."
        if not ltx_paths:
            return "❌ Descarga LTX primero."
        get_contestant_ref_for_tts()

        frag = frags[frag_idx]
        if nuevo_texto and nuevo_texto.strip():
            frag["texto"] = nuevo_texto.strip()
            frag["duracion_estimada_s"] = max(3.0, len(frag["texto"].split()) / 2.5)
            vp.estado[etapa]["guion_json"] = guion
            vp.save()
        texto = str(frag.get("texto", "")).strip()
        if not texto:
            return "❌ El fragmento no tiene texto."
        hablante = _norm_hablante(frag.get("hablante"))
        log(f"[Regen] {etapa} frag {frag_idx} ({hablante}): «{texto[:60]}»")

        _wav, sig = _dialogo_audio_fragmento(vp, etapa, frag_idx, hablante, texto,
                                             _fnum(frag.get("duracion_estimada_s"), 5.0),
                                             plantilla_slug, fast_mode_tts, force=True)
        try:
            qwen3_tts.unload()
        except Exception:
            pass
        clip = _dialogo_clip_fragmento(vp, plantilla_slug, personaje_key, etapa,
                                       frag_idx, hablante, ltx_paths, fps, sig)
        if not clip:
            return f"❌ No se pudo generar el clip del fragmento {frag_idx}."

        # Reensamblar el video de la etapa si todos los clips existen
        items = _items_validos_guion(guion)
        clips_por_idx = {}
        for it in items:
            p = _frag_paths(vp, etapa, it["idx"], it["hablante"])["mp4"]
            if os.path.exists(p) and os.path.getsize(p) > 10000:
                clips_por_idx[it["idx"]] = p
        extra = ""
        if len(clips_por_idx) == len(items):
            out_final = _ensamblar_etapa_dialogos(
                vp, etapa, [clips_por_idx[it["idx"]] for it in items], fps)
            _registrar_etapa_dialogos(vp, etapa, items, clips_por_idx, out_final)
            extra = f"\n🎞️ Video de {etapa} reensamblado."
        else:
            extra = (f"\nℹ️ Faltan {len(items)-len(clips_por_idx)} clips: "
                     f"el video de {etapa} se ensambla al ejecutar la etapa.")
        return (f"✅ Fragmento {frag_idx} regenerado ({hablante}, "
                f"{_ffprobe_duration(_frag_paths(vp, etapa, frag_idx, hablante)['wav']) or 0:.1f}s).{extra}")
    except Exception as e:
        import traceback
        log(f"[Regen] ❌ {e}\n{traceback.format_exc()[-1200:]}")
        return f"❌ Error regenerando fragmento: {e}"


# ═══════════════════════════════════════════════════════════════
# v7 — PREVIEW SOLO IMÁGENES · REGENERAR CLIP LIPSYNC INDIVIDUAL · ESTADO PARA EL TIMELINE
# ═══════════════════════════════════════════════════════════════
def _dialogo_preview_fragmento(vp, plantilla_slug, etapa, idx, hablante, sig, fps) -> str:
    # Clip de PREVIEW de un fragmento: imagen fija del hablante + su audio (misma duración que tendrá el lipsync).
    fp = _frag_paths(vp, etapa, idx, hablante)
    img = _imagen_hablante_segura(vp, plantilla_slug, hablante, idx)
    if not img:
        log(f"  [{idx:03d}] ❌ sin imagen para el preview de {hablante}.")
        return ""
    try:
        img_mt = os.path.getmtime(img)
    except Exception:
        img_mt = 0
    sig_p = f"{sig}|{img}|{img_mt:.0f}"
    if (os.path.exists(fp["prev"]) and os.path.getsize(fp["prev"]) > 1000
            and _read_txt(fp["prev_sig"]) == sig_p):
        return fp["prev"]
    _rm(fp["prev"], fp["prev_sig"])
    real_dur = min(_ffprobe_duration(fp["wav"]) or 5.0, DIALOGO_MAX_AUDIO_S)
    if not _clip_estatico_dialogo(img, fp["wav"], fp["prev"], real_dur, fps):
        log(f"  [{idx:03d}] ❌ no se pudo crear el preview de {hablante}.")
        return ""
    _write_txt(fp["prev_sig"], sig_p)
    return fp["prev"]

def _concat_dialogo_clips(clips: list, concat_list: str, out_mp4: str, fps: int = 24) -> str:
    with open(concat_list, "w", encoding="utf-8") as f:
        for c in clips:
            f.write(f"file '{os.path.abspath(c)}'\n")
    cmd = (f"ffmpeg -y -f concat -safe 0 -i '{concat_list}' "
           f"-c:v libx264 -r {fps} -pix_fmt yuv420p -c:a aac -b:a 192k '{out_mp4}' -loglevel error")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("FFmpeg concat (preview) falló:\n" + (r.stderr or "")[-3000:])
    return out_mp4

def _preview_etapa_dialogos(vp, plantilla_slug, etapa, items, sigs, fps=24) -> str:
    # Ensambla {etapa}_preview.mp4. NO toca estado[etapa]["video_final"] (que es el video con lipsync
    # que usa el render final): el preview es solo para validar el guion en audio.
    log(f"[{etapa.upper()}] PREVIEW: {len(items)} fragmentos con imagen fija + audio (sin LTX)...")
    clips = []
    for k, it in enumerate(items):
        c = _dialogo_preview_fragmento(vp, plantilla_slug, etapa, it["idx"], it["hablante"],
                                       sigs[it["idx"]], fps)
        if c:
            clips.append(c)
    if not clips:
        raise RuntimeError(f"No se pudo crear ningún clip de preview para {etapa}.")
    etapa_dir = os.path.join(vp.video_dir, f"{etapa}_dialogos")
    out = os.path.join(etapa_dir, f"{etapa}_preview.mp4")
    _concat_dialogo_clips(clips, os.path.join(etapa_dir, "concat_preview.txt"), out, fps)
    st = vp.estado.setdefault(etapa, {})
    st["video_preview"] = out
    st["n_fragmentos"] = len(items)
    vp.save()
    log(f"[{etapa.upper()}] ✅ PREVIEW listo → {out} ({_ffprobe_duration(out) or 0:.2f}s). "
        f"Si el guion suena bien: «🎬 Continuar con lipsync».")
    return out

def _reensamblar_si_completo(vp, etapa, guion, fps=24) -> str:
    items = _items_validos_guion(guion)
    clips_por_idx = {}
    for it in items:
        p = _frag_paths(vp, etapa, it["idx"], it["hablante"])["mp4"]
        if os.path.exists(p) and os.path.getsize(p) > 10000:
            clips_por_idx[it["idx"]] = p
    if items and len(clips_por_idx) == len(items):
        out_final = _ensamblar_etapa_dialogos(
            vp, etapa, [clips_por_idx[it["idx"]] for it in items], fps)
        _registrar_etapa_dialogos(vp, etapa, items, clips_por_idx, out_final)
        return f"🎞️ Video de {etapa} reensamblado."
    return (f"ℹ️ Faltan {len(items) - len(clips_por_idx)} clips lipsync: el video de {etapa} "
            f"se ensambla al completar la etapa.")

def regenerar_clip_lipsync_fragmento(vp, plantilla_slug, personaje_key, etapa, frag_idx,
                                     ltx_paths, fps=24, fast_mode_tts=False) -> str:
    # Regenera SOLO el clip lipsync (LTX) de UN fragmento, con seed nueva y REUTILIZANDO su audio
    # (el audio solo se rehace si falta o si el texto del guion cambió).
    # Si LTX falla se conserva el clip anterior.
    try:
        guion = (vp.estado.get(etapa, {}) or {}).get("guion_json") or {}
        frags = guion.get("fragmentos", []) or []
        if not frags:
            return f"❌ No hay guion guardado para {etapa}."
        frag_idx = int(frag_idx)
        if not (0 <= frag_idx < len(frags)):
            return f"❌ Índice {frag_idx} fuera de rango (0..{len(frags)-1})."
        if not ltx_paths:
            return "❌ Descarga LTX primero."
        frag = frags[frag_idx]
        texto = str(frag.get("texto", "")).strip()
        if not texto:
            return "❌ El fragmento no tiene texto."
        hablante = _norm_hablante(frag.get("hablante"))
        fp = _frag_paths(vp, etapa, frag_idx, hablante)

        # ── audio: reutilizar si es vigente; si falta o el texto cambió → rehacerlo ──
        slug = vp.estado.get("plantilla_slug", plantilla_slug)
        wav_ok = os.path.exists(fp["wav"]) and os.path.getsize(fp["wav"]) > 8000
        sig_wav = _read_txt(fp["wav_sig"])
        vigentes = {_frag_sig(hablante, texto, False, slug), _frag_sig(hablante, texto, True, slug)}
        if wav_ok and sig_wav in vigentes:
            sig = sig_wav
        else:
            log(f"[Regen lipsync] {etapa} frag {frag_idx}: audio ausente/desactualizado → se rehace.")
            get_contestant_ref_for_tts()
            _w, sig = _dialogo_audio_fragmento(vp, etapa, frag_idx, hablante, texto,
                                               _fnum(frag.get("duracion_estimada_s"), 5.0),
                                               plantilla_slug, fast_mode_tts, force=True)
            try:
                qwen3_tts.unload()
            except Exception:
                pass

        # ── clip: se conserva el anterior hasta que el nuevo salga bien ──
        prev_mp4 = fp["mp4"] + ".prev"
        had_sig, had_fb = _read_txt(fp["mp4_sig"]), os.path.exists(fp["fb"])
        if os.path.exists(fp["mp4"]):
            shutil.copyfile(fp["mp4"], prev_mp4)
        _rm(fp["mp4"], fp["mp4_sig"], fp["fb"])
        log(f"[Regen lipsync] {etapa} frag {frag_idx} ({hablante}): nuevo clip LTX con seed nueva...")
        clip = _dialogo_clip_fragmento(vp, plantilla_slug, personaje_key, etapa, frag_idx,
                                       hablante, ltx_paths, fps, sig, permitir_respaldo=False)
        if not clip:
            if os.path.exists(prev_mp4):
                shutil.move(prev_mp4, fp["mp4"])
                if had_sig: _write_txt(fp["mp4_sig"], had_sig)
                if had_fb:  _write_txt(fp["fb"], "estatico")
            return ("❌ El lipsync LTX falló; se conservó el clip anterior. "
                    "Revisa la consola (run.log) y reintenta.")
        _rm(prev_mp4)

        extra = _reensamblar_si_completo(vp, etapa, guion, fps)
        dur = _ffprobe_duration(fp["wav"]) or 0
        return f"✅ Lipsync del fragmento {frag_idx} regenerado ({hablante}, {dur:.1f}s).\n{extra}"
    except Exception as e:
        import traceback
        log(f"[Regen lipsync] ❌ {e}\n{traceback.format_exc()[-1200:]}")
        return f"❌ Error regenerando lipsync: {e}"

def estado_fragmentos_dialogo(vp, etapa: str) -> list:
    # Estado de cada fragmento del guion para pintar el timeline de la UI.
    # estado: lipsync | fallback | preview | audio | pendiente
    guion = (vp.estado.get(etapa, {}) or {}).get("guion_json") or {}
    slug = vp.estado.get("plantilla_slug", "")
    out = []
    for it in _items_validos_guion(guion):
        fp = _frag_paths(vp, etapa, it["idx"], it["hablante"])
        wav_ok = os.path.exists(fp["wav"]) and os.path.getsize(fp["wav"]) > 8000
        sig_wav = _read_txt(fp["wav_sig"])
        vigentes = {_frag_sig(it["hablante"], it["texto"], False, slug),
                    _frag_sig(it["hablante"], it["texto"], True, slug)}
        desact = bool(wav_ok and sig_wav and sig_wav not in vigentes)
        clip_ok = os.path.exists(fp["mp4"]) and os.path.getsize(fp["mp4"]) > 10000
        prev_ok = os.path.exists(fp["prev"]) and os.path.getsize(fp["prev"]) > 1000
        if clip_ok and os.path.exists(fp["fb"]):
            estado = "fallback"
        elif clip_ok:
            estado = "lipsync"
        elif prev_ok:
            estado = "preview"
        elif wav_ok:
            estado = "audio"
        else:
            estado = "pendiente"
        out.append({
            "idx": it["idx"], "hablante": it["hablante"], "texto": it["texto"],
            "emocion": it["emocion"], "estado": estado, "desactualizado": desact,
            "dur": (_ffprobe_duration(fp["wav"]) if wav_ok else None),
            "audio": fp["wav"] if wav_ok else "",
            "preview": fp["prev"] if prev_ok else "",
            "clip": fp["mp4"] if clip_ok else "",
        })
    return out


log("CELDA 16 COMPLETADA.")

[18:33:57] CELDA 16 COMPLETADA.


In [16]:
# ═══════════════════════════════════════════════════════════════
# CELDA 17: RENDER FINAL + PUBLICACIÓN COMO KAGGLE DATASET
# - Concatena Etapas 2, 3, 4 (según las que estén habilitadas)
#   con el video de la Etapa 3 como eje central (nunca omitible)
# - Verificación frame-perfect con ffprobe antes de la entrega final
# - Publica modelos como Kaggle Dataset propio para persistencia gratuita
# ═══════════════════════════════════════════════════════════════

def render_video_final(
    vp: VideoProject,
    etapas_habilitadas: dict,
    fps: int = 24,
) -> str:
    """
    Ensambla todas las etapas habilitadas en el video final del canal.
    etapas_habilitadas = {"etapa2": True/False, "etapa3": True, "etapa4": True/False}
    """
    parts = []

    for etapa in ["etapa2", "etapa3", "etapa4"]:
        if not etapas_habilitadas.get(etapa, False):
            continue
        v_path = vp.estado.get(etapa, {}).get("video_final", "")
        if not v_path or not os.path.exists(v_path):
            if etapa == "etapa3":
                raise RuntimeError("Etapa 3 no tiene video_final. Ejecútala primero.")
            log(f"[RENDER FINAL] {etapa} omitida (no hay video_final o está deshabilitada).")
            continue
        parts.append(v_path)
        log(f"[RENDER FINAL] Incluida {etapa}: {v_path} ({_ffprobe_duration(v_path):.2f}s)")

    if not parts:
        raise RuntimeError("No hay partes de video para renderizar.")

    out_dir = os.path.join(vp.video_dir)
    concat_list = os.path.join(out_dir, "_final_concat.txt")
    with open(concat_list, "w", encoding="utf-8") as f:
        for p in parts:
            f.write(f"file '{os.path.abspath(p)}'\n")

    out_final = os.path.join(out_dir, "VIDEO_FINAL.mp4")
    cmd = (
        f"ffmpeg -y -f concat -safe 0 -i '{concat_list}' "
        f"-c:v libx264 -r {fps} -pix_fmt yuv420p "
        f"-c:a aac -b:a 192k '{out_final}'"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("FFmpeg render final falló:\n" + (r.stderr or "")[-3000:])

    # ── Verificación con ffprobe ──────────────────────────────────────────────
    v_dur = _ffprobe_duration(out_final) or 0.0
    expected = sum(_ffprobe_duration(p) or 0 for p in parts)
    diff_ms = abs(v_dur - expected) * 1000
    log(f"[RENDER FINAL] OK → {out_final}")
    log(f"  Duración video: {v_dur:.3f}s | Esperada: {expected:.3f}s | Diff: {diff_ms:.1f}ms")
    if diff_ms > 100:
        log(f"  ⚠ Diferencia {diff_ms:.0f}ms superior a 100ms — revisar clips individuales.")

    vp.estado["video_final"] = out_final
    vp.save()
    return out_final

def publicar_modelos_como_dataset(dataset_slug: str = "audiciones-modelos-ia") -> str:
    """
    Publica los modelos GGUF descargados (Qwen-Image-2.1, VAE, encoder, TTS)
    como un Kaggle Dataset privado propio del usuario.
    La próxima sesión, se detectarán en /kaggle/input sin necesitar descargarlos.
    Esto hace que los 20 GB de /kaggle/working NO sean el cuello de botella.
    """
    return publish_dir_as_kaggle_dataset(
        dataset_slug=dataset_slug,
        src_dir=LOCAL_MODELS_DIR,
        title="Modelos IA - Sistema Audiciones",
        update_existing=True,
    )

log("CELDA 17 COMPLETADA.")

[18:34:06] CELDA 17 COMPLETADA.


In [17]:
# ═══ CELDA 17 (v8): COMPATIBILIDAD huggingface_hub — SIN tocar el kernel ═══
# ANTES: pip install "huggingface_hub<1.0" + borrar transformers/gradio de sys.modules.
#   Eso dejaba un huggingface_hub demasiado viejo para el transformers instalado y el
#   subproceso de LTX moría a los ~7 s con:
#       ImportError: cannot import name 'is_offline_mode' from 'huggingface_hub'
#   (la versión anterior estable NO tenía esta celda). Los parches HfFolder para gradio ya
#   existen en la Celda 12b (script LTX) y en la Celda 18 (UI), así que no hace falta bajar la versión.
# AHORA: solo se comprueba que el subproceso LTX pueda importar transformers + mmgp.
#   Si no puede, ensure_ltx_env() (Celda 13) instala huggingface_hub>=1.0 en una carpeta
#   aislada que SOLO ve el subproceso LTX. Qwen-TTS (venv propio) y el kernel no se modifican.
import importlib.metadata as _md

def _ver(pkg):
    try:
        return _md.version(pkg)
    except Exception:
        return "no instalado"

print(f"ℹ️ kernel: huggingface_hub={_ver('huggingface_hub')} | transformers={_ver('transformers')} | gradio={_ver('gradio')}")
try:
    _pp = ensure_ltx_env(force=True)
    print("✅ Entorno LTX-2.5 OK (transformers + mmgp importables en el subproceso)"
          + (f" — huggingface_hub aislado en {_pp}" if _pp else ""))
except Exception as _e:
    print(f"❌ Entorno LTX-2.5 con problemas:\n{_e}")


ℹ️ kernel: huggingface_hub=2.2.0 | transformers=5.16.1 | gradio=6.30.0
[18:34:17] [LTX25] ⚠ Conflicto huggingface_hub/transformers → instalando 'huggingface_hub>=1.5.0,<2.0.0' aislado en /kaggle/tmp/ltx_hf_compat (solo para el subproceso LTX)...
[18:34:45] [LTX25] ✅ Entorno LTX corregido (huggingface_hub aislado, --no-deps).
✅ Entorno LTX-2.5 OK (transformers + mmgp importables en el subproceso) — huggingface_hub aislado en /kaggle/tmp/ltx_hf_compat


In [18]:
# ═══ FIX DEFINITIVO huggingface_hub ═══
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub>=0.34.0,<1.0.0"
], capture_output=True)

for mod in list(sys.modules.keys()):
    if any(x in mod.lower() for x in ["huggingface_hub", "gradio", "transformers"]):
        del sys.modules[mod]

try:
    from huggingface_hub.utils import HfFolder
    print("✅ huggingface_hub>=0.34 compatible con transformers+gradio")
except ImportError:
    print("⚠ HfFolder no encontrado, aplicando patch manual...")
    # El patch de Celda 6 ya lo cubre

✅ huggingface_hub>=0.34 compatible con transformers+gradio


In [19]:
import importlib.metadata as _md, huggingface_hub
print("en memoria:", huggingface_hub.__version__, "| en disco:", _md.version("huggingface_hub"))
from huggingface_hub import hf_hub_download   # reemplaza la función vieja de las celdas 3 y 12
print("✅ hf_hub_download re-enlazado")

en memoria: 0.36.2 | en disco: 0.36.2
✅ hf_hub_download re-enlazado


In [20]:
import subprocess, os

QWEN_PY = "/tmp/qwen3_tts_env/bin/python"

if os.path.exists(QWEN_PY):
    r = subprocess.run([QWEN_PY, "-m", "pip", "install", "-q", "hf_transfer"],
                       capture_output=True, text=True)
    print("venv Qwen:", "✅ OK" if r.returncode == 0 else f"❌ {r.stderr[-500:]}")
else:
    print("⚠ El venv de Qwen aún no existe (ejecuta primero la celda de Qwen)")

r = subprocess.run(["pip", "install", "-q", "hf_transfer"], capture_output=True, text=True)
print("entorno principal:", "✅ OK" if r.returncode == 0 else f"❌ {r.stderr[-500:]}")

venv Qwen: ✅ OK
entorno principal: ✅ OK


In [21]:
import subprocess, time
import gradio as gr

gr.close_all()                                   # cierra todos los servidores Gradio del kernel
time.sleep(2)
subprocess.run("pkill -f frpc_linux_amd64", shell=True)   # elimina túneles que hayan quedado
time.sleep(1)

print(subprocess.run("ss -ltnp | grep -E ':786[0-9]' || echo 'puertos 786x libres ✅'",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("ps aux | grep frpc | grep -v grep || echo 'sin túneles ✅'",
                     shell=True, capture_output=True, text=True).stdout)

puertos 786x libres ✅

sin túneles ✅



In [22]:
import subprocess, sys, os

TARGET = "/kaggle/tmp/ltx_hf_compat"
TF_VER = "5.14.1"
os.makedirs(TARGET, exist_ok=True)

def check():
    env = os.environ.copy()
    env["PYTHONPATH"] = TARGET + os.pathsep + env.get("PYTHONPATH", "")
    r = subprocess.run(
        [sys.executable, "-c",
         "import transformers as t; print(t.__version__); from mmgp import offload, quant_router"],
        capture_output=True, text=True, env=env, timeout=900)
    return r.returncode == 0, ((r.stdout or "") + (r.stderr or ""))

ok, out = check()
if ok and out.strip().splitlines()[0] == TF_VER:
    print(f"✅ Ya fijado: transformers {TF_VER} para el subproceso LTX")
else:
    intentos = [
        ("sin dependencias", ["--no-deps", f"transformers=={TF_VER}"]),
        ("con dependencias", [f"transformers=={TF_VER}", "huggingface_hub>=1.5.0,<2.0.0"]),
    ]
    for etiqueta, specs in intentos:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                            "--no-cache-dir", "--target", TARGET] + specs,
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(f"❌ pip ({etiqueta}):", (r.stderr or r.stdout)[-600:]); continue
        ok, out = check()
        if ok:
            print(f"✅ transformers {out.strip().splitlines()[0]} aislado ({etiqueta})"); break
        print(f"⚠ Import falló ({etiqueta}):\n{out[-800:]}")
    else:
        print("❌ No se pudo fijar. Pégame el último mensaje de arriba.")

⚠ Import falló (sin dependencias):
formers/utils/versions.py", line 116, in require_version_core
    return require_version(requirement, hint)
  File "/kaggle/tmp/ltx_hf_compat/transformers/utils/versions.py", line 110, in require_version
    _compare_versions(op, got_ver, want_ver, requirement, pkg, hint)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/tmp/ltx_hf_compat/transformers/utils/versions.py", line 43, in _compare_versions
    raise ImportError(
        f"{requirement} is required for a normal functioning of this module, but found {pkg}=={got_ver}.{hint}"
    )
ImportError: tokenizers>=0.22.0,<=0.23.0 is required for a normal functioning of this module, but found tokenizers==0.23.1.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

✅ transformers 5.14.1 aislado (con dependencias)


In [24]:
import os, subprocess, hashlib
W = "/kaggle/working/Wan2GP"
r = subprocess.run(["git","-C",W,"rev-parse","--short=10","HEAD"], capture_output=True, text=True)
print("Commit Wan2GP:", r.stdout.strip() or "sin .git", "(validado: 0e58385fbd)")
esperado = {"wgp.py":"a33acb7b9a", "requirements.txt":"23a261e975", "models/ltx2/ltx2_handler.py":"f65f7785f3"}
for rel, exp in esperado.items():
    p = os.path.join(W, rel)
    got = hashlib.md5(open(p,"rb").read()).hexdigest()[:10] if os.path.exists(p) else "NO EXISTE"
    print(f"{'✅' if got==exp else '⚠️'} {rel}: {got} (esperado {exp})")
print(subprocess.run("pip show mmgp | sed -n 1,2p", shell=True, capture_output=True, text=True).stdout)

# Monitor de RAM desacoplado → /kaggle/working/mem_watch.log (cada 3 s)
open("/tmp/memwatch.sh","w").write('''#!/bin/bash
while true; do
  echo "$(date +%T) $(free -m | awk 'NR==2{print "used="$3"MB avail="$7"MB"}') | $(ps -eo rss,comm --sort=-rss | sed -n 2,3p | tr '\\n' ' ')" >> /kaggle/working/mem_watch.log
  sleep 3
done
''')
if subprocess.run(["pgrep","-f","memwatch.sh"], capture_output=True).returncode != 0:
    subprocess.Popen(["bash","/tmp/memwatch.sh"], start_new_session=True,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Monitor activo → /kaggle/working/mem_watch.log")

Commit Wan2GP: b8b18f8114 (validado: 0e58385fbd)
⚠️ wgp.py: c1b567ddf6 (esperado a33acb7b9a)
⚠️ requirements.txt: 6952c33959 (esperado 23a261e975)
⚠️ models/ltx2/ltx2_handler.py: f01208055c (esperado f65f7785f3)
Name: mmgp
Version: 3.8.2

Monitor activo → /kaggle/working/mem_watch.log


In [25]:
# ═══════════════════════════════════════════════════════════════
# CELDA 18 (v7): UI GRADIO — ESTILO AIQUEST + VOZ CONCURSANTE
#
# NOVEDADES v7 (todo lo demás intacto):
#   - 🖼️ Preview con imágenes en Etapas 2 y 4 (audio + imagen, sin lipsync) y
#     🎬 «Continuar con lipsync» (reutiliza guion y audios ya validados)
#   - 🧩 Timeline por fragmentos con botón 🔄 «Regenerar lipsync (LTX)» en Etapa 2, 3 y 4
#   - 🚀 AUTOMATIZACIÓN: E2+E4 juntas · TODO con casillas para excluir etapas · solo canción
#   - 🧪 Botón de diagnóstico del entorno LTX (arregla el conflicto de huggingface_hub)
#
# CAMBIOS vs v5:
#   - Nueva sección "🎙️ Voz del Concursante" encima de Etapas 2 y 4:
#       · Subir audio → auto-transcribir → preview clonado → validar
#       · O bien: escribir descripción → VoiceDesign preview → adoptar como ref
#       · La voz adoptada clona consistentemente todo el guion
#   - Estado de voz del concursante visible en panel de progreso
#   - Auto-transcripción de jueces al arrancar Etapa 2/4
#   - Preview de audio de prueba antes de comprometerse con una voz
#   - Botón "Transcribir jueces ahora" para verificar antes de ejecutar
#   - NINGÚN cambio en funcionalidades de imagen o LTX/video
# ═══════════════════════════════════════════════════════════════


# ── HfFolder backward-compatibility patch ───────────────────
try:
    import huggingface_hub.utils as hf_utils
    if not hasattr(hf_utils, 'HfFolder'):
        class HfFolder:
            @staticmethod
            def get_token():
                try:
                    from huggingface_hub import get_token
                    return get_token() or os.environ.get('HF_TOKEN')
                except Exception:
                    return os.environ.get('HF_TOKEN')
            @staticmethod
            def save_token(token):
                pass
        setattr(hf_utils, 'HfFolder', HfFolder)
        if 'huggingface_hub.utils' in sys.modules:
            setattr(sys.modules['huggingface_hub.utils'], 'HfFolder', HfFolder)
        print("✅ HfFolder patch aplicado solo para compatibilidad legacy del proyecto")
    else:
        print(f"✅ HfFolder ya disponible (huggingface_hub {hf_utils.__name__})")
except Exception as e:
    print(f"⚠ HfFolder patch warning: {e}")



import gradio as gr
import sys
import os
import time
import json
import shutil
import re

# ── Estado global ─────────────────────────────────────────────
_UI_STATE    = load_ui_state()
_CUR_VP_DIR  = _UI_STATE.get("current_video_dir", "")
_CUR_TPL     = _UI_STATE.get("current_template_slug", "")
_LTX_PATHS   = globals().get("_LTX_PATHS", {})

def _get_vp() -> VideoProject:
    if not _CUR_VP_DIR or not os.path.isdir(_CUR_VP_DIR):
        raise gr.Error("No hay video activo. Crea un proyecto primero.")
    return VideoProject(_CUR_VP_DIR)

def _set_vp(vp_dir: str, tpl_slug: str):
    global _CUR_VP_DIR, _CUR_TPL
    _CUR_VP_DIR = vp_dir
    _CUR_TPL    = tpl_slug
    _UI_STATE["current_video_dir"]     = vp_dir
    _UI_STATE["current_template_slug"] = tpl_slug
    save_ui_state(_UI_STATE)


# ═══════════════════════════════════════════════════════════════
# HANDLERS ETAPA 1 (sin cambios — imagen/LTX estable)
# ═══════════════════════════════════════════════════════════════

def _etapa1_common_params():
    vp = _get_vp()
    params = vp.params.get("etapa1", {})
    return {
        "vp": vp,
        "tpl": _CUR_TPL,
        "diffusion_quant": params.get("diffusion_quant", "Q8_0"),
        "encoder_quant": params.get("encoder_quant", "Q4_K_M"),
        "mmproj_quant": params.get("mmproj_quant", "Q8_0"),
        "steps_A": int(params.get("steps_A", 30)),
        "steps_edit": int(params.get("steps_edit", 30)),
        "cfg": float(params.get("cfg", 2.0)),
        "seed": int(params.get("seed", -1)),
    }


def ui_etapa1_paso(key, upload_file, personaje, p_reemplazar, edad, vestuario, planos):
    """
    Genera UNA sola imagen de Etapa 1, o usa la subida manualmente.
    key: A, B, C, D, E, F, G
    """
    try:
        cp = _etapa1_common_params()
        vp, tpl = cp["vp"], cp["tpl"]
        if not tpl:
            return f"❌ Selecciona plantilla primero.", None

        img_dir = os.path.join(vp.video_dir, "etapa1_imagenes")
        os.makedirs(img_dir, exist_ok=True)

        fname_map = {
            "A": "concursante_base.png",
            "B": "concursante_cuerpo_completo.png",
            "C": "concursante_plano_cercano.png",
            "D": "concursante_closeup.png",
            "E": "concursante_plano_lejano_1.png",
            "F": "concursante_plano_lejano_2.png",
            "G": "concursante_plano_lejano_3.png",
        }
        out_path = os.path.join(img_dir, fname_map[key])

        # ── UPLOAD MANUAL: si hay archivo, copiar y saltar generación ──
        if upload_file is not None:
            src = upload_file if isinstance(upload_file, str) else getattr(upload_file, "name", "")
            if src and os.path.exists(src):
                shutil.copyfile(src, out_path)
                vp.estado.setdefault("aprobados", {})[f"etapa1_{key}"] = True
                vp.save()
                log(f"[ETAPA1] {key} subida manualmente → {out_path}")
                return f"✅ {key} cargada manualmente.", out_path
            else:
                return f"⚠ Archivo inválido para {key}.", None

        # ── GENERACIÓN CON IA ──
        mp = ensure_qwen_image21_models(cp["diffusion_quant"], cp["encoder_quant"], cp["mmproj_quant"])
        sd = SDQwenImage21(SDCLI_PATH, mp)

        W_9_16, H_9_16 = 832, 1472
        W_16_9, H_16_9 = 1376, 768

        if key == "A":
            p1 = render_prompt_1(personaje, int(edad), str(vestuario))
            sd.generate_text2img(p1, out_path, W_9_16, H_9_16, cp["steps_A"], cp["cfg"], seed=cp["seed"])

        elif key == "B":
            ref_a = os.path.join(img_dir, fname_map["A"])
            if not os.path.exists(ref_a):
                return "❌ Necesitas la imagen A primero.", None
            sd.edit_with_refs(PROMPT_2_CUERPO_COMPLETO, [ref_a], out_path, W_9_16, H_9_16, cp["steps_edit"], cp["cfg"], seed=cp["seed"])

        elif key in ("C", "E", "F", "G"):
            ref_b = os.path.join(img_dir, fname_map["B"])
            if not os.path.exists(ref_b):
                return "❌ Necesitas la imagen B primero.", None
            if key == "C":
                tpl_img = os.path.join(cargar_plantilla(tpl)["_base_dir"], PLANTILLA_SUBDIRS["escenario_cercano"], "plantilla_plano_cercano.png")
            else:
                idx = {"E": 1, "F": 2, "G": 3}[key]
                tpl_img = os.path.join(cargar_plantilla(tpl)["_base_dir"], PLANTILLA_SUBDIRS["escenario_lejano"], f"plantilla_plano_lejano_{idx}.png")
            if not os.path.exists(tpl_img):
                return f"❌ Falta plantilla: {os.path.basename(tpl_img)}", None
            p3 = render_prompt_3(personaje, p_reemplazar)
            sd.edit_with_refs(p3, [ref_b, tpl_img], out_path, W_16_9, H_16_9, cp["steps_edit"], cp["cfg"], seed=cp["seed"])

        elif key == "D":
            ref_c = os.path.join(img_dir, fname_map["C"])
            if not os.path.exists(ref_c):
                return "❌ Necesitas la imagen C primero.", None
            sd.edit_with_refs(PROMPT_4_CLOSEUP, [ref_c], out_path, W_16_9, H_16_9, cp["steps_edit"], cp["cfg"], seed=cp["seed"])

        vp.estado.setdefault("aprobados", {})[f"etapa1_{key}"] = os.path.exists(out_path)
        vp.save()
        log(f"[ETAPA1] {key} generada → {out_path}")
        return f"✅ Imagen {key} generada.", out_path

    except Exception as e:
        return f"❌ Error {key}: {e}", None

def ui_etapa1_continuar_desde(start_key, personaje, p_reemplazar, edad, vestuario, planos):
    keys_order = ["A", "B", "C", "D", "E", "F", "G"]
    try:
        idx_start = keys_order.index(start_key)
    except ValueError:
        return f"❌ Key inválida: {start_key}"
    resultados = []
    for k in keys_order[idx_start:]:
        img_dir = os.path.join(_get_vp().video_dir, "etapa1_imagenes")
        fname_map = {"A":"concursante_base.png","B":"concursante_cuerpo_completo.png",
                     "C":"concursante_plano_cercano.png","D":"concursante_closeup.png",
                     "E":"concursante_plano_lejano_1.png","F":"concursante_plano_lejano_2.png",
                     "G":"concursante_plano_lejano_3.png"}
        existing = os.path.join(img_dir, fname_map[k])
        if os.path.exists(existing) and os.path.getsize(existing) > 5000:
            resultados.append(f"{k}: ✅ ya existe")
            continue
        msg, _ = ui_etapa1_paso(k, None, personaje, p_reemplazar, edad, vestuario, planos)
        resultados.append(f"{k}: {msg}")
        if "❌" in msg:
            break
    return "\n".join(resultados)

def ui_etapa1_todo(personaje, p_reemplazar, edad, vestuario, planos,
                   dq, eq, mq, stA, stE, cfg_v, seed_v):
    if not _CUR_TPL:
        return "❌ Selecciona plantilla.", []
    try:
        vp = _get_vp()
        vp.params.setdefault("etapa1", {}).update({
            "edad": int(edad), "vestuario": str(vestuario),
            "personaje": personaje, "personaje_plantilla": p_reemplazar,
            "planos_lejanos_a_usar": int(planos),
            "diffusion_quant": dq, "encoder_quant": eq, "mmproj_quant": mq,
            "steps_A": int(stA), "steps_edit": int(stE),
            "cfg": float(cfg_v), "seed": int(seed_v),
        })
        vp.save()
        paths = etapa1_generar_imagenes(
            vp, _CUR_TPL, personaje, p_reemplazar,
            edad=int(edad), vestuario=str(vestuario),
            planos_lejanos_a_usar=int(planos),
            diffusion_quant=dq, encoder_quant=eq, mmproj_quant=mq,
            steps_A=int(stA), steps_edit=int(stE),
            cfg=float(cfg_v), seed=int(seed_v),
        )
        gallery = [paths[k] for k in ["A","B","C","D","E","F","G"]
                   if k in paths and os.path.exists(paths[k])]
        return "✅ Etapa 1 completa.", gallery
    except Exception as e:
        return f"❌ Error: {e}", []


# ═══════════════════════════════════════════════════════════════
# HANDLERS VOZ DEL CONCURSANTE (NUEVOS — Celda 18 v6)
# ═══════════════════════════════════════════════════════════════

def ui_upload_contestant_audio(audio_file):
    """Procesa el audio subido: convierte + transcribe + actualiza estado."""
    if audio_file is None:
        return "❌ No se recibió archivo de audio.", "", "", gr.update(interactive=False)
    src = audio_file if isinstance(audio_file, str) else getattr(audio_file, "name", "")
    if not src or not os.path.exists(src):
        return "❌ Archivo inválido.", "", "", gr.update(interactive=False)
    texto, idioma, msg = process_contestant_upload(src)
    return msg, texto, idioma, gr.update(interactive=bool(texto))


def ui_generate_clone_preview(ref_text_box, preview_text_box):
    """Genera preview clonando el audio subido del concursante."""
    ref_text   = ref_text_box or CONTESTANT_VOICE_STATE.get("ref_text", "")
    prev_text  = preview_text_box.strip() if preview_text_box and preview_text_box.strip() else PREVIEW_TEXT_ES
    wav_path, msg = generate_contestant_clone_preview(
        ref_wav=CONTESTANT_REF_WAV,
        ref_text=ref_text,
        preview_text=prev_text,
    )
    return msg, wav_path if wav_path and os.path.exists(wav_path) else None


def ui_generate_design_preview(design_instruct_box, preview_text_box):
    """Genera preview con VoiceDesign desde descripción texto."""
    instruct  = design_instruct_box.strip() if design_instruct_box else ""
    prev_text = preview_text_box.strip() if preview_text_box and preview_text_box.strip() else PREVIEW_TEXT_ES
    if not instruct:
        return "❌ Escribe una descripción de voz.", None
    wav_path, msg = generate_voice_design_preview(instruct, prev_text)
    return msg, wav_path if wav_path and os.path.exists(wav_path) else None


def ui_adopt_design_as_ref(design_instruct_box):
    """Adopta el preview de VoiceDesign como referencia de clonación."""
    instruct = design_instruct_box.strip() if design_instruct_box else ""
    ref_wav, msg = adopt_design_as_reference(instruct)
    return msg, gr.update(interactive=True)


def ui_validate_contestant_voice():
    """Marca la voz como validada."""
    ok, msg = validate_contestant_voice()
    color = "✅" if ok else "❌"
    return msg


def ui_transcribe_judges(tpl_slug):
    """Transcribe los audios de todos los jueces en la plantilla."""
    if not tpl_slug:
        return "❌ Selecciona una plantilla primero."
    resultados = []
    for i in range(1, 5):
        wav = get_juez_ref_wav(tpl_slug, i)
        if not wav or not os.path.exists(wav):
            resultados.append(f"Juez {i}: ⚠ sin audio en plantilla")
            continue
        # Verificar si ya existe transcripción
        existing = get_juez_ref_text(tpl_slug, i)
        if existing:
            resultados.append(f"Juez {i}: ✅ ya transcrito → «{existing[:60]}...»")
            continue
        texto = auto_transcribe_juez(tpl_slug, i)
        if texto:
            resultados.append(f"Juez {i}: ✅ «{texto[:70]}...»")
        else:
            resultados.append(f"Juez {i}: ❌ transcripción fallida")
    return "\n".join(resultados)


def ui_show_voice_state():
    """Muestra el estado actual de la voz del concursante."""
    state = load_contestant_voice_state()
    lines = [
        f"Modo: {state.get('mode') or 'no configurado'}",
        f"Referencia: {os.path.basename(state.get('ref_wav_path','')) or 'ninguna'}",
        f"Transcripción: «{state.get('ref_text','')[:100]}{'...' if len(state.get('ref_text',''))>100 else ''}»",
        f"Diseño: {state.get('design_instruct','')[:80] or 'N/A'}",
        f"✅ VALIDADA: {state.get('validated', False)}",
    ]
    return "\n".join(lines)


# ═══════════════════════════════════════════════════════════════
# HANDLERS CLIPS 5s Y DEMÁS (sin cambios vs v5)
# ═══════════════════════════════════════════════════════════════

def ui_gen_5s_individual(img_key, personaje, seed_v):
    """Genera UN clip de 5s. Retorna (status, gallery_list)."""
    try:
        vp = _get_vp()
        if not _LTX_PATHS:
            return "❌ Descarga primero los modelos LTX.", []

        img_dir = os.path.join(vp.video_dir, "etapa1_imagenes")
        fname_map = {
            "E": "concursante_plano_lejano_1.png",
            "F": "concursante_plano_lejano_2.png",
            "G": "concursante_plano_lejano_3.png",
        }
        if img_key not in fname_map:
            return f"❌ Key inválida: {img_key}", []

        img_path = os.path.join(img_dir, fname_map[img_key])
        if not os.path.exists(img_path):
            return f"❌ Imagen {img_key} no existe. Genera Etapa 1 primero.", []

        out_dir = os.path.join(vp.video_dir, "etapa3_cancion", "planos_5s")
        os.makedirs(out_dir, exist_ok=True)

        prompt_str, direccion = render_prompt_5s(personaje)
        out_mp4 = os.path.join(out_dir, f"far_{img_key}_v1.mp4")

        generar_clip_5s(
            vp, img_path, prompt_str, out_mp4, _LTX_PATHS,
            seed=int(seed_v), fps=24, width=1152, height=704,  # 1280×704, 1152×704, 1920×1088
        )

        vp.estado.setdefault("etapa3", {}).setdefault("clips_5s", [])
        if out_mp4 not in vp.estado["etapa3"]["clips_5s"]:
            vp.estado["etapa3"]["clips_5s"].append(out_mp4)
        vp.save()

        log(f"[5s] {img_key} generado → {out_mp4}")

        # ── Gallery acepta list[str] de imágenes o list[tuple(path,caption)] ──
        # Para video, devolver la imagen de referencia usada como preview
        gallery_items = []
        if os.path.exists(img_path):
            gallery_items.append((img_path, f"Ref {img_key}"))
        # Añadir frame preview del video generado si existe
        preview_png = out_mp4.replace(".mp4", "_preview.png")
        if not os.path.exists(preview_png) and os.path.exists(out_mp4):
            try:
                import subprocess as _sp
                _sp.run(
                    f"ffmpeg -y -i '{out_mp4}' -vframes 1 -q:v 2 '{preview_png}'",
                    shell=True, capture_output=True, timeout=10
                )
            except Exception:
                pass
        if os.path.exists(preview_png):
            sz_kb = os.path.getsize(out_mp4) // 1024
            gallery_items.append((preview_png, f"Frame 1 — {sz_kb}KB"))

        return (f"✅ Clip 5s {img_key} generado ({direccion}) "
                f"→ {os.path.basename(out_mp4)}"), gallery_items

    except Exception as e:
        import traceback
        log(f"[UI gen_5s_{img_key}] ❌ {e}\n{traceback.format_exc()[-1000:]}")
        return f"❌ Error: {e}", []


def ui_gen_5s_todos(personaje, seed_v):
    """Genera todos los clips 5s. Retorna (status, gallery_list)."""
    try:
        vp = _get_vp()
        if not _LTX_PATHS:
            return "❌ Descarga primero los modelos LTX.", []

        clips = etapa3_paso1_generar_clips_5s(
            vp, _CUR_TPL, personaje, _LTX_PATHS, seed=int(seed_v)
        )

        # Gallery: previews de todos los clips generados
        gallery_items = []
        for clip_path in clips:
            preview = clip_path.replace(".mp4", "_preview.png")
            if not os.path.exists(preview) and os.path.exists(clip_path):
                try:
                    import subprocess as _sp
                    _sp.run(
                        f"ffmpeg -y -i '{clip_path}' -vframes 1 -q:v 2 '{preview}'",
                        shell=True, capture_output=True, timeout=10
                    )
                except Exception:
                    pass
            if os.path.exists(preview):
                sz_kb = os.path.getsize(clip_path) // 1024 if os.path.exists(clip_path) else 0
                gallery_items.append((preview, f"{os.path.basename(clip_path)} — {sz_kb}KB"))
            elif os.path.exists(clip_path.replace("far_", "").replace(".mp4", ".png")):
                gallery_items.append(clip_path.replace("far_", "").replace(".mp4", ".png"))

        return f"✅ {len(clips)} clips 5s generados.", gallery_items

    except Exception as e:
        return f"❌ Error: {e}", []


# ═══════════════════════════════════════════════════════════════
# RESTO DE HANDLERS (sin cambios vs v5)
# ═══════════════════════════════════════════════════════════════

def ui_refresh_tpl():
    opts = listar_plantillas()
    choices = [(n, s) for (n, s) in opts]
    val = _CUR_TPL if _CUR_TPL else (choices[0][1] if choices else None)
    return gr.update(choices=choices, value=val)

def ui_crear_plantilla(nombre):
    if not nombre or not nombre.strip():
        return "❌ Escribe un nombre.", gr.update()
    base = crear_plantilla(nombre.strip())
    return f"✅ Plantilla creada: {os.path.basename(base)}", ui_refresh_tpl()

def ui_subir_assets_juez_independiente(tpl_slug, j1_img, j1_audios, j2_img, j2_audios,
                                        j3_img, j3_audios, j4_img, j4_audios,
                                        esc_cercano, escs_lejanos, reac1, reac2):
    if not tpl_slug:
        return "❌ Selecciona plantilla."
    try:
        tpl = cargar_plantilla(tpl_slug)
        base = tpl["_base_dir"]
        resultados = []
        img_dir = os.path.join(base, PLANTILLA_SUBDIRS["jueces_imagenes"])
        voz_dir = os.path.join(base, PLANTILLA_SUBDIRS["voces_jueces"])
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(voz_dir, exist_ok=True)
        for j_num, img_f, aud_fs in [(1,j1_img,j1_audios),(2,j2_img,j2_audios),
                                      (3,j3_img,j3_audios),(4,j4_img,j4_audios)]:
            if img_f is not None:
                src = img_f if isinstance(img_f, str) else getattr(img_f, "name", "")
                if src and os.path.exists(src):
                    shutil.copyfile(src, os.path.join(img_dir, f"juez_{j_num}.png"))
                    resultados.append(f"Juez {j_num}: imagen ✅")
            if aud_fs is not None:
                if not isinstance(aud_fs, list): aud_fs = [aud_fs]
                wavs = []
                for af in aud_fs:
                    if af is None: continue
                    src = af if isinstance(af, str) else getattr(af, "name", "")
                    if src and os.path.exists(src):
                        tmp = os.path.join(voz_dir, f"j{j_num}_tmp_{len(wavs)}.wav")
                        ffmpeg_to_wav_mono_24k(src, tmp)
                        wavs.append(tmp)
                if wavs:
                    dst = os.path.join(voz_dir, f"juez_{j_num}.wav")
                    if len(wavs) == 1:
                        shutil.copyfile(wavs[0], dst)
                    else:
                        cl = os.path.join(voz_dir, f"j{j_num}_c.txt")
                        with open(cl, "w") as f:
                            for w in wavs: f.write(f"file '{os.path.abspath(w)}'\n")
                        subprocess.run(
                            f"ffmpeg -y -f concat -safe 0 -i '{cl}' "
                            f"-ac 1 -ar 24000 -acodec pcm_s16le '{dst}'",
                            shell=True, capture_output=True)
                        try: os.remove(cl)
                        except: pass
                    for w in wavs:
                        try:
                            if w != dst and os.path.exists(w): os.remove(w)
                        except: pass
                    # Invalidar transcripción anterior del juez (si existe)
                    txt_old = os.path.join(JUECES_REF_DIR, f"juez_{j_num}_ref.txt")
                    if os.path.exists(txt_old):
                        try: os.remove(txt_old)
                        except: pass
                    resultados.append(f"Juez {j_num}: {len(wavs)} audio(s) ✅ (transcripción pendiente)")
        cdir = os.path.join(base, PLANTILLA_SUBDIRS["escenario_cercano"])
        os.makedirs(cdir, exist_ok=True)
        if esc_cercano:
            src = esc_cercano if isinstance(esc_cercano, str) else esc_cercano.name
            if src and os.path.exists(src):
                shutil.copyfile(src, os.path.join(cdir, "plantilla_plano_cercano.png"))
                resultados.append("Escenario cercano ✅")
        fdir = os.path.join(base, PLANTILLA_SUBDIRS["escenario_lejano"])
        os.makedirs(fdir, exist_ok=True)
        if escs_lejanos:
            if not isinstance(escs_lejanos, list): escs_lejanos = [escs_lejanos]
            for i, ef in enumerate(escs_lejanos, 1):
                if ef and i <= 3:
                    src = ef if isinstance(ef, str) else getattr(ef, "name", "")
                    if src and os.path.exists(src):
                        shutil.copyfile(src, os.path.join(fdir, f"plantilla_plano_lejano_{i}.png"))
            resultados.append("Escenarios lejanos ✅")
        r1d = os.path.join(base, PLANTILLA_SUBDIRS["reacciones_primera_mitad"])
        r2d = os.path.join(base, PLANTILLA_SUBDIRS["reacciones_segunda_mitad"])
        os.makedirs(r1d, exist_ok=True); os.makedirs(r2d, exist_ok=True)
        for rf_list, rd in [(reac1, r1d), (reac2, r2d)]:
            if rf_list:
                if not isinstance(rf_list, list): rf_list = [rf_list]
                for rf in rf_list:
                    if rf is None: continue
                    src = rf if isinstance(rf, str) else getattr(rf, "name", "")
                    if src and os.path.exists(src):
                        shutil.copyfile(src, os.path.join(rd, os.path.basename(src)))
        meta = cargar_plantilla(tpl_slug)
        meta["planos_lejanos_disponibles"] = sum(
            1 for i in [1,2,3] if os.path.exists(
                os.path.join(base, PLANTILLA_SUBDIRS["escenario_lejano"], f"plantilla_plano_lejano_{i}.png")))
        guardar_plantilla(meta)
        return "✅ Assets guardados:\n" + "\n".join(resultados)
    except Exception as e:
        return f"❌ Error: {e}"

def ui_nuevo_video(tpl_slug, personaje):
    if not tpl_slug:
        return "❌ Selecciona plantilla.", "", gr.update()
    try:
        vp = create_new_video_project(tpl_slug, personaje)
        vp.estado["plantilla_slug"] = tpl_slug
        vp.estado["personaje"] = personaje
        vp.estado["creado_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        vp.save()
        _set_vp(vp.video_dir, tpl_slug)
        return f"✅ Video activo: {os.path.basename(vp.video_dir)}", vp.video_dir, ui_refresh_tpl()
    except Exception as e:
        return f"❌ Error: {e}", "", gr.update()


def ui_descargar_ltx(preset_label: str):
    global _LTX_PATHS, _LTX_ENGINE
    try:
        if not preset_label or preset_label not in LTX23_PRESETS:
            return f"❌ Preset inválido: '{preset_label}'"

        log(f"[UI] Descargando modelos LTX-2.5 — preset: {preset_label}")
        paths = ensure_ltx23_models(preset_label)

        if not paths or not isinstance(paths, dict):
            return "❌ ensure_ltx23_models() no devolvió rutas válidas."

        _LTX_PATHS = paths

        # Motor LTX ahora es Wan2GP (Celda 13 nueva). No crear SDLTXVideo23.
        # Verificar que el backend esté disponible:
        try:
            backend = get_ltx_backend()
            if not backend.is_available():
                log("[UI] ⚠ Backend LTX-2.5 no disponible. Verifica Celda 13.")
        except Exception:
            log("[UI] ⚠ Backend LTX-2.5 no disponible. Verifica Celda 13.")

        p = _LTX_PATHS
        lineas = [
            "✅ LTX-2.5 listo:",
            f"  Preset:      {p.get('_preset', preset_label)}",
            f"  Transformer: {os.path.basename(p.get('transformer', '?'))}",
            f"  Encoder:     {os.path.basename(p.get('encoder', '?'))}",
            f"  Video VAE:   {os.path.basename(p.get('video_vae', '?'))}",
            f"  Audio VAE:   {os.path.basename(p.get('audio_vae', '?'))}",
            f"  Embeddings:  {os.path.basename(p.get('embeddings_connector', '?'))}",
            f"  VRAM est.:   {p.get('_vram_estimado', 0):.0f} GB",
            f"  Steps:       {p.get('_steps', 8)} | CFG: {p.get('_cfg', 1.0)}",
            f"  /tmp libre:  {free_gb('/kaggle/tmp'):.1f} GB",
        ]
        return "\n".join(lineas)

    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        log(f"[UI] Error en ui_descargar_ltx:\n{tb}")
        return f"❌ Error: {e}"

def ui_regenerar_imagen(tpl, personaje, p_reemplazar, img_key, seed_v):
    try:
        vp = _get_vp(); tpl = tpl or _CUR_TPL
        params = vp.params.get("etapa1", {})
        paths = etapa1_generar_imagenes(
            vp, tpl, personaje, p_reemplazar,
            edad=int(params.get("edad", 28)),
            vestuario=params.get("vestuario", "vestido elegante de gala"),
            planos_lejanos_a_usar=int(params.get("planos_lejanos_a_usar", 3)),
            diffusion_quant=params.get("diffusion_quant","Q4_K"),
            encoder_quant=params.get("encoder_quant","Q4_K_M"),
            mmproj_quant=params.get("mmproj_quant","Q8_0"),
            steps_A=int(params.get("steps_A",18)),
            steps_edit=int(params.get("steps_edit",18)),
            cfg=float(params.get("cfg",6.0)), seed=int(seed_v))
        out = paths.get(img_key)
        return f"✅ {img_key} regenerada.", out if out and os.path.exists(out) else None
    except Exception as e:
        return f"❌ Error: {e}", None

def ui_subir_cancion(song_file):
    if not song_file: return "❌ No se subió audio."
    try:
        vp = _get_vp()
        src = song_file if isinstance(song_file, str) else song_file.name
        out = os.path.join(vp.video_dir, "etapa3_cancion", "song.wav")
        os.makedirs(os.path.dirname(out), exist_ok=True)
        ffmpeg_to_wav_mono_24k(src, out)
        vp.estado.setdefault("etapa3", {})["song_wav"] = out
        vp.save()
        dur = _ffprobe_duration(out) or 0
        return f"✅ Canción lista: {out} ({dur:.1f}s)"
    except Exception as e:
        return f"❌ Error: {e}"

def ui_regen_5s(tpl, personaje, clip_path, seed_v):
    try:
        vp = _get_vp(); tpl = tpl or _CUR_TPL
        if not _LTX_PATHS: return "❌ Descarga LTX.", None
        new = etapa3_regenerar_clip_5s(vp, clip_path, personaje, _LTX_PATHS, seed=int(seed_v))
        return "✅ Clip 5s regenerado.", new
    except Exception as e: return f"❌ Error: {e}", None

def ui_build_tl(tpl, personaje):
    try:
        tpl = tpl or _CUR_TPL
        result, html = ui_build_timeline(tpl, personaje)
        return result, html
    except Exception as e:
        return f"❌ Error: {e}", ""

def ui_renderizar_cancion(tpl, personaje):
    try:
        vp = _get_vp(); tpl = tpl or _CUR_TPL
        if not _LTX_PATHS: return "❌ Descarga LTX.", None
        out = etapa3_paso3_renderizar_secuencia(vp, tpl, personaje, _LTX_PATHS)
        return f"✅ Canción renderizada: {os.path.basename(out)}", out
    except Exception as e:
        return f"❌ Error: {e}", None

def ui_regen_5s(tpl, personaje, clip_path, seed_v):
    try:
        vp = _get_vp(); tpl = tpl or _CUR_TPL
        if not _LTX_PATHS: return "❌ Descarga LTX.", None
        new = etapa3_regenerar_clip_5s(vp, clip_path, personaje, _LTX_PATHS, seed=int(seed_v))
        return "✅ Clip 5s regenerado.", new
    except Exception as e:
        return f"❌ Error: {e}", None

import threading
_GPU_LOCK = threading.Lock()      # v7: evita lanzar dos tareas de GPU a la vez (LTX/Qwen)
_BUSY_MSG = "⏳ Hay otra tarea de GPU en curso (generación o regeneración). Espera a que termine."

def _ui_run_etapa(etapa, tpl, personaje, historia, model_id, fast_mode_tts, modo, solo_preview=False):
    # modo: "nuevo" → guion NUEVO (xKiro) | "actual" → usa el guion guardado/editado |
    #       "auto"  → reutiliza el guion si existe, si no lo genera
    # solo_preview=True → TTS + video SOLO con imágenes (no usa LTX ni lipsync)
    if not _GPU_LOCK.acquire(blocking=False):
        return _BUSY_MSG, None
    try:
        vp = _get_vp(); tpl = tpl or _CUR_TPL
        if not _LTX_PATHS and not solo_preview:
            return "❌ Descarga LTX.", None
        out = ejecutar_etapa_dialogos(
            vp, tpl, personaje, etapa, _LTX_PATHS or {},
            historia_base=historia or "", model_id=model_id,
            fast_mode_tts=bool(fast_mode_tts),
            regenerar_guion=(modo == "nuevo"),
            usar_guion_actual=(modo == "actual"),
            solo_preview=bool(solo_preview),
        )
        origen = {"nuevo": "guion NUEVO", "actual": "guion ACTUAL"}.get(modo, "guion existente o nuevo")
        if solo_preview:
            return (f"✅ {etapa.capitalize()} PREVIEW listo ({origen}): audio + imágenes, sin lipsync.\n"
                    f"Escúchalo; si el guion suena bien pulsa «🎬 Continuar con lipsync».", out)
        return f"✅ {etapa.capitalize()} completada ({origen}).", out
    except Exception as e:
        import traceback
        log(f"[UI {etapa}] ❌ {e}\n{traceback.format_exc()[-1500:]}")
        return f"❌ Error: {e}", None
    finally:
        _GPU_LOCK.release()

def ui_etapa2(tpl, personaje, historia, model_id, fast_mode_tts):
    return _ui_run_etapa("etapa2", tpl, personaje, historia, model_id, fast_mode_tts, "nuevo")

def ui_etapa2_guion_actual(tpl, personaje, historia, model_id, fast_mode_tts):
    return _ui_run_etapa("etapa2", tpl, personaje, historia, model_id, fast_mode_tts, "actual")

def ui_etapa4(tpl, personaje, model_id, fast_mode_tts):
    return _ui_run_etapa("etapa4", tpl, personaje, "", model_id, fast_mode_tts, "nuevo")

def ui_etapa4_guion_actual(tpl, personaje, model_id, fast_mode_tts):
    return _ui_run_etapa("etapa4", tpl, personaje, "", model_id, fast_mode_tts, "actual")

def ui_generar_solo_guion(etapa, tpl, personaje, historia, model_id):
    # Genera el guion (sin TTS ni video) y lo deja cargado en el editor.
    try:
        vp = _get_vp()
        data = generar_guion_con_respaldo(vp, personaje, etapa,
                                          historia_base=historia or "", model_id=model_id)
        n = len(data.get("fragmentos", []))
        dur = _fnum(data.get("duracion_total_estimada_min"))
        msg = (f"✅ Guion {etapa} generado y guardado: {n} fragmentos (~{dur:.1f} min).\n"
               f"Revísalo/edítalo en '✏️ EDITAR GUION MANUALMENTE' y luego pulsa "
               f"'⏯️ Generar con guion actual'.")
        return msg, json.dumps(data, ensure_ascii=False, indent=2), etapa
    except Exception as e:
        import traceback
        log(f"[UI guion {etapa}] ❌ {e}\n{traceback.format_exc()[-1200:]}")
        return f"❌ Error generando guion: {e}", gr.update(), gr.update()

def ui_regen_frag(tpl, personaje, etapa_sel, frag_idx, nuevo_texto):
    try:
        vp = _get_vp(); tpl = tpl or _CUR_TPL
        if not _LTX_PATHS: return "❌ Descarga LTX."
        return regenerar_fragmento_dialogo(
            vp, tpl, personaje, etapa_sel,
            int(frag_idx), nuevo_texto=nuevo_texto,
            ltx_paths=_LTX_PATHS,
        )
    except Exception as e:
        return f"❌ Error: {e}"

def ui_render_final(tpl, etapa2_on, etapa4_on):
    try:
        vp = _get_vp()
        out = render_video_final(vp, {"etapa2": bool(etapa2_on), "etapa3": True, "etapa4": bool(etapa4_on)})
        return f"✅ Video final: {os.path.basename(out)}", out
    except Exception as e:
        return f"❌ Error: {e}", None

def ui_publicar_dataset():
    try: return publicar_modelos_como_dataset()
    except Exception as e: return f"❌ Error: {e}"

def ui_ver_estado():
    try:
        vp = _get_vp()
        voice_st = load_contestant_voice_state()
        lineas = [
            f"Video: {os.path.basename(vp.video_dir)}",
            f"Aprobados: {vp.estado.get('aprobados',{})}",
            f"Timeline: {len(vp.timeline_items)} clips",
            f"",
            f"🎙️ Voz concursante: {voice_st.get('mode') or 'no config'} | validada={voice_st.get('validated',False)}",
            f"   Ref: {os.path.basename(voice_st.get('ref_wav_path','')) or 'ninguna'}",
        ]
        for et in ["etapa2", "etapa3", "etapa4"]:
            lineas.append(f"{et}: {vp.estado.get(et,{}).get('video_final','(sin video)')}")
        lineas.append(f"FINAL: {vp.estado.get('video_final','(sin render)')}")
        lineas.append(f"/working: {free_gb('/kaggle/working'):.2f} GB | /tmp: {free_gb('/kaggle/tmp'):.2f} GB")
        return "\n".join(lineas)
    except Exception as e:
        return f"Error: {e}"

def ui_ver_guion_json():
    try:
        vp = _get_vp()
        g2 = vp.estado.get("etapa2", {}).get("guion_json", {})
        g4 = vp.estado.get("etapa4", {}).get("guion_json", {})
        return (json.dumps(g2, ensure_ascii=False, indent=2) +
                "\n\n--- ETAPA 4 ---\n\n" +
                json.dumps(g4, ensure_ascii=False, indent=2))
    except Exception as e:
        return str(e)

# ── Helpers de reanudación ───────────────────────────────────

def _video_mtime(p):
    try:
        for fn in ["estado.json","timeline.json"]:
            fp = os.path.join(p, fn)
            if os.path.exists(fp): return os.path.getmtime(fp)
        return os.path.getmtime(p)
    except: return time.time()

def _list_video_dirs():
    if not os.path.isdir(VIDEOS_DIR): return []
    dirs = []
    for name in os.listdir(VIDEOS_DIR):
        p = os.path.join(VIDEOS_DIR, name)
        if os.path.isdir(p) and (os.path.exists(os.path.join(p,"estado.json")) or
                                  os.path.exists(os.path.join(p,"timeline.json"))):
            dirs.append(p)
    dirs.sort(key=_video_mtime, reverse=True)
    return dirs

def _video_label(p):
    try: mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(_video_mtime(p)))
    except: mtime = "?"
    return f"{os.path.basename(p)} · {mtime}"

def _gallery_images(vp_dir):
    img_dir = os.path.join(vp_dir, "etapa1_imagenes")
    keys = [("A","concursante_base.png"),("B","concursante_cuerpo_completo.png"),
            ("C","concursante_plano_cercano.png"),("D","concursante_closeup.png"),
            ("E","concursante_plano_lejano_1.png"),("F","concursante_plano_lejano_2.png"),
            ("G","concursante_plano_lejano_3.png")]
    return [(os.path.join(img_dir,fn),lb) for lb,fn in keys
            if os.path.exists(os.path.join(img_dir,fn))]

def _safe_dur(path):
    try:
        d = _ffprobe_duration(path)
        return f"{d:.1f}s" if d else "?"
    except: return "?"

def _progress_text(vp_dir):
    if not vp_dir or not os.path.isdir(vp_dir): return "No hay video activo."
    try: vp = VideoProject(vp_dir)
    except Exception as e: return f"Error: {e}"
    imgs = _gallery_images(vp_dir)
    labels = [x[1] for x in imgs]
    missing = [k for k in ["A","B","C","D","E","F","G"] if k not in labels]
    e2 = vp.estado.get("etapa2",{}) or {}
    g2 = (e2.get("guion_json",{}) or {}).get("fragmentos",[]) or []
    e3 = vp.estado.get("etapa3",{}) or {}
    song = e3.get("song_wav","")
    song_ok = bool(song and os.path.exists(song))
    clips5 = [c for c in (e3.get("clips_5s",[]) or []) if os.path.exists(c)]
    e4 = vp.estado.get("etapa4",{}) or {}
    g4 = (e4.get("guion_json",{}) or {}).get("fragmentos",[]) or []
    fv = vp.estado.get("video_final","")
    # Voz del concursante
    voice_st = load_contestant_voice_state()
    voz_ok = voice_st.get("validated", False)
    voz_info = f"{'✅' if voz_ok else '❌'} {voice_st.get('mode') or 'no configurada'}"
    L = [
        f"📂 {os.path.basename(vp_dir)}",
        f"🎭 {vp.estado.get('personaje','?')} | 📁 {vp.estado.get('plantilla_slug',_CUR_TPL or '?')}",
        f"🎙️ Voz concursante: {voz_info}",
        "",
        f"🖼️ E1: {len(imgs)}/7 | Faltan: {', '.join(missing) if missing else 'ninguna'}",
        f"🎙️ E2: {'✅' if g2 else '❌'} ({len(g2)} frags) | 🎵 E3: canción {'✅' if song_ok else '❌'} | clips5s: {len(clips5)}",
        f"🏆 E4: {'✅' if g4 else '❌'} ({len(g4)} frags) | 🎞️ FINAL: {'✅ '+os.path.basename(fv) if fv and os.path.exists(fv) else '❌'}",
    ]
    try:
        L.append(f"💾 /working: {free_gb('/kaggle/working'):.2f} GB | /tmp: {free_gb('/kaggle/tmp'):.2f} GB")
    except: pass
    return "\n".join(L)

def ui_listar_videos():
    dirs = _list_video_dirs()
    choices = [(_video_label(p),p) for p in dirs]
    val = _CUR_VP_DIR if _CUR_VP_DIR and os.path.isdir(_CUR_VP_DIR) else (dirs[0] if dirs else None)
    if val and all(c[1]!=val for c in choices):
        choices.insert(0, (_video_label(val), val))
    return gr.update(choices=choices, value=val)

def ui_scan_ultimo_video():
    global _CUR_VP_DIR, _CUR_TPL
    dirs = _list_video_dirs()
    choices = [(_video_label(p),p) for p in dirs]
    sel = _CUR_VP_DIR if _CUR_VP_DIR and os.path.isdir(_CUR_VP_DIR) else (dirs[0] if dirs else None)
    if sel and all(c[1]!=sel for c in choices):
        choices.insert(0, (_video_label(sel), sel))
    if sel:
        try:
            vp = VideoProject(sel)
            tpl = vp.estado.get("plantilla_slug") or _CUR_TPL
            if tpl: _CUR_TPL = tpl
        except: pass
        _set_vp(sel, _CUR_TPL or "")
        return _progress_text(sel), _gallery_images(sel), gr.update(choices=choices, value=sel), sel
    return "No hay videos.\n1) Crea plantilla.\n2) Crea video.", [], gr.update(choices=choices, value=None), ""

def ui_cargar_video_seleccionado(video_path):
    global _CUR_VP_DIR, _CUR_TPL
    if not video_path or not os.path.isdir(video_path):
        return "❌ Video no existe.", [], "Ninguno", ""
    try:
        vp = VideoProject(video_path)
        tpl = vp.estado.get("plantilla_slug") or _CUR_TPL
        if tpl: _CUR_TPL = tpl
    except: pass
    _set_vp(video_path, _CUR_TPL or "")
    return _progress_text(video_path), _gallery_images(video_path), os.path.basename(video_path), video_path

def ui_refrescar_progreso():
    if _CUR_VP_DIR and os.path.isdir(_CUR_VP_DIR):
        return _progress_text(_CUR_VP_DIR), _gallery_images(_CUR_VP_DIR)
    return "No hay video activo.", []

def ui_initial_load():
    prog, gal, dd, ruta = ui_scan_ultimo_video()
    return prog, gal, dd, ruta, ui_refresh_tpl()


def ui_guardar_guion_editado(etapa_sel, guion_texto):
    try:
        vp = _get_vp()
        return guardar_guion_editado(vp, etapa_sel, guion_texto)
    except Exception as e:
        return f"❌ Error: {e}"

# ═══════════════════════════════════════════════════════════════
# v7 — NUEVOS HANDLERS: preview con imágenes · timeline por fragmento ·
#      regenerar clip lipsync · automatización por etapas · diagnóstico LTX
# ═══════════════════════════════════════════════════════════════
def _bump(n):
    try:
        return int(n) + 1
    except Exception:
        return 1

def _bump3(a, b, c):
    return _bump(a), _bump(b), _bump(c)

# ── Preview con imágenes (Etapas 2 y 4) ──────────────────────
def ui_etapa2_preview(tpl, personaje, historia, model_id, fast_mode_tts):
    return _ui_run_etapa("etapa2", tpl, personaje, historia, model_id, fast_mode_tts,
                         "auto", solo_preview=True)

def ui_etapa4_preview(tpl, personaje, model_id, fast_mode_tts):
    return _ui_run_etapa("etapa4", tpl, personaje, "", model_id, fast_mode_tts,
                         "auto", solo_preview=True)

def ui_probar_entorno_ltx():
    try:
        pp = ensure_ltx_env(force=True)
        msg = "✅ Entorno LTX-2.5 OK: el subproceso puede importar transformers + mmgp."
        if pp:
            msg += f"\n   Se usa un huggingface_hub compatible aislado en: {pp}"
        return msg
    except Exception as e:
        return f"❌ {e}"

# ── Filas del timeline por fragmentos ────────────────────────
_ESTADO_DIALOGO = {
    "lipsync":   "🎬 Lipsync LTX ✅",
    "fallback":  "⚠️ Lipsync FALLÓ → clip estático",
    "preview":   "🖼️ Preview (imagen + audio)",
    "audio":     "🔊 Solo audio",
    "pendiente": "⏳ Pendiente",
}

def ui_filas_timeline_dialogo(etapa):
    try:
        vp = _get_vp()
    except Exception:
        return {"msg": "Crea o carga un video para ver el timeline.", "rows": []}
    try:
        filas = estado_fragmentos_dialogo(vp, etapa)
    except Exception as e:
        return {"msg": f"❌ No se pudo leer el timeline: {e}", "rows": []}
    if not filas:
        return {"msg": "Aún no hay guion en esta etapa. Pulsa «📝 Generar solo guion».", "rows": []}
    cuenta = {}
    for f in filas:
        cuenta[f["estado"]] = cuenta.get(f["estado"], 0) + 1
    resumen = " · ".join(f"{_ESTADO_DIALOGO.get(k, k)}: **{v}**" for k, v in cuenta.items())
    rows = []
    for f in filas:
        txt = f["texto"].replace("\n", " ")
        if len(txt) > 170:
            txt = txt[:170] + "…"
        dur = f"{f['dur']:.1f}s" if f.get("dur") else "—"
        extra = " · ✏️ el texto cambió: audio desactualizado" if f["desactualizado"] else ""
        md = (f"**#{f['idx']:02d} · {f['hablante']}** · ⏱ {dur} · "
              f"{_ESTADO_DIALOGO.get(f['estado'], f['estado'])}{extra}\n\n«{txt}»")
        video = f["clip"] or f["preview"] or ""
        rows.append({"idx": f["idx"], "md": md, "video": video,
                     "audio": (f["audio"] if not video else "")})
    return {"msg": f"{len(filas)} fragmentos — {resumen}", "rows": rows}

_TIPO_CLIP = {"reaccion": "😮 reacción", "plano_lejano_5s": "🎥 plano lejano 5s",
              "lipsync_15s": "🎤 lipsync", "dialogo": "💬 diálogo"}
_ESTADO_CLIP = {"pendiente": "⏳ pendiente", "completado": "✅ listo",
                "completado_fallback": "⚠️ lipsync FALLÓ → imagen estática"}

def ui_filas_timeline_cancion():
    try:
        vp = _get_vp()
    except Exception:
        return {"msg": "Crea o carga un video para ver el timeline.", "rows": []}
    try:
        filas = etapa3_filas_timeline(vp)
    except Exception as e:
        return {"msg": f"❌ No se pudo leer el timeline: {e}", "rows": []}
    if not filas:
        return {"msg": "Aún no hay timeline. Pulsa «📊 Construir timeline».", "rows": []}
    n_lip = sum(1 for f in filas if f["tipo"] == "lipsync_15s")
    n_ok = sum(1 for f in filas if f["tipo"] == "lipsync_15s" and f["estado"] == "completado")
    rows = []
    for f in filas:
        es_lip = (f["tipo"] == "lipsync_15s")
        md = (f"**{f['clip_id']} · {_TIPO_CLIP.get(f['tipo'], f['tipo'])}** · "
              f"{f['start']:.1f}s → {f['end']:.1f}s ({f['dur']:.1f}s) · "
              f"{_ESTADO_CLIP.get(f['estado'], f['estado'])}")
        rows.append({"clip_id": f["clip_id"], "md": md, "video": f["video"], "lipsync": es_lip})
    return {"msg": f"{len(filas)} clips en el timeline — lipsync LTX listos: **{n_ok}/{n_lip}**. "
                   f"Los planos de 5s son compartidos: se regeneran en «Regenerar clip 5s específico».",
            "rows": rows}

# ── Regenerar un clip lipsync ────────────────────────────────
def ui_regen_clip_dialogo(etapa, idx, tpl, personaje):
    if not _GPU_LOCK.acquire(blocking=False):
        return _BUSY_MSG
    try:
        vp = _get_vp(); tpl = tpl or _CUR_TPL
        if not _LTX_PATHS:
            return "❌ Descarga LTX."
        return regenerar_clip_lipsync_fragmento(vp, tpl, personaje, etapa, int(idx), _LTX_PATHS)
    except Exception as e:
        import traceback
        log(f"[UI regen lipsync {etapa} #{idx}] ❌ {e}\n{traceback.format_exc()[-1200:]}")
        return f"❌ Error: {e}"
    finally:
        _GPU_LOCK.release()

def ui_regen_clip_cancion(clip_id, tpl, personaje):
    if not _GPU_LOCK.acquire(blocking=False):
        return _BUSY_MSG
    try:
        vp = _get_vp(); tpl = tpl or _CUR_TPL
        if not _LTX_PATHS:
            return "❌ Descarga LTX."
        out, reens = etapa3_regenerar_clip_lipsync(vp, tpl, personaje, clip_id, _LTX_PATHS)
        extra = ("🎞️ Canción reensamblada." if reens else
                 "ℹ️ Aún faltan lipsync por renderizar: pulsa «▶️ Renderizar Etapa 3» para ensamblar.")
        return f"✅ Lipsync {clip_id} regenerado.\n{extra}"
    except Exception as e:
        import traceback
        log(f"[UI regen lipsync canción {clip_id}] ❌ {e}\n{traceback.format_exc()[-1200:]}")
        return f"❌ Error: {e}"
    finally:
        _GPU_LOCK.release()

def _mk_regen_dialogo(etapa, idx):
    def _fn(tpl, personaje):
        return ui_regen_clip_dialogo(etapa, idx, tpl, personaje)
    return _fn

def _mk_regen_cancion(clip_id):
    def _fn(tpl, personaje):
        return ui_regen_clip_cancion(clip_id, tpl, personaje)
    return _fn

# ── Timelines dinámicos (se llaman DENTRO del gr.Blocks) ─────
def _make_dialog_timeline(etapa, tick, status_box, tpl_c, pj_c):
    @gr.render(inputs=[tick])
    def _render_dialog_tl(_t):
        data = ui_filas_timeline_dialogo(etapa)
        gr.Markdown(data["msg"])
        for r in data["rows"]:
            with gr.Row(equal_height=True):
                with gr.Column(scale=5, min_width=260):
                    gr.Markdown(r["md"])
                with gr.Column(scale=3, min_width=220):
                    if r["video"]:
                        gr.Video(value=r["video"], height=150, interactive=False, show_label=False)
                    elif r["audio"]:
                        gr.Audio(value=r["audio"], type="filepath", interactive=False, show_label=False)
                with gr.Column(scale=2, min_width=170):
                    btn = gr.Button("🔄 Regenerar lipsync (LTX)", size="sm", variant="secondary")
                    btn.click(fn=_mk_regen_dialogo(etapa, r["idx"]),
                              inputs=[tpl_c, pj_c], outputs=[status_box]
                    ).then(fn=_bump, inputs=[tick], outputs=[tick])

def _make_song_timeline(tick, status_box, tpl_c, pj_c):
    @gr.render(inputs=[tick])
    def _render_song_tl(_t):
        data = ui_filas_timeline_cancion()
        gr.Markdown(data["msg"])
        for r in data["rows"]:
            if not r["lipsync"]:
                gr.Markdown(r["md"])
                continue
            with gr.Row(equal_height=True):
                with gr.Column(scale=5, min_width=260):
                    gr.Markdown(r["md"])
                with gr.Column(scale=3, min_width=220):
                    if r["video"]:
                        gr.Video(value=r["video"], height=150, interactive=False, show_label=False)
                with gr.Column(scale=2, min_width=170):
                    btn = gr.Button("🔄 Regenerar lipsync (LTX)", size="sm", variant="secondary")
                    btn.click(fn=_mk_regen_cancion(r["clip_id"]),
                              inputs=[tpl_c, pj_c], outputs=[status_box]
                    ).then(fn=_bump, inputs=[tick], outputs=[tick])

# ── AUTOMATIZACIÓN por etapas ────────────────────────────────
_MODOS_DIALOGO = {
    "Completo (audio + lipsync LTX)": "completo",
    "Solo preview con imágenes (sin lipsync)": "preview",
}
_MODOS_GUION = {
    "Reutilizar guion si existe (si no, generar nuevo)": "auto",
    "Generar guion NUEVO": "nuevo",
    "Usar guion ACTUAL (editado)": "actual",
}

def ui_automatizar(tpl, personaje, historia, model_e2, model_e4, fast_e2, fast_e4,
                   inc_e2, inc_cancion, inc_e4, modo_dialogo_lbl, modo_guion_lbl,
                   reconstruir_tl, render_final):
    # Generador: va mostrando el progreso. Cada etapa es independiente: si una falla se anota
    # y se continúa con las demás. Orden: Etapa 2 → Canción → Etapa 4.
    if not (inc_e2 or inc_cancion or inc_e4):
        yield "❌ Marca al menos una etapa.", gr.update()
        return
    if not _GPU_LOCK.acquire(blocking=False):
        yield _BUSY_MSG, gr.update()
        return
    lines, last_video = [], None
    def _snap():
        return "\n".join(lines)
    try:
        vp = _get_vp(); tpl = tpl or _CUR_TPL
        modo_d = _MODOS_DIALOGO.get(modo_dialogo_lbl, "completo")
        modo_g = _MODOS_GUION.get(modo_guion_lbl, "auto")
        preview = (modo_d == "preview")
        necesita_ltx = bool(inc_cancion) or (bool(inc_e2 or inc_e4) and not preview)
        if necesita_ltx and not _LTX_PATHS:
            yield "❌ Descarga/inicializa LTX-2.5 primero (hace falta para lipsync y canción).", gr.update()
            return

        nombres = [n for n, a in (("Etapa 2", inc_e2), ("Canción", inc_cancion), ("Etapa 4", inc_e4)) if a]
        lines.append("📋 Plan: " + " → ".join(nombres) +
                     (f" | diálogos: {'PREVIEW con imágenes' if preview else 'completo con lipsync'}"
                      if (inc_e2 or inc_e4) else ""))
        yield _snap(), gr.update()

        def _run_dialog(etapa, hist, mid, fast):
            return ejecutar_etapa_dialogos(
                vp, tpl, personaje, etapa, _LTX_PATHS or {},
                historia_base=hist or "", model_id=mid, fast_mode_tts=bool(fast),
                regenerar_guion=(modo_g == "nuevo"), usar_guion_actual=(modo_g == "actual"),
                solo_preview=preview)

        def _run_cancion():
            song = vp.estado.get("etapa3", {}).get("song_wav", "")
            if not song or not os.path.exists(song):
                raise RuntimeError("Falta la canción: súbela y pulsa «💾 Guardar canción».")
            far5_dir = os.path.join(vp.video_dir, "etapa3_cancion", "planos_5s")
            tiene5 = os.path.isdir(far5_dir) and any(f.lower().endswith(".mp4") for f in os.listdir(far5_dir))
            if not tiene5:
                log("[AUTO] No hay clips 5s → generando (PASO 3.1)...")
                etapa3_paso1_generar_clips_5s(vp, tpl, personaje, _LTX_PATHS)
            if reconstruir_tl or not vp.timeline_items:
                log("[AUTO] Construyendo timeline de la canción (PASO 3.2)...")
                build_song_timeline(vp, tpl, personaje, song_audio_path=song,
                                    use_far_planes=["E", "F", "G"])
            return etapa3_paso3_renderizar_secuencia(vp, tpl, personaje, _LTX_PATHS)

        pasos = []
        if inc_e2:
            pasos.append(("Etapa 2", lambda: _run_dialog("etapa2", historia, model_e2, fast_e2)))
        if inc_cancion:
            pasos.append(("Canción", _run_cancion))
        if inc_e4:
            pasos.append(("Etapa 4", lambda: _run_dialog("etapa4", "", model_e4, fast_e4)))

        for nombre, fn in pasos:
            lines.append(f"▶️ {nombre}...")
            yield _snap(), gr.update()
            try:
                res = fn()
                if res and os.path.exists(str(res)):
                    last_video = res
                lines.append(f"   ✅ {nombre} lista → {os.path.basename(str(res))}")
            except Exception as e:
                import traceback
                log(f"[AUTO {nombre}] ❌ {e}\n{traceback.format_exc()[-1500:]}")
                lines.append(f"   ❌ {nombre}: {e}")
            yield _snap(), (last_video if last_video else gr.update())

        if render_final:
            if preview and (inc_e2 or inc_e4):
                lines.append("ℹ️ VIDEO FINAL omitido: en modo preview los videos son solo de validación.")
            else:
                try:
                    vf = render_video_final(vp, {"etapa2": bool(inc_e2), "etapa3": bool(inc_cancion),
                                                 "etapa4": bool(inc_e4)})
                    last_video = vf
                    lines.append(f"🏁 VIDEO FINAL → {os.path.basename(vf)}")
                except Exception as e:
                    lines.append(f"❌ VIDEO FINAL: {e}")
        lines.append("✔️ Automatización terminada.")
        yield _snap(), (last_video if last_video else gr.update())
    except Exception as e:
        import traceback
        log(f"[AUTO] ❌ {e}\n{traceback.format_exc()[-1500:]}")
        lines.append(f"❌ Error: {e}")
        yield _snap(), gr.update()
    finally:
        _GPU_LOCK.release()

def ui_auto_e2_e4(tpl, personaje, historia, model_e2, model_e4, fast_e2, fast_e4,
                  modo_dialogo_lbl, modo_guion_lbl, render_final):
    yield from ui_automatizar(tpl, personaje, historia, model_e2, model_e4, fast_e2, fast_e4,
                              True, False, True, modo_dialogo_lbl, modo_guion_lbl, False, render_final)

def ui_auto_solo_cancion(tpl, personaje, reconstruir_tl):
    yield from ui_automatizar(tpl, personaje, "", "", "", False, False,
                              False, True, False, "", "", reconstruir_tl, False)

# ═══════════════════════════════════════════════════════════════
# v8 — PARCHE ADITIVO (solo UI). No modifica la lógica de las celdas anteriores:
#   1) Selección múltiple + «Regenerar seleccionados» en los timelines de E2, E3 y E4.
#   2) Contexto (prompt) para el guion de la Etapa 4 y casillas de contexto en Automatización.
# Cómo funciona: envuelve por nombre funciones ya existentes y SOLO actúa si hay un contexto
# escrito. Con las casillas vacías el comportamiento es idéntico al de antes.
# Las definiciones originales de _make_dialog_timeline / _make_song_timeline quedan arriba
# sin tocar; las de este bloque (mismo nombre) las sustituyen al construir la UI.
# ═══════════════════════════════════════════════════════════════
_CTX = {"e4_ui": "", "e2_auto": "", "e4_auto": "", "auto_run": False}

def _ctx_setter(clave):
    def _fn(txt):
        _CTX[clave] = (txt or "").strip()
    return _fn

def _ctx_cargar():
    # Al (re)cargar la página muestra lo que está realmente activo (evita contextos "fantasma").
    return _CTX["e4_ui"], _CTX["e2_auto"], _CTX["e4_auto"]

def _ctx_e4_activo():
    # Dentro de una automatización manda la casilla de la automatización; si está vacía,
    # se usa la de la sección Etapa 4.
    if _CTX["auto_run"] and _CTX["e4_auto"]:
        return _CTX["e4_auto"]
    return _CTX["e4_ui"]

# — Contexto para el guion de la ETAPA 4 (el prompt original no admitía historia) —
if not getattr(build_dialogo_final_system_prompt, "_ctx_wrapped", False):
    _build_final_orig = build_dialogo_final_system_prompt
    def _build_final_ctx(personaje_key, *a, **k):
        base = _build_final_orig(personaje_key, *a, **k)
        ctx = _ctx_e4_activo()
        if not ctx:
            return base
        marca = "REGLAS OBLIGATORIAS E INVIOLABLES:"
        bloque = f"Contexto a incorporar (historia / actuación del concursante): {ctx}\n\n"
        log(f"[CTX E4] Contexto aplicado al guion de la Etapa 4 ({len(ctx)} caracteres).")
        return base.replace(marca, bloque + marca, 1) if marca in base else base + "\n\n" + bloque
    _build_final_ctx._ctx_wrapped = True
    build_dialogo_final_system_prompt = _build_final_ctx

# — Contexto de la ETAPA 2 desde Automatización (solo si esa casilla tiene texto) —
if not getattr(build_dialogo_inicial_system_prompt, "_ctx_wrapped", False):
    _build_inicial_orig = build_dialogo_inicial_system_prompt
    def _build_inicial_ctx(personaje_key, historia_base="", *a, **k):
        if _CTX["auto_run"] and _CTX["e2_auto"]:
            historia_base = _CTX["e2_auto"]
            log(f"[CTX E2] Contexto de Automatización aplicado ({len(historia_base)} caracteres).")
        return _build_inicial_orig(personaje_key, historia_base, *a, **k)
    _build_inicial_ctx._ctx_wrapped = True
    build_dialogo_inicial_system_prompt = _build_inicial_ctx

# — Marca "estoy dentro de una automatización" (mismo nombre → el cableado existente lo recoge) —
import functools
if not getattr(ui_automatizar, "_ctx_wrapped", False):
    _ui_automatizar_base = ui_automatizar
    @functools.wraps(_ui_automatizar_base)
    def ui_automatizar(*args, **kwargs):
        _CTX["auto_run"] = True
        try:
            yield from _ui_automatizar_base(*args, **kwargs)
        finally:
            _CTX["auto_run"] = False
    ui_automatizar._ctx_wrapped = True

# — Regeneración por lote (reutiliza los handlers individuales ya existentes) —
def _regen_lote(sel, regen_fn, nombre_fn):
    if not sel:
        yield "ℹ️ No hay nada seleccionado. Marca uno o más con la casilla «Elegir»."
        return
    lineas = [f"🔄 Regenerando {len(sel)} seleccionado(s), uno tras otro (seed nueva en cada uno)..."]
    yield "\n".join(lineas)
    ok = 0
    for k, item in enumerate(sel, 1):
        lineas.append(f"▶️ ({k}/{len(sel)}) {nombre_fn(item)}...")
        yield "\n".join(lineas)
        res = str(regen_fn(item) or "").strip()
        if res == _BUSY_MSG:
            lineas.append(f"{_BUSY_MSG}\n⛔ Lote detenido.")
            yield "\n".join(lineas)
            return
        bien = res.startswith("✅")
        ok += 1 if bien else 0
        primera = (res.splitlines() or [""])[0][:200] or ("✅ listo" if bien else "❌ sin respuesta")
        lineas.append("   " + primera)
        yield "\n".join(lineas)
    lineas.append(f"✔️ Lote terminado: {ok}/{len(sel)} regenerados.")
    yield "\n".join(lineas)

def _mk_regen_lote_dialogo(etapa, idxs):
    def _fn(tpl, personaje, *marcas):
        sel = [i for i, m in zip(idxs, marcas) if m]
        yield from _regen_lote(sel, lambda i: ui_regen_clip_dialogo(etapa, i, tpl, personaje),
                               lambda i: f"fragmento #{int(i):02d}")
    return _fn

def _mk_regen_lote_cancion(clip_ids):
    def _fn(tpl, personaje, *marcas):
        sel = [c for c, m in zip(clip_ids, marcas) if m]
        yield from _regen_lote(sel, lambda c: ui_regen_clip_cancion(c, tpl, personaje),
                               lambda c: f"clip {c}")
    return _fn

def _mk_marcar(n, valor):
    def _fn():
        return tuple(valor for _ in range(n)) if n > 1 else valor
    return _fn

def _barra_lote(barra, chks, fn_lote, tick, status_box, tpl_c, pj_c):
    if not chks:
        return
    try:
        with barra:
            b_all  = gr.Button("☑️ Marcar todos", size="sm", variant="secondary")
            b_none = gr.Button("⬜ Desmarcar todos", size="sm", variant="secondary")
            b_go   = gr.Button("🔄 Regenerar seleccionados (LTX)", size="sm", variant="primary")
        b_all.click(fn=_mk_marcar(len(chks), True), inputs=[], outputs=chks)
        b_none.click(fn=_mk_marcar(len(chks), False), inputs=[], outputs=chks)
        b_go.click(fn=fn_lote, inputs=[tpl_c, pj_c] + chks, outputs=[status_box]
                   ).then(fn=_bump, inputs=[tick], outputs=[tick])
    except Exception as e:
        log(f"[UI v8] ⚠ No se pudo crear la barra de selección múltiple: {e}")

# — Timelines con casilla «Elegir» por fragmento (mismas firmas que los originales) —
def _make_dialog_timeline(etapa, tick, status_box, tpl_c, pj_c):
    @gr.render(inputs=[tick])
    def _render_dialog_tl(_t):
        data = ui_filas_timeline_dialogo(etapa)
        gr.Markdown(data["msg"])
        barra = gr.Row() if data["rows"] else None      # barra superior; se rellena al final
        chks = []
        for r in data["rows"]:
            with gr.Row(equal_height=True):
                with gr.Column(scale=1, min_width=80):
                    chks.append(gr.Checkbox(label="Elegir", value=False))
                with gr.Column(scale=5, min_width=260):
                    gr.Markdown(r["md"])
                with gr.Column(scale=3, min_width=220):
                    if r["video"]:
                        gr.Video(value=r["video"], height=150, interactive=False, show_label=False)
                    elif r["audio"]:
                        gr.Audio(value=r["audio"], type="filepath", interactive=False, show_label=False)
                with gr.Column(scale=2, min_width=170):
                    btn = gr.Button("🔄 Regenerar lipsync (LTX)", size="sm", variant="secondary")
                    btn.click(fn=_mk_regen_dialogo(etapa, r["idx"]),
                              inputs=[tpl_c, pj_c], outputs=[status_box]
                    ).then(fn=_bump, inputs=[tick], outputs=[tick])
        _barra_lote(barra, chks, _mk_regen_lote_dialogo(etapa, [r["idx"] for r in data["rows"]]),
                    tick, status_box, tpl_c, pj_c)

def _make_song_timeline(tick, status_box, tpl_c, pj_c):
    @gr.render(inputs=[tick])
    def _render_song_tl(_t):
        data = ui_filas_timeline_cancion()
        gr.Markdown(data["msg"])
        barra = gr.Row() if any(r["lipsync"] for r in data["rows"]) else None
        chks, ids = [], []
        for r in data["rows"]:
            if not r["lipsync"]:
                gr.Markdown(r["md"])
                continue
            with gr.Row(equal_height=True):
                with gr.Column(scale=1, min_width=80):
                    chks.append(gr.Checkbox(label="Elegir", value=False))
                    ids.append(r["clip_id"])
                with gr.Column(scale=5, min_width=260):
                    gr.Markdown(r["md"])
                with gr.Column(scale=3, min_width=220):
                    if r["video"]:
                        gr.Video(value=r["video"], height=150, interactive=False, show_label=False)
                with gr.Column(scale=2, min_width=170):
                    btn = gr.Button("🔄 Regenerar lipsync (LTX)", size="sm", variant="secondary")
                    btn.click(fn=_mk_regen_cancion(r["clip_id"]),
                              inputs=[tpl_c, pj_c], outputs=[status_box]
                    ).then(fn=_bump, inputs=[tick], outputs=[tick])
        _barra_lote(barra, chks, _mk_regen_lote_cancion(ids), tick, status_box, tpl_c, pj_c)


# ═══════════════════════════════════════════════════════════════
# v9 — PARCHE ANTI-REGENERACIÓN (solo UI, aditivo; no cambia ninguna otra celda)
#   Regla: un botón de etapa SOLO genera clips si esa etapa NO tiene ninguno.
#   Si ya tiene clips → únicamente ENSAMBLA/RENDERIZA lo que existe.
#   Para rehacer algo se usan los botones de regenerar (individual o «seleccionados»).
#   Cómo: envuelve por nombre 2 funciones de pipeline ya existentes. Las originales
#   quedan intactas y se llaman tal cual cuando la etapa está vacía.
# ═══════════════════════════════════════════════════════════════
def _v9_clip_ok(p):
    try:
        return bool(p) and os.path.exists(p) and os.path.getsize(p) > 10000
    except Exception:
        return False

# ── Etapa 3 (canción): render = ensamblar, nunca regenerar lo que ya existe ──────────
if not getattr(etapa3_paso3_renderizar_secuencia, "_v9", False):
    _v9_orig_render3 = etapa3_paso3_renderizar_secuencia
    _V9_MARK = "v9-reuse"

    def etapa3_paso3_renderizar_secuencia(vp, plantilla_slug, personaje_key, ltx_paths,
                                          fps=24, forzar_lipsync=False):
        _orig = lambda: _v9_orig_render3(vp, plantilla_slug, personaje_key, ltx_paths, fps, forzar_lipsync)
        song_wav = vp.estado.get("etapa3", {}).get("song_wav", "")
        if forzar_lipsync or not song_wav or not os.path.exists(song_wav) or not vp.timeline_items:
            return _orig()                       # forzado, o el original ya da su error claro
        clips_dir = os.path.join(vp.video_dir, "etapa3_cancion", "clips_renderizados")
        song_dur = _ffprobe_duration(song_wav) or 0.0
        existentes, faltan, t = [], [], 0.0
        for it in vp.timeline_items:
            dur = float(it.get("dur_s", 0))
            if it.get("tipo") == "lipsync_15s":
                if _v9_clip_ok(os.path.join(clips_dir, f"{it['clip_id']}.mp4")):
                    existentes.append(it)
                elif min(dur, max(0.1, song_dur - t)) >= 0.5:   # igual que el render: sin audio no cuenta
                    faltan.append(it["clip_id"])
            t += dur
        if not existentes:
            return _orig()                       # etapa vacía → genera normalmente
        if faltan:
            raise RuntimeError(
                f"La canción ya tiene {len(existentes)} lipsync, pero faltan {len(faltan)} "
                f"({', '.join(faltan[:8])}{'…' if len(faltan) > 8 else ''}). No se genera nada "
                f"automáticamente: márcalos en el timeline y pulsa «Regenerar seleccionados».")
        # Solo ensamblar: los lipsync existentes se marcan «vigentes» durante esta llamada
        guardado = [(it, it.get("estado"), it.get("sig")) for it in existentes]
        sig_prev = globals()["_sig_lipsync_cancion"]
        globals()["_sig_lipsync_cancion"] = lambda *a, **k: _V9_MARK
        for it in existentes:
            it["estado"], it["sig"] = "completado", _V9_MARK
        try:
            log("[PASO 3.3] Lipsync ya generados → solo se ensambla (no se regenera nada).")
            return _v9_orig_render3(vp, plantilla_slug, personaje_key, ltx_paths, fps, False)
        finally:
            globals()["_sig_lipsync_cancion"] = sig_prev
            for it, est, sg in guardado:         # deja el estado exactamente como estaba
                it["estado"] = est
                if sg is None:
                    it.pop("sig", None)
                else:
                    it["sig"] = sg
            try:
                vp.save()
            except Exception:
                pass
    etapa3_paso3_renderizar_secuencia._v9 = True

# ── Etapas 2 y 4 (diálogos): si ya hay clips lipsync, solo ensamblar ────────────────
#    «Ejecutar completa (guion NUEVO)» y el PREVIEW pasan directo al original (sin cambios).
if not getattr(ejecutar_etapa_dialogos, "_v9", False):
    _v9_orig_dialogos = ejecutar_etapa_dialogos

    def ejecutar_etapa_dialogos(vp, plantilla_slug, personaje_key, etapa, ltx_paths, *args, **kw):
        if kw.get("regenerar_guion") or kw.get("solo_preview"):
            return _v9_orig_dialogos(vp, plantilla_slug, personaje_key, etapa, ltx_paths, *args, **kw)
        guion = (vp.estado.get(etapa, {}) or {}).get("guion_json") or {}
        items = _items_validos_guion(guion)
        hay = [it for it in items
               if _v9_clip_ok(_frag_paths(vp, etapa, it["idx"], it["hablante"])["mp4"])]
        if not hay:
            return _v9_orig_dialogos(vp, plantilla_slug, personaje_key, etapa, ltx_paths, *args, **kw)
        if len(hay) < len(items):
            falt = [it["idx"] for it in items if it not in hay]
            raise RuntimeError(
                f"{etapa} ya tiene {len(hay)} clip(s) lipsync, pero faltan {len(falt)} "
                f"(fragmentos {', '.join('#%d' % i for i in falt[:8])}{'…' if len(falt) > 8 else ''}). "
                f"No se genera nada automáticamente: usa «🔄 Regenerar lipsync» o "
                f"«Regenerar seleccionados» en esos fragmentos.")
        log(f"[{etapa.upper()}] Ya tiene {len(hay)} clips lipsync → solo se ensambla (no se regenera nada).")
        _reensamblar_si_completo(vp, etapa, guion, kw.get("fps", 24))
        return (vp.estado.get(etapa, {}) or {}).get("video_final", "")
    ejecutar_etapa_dialogos._v9 = True


# ═══════════════════════════════════════════════════════════════
# CSS AIQUEST MODERNO
# ═══════════════════════════════════════════════════════════════
AIQUEST_CSS = '''
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: "Inter", sans-serif !important; }
.gradio-container { max-width: 1120px !important; margin: auto !important; }
.brand-header {
    text-align: center;
    background: linear-gradient(135deg, #6a11cb 0%, #2575fc 100%);
    padding: 28px; border-radius: 15px; margin-bottom: 20px;
    box-shadow: 0 10px 30px rgba(106,17,203,0.3);
}
.brand-title   { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle{ color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 12px; }
.voice-section {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    border: 2px solid #6a11cb; border-radius: 12px; padding: 16px; margin: 12px 0;
}
button.primary {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    color: white !important; font-weight: 600 !important; border-radius: 12px !important;
}
button.secondary { border-radius: 10px !important; }
.voice-validated { color: #4caf50 !important; font-weight: 700 !important; }
.voice-pending   { color: #ff9800 !important; font-weight: 600 !important; }
'''


# ═══════════════════════════════════════════════════════════════
# UI GRADIO v6
# ═══════════════════════════════════════════════════════════════
with gr.Blocks(css=AIQUEST_CSS, theme=gr.themes.Soft(),
               title="🎤 Audiciones — Got Talent / La Voz") as demo:

    gr.HTML('''<div class="brand-header">
        <div class="brand-title">🎤 Sistema de Audiciones</div>
        <div class="brand-subtitle">
            Generación granular · Upload manual · Voz clonada consistente ·
            LTX-2.5 Wan2GP · Reanudación automática
        </div>
    </div>''')

    # ────────────────────────────────────────────────────────
    # REANUDAR
    # ────────────────────────────────────────────────────────
    with gr.Accordion("🔄 REANUDAR VIDEO EXISTENTE", open=True):
        with gr.Row():
            video_selector = gr.Dropdown(
                label="📂 Videos disponibles", choices=[], value=None,
                allow_custom_value=True, scale=4)
            btn_cargar  = gr.Button("▶️ Cargar",   variant="primary",   scale=1)
            btn_refprog = gr.Button("🔄 Refrescar", variant="secondary", scale=1)
            btn_listar  = gr.Button("📋 Listar",   variant="secondary", scale=1)
        progreso_box     = gr.Textbox(label="Estado del proyecto", value="Cargando...",
                                      lines=18, interactive=False)
        progreso_gallery = gr.Gallery(label="Imágenes Etapa 1", columns=4, height=200)
        with gr.Row():
            video_activo_label = gr.Textbox(
                label="Video activo",
                value=os.path.basename(_CUR_VP_DIR) if _CUR_VP_DIR else "Ninguno",
                interactive=False, scale=4)
            btn_abrir = gr.Button("📁 Ruta", scale=1)
        ruta_label = gr.Textbox(label="Ruta completa", value=_CUR_VP_DIR, interactive=False)

    gr.Markdown("---")

    # ────────────────────────────────────────────────────────
    # PLANTILLA + NUEVO VIDEO
    # ────────────────────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=3):
            tpl_dd = gr.Dropdown(label="📁 Plantilla activa", choices=[], value=None,
                                 allow_custom_value=True)
            with gr.Row():
                new_tpl_box   = gr.Textbox(label="Nueva plantilla", value="Mi Plantilla")
                btn_crear_tpl = gr.Button("➕ Crear",  variant="primary")
                btn_refresh_tpl = gr.Button("🔄")
        with gr.Column(scale=2):
            personaje    = gr.Dropdown(label="🎭 Concursante",
                                       choices=list(PERSONAJE_OPCIONES.keys()), value="Mujer")
            p_reemplazar = gr.Dropdown(label="🔄 Reemplazar en plantilla",
                                       choices=list(PERSONAJE_OPCIONES.keys()), value="Mujer")
            btn_nuevo_video = gr.Button("🎬 Crear NUEVO video", variant="primary")
    with gr.Row():
        status_main   = gr.Textbox(label="Estado",      interactive=False, scale=5)
        video_dir_box = gr.Textbox(label="Directorio",  value=_CUR_VP_DIR,
                                   interactive=False, scale=3)

    # ────────────────────────────────────────────────────────
    # ASSETS JUECES INDEPENDIENTES
    # ────────────────────────────────────────────────────────
    with gr.Accordion("📂 Assets de Plantilla (Jueces independientes)", open=False):
        gr.Markdown("### 👨‍⚖️ Jueces — Imagen + Audio(s) por separado")
        gr.Markdown(
            "⚠️ **Al subir audios de jueces, la transcripción previa se borra** "
            "(se re-transcribirá automáticamente al ejecutar Etapa 2/4 "
            "o usando el botón de abajo)."
        )
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("**Juez 1**")
                j1_img = gr.File(label="Imagen", file_types=[".png",".jpg",".jpeg"])
                j1_aud = gr.File(label="Audio(s)", file_count="multiple",
                                 file_types=[".wav",".mp3",".m4a"])
            with gr.Column(scale=1):
                gr.Markdown("**Juez 2**")
                j2_img = gr.File(label="Imagen", file_types=[".png",".jpg",".jpeg"])
                j2_aud = gr.File(label="Audio(s)", file_count="multiple",
                                 file_types=[".wav",".mp3",".m4a"])
            with gr.Column(scale=1):
                gr.Markdown("**Juez 3**")
                j3_img = gr.File(label="Imagen", file_types=[".png",".jpg",".jpeg"])
                j3_aud = gr.File(label="Audio(s)", file_count="multiple",
                                 file_types=[".wav",".mp3",".m4a"])
            with gr.Column(scale=1):
                gr.Markdown("**Juez 4**")
                j4_img = gr.File(label="Imagen", file_types=[".png",".jpg",".jpeg"])
                j4_aud = gr.File(label="Audio(s)", file_count="multiple",
                                 file_types=[".wav",".mp3",".m4a"])
        gr.Markdown("---")
        with gr.Row():
            esc_cercano  = gr.File(label="plano_cercano.png",
                                   file_types=[".png",".jpg"])
            escs_lejanos = gr.File(label="planos_lejanos 1..3",
                                   file_count="multiple", file_types=[".png",".jpg"])
        with gr.Row():
            reac1 = gr.File(label="Reacciones 1ª mitad", file_count="multiple",
                            file_types=[".mp4"])
            reac2 = gr.File(label="Reacciones 2ª mitad", file_count="multiple",
                            file_types=[".mp4"])
        with gr.Row():
            btn_assets = gr.Button("💾 Guardar TODOS los assets", variant="secondary")
            btn_transcribe_judges = gr.Button("🗣️ Transcribir audios de jueces ahora",
                                              variant="secondary")
        assets_status    = gr.Textbox(label="Estado assets/transcripción",
                                      interactive=False, lines=8)

    # ────────────────────────────────────────────────────────
    # ETAPA 1 — IMÁGENES (sin cambios)
    # ────────────────────────────────────────────────────────
    with gr.Accordion("🖼️ ETAPA 1 — Imágenes A→G (Granular + Upload)", open=True):
        gr.Markdown("### ⚙️ Parámetros")
        with gr.Row():
            edad     = gr.Number(label="Edad", value=28, precision=0)
            vestuario = gr.Textbox(label="Vestuario", value="vestido elegante de gala")
            planos   = gr.Radio(label="Planos lejanos", choices=[1,2,3], value=2)
        with gr.Row():
            dq = gr.Dropdown(label="Diffusion quant",
                             choices=list(QWEN_IMAGE_DIFFUSION_QUANTS.keys()),
                             value=list(QWEN_IMAGE_DIFFUSION_QUANTS.keys())[3],
                             allow_custom_value=True)
            eq = gr.Dropdown(label="Encoder quant",
                             choices=list(QWEN_VL_ENCODERS.keys()),
                             value=list(QWEN_VL_ENCODERS.keys())[0],
                             allow_custom_value=True)
            mq = gr.Dropdown(label="mmproj quant",
                             choices=list(QWEN_VL_MMPROJ.keys()),
                             value=list(QWEN_VL_MMPROJ.keys())[0],
                             allow_custom_value=True)
        with gr.Row():
            stA   = gr.Slider(5,40,value=18,step=1,label="Steps A")
            stE   = gr.Slider(5,40,value=18,step=1,label="Steps edit")
            cfg_v = gr.Slider(1.0,10.0,value=6.0,step=0.5,label="CFG")
            seed_v = gr.Number(label="Seed (-1=random)", value=-1, precision=0)
        gr.Markdown("---")
        gr.Markdown("### 🔑 PASO A — Imagen Base")
        with gr.Row():
            upload_A = gr.File(label="📤 Subir A", file_types=[".png",".jpg",".jpeg"])
            btn_A    = gr.Button("▶️ Generar/Subir A", variant="primary")
            btn_cont_A = gr.Button("⏩ Continuar desde A", variant="secondary")
        status_A  = gr.Textbox(label="Estado A", interactive=False)
        preview_A = gr.Image(label="Preview A", height=200)

        gr.Markdown("### 🔑 PASO B — Cuerpo Completo")
        with gr.Row():
            upload_B = gr.File(label="📤 Subir B", file_types=[".png",".jpg",".jpeg"])
            btn_B    = gr.Button("▶️ Generar/Subir B", variant="primary")
            btn_cont_B = gr.Button("⏩ Continuar desde B", variant="secondary")
        status_B  = gr.Textbox(label="Estado B", interactive=False)
        preview_B = gr.Image(label="Preview B", height=200)

        gr.Markdown("### PASO C — Plano Cercano en Escenario")
        with gr.Row():
            upload_C = gr.File(label="📤 Subir C", file_types=[".png",".jpg",".jpeg"])
            btn_C    = gr.Button("▶️ Generar/Subir C", variant="primary")
            btn_cont_C = gr.Button("⏩ Continuar desde C", variant="secondary")
        status_C  = gr.Textbox(label="Estado C", interactive=False)
        preview_C = gr.Image(label="Preview C", height=200)

        gr.Markdown("### PASO D — Close-up")
        with gr.Row():
            upload_D = gr.File(label="📤 Subir D", file_types=[".png",".jpg",".jpeg"])
            btn_D    = gr.Button("▶️ Generar/Subir D", variant="primary")
            btn_cont_D = gr.Button("⏩ Continuar desde D", variant="secondary")
        status_D  = gr.Textbox(label="Estado D", interactive=False)
        preview_D = gr.Image(label="Preview D", height=200)

        gr.Markdown("### PASOS E/F/G — Planos Lejanos")
        with gr.Row():
            upload_E = gr.File(label="📤 Subir E", file_types=[".png",".jpg",".jpeg"])
            btn_E    = gr.Button("▶️ E", variant="primary")
        with gr.Row():
            upload_F = gr.File(label="📤 Subir F", file_types=[".png",".jpg",".jpeg"])
            btn_F    = gr.Button("▶️ F", variant="primary")
        with gr.Row():
            upload_G = gr.File(label="📤 Subir G", file_types=[".png",".jpg",".jpeg"])
            btn_G    = gr.Button("▶️ G", variant="primary")
            btn_cont_E = gr.Button("⏩ Continuar desde E", variant="secondary")
        status_EFG = gr.Textbox(label="Estado E/F/G", interactive=False)

        gr.Markdown("---")
        gr.Markdown("### 🚀 MODO RÁPIDO: Ejecutar Etapa 1 Completa")
        btn_e1_todo = gr.Button("▶️ EJECUTAR ETAPA 1 COMPLETA", variant="primary", size="lg")
        e1_status   = gr.Textbox(label="Estado Etapa 1", interactive=False)
        e1_gallery  = gr.Gallery(label="Galería A..G", columns=4, height=280)

    # ────────────────────────────────────────────────────────
    # LTX-2.5
    # ────────────────────────────────────────────────────────
    with gr.Accordion("⬇️ Modelos LTX-2.5 (Wan2GP)", open=False):
        ltx_preset_dd = gr.Dropdown(
            label="Preset", choices=list(LTX23_PRESETS.keys()),
            value=DEFAULT_LTX23_PRESET)
        for k, v in LTX23_PRESETS.items():
            gr.Markdown(f"- **{k}**: {v['desc']}")
        btn_ltx_dl = gr.Button("⬇️ Descargar e inicializar LTX-2.5", variant="secondary")
        ltx_status = gr.Textbox(label="Estado LTX", interactive=False, lines=7)
        with gr.Row():
            btn_ltx_check = gr.Button("🧪 Probar entorno LTX (diagnóstico lipsync)", variant="secondary")
        ltx_check_status = gr.Textbox(label="Diagnóstico entorno LTX", interactive=False, lines=4)

    # ════════════════════════════════════════════════════════
    # 🎙️ VOZ DEL CONCURSANTE — SECCIÓN CENTRAL (encima de E2 y E4)
    # Esta sección configura la voz que se usará en todo el guion.
    # ════════════════════════════════════════════════════════
    with gr.Accordion(
        "🎙️ VOZ DEL CONCURSANTE — Configuración (requerida para Etapas 2 y 4)",
        open=True
    ):
        gr.HTML('''<div style="background:linear-gradient(135deg,#1a1a2e,#16213e);
                   border:2px solid #6a11cb;border-radius:12px;padding:14px;margin-bottom:10px;">
            <p style="color:#b0b8ff;margin:0;font-size:0.95em;">
            ✨ <b>Pipeline de voz consistente:</b> Cualquier voz que configures aquí
            se usará mediante <b>clonación</b> en TODOS los fragmentos del guion
            (tanto Etapa 2 como Etapa 4), garantizando consistencia total.
            </p></div>''')

        voice_state_box = gr.Textbox(
            label="📊 Estado actual de la voz",
            value=ui_show_voice_state(),
            interactive=False, lines=5
        )
        btn_show_voice_state = gr.Button("🔄 Actualizar estado de voz", variant="secondary")

        # ── OPCIÓN A: Subir audio del concursante ────────────
        with gr.Tab("🎤 Opción A — Subir mi audio (clonación directa)"):
            gr.Markdown(
                "Sube un audio claro de **mínimo 3 segundos**. "
                "Se transcribirá automáticamente para mejorar la calidad de clonación."
            )
            with gr.Row():
                contestant_audio_upload = gr.File(
                    label="🎵 Audio del concursante",
                    file_types=[".wav",".mp3",".m4a",".flac",".ogg"],
                    scale=3
                )
                btn_process_contestant = gr.Button(
                    "⚙️ Procesar y transcribir", variant="primary", scale=1
                )
            contestant_upload_status = gr.Textbox(
                label="Estado del procesamiento",
                interactive=False, lines=5
            )
            with gr.Row():
                contestant_transcription_box = gr.Textbox(
                    label="📝 Transcripción detectada (editable)",
                    placeholder="Aquí aparecerá la transcripción automática...",
                    lines=3, interactive=True, scale=4
                )
                contestant_lang_box = gr.Textbox(
                    label="Idioma", interactive=False, scale=1
                )
            gr.Markdown("#### 🔊 Preview de clonación — valida antes de comprometerte")
            preview_text_clone = gr.Textbox(
                label="Texto para el preview",
                value=PREVIEW_TEXT_ES,
                lines=2, interactive=True
            )
            with gr.Row():
                btn_gen_clone_preview = gr.Button(
                    "▶️ Generar preview clonado", variant="primary"
                )
                btn_validate_clone = gr.Button(
                    "✅ Validar esta voz para el guion", variant="primary"
                )
            clone_preview_status = gr.Textbox(
                label="Estado del preview", interactive=False, lines=3
            )
            clone_preview_audio = gr.Audio(
                label="🔊 Preview clonado (escucha antes de validar)",
                type="filepath"
            )

        # ── OPCIÓN B: Diseñar voz con descripción ─────────────
        with gr.Tab("🎨 Opción B — Diseñar voz (VoiceDesign → Referencia)"):
            gr.Markdown(
                "Describe la voz que quieres. "
                "Se generará una **muestra de audio**. "
                "Si te gusta, la adoptamos como referencia y el guion completo "
                "usará **clonación** de esa voz para consistencia máxima."
            )
            gr.Markdown("**Ejemplos de descripción:**")
            gr.Markdown(
                "- `A young female, cheerful, singing naturally, Spanish speaking`\n"
                "- `A teenage girl, shy but expressive, soft and emotional, Spanish`\n"
                "- `A mature woman, warm and determined, clear pronunciation, Spanish`\n"
                "- `A young male, confident, energetic, moderate pitch, Spanish`"
            )
            design_instruct_box = gr.Textbox(
                label="✏️ Descripción de la voz (en inglés para mejores resultados)",
                placeholder="A young female, cheerful, speaking clearly with natural warmth, Spanish speaking",
                lines=3, interactive=True
            )
            preview_text_design = gr.Textbox(
                label="Texto para el preview",
                value=PREVIEW_TEXT_ES,
                lines=2, interactive=True
            )
            with gr.Row():
                btn_gen_design_preview = gr.Button(
                    "▶️ Generar preview VoiceDesign", variant="primary"
                )
                btn_adopt_design = gr.Button(
                    "✅ Usar esta voz como referencia y validar",
                    variant="primary"
                )
            design_preview_status = gr.Textbox(
                label="Estado del diseño", interactive=False, lines=4
            )
            design_preview_audio = gr.Audio(
                label="🔊 Preview de voz diseñada (escucha y compara)",
                type="filepath"
            )
            gr.Markdown(
                "ℹ️ Al pulsar **'Usar esta voz como referencia'**, "
                "el audio del preview se convierte en la referencia de clonación. "
                "A partir de ese momento, el guion completo clona esa voz."
            )

        # ── Estado final de validación ────────────────────────
        gr.Markdown("---")
        with gr.Row():
            btn_validate_final = gr.Button(
                "✅ CONFIRMAR voz como lista para el guion",
                variant="primary", size="lg"
            )
            btn_voice_state_refresh = gr.Button("🔄 Refrescar estado", variant="secondary")
        voice_validation_final_box = gr.Textbox(
            label="✅ Estado de validación final",
            interactive=False, lines=4
        )

    # ════════════════════════════════════════════════════════
    # ETAPA 3 — CANCIÓN
    # ════════════════════════════════════════════════════════
    with gr.Accordion("🎵 ETAPA 3 — Canción con Lipsync", open=True):
        with gr.Row():
            song_file  = gr.File(label="Audio canción",
                                 file_types=[".wav",".mp3",".m4a",".flac"])
            btn_song   = gr.Button("💾 Guardar canción")
            song_status = gr.Textbox(label="Estado canción", interactive=False)

        gr.Markdown("### 🎬 PASO 3.1 — Clips 5s (validación individual)")
        with gr.Row():
            seed_5s       = gr.Number(label="Seed", value=-1, precision=0)
            btn_gen5s_E   = gr.Button("▶️ Clip E", variant="primary")
            btn_gen5s_F   = gr.Button("▶️ Clip F", variant="primary")
            btn_gen5s_G   = gr.Button("▶️ Clip G", variant="primary")
            btn_gen5s_all = gr.Button("🚀 Todos los clips 5s", variant="secondary")
        clips5s_status  = gr.Textbox(label="Estado clips 5s", interactive=False)
        clips5s_gallery = gr.Gallery(label="Clips 5s generados", columns=3, height=200)

        with gr.Accordion("🔄 Regenerar clip 5s específico", open=False):
            with gr.Row():
                clip_path_in  = gr.Textbox(label="Ruta del clip")
                seed_regen5s  = gr.Number(label="Seed", value=-1, precision=0)
                btn_regen5s   = gr.Button("🔄 Regenerar")
            regen5s_status = gr.Textbox(label="Estado", interactive=False)

        gr.Markdown("### 📊 PASO 3.2 — Timeline")
        btn_tl     = gr.Button("📊 Construir timeline", variant="secondary")
        tl_status  = gr.Textbox(label="Estado timeline", interactive=False)
        tl_html    = gr.HTML(
            value="<div style='background:#1a1a1a;border:1px solid #444;"
                  "border-radius:8px;padding:8px;min-height:50px;"
                  "color:#aaa;font-family:monospace;'>Timeline aparecerá aquí.</div>"
        )

        gr.Markdown("### 🧩 Timeline por fragmentos (regenerar lipsync clip a clip)")
        gr.Markdown("Cada clip lipsync muestra su video y un botón **🔄 Regenerar lipsync (LTX)** que rehace "
                    "solo ese tramo con seed nueva (si LTX falla se conserva el anterior). "
                    "Al pulsar «Renderizar Etapa 3», los lipsync ya hechos se reutilizan.")
        tick_e3 = gr.State(0)
        btn_tl_frag = gr.Button("🔄 Actualizar fragmentos", size="sm", variant="secondary")
        regen_tl_st_e3 = gr.Textbox(label="Estado de regeneración (canción)", interactive=False, lines=2)
        _make_song_timeline(tick_e3, regen_tl_st_e3, tpl_dd, personaje)

        gr.Markdown("### ▶️ PASO 3.3 — Renderizar")
        btn_render3  = gr.Button("▶️ Renderizar Etapa 3", variant="primary")
        e3_status    = gr.Textbox(label="Estado Etapa 3", interactive=False)
        e3_video     = gr.Video(label="Preview canción", height=240)

    # ════════════════════════════════════════════════════════
    # ETAPA 2 — DIÁLOGOS INICIALES
    # ════════════════════════════════════════════════════════
    with gr.Accordion("🎙️ ETAPA 2 — Diálogos iniciales", open=False):
        gr.HTML('''<div style="background:#0d2137;border-left:4px solid #2575fc;
                   padding:10px;border-radius:6px;margin-bottom:12px;">
            <p style="color:#7eb8f7;margin:0;font-size:0.9em;">
            🎙️ <b>Requiere voz del concursante validada</b> (sección de arriba).<br>
            Los jueces usarán sus audios de plantilla (auto-transcritos).<br>
            Si no hay audio de juez, se aplica VoiceDesign fallback automático.
            </p></div>''')

        historia_box = gr.Textbox(
            label="Historia personal del concursante",
            placeholder="Ej: 'Perdió su trabajo, cuida a sus hijos sola, "
                        "la música es su única salida...'",
            lines=4
        )
        with gr.Row():
            model_id_e2  = gr.Dropdown(
                label="Modelo xKiro para guion",
                choices=list(XKIRO_MODEL_OPTIONS.values()),
                value=DEFAULT_XKIRO_TEXT_MODEL_ID,
                allow_custom_value=True, scale=3
            )
            fast_mode_e2 = gr.Checkbox(
                label="Fast Mode TTS (sin transcripción ref)",
                value=False, scale=1
            )
        with gr.Row():
            btn_e2_regen_guion = gr.Button("📝 Generar solo guion (sin TTS)",
                                           variant="secondary")
            btn_e2_actual      = gr.Button("⏯️ Generar Etapa 2 con guion actual",
                                           variant="primary")
            btn_e2             = gr.Button("▶️ Ejecutar completa (guion NUEVO + TTS + video)",
                                           variant="secondary")
        with gr.Row():
            btn_e2_preview   = gr.Button("🖼️ Preview con imágenes (audio + imagen, sin lipsync)",
                                         variant="secondary")
            btn_e2_continuar = gr.Button("🎬 Continuar con lipsync (usa guion y audios del preview)",
                                         variant="primary")
        e2_status = gr.Textbox(label="Estado Etapa 2", interactive=False, lines=3)
        e2_preview_video = gr.Video(label="🖼️ Preview con imágenes (Etapa 2)", height=240)
        e2_video  = gr.Video(label="Preview Etapa 2", height=240)

        gr.Markdown("### 🧩 Timeline de fragmentos — Etapa 2")
        gr.Markdown("Cada fragmento muestra su estado y su clip. **🔄 Regenerar lipsync** rehace solo el clip LTX "
                    "de ese fragmento (reutiliza su audio) y reensambla el video de la etapa.")
        tick_e2 = gr.State(0)
        btn_tl_e2 = gr.Button("🔄 Actualizar timeline E2", size="sm", variant="secondary")
        regen_tl_st_e2 = gr.Textbox(label="Estado de regeneración (E2)", interactive=False, lines=2)
        _make_dialog_timeline("etapa2", tick_e2, regen_tl_st_e2, tpl_dd, personaje)

        with gr.Accordion("🔄 Regenerar fragmento individual (E2)", open=False):
            with gr.Row():
                etapa_sel_e2  = gr.Dropdown(label="Etapa", choices=["etapa2","etapa4"],
                                            value="etapa2")
                frag_idx_n    = gr.Number(label="Índice fragmento (0-based)",
                                          value=0, precision=0)
            nuevo_txt = gr.Textbox(label="Nuevo texto (vacío=usar original)", lines=3)
            btn_regen_frag  = gr.Button("🔄 Regenerar fragmento", variant="secondary")
            regen_frag_st   = gr.Textbox(label="Estado", interactive=False)

        with gr.Row():
            btn_ver_guion = gr.Button("📄 Ver guion JSON (E2 + E4)")
        guion_json_box = gr.Code(label="Guion JSON", language="json", lines=20)

        with gr.Accordion("✏️ EDITAR GUION MANUALMENTE", open=False):
            gr.Markdown(
                    "Edita el JSON directamente. Cambia textos, emociones o duraciones.\n"
                    "Al guardar, pulsa **'⏯️ Generar Etapa con guion actual'** para usar "
                    "**este guion editado** (no se llama a la IA).\n"
                    "⚠ 'Ejecutar completa' genera un guion NUEVO (el anterior queda respaldado).\n"
                    "Solo se rehacen audio/video de los fragmentos cuyo texto cambió."
            )
            with gr.Row():
                etapa_edit_sel = gr.Dropdown(
                    label="Etapa a editar",
                    choices=["etapa2", "etapa4"],
                    value="etapa2", scale=1
                )
                btn_cargar_guion_edit = gr.Button(
                    "📥 Cargar guion actual", variant="secondary", scale=1
                )
            guion_edit_box = gr.Code(
                label="Guion JSON editable",
                language="json",
                lines=25,
                interactive=True
            )
            with gr.Row():
                btn_guardar_guion_edit = gr.Button(
                    "💾 Guardar guion editado", variant="primary"
                )
                guion_edit_status = gr.Textbox(label="Estado", interactive=False)

            # Wiring
            btn_cargar_guion_edit.click(
                fn=lambda et: json.dumps(
                    (_get_vp().estado.get(et, {}).get("guion_json", {})),
                    ensure_ascii=False, indent=2
                ),
                inputs=[etapa_edit_sel],
                outputs=[guion_edit_box]
            )
            btn_guardar_guion_edit.click(
                fn=ui_guardar_guion_editado,
                inputs=[etapa_edit_sel, guion_edit_box],
                outputs=[guion_edit_status]
            )


    # ════════════════════════════════════════════════════════
    # ETAPA 4 — DIÁLOGOS FINALES
    # ════════════════════════════════════════════════════════
    with gr.Accordion("🏆 ETAPA 4 — Diálogos finales de los jueces", open=False):
        gr.HTML('''<div style="background:#0d2137;border-left:4px solid #6a11cb;
                   padding:10px;border-radius:6px;margin-bottom:12px;">
            <p style="color:#b0b8ff;margin:0;font-size:0.9em;">
            🎙️ <b>Requiere voz del concursante validada</b> (sección de arriba).<br>
            En esta etapa hablan principalmente los jueces (≥70% del tiempo).<br>
            La misma referencia del concursante se usa si éste responde brevemente.
            </p></div>''')

        ctx_e4_box = gr.Textbox(
            label="Contexto para el guion (opcional) — historia del concursante / cómo fue su actuación",
            placeholder="Ej: 'Perdió su trabajo, cuida a sus hijos sola, la música es su única salida... "
                        "cantó con la voz quebrada y el público se puso de pie...'",
            lines=4,
            info="Se usa al generar un guion NUEVO de la Etapa 4. Vacío = comportamiento de siempre."
        )

        with gr.Row():
            model_id_e4  = gr.Dropdown(
                label="Modelo xKiro para guion",
                choices=list(XKIRO_MODEL_OPTIONS.values()),
                value=DEFAULT_XKIRO_TEXT_MODEL_ID,
                allow_custom_value=True, scale=3
            )
            fast_mode_e4 = gr.Checkbox(
                label="Fast Mode TTS", value=False, scale=1
            )
        with gr.Row():
            btn_e4_regen_guion = gr.Button("📝 Generar solo guion (sin TTS)",
                                           variant="secondary")
            btn_e4_actual      = gr.Button("⏯️ Generar Etapa 4 con guion actual",
                                           variant="primary")
            btn_e4               = gr.Button("▶️ Ejecutar completa (guion NUEVO + TTS + video)",
                                           variant="secondary")
        with gr.Row():
            btn_e4_preview   = gr.Button("🖼️ Preview con imágenes (audio + imagen, sin lipsync)",
                                         variant="secondary")
            btn_e4_continuar = gr.Button("🎬 Continuar con lipsync (usa guion y audios del preview)",
                                         variant="primary")
        e4_status = gr.Textbox(label="Estado Etapa 4", interactive=False, lines=3)
        e4_preview_video = gr.Video(label="🖼️ Preview con imágenes (Etapa 4)", height=240)
        e4_video  = gr.Video(label="Preview Etapa 4", height=240)

        gr.Markdown("### 🧩 Timeline de fragmentos — Etapa 4")
        gr.Markdown("Cada fragmento muestra su estado y su clip. **🔄 Regenerar lipsync** rehace solo el clip LTX "
                    "de ese fragmento (reutiliza su audio) y reensambla el video de la etapa.")
        tick_e4 = gr.State(0)
        btn_tl_e4 = gr.Button("🔄 Actualizar timeline E4", size="sm", variant="secondary")
        regen_tl_st_e4 = gr.Textbox(label="Estado de regeneración (E4)", interactive=False, lines=2)
        _make_dialog_timeline("etapa4", tick_e4, regen_tl_st_e4, tpl_dd, personaje)

        with gr.Accordion("🔄 Regenerar fragmento individual (E4)", open=False):
            with gr.Row():
                etapa_sel_e4  = gr.Dropdown(label="Etapa", choices=["etapa2","etapa4"],
                                            value="etapa4")
                frag_idx_e4   = gr.Number(label="Índice fragmento (0-based)",
                                           value=0, precision=0)
            nuevo_txt_e4 = gr.Textbox(label="Nuevo texto (vacío=usar original)", lines=3)
            btn_regen_frag_e4 = gr.Button("🔄 Regenerar fragmento E4", variant="secondary")
            regen_frag_st_e4  = gr.Textbox(label="Estado", interactive=False)

    # ════════════════════════════════════════════════════════
    # RENDER FINAL
    # ════════════════════════════════════════════════════════
    # ════════════════════════════════════════════════════════
    # 🚀 AUTOMATIZACIÓN (v7)
    # ════════════════════════════════════════════════════════
    with gr.Accordion("🚀 AUTOMATIZACIÓN — generar varias etapas de una vez", open=False):
        gr.Markdown(
            "Marca qué etapas generar y pulsa **Generar lo marcado**. Si desmarcas una (o las dos de diálogos) "
            "esa etapa se omite; con solo «Canción» marcada se genera únicamente la canción. "
            "Cada etapa es independiente: si una falla se anota y se sigue con las demás. "
            "Orden: Etapa 2 → Canción → Etapa 4. Requiere voz validada (E2/E4) y canción guardada (Canción).\n\n"
            "💡 Para **validar primero y rehacer después**: modo «Solo preview con imágenes», revisa el timeline y "
            "luego «🎬 Continuar con lipsync» en cada etapa (reutiliza guion y audios)."
        )
        with gr.Row():
            auto_e2 = gr.Checkbox(label="🎙️ Etapa 2 (diálogos iniciales)", value=True)
            auto_e3 = gr.Checkbox(label="🎵 Canción (Etapa 3)", value=True)
            auto_e4 = gr.Checkbox(label="🏆 Etapa 4 (jueces finales)", value=True)
        with gr.Row():
            auto_modo_dialogo = gr.Radio(label="Modo de diálogos (E2/E4)",
                                         choices=list(_MODOS_DIALOGO.keys()),
                                         value=list(_MODOS_DIALOGO.keys())[0])
            auto_modo_guion = gr.Radio(label="Guion de E2/E4",
                                       choices=list(_MODOS_GUION.keys()),
                                       value=list(_MODOS_GUION.keys())[0])
        with gr.Row():
            auto_ctx_e2 = gr.Textbox(
                label="📝 Contexto del guion — Etapa 2 (opcional)",
                lines=4,
                placeholder="Ej: historia personal del concursante...",
                info="Si lo rellenas, sustituye a «Historia personal» de la Etapa 2 solo dentro de la automatización."
            )
            auto_ctx_e4 = gr.Textbox(
                label="📝 Contexto del guion — Etapa 4 (opcional)",
                lines=4,
                placeholder="Ej: la misma historia + cómo fue la actuación...",
                info="Si está vacío se usa el contexto de la sección Etapa 4."
            )
        gr.Markdown("ℹ️ Los contextos solo se aplican cuando se **escribe un guion nuevo**: elige "
                    "«Generar guion NUEVO» (con «Reutilizar» y un guion ya existente no se vuelve a escribir).")
        with gr.Row():
            auto_reconstruir_tl = gr.Checkbox(
                label="Reconstruir el timeline de la canción (si no, se conserva el que ya validaste)",
                value=False)
            auto_render_final = gr.Checkbox(
                label="Renderizar VIDEO FINAL al terminar (solo con diálogos en modo Completo)", value=False)
        with gr.Row():
            btn_auto_todo = gr.Button("🚀 Generar lo marcado", variant="primary", size="lg")
            btn_auto_e2e4 = gr.Button("🎙️ Solo Etapas 2 y 4 (juntas)", variant="secondary")
            btn_auto_cancion = gr.Button("🎵 Solo canción", variant="secondary")
        auto_status = gr.Textbox(label="Progreso de la automatización", interactive=False, lines=10)
        auto_video  = gr.Video(label="Último video generado", height=300)

    with gr.Accordion("🎞️ RENDER FINAL", open=True):
        with gr.Row():
            etapa2_on = gr.Checkbox(label="Incluir Etapa 2 (diálogos iniciales)", value=True)
            gr.Markdown("**Etapa 3: siempre incluida**")
            etapa4_on = gr.Checkbox(label="Incluir Etapa 4 (jueces finales)", value=True)
        btn_final    = gr.Button("🏁 Renderizar VIDEO FINAL", variant="primary", size="lg")
        final_status = gr.Textbox(label="Estado", interactive=False)
        final_video  = gr.Video(label="VIDEO FINAL", height=360)

    # ════════════════════════════════════════════════════════
    # UTILIDADES
    # ════════════════════════════════════════════════════════
    with gr.Accordion("🛠️ Utilidades", open=False):
        with gr.Row():
            btn_estado  = gr.Button("📋 Ver estado completo")
            btn_dataset = gr.Button("☁️ Publicar → Kaggle Dataset")
        util_status = gr.Textbox(label="Info", interactive=False, lines=10)

    # ════════════════════════════════════════════════════════
    # LOGS
    # ════════════════════════════════════════════════════════
    with gr.Accordion("📟 Consola (run.log)", open=True):
        logs_box = gr.Textbox(label="Logs", value=tail_log(),
                              lines=28, max_lines=50, interactive=False)
    timer = gr.Timer(2.5)

    # ════════════════════════════════════════════════════════
    # WIRING — TODOS LOS EVENTOS
    # ════════════════════════════════════════════════════════

  # ── REEMPLAZAR POR (carga mínima inmediata):
    demo.load(
        fn=ui_initial_load,
        inputs=[],
        outputs=[progreso_box, progreso_gallery, video_selector, ruta_label, tpl_dd],
    )

    # Reanudación
    btn_cargar.click(
        fn=ui_cargar_video_seleccionado, inputs=[video_selector],
        outputs=[progreso_box, progreso_gallery, video_activo_label, ruta_label]
    ).then(fn=ui_refresh_tpl, inputs=[], outputs=[tpl_dd]).then(
        fn=_bump3, inputs=[tick_e2, tick_e3, tick_e4], outputs=[tick_e2, tick_e3, tick_e4])
    btn_refprog.click(fn=ui_refrescar_progreso, inputs=[],
                      outputs=[progreso_box, progreso_gallery])
    btn_listar.click(fn=ui_listar_videos, inputs=[], outputs=[video_selector])
    btn_abrir.click(fn=lambda: _CUR_VP_DIR or "", inputs=[], outputs=[ruta_label])

    # Plantillas
    btn_refresh_tpl.click(ui_refresh_tpl, [], [tpl_dd])
    btn_crear_tpl.click(ui_crear_plantilla, [new_tpl_box], [status_main, tpl_dd])
    btn_assets.click(
        ui_subir_assets_juez_independiente,
        [tpl_dd, j1_img, j1_aud, j2_img, j2_aud, j3_img, j3_aud, j4_img, j4_aud,
         esc_cercano, escs_lejanos, reac1, reac2],
        [assets_status]
    )
    btn_transcribe_judges.click(
        fn=ui_transcribe_judges, inputs=[tpl_dd], outputs=[assets_status]
    )

    # Nuevo video
    btn_nuevo_video.click(
        ui_nuevo_video, [tpl_dd, personaje],
        [status_main, video_dir_box, tpl_dd]
    ).then(ui_scan_ultimo_video, [],
           [progreso_box, progreso_gallery, video_selector, ruta_label]).then(
        _bump3, [tick_e2, tick_e3, tick_e4], [tick_e2, tick_e3, tick_e4])

    # LTX
    btn_ltx_dl.click(ui_descargar_ltx, [ltx_preset_dd], [ltx_status])

    # ── ETAPA 1 GRANULAR ──────────────────────────────────
    e1_inputs = [personaje, p_reemplazar, edad, vestuario, planos]

    def _make_paso_fn(key):
        def paso_wrapper(upload_file, personaje, p_reemplazar, edad, vestuario, planos):
            return ui_etapa1_paso(key, upload_file, personaje, p_reemplazar, edad, vestuario, planos)
        return paso_wrapper

    def _make_cont_fn(start_key):
        def cont_wrapper(personaje, p_reemplazar, edad, vestuario, planos):
            return ui_etapa1_continuar_desde(start_key, personaje, p_reemplazar, edad, vestuario, planos)
        return cont_wrapper

    btn_A.click(_make_paso_fn("A"), [upload_A]+e1_inputs, [status_A, preview_A]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_B.click(_make_paso_fn("B"), [upload_B]+e1_inputs, [status_B, preview_B]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_C.click(_make_paso_fn("C"), [upload_C]+e1_inputs, [status_C, preview_C]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_D.click(_make_paso_fn("D"), [upload_D]+e1_inputs, [status_D, preview_D]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_E.click(_make_paso_fn("E"), [upload_E]+e1_inputs, [status_EFG, preview_A]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_F.click(_make_paso_fn("F"), [upload_F]+e1_inputs, [status_EFG, preview_A]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_G.click(_make_paso_fn("G"), [upload_G]+e1_inputs, [status_EFG, preview_A]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])

    btn_cont_A.click(_make_cont_fn("A"), e1_inputs, [e1_status]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_cont_B.click(_make_cont_fn("B"), e1_inputs, [e1_status]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_cont_C.click(_make_cont_fn("C"), e1_inputs, [e1_status]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_cont_D.click(_make_cont_fn("D"), e1_inputs, [e1_status]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_cont_E.click(_make_cont_fn("E"), e1_inputs, [e1_status]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])

    btn_e1_todo.click(
        ui_etapa1_todo,
        e1_inputs + [dq, eq, mq, stA, stE, cfg_v, seed_v],
        [e1_status, e1_gallery]
    ).then(ui_refrescar_progreso, [], [progreso_box, progreso_gallery])

    # ── VOZ DEL CONCURSANTE ───────────────────────────────
    btn_show_voice_state.click(
        fn=ui_show_voice_state, inputs=[],
        outputs=[voice_state_box]
    )
    btn_voice_state_refresh.click(
        fn=ui_show_voice_state, inputs=[],
        outputs=[voice_state_box]
    )

    # Opción A: subir audio
    btn_process_contestant.click(
        fn=ui_upload_contestant_audio,
        inputs=[contestant_audio_upload],
        outputs=[contestant_upload_status,
                 contestant_transcription_box,
                 contestant_lang_box,
                 btn_gen_clone_preview]
    ).then(fn=ui_show_voice_state, inputs=[], outputs=[voice_state_box])

    btn_gen_clone_preview.click(
        fn=ui_generate_clone_preview,
        inputs=[contestant_transcription_box, preview_text_clone],
        outputs=[clone_preview_status, clone_preview_audio]
    )

    btn_validate_clone.click(
        fn=ui_validate_contestant_voice,
        inputs=[],
        outputs=[voice_validation_final_box]
    ).then(fn=ui_show_voice_state, inputs=[], outputs=[voice_state_box]).then(
        fn=ui_refrescar_progreso, inputs=[], outputs=[progreso_box, progreso_gallery])

    # Opción B: VoiceDesign
    btn_gen_design_preview.click(
        fn=ui_generate_design_preview,
        inputs=[design_instruct_box, preview_text_design],
        outputs=[design_preview_status, design_preview_audio]
    )

    btn_adopt_design.click(
        fn=ui_adopt_design_as_ref,
        inputs=[design_instruct_box],
        outputs=[design_preview_status, btn_validate_final]
    ).then(
        fn=ui_validate_contestant_voice,
        inputs=[],
        outputs=[voice_validation_final_box]
    ).then(fn=ui_show_voice_state, inputs=[], outputs=[voice_state_box]).then(
        fn=ui_refrescar_progreso, inputs=[], outputs=[progreso_box, progreso_gallery])

    # Validación final (botón grande)
    btn_validate_final.click(
        fn=ui_validate_contestant_voice,
        inputs=[],
        outputs=[voice_validation_final_box]
    ).then(fn=ui_show_voice_state, inputs=[], outputs=[voice_state_box]).then(
        fn=ui_refrescar_progreso, inputs=[], outputs=[progreso_box, progreso_gallery])

    # ── ETAPA 3 GRANULAR ─────────────────────────────────
    btn_song.click(ui_subir_cancion, [song_file], [song_status])

    def _make_gen5s_fn(key):
        def gen5s_wrapper(personaje, seed_v):
            return ui_gen_5s_individual(key, personaje, seed_v)
        return gen5s_wrapper

    btn_gen5s_E.click(_make_gen5s_fn("E"), [personaje, seed_5s],
                      [clips5s_status, clips5s_gallery]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_gen5s_F.click(_make_gen5s_fn("F"), [personaje, seed_5s],
                      [clips5s_status, clips5s_gallery]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_gen5s_G.click(_make_gen5s_fn("G"), [personaje, seed_5s],
                      [clips5s_status, clips5s_gallery]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])
    btn_gen5s_all.click(ui_gen_5s_todos, [personaje, seed_5s],
                        [clips5s_status, clips5s_gallery]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery])

    btn_regen5s.click(ui_regen_5s, [tpl_dd, personaje, clip_path_in, seed_regen5s],
                      [regen5s_status, clip_path_in])
    btn_tl.click(ui_build_tl, [tpl_dd, personaje], [tl_status, tl_html]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(_bump, [tick_e3], [tick_e3])
    btn_render3.click(ui_renderizar_cancion, [tpl_dd, personaje],
                      [e3_status, e3_video]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(_bump, [tick_e3], [tick_e3])

    # ── ETAPA 2 ───────────────────────────────────────────
    btn_e2.click(
        fn=ui_etapa2,
        inputs=[tpl_dd, personaje, historia_box, model_id_e2, fast_mode_e2],
        outputs=[e2_status, e2_video]
    ).then(ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(_bump, [tick_e2], [tick_e2])

    btn_e2_regen_guion.click(
        fn=lambda tpl, personaje, historia, mid: ui_generar_solo_guion(
            "etapa2", tpl, personaje, historia, mid),
        inputs=[tpl_dd, personaje, historia_box, model_id_e2],
        outputs=[e2_status, guion_edit_box, etapa_edit_sel]
    ).then(_bump, [tick_e2], [tick_e2])

    btn_e2_actual.click(
        fn=ui_etapa2_guion_actual,
        inputs=[tpl_dd, personaje, historia_box, model_id_e2, fast_mode_e2],
        outputs=[e2_status, e2_video]
    ).then(ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(_bump, [tick_e2], [tick_e2])

    btn_regen_frag.click(
        fn=ui_regen_frag,
        inputs=[tpl_dd, personaje, etapa_sel_e2, frag_idx_n, nuevo_txt],
        outputs=[regen_frag_st]
    ).then(_bump, [tick_e2], [tick_e2]).then(_bump, [tick_e4], [tick_e4])

    # ── ETAPA 4 ───────────────────────────────────────────
    btn_e4.click(
        fn=ui_etapa4,
        inputs=[tpl_dd, personaje, model_id_e4, fast_mode_e4],
        outputs=[e4_status, e4_video]
    ).then(ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(_bump, [tick_e4], [tick_e4])

    btn_e4_regen_guion.click(
        fn=lambda tpl, personaje, mid: ui_generar_solo_guion(
            "etapa4", tpl, personaje, "", mid),
        inputs=[tpl_dd, personaje, model_id_e4],
        outputs=[e4_status, guion_edit_box, etapa_edit_sel]
    ).then(_bump, [tick_e4], [tick_e4])

    btn_e4_actual.click(
        fn=ui_etapa4_guion_actual,
        inputs=[tpl_dd, personaje, model_id_e4, fast_mode_e4],
        outputs=[e4_status, e4_video]
    ).then(ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(_bump, [tick_e4], [tick_e4])

    btn_regen_frag_e4.click(
        fn=ui_regen_frag,
        inputs=[tpl_dd, personaje, etapa_sel_e4, frag_idx_e4, nuevo_txt_e4],
        outputs=[regen_frag_st_e4]
    ).then(_bump, [tick_e2], [tick_e2]).then(_bump, [tick_e4], [tick_e4])

    btn_ver_guion.click(fn=ui_ver_guion_json, inputs=[], outputs=[guion_json_box])

    # ── RENDER FINAL ──────────────────────────────────────
    btn_final.click(
        fn=ui_render_final,
        inputs=[tpl_dd, etapa2_on, etapa4_on],
        outputs=[final_status, final_video]
    ).then(ui_refrescar_progreso, [], [progreso_box, progreso_gallery])

    # ── UTILIDADES ────────────────────────────────────────
    btn_estado.click(fn=ui_ver_estado, inputs=[], outputs=[util_status])
    btn_dataset.click(fn=ui_publicar_dataset, inputs=[], outputs=[util_status])

    # ══════════════════════════════════════════════════════
    # v7 — WIRING NUEVO
    # ══════════════════════════════════════════════════════
    # Diagnostico del entorno LTX
    btn_ltx_check.click(ui_probar_entorno_ltx, [], [ltx_check_status])

    # Preview con imagenes + Continuar con lipsync (E2)
    _e2_in = [tpl_dd, personaje, historia_box, model_id_e2, fast_mode_e2]
    btn_e2_preview.click(ui_etapa2_preview, _e2_in, [e2_status, e2_preview_video]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(
        _bump, [tick_e2], [tick_e2])
    btn_e2_continuar.click(ui_etapa2_guion_actual, _e2_in, [e2_status, e2_video]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(
        _bump, [tick_e2], [tick_e2])

    # Preview con imagenes + Continuar con lipsync (E4)
    _e4_in = [tpl_dd, personaje, model_id_e4, fast_mode_e4]
    btn_e4_preview.click(ui_etapa4_preview, _e4_in, [e4_status, e4_preview_video]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(
        _bump, [tick_e4], [tick_e4])
    btn_e4_continuar.click(ui_etapa4_guion_actual, _e4_in, [e4_status, e4_video]).then(
        ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(
        _bump, [tick_e4], [tick_e4])

    # Refresco manual de los timelines por fragmentos
    btn_tl_e2.click(_bump, [tick_e2], [tick_e2])
    btn_tl_e4.click(_bump, [tick_e4], [tick_e4])
    btn_tl_frag.click(_bump, [tick_e3], [tick_e3])

    # Automatizacion
    btn_auto_todo.click(
        ui_automatizar,
        [tpl_dd, personaje, historia_box, model_id_e2, model_id_e4, fast_mode_e2, fast_mode_e4,
         auto_e2, auto_e3, auto_e4, auto_modo_dialogo, auto_modo_guion,
         auto_reconstruir_tl, auto_render_final],
        [auto_status, auto_video]
    ).then(ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(
        _bump3, [tick_e2, tick_e3, tick_e4], [tick_e2, tick_e3, tick_e4])
    btn_auto_e2e4.click(
        ui_auto_e2_e4,
        [tpl_dd, personaje, historia_box, model_id_e2, model_id_e4, fast_mode_e2, fast_mode_e4,
         auto_modo_dialogo, auto_modo_guion, auto_render_final],
        [auto_status, auto_video]
    ).then(ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(
        _bump3, [tick_e2, tick_e3, tick_e4], [tick_e2, tick_e3, tick_e4])
    btn_auto_cancion.click(
        ui_auto_solo_cancion,
        [tpl_dd, personaje, auto_reconstruir_tl],
        [auto_status, auto_video]
    ).then(ui_refrescar_progreso, [], [progreso_box, progreso_gallery]).then(
        _bump3, [tick_e2, tick_e3, tick_e4], [tick_e2, tick_e3, tick_e4])

    # ── v8: contexto de guiones (E4 y Automatización) ──────
    ctx_e4_box.change(fn=_ctx_setter("e4_ui"), inputs=[ctx_e4_box], outputs=None,
                      queue=False, show_progress="hidden", api_name=False)
    auto_ctx_e2.change(fn=_ctx_setter("e2_auto"), inputs=[auto_ctx_e2], outputs=None,
                       queue=False, show_progress="hidden", api_name=False)
    auto_ctx_e4.change(fn=_ctx_setter("e4_auto"), inputs=[auto_ctx_e4], outputs=None,
                       queue=False, show_progress="hidden", api_name=False)
    demo.load(fn=_ctx_cargar, inputs=[], outputs=[ctx_e4_box, auto_ctx_e2, auto_ctx_e4],
              show_progress="hidden", api_name=False)
    # ── Logs (auto-refresh cada 2.5s) ─────────────────────
    timer.tick(fn=lambda: tail_log(12000), inputs=[], outputs=[logs_box])


# ════════════════════════════════════════════════════════════════
# LAUNCH — CSS y theme van aquí en Gradio 6
# ════════════════════════════════════════════════════════════════
demo.queue(max_size=10).launch(share=True, show_error=True)

✅ HfFolder ya disponible (huggingface_hub huggingface_hub.utils)


/tmp/ipykernel_1614/2423657221.py:1513: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=AIQUEST_CSS, theme=gr.themes.Soft(),


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://719725217879d55648.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[14:16:33] [UI] Descargando modelos LTX-2.5 — preset: ⚡ RÁPIDO — LTX-2.5 Distilled INT8 (~25 GB /tmp, 8 steps)
[14:28:21] [PASO 3.3] Lipsync ya generados → solo se ensambla (no se regenera nada).
[14:28:56] [PASO 3.3] c0010 lipsync vigente → reutilizado (no se regenera).
[14:29:08] [PASO 3.3] c0014 lipsync vigente → reutilizado (no se regenera).
[14:29:19] [PASO 3.3] c0018 lipsync vigente → reutilizado (no se regenera).
[14:29:29] [PASO 3.3] c0022 lipsync vigente → reutilizado (no se regenera).
[14:29:42] [PASO 3.3] c0026 lipsync vigente → reutilizado (no se regenera).
[14:29:53] [PASO 3.3] c0030 lipsync vigente → reutilizado (no se regenera).
[14:30:03] [PASO 3.3] c0034 lipsync vigente → reutilizado (no se regenera).
[14:30:14] [PASO 3.3] c0038 lipsync vigente → reutilizado (no se regenera).
[14:30:25] [PASO 3.3] c0042 lipsync vigente → reutilizado (no se regenera).
[14:33:37] [PASO 3.3] Video: 244.542s | Audio: 244.800s | diff: 258.3ms
[14:33:45] [ETAPA 3] ✅ COMPLETADA → /kaggle/work

In [23]:
# ═══════════════════════════════════════════════════════════════
# CELDA FINAL: 📺 SUBIDA A YOUTUBE (multi-canal) + xKiro
#  - Varios canales (cada uno con su propia API/OAuth) guardados en JSON
#  - xKiro genera títulos (eliges 1), descripción, tags y hashtags
#  - Todo editable antes de subir; ajustes por canal que se autorrellenan
#  - Subida resumible, miniatura, lista de reproducción, programación
# No instala nada ni modifica otras celdas.
# ═══════════════════════════════════════════════════════════════
import os, sys, json, re, time, mimetypes, subprocess, threading, tempfile
from datetime import datetime, timezone
import requests
import gradio as gr

CONFIG_DIR  = "/kaggle/working/yt_uploader"          # ← contiene secretos/tokens
CONFIG_PATH = os.path.join(CONFIG_DIR, "config.json")
os.makedirs(CONFIG_DIR, exist_ok=True)
_CFG_LOCK = threading.Lock()

OAUTH_SCOPE = "https://www.googleapis.com/auth/youtube"
YT_API      = "https://www.googleapis.com/youtube/v3"
CHUNK       = 8 * 1024 * 1024                         # múltiplo de 256 KiB

# ───────────────────────── Config persistente ─────────────────────────
def _load_cfg():
    try:
        with open(CONFIG_PATH, encoding="utf-8") as f:
            cfg = json.load(f)
    except Exception:
        cfg = {}
    cfg.setdefault("channels", {})
    cfg.setdefault("history", [])
    cfg.setdefault("last_channel", "")
    cfg.setdefault("last_client", {})
    return cfg

def _save_cfg(cfg):
    with _CFG_LOCK:
        tmp = CONFIG_PATH + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(cfg, f, ensure_ascii=False, indent=2)
        os.replace(tmp, CONFIG_PATH)
        try: os.chmod(CONFIG_PATH, 0o600)
        except Exception: pass

DEFAULT_PROFILE = {
    "privacy": "private", "category": "22", "language": "es",
    "tags": "", "footer": "",
    "kids": False, "synthetic": True, "embeddable": True, "stats": True,
    "license": "youtube", "notify": False, "playlist": "",
    "ai_context": "", "timezone": "UTC", "ai_model": "", "n_titles": 6,
}
PROF_KEYS = ["privacy", "category", "language", "tags", "footer", "kids", "synthetic",
             "embeddable", "stats", "license", "notify", "playlist", "ai_context",
             "timezone", "ai_model", "n_titles"]

def _profile(alias):
    p = dict(DEFAULT_PROFILE)
    p.update(_load_cfg()["channels"].get(alias, {}).get("defaults", {}))
    return p

def _ch_choices():
    cfg = _load_cfg()
    return [(f"{a} — {c.get('channel_title') or 'sin verificar'}", a) for a, c in cfg["channels"].items()]

# ───────────────────────── Utilidades ─────────────────────────
def _clean_text(s):
    return (s or "").replace("<", "").replace(">", "").strip()

def _parse_tags(s, limit=450):
    raw = re.split(r"[,\n]", s or "")
    out, seen, total = [], set(), 0
    for t in raw:
        t = t.strip().lstrip("#").strip()
        t = _clean_text(t)
        if not t or t.lower() in seen:
            continue
        cost = len(t) + (2 if " " in t else 0) + 1
        if total + cost > limit:
            break
        seen.add(t.lower()); out.append(t); total += cost
    return out

def _yt_err(r):
    try:
        e = r.json().get("error", {})
        reasons = ",".join(x.get("reason", "") for x in e.get("errors", []) if isinstance(x, dict))
        return f"HTTP {r.status_code}: {e.get('message', r.text[:300])} [{reasons}]"
    except Exception:
        return f"HTTP {r.status_code}: {r.text[:300]}"

_TOK = {}
def _token(alias):
    ch = _load_cfg()["channels"].get(alias)
    if not ch:
        raise RuntimeError(f"Canal '{alias}' no encontrado.")
    c = _TOK.get(alias)
    if c and c["exp"] > time.time() + 90:
        return c["tok"]
    r = requests.post("https://oauth2.googleapis.com/token", data={
        "client_id": ch["client_id"], "client_secret": ch["client_secret"],
        "refresh_token": ch["refresh_token"], "grant_type": "refresh_token"}, timeout=30)
    j = r.json()
    if "access_token" not in j:
        raise RuntimeError(f"No se pudo renovar el token de '{alias}': {j.get('error')} "
                           f"{j.get('error_description', '')}. Reconecta el canal en la pestaña 🔑.")
    _TOK[alias] = {"tok": j["access_token"], "exp": time.time() + int(j.get("expires_in", 3500))}
    return j["access_token"]

def _auth(alias):
    return {"Authorization": f"Bearer {_token(alias)}"}

def _channel_info(alias):
    r = requests.get(f"{YT_API}/channels", params={"part": "snippet", "mine": "true"},
                     headers=_auth(alias), timeout=30)
    if r.status_code != 200:
        raise RuntimeError(_yt_err(r))
    items = r.json().get("items", [])
    if not items:
        return None
    return {"id": items[0]["id"], "title": items[0]["snippet"]["title"]}

# ───────────────────────── xKiro ─────────────────────────
def _xk_models():
    m = globals().get("XKIRO_MODEL_OPTIONS")
    if isinstance(m, dict) and m:
        return dict(m)
    d = globals().get("DEFAULT_XKIRO_TEXT_MODEL_ID")
    return {str(d): d} if d else {}

def _xk_json(system_prompt, user_prompt, model_id):
    fn = globals().get("call_xkiro_json")
    if callable(fn):
        return fn(system_prompt, user_prompt, model_id=model_id, temperature=0.8, max_tokens=3500)
    from openai import OpenAI
    key = os.environ.get("XKIRO_API_KEY", "").strip()
    if not key:
        raise RuntimeError("Falta XKIRO_API_KEY.")
    cli = OpenAI(api_key=key, base_url=globals().get("XKIRO_BASE_URL", "https://api.xkiro.com/v1"))
    resp = cli.chat.completions.create(
        model=model_id, temperature=0.8, max_tokens=3500,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}])
    txt = (resp.choices[0].message.content or "").strip()
    txt = re.sub(r"^```(?:json)?", "", txt).rstrip("`").strip()
    return json.loads(txt)

def ai_generate(prompt, ctx, model_name, n, tags_cur, footer, lang):
    if not (prompt or "").strip():
        raise gr.Error("Escribe el prompt / idea del video.")
    models = _xk_models()
    model_id = models.get(model_name) or globals().get("DEFAULT_XKIRO_TEXT_MODEL_ID")
    if not model_id:
        raise gr.Error("No hay modelo xKiro disponible (ejecuta la celda de xKiro primero).")
    n = int(n or 6)
    system = ("Eres un experto en SEO y estrategia de YouTube. "
              "Respondes ÚNICAMENTE con JSON válido, sin markdown ni texto extra.")
    user = f"""IDEA / CONTEXTO DEL VIDEO:
{prompt.strip()}

CONTEXTO DEL CANAL (tono, público, estilo):
{(ctx or '').strip() or 'No especificado'}

IDIOMA DE SALIDA (código): {lang or 'es'}

Genera:
- "titles": {n} títulos distintos entre sí (curiosidad, directo, emocional, SEO...). Ideal ≤ 70 caracteres, NUNCA más de 100. Sin clickbait falso, sin los símbolos < >.
- "description": descripción lista para publicar. Las 2 primeras líneas son un gancho con palabras clave. 150–300 palabras, párrafos cortos. NO inventes enlaces, datos ni marcas de tiempo. Sin hashtags dentro del texto.
- "tags": 12 a 20 etiquetas relevantes (sin #).
- "hashtags": 3 hashtags (con #).

Formato exacto:
{{"titles": ["..."], "description": "...", "tags": ["..."], "hashtags": ["#..."]}}"""
    try:
        data = _xk_json(system, user, model_id)
    except Exception as e:
        raise gr.Error(f"xKiro falló: {e}")
    titles, seen = [], set()
    for t in data.get("titles", []):
        t = _clean_text(str(t)).strip('"“”')[:100]
        if t and t.lower() not in seen:
            seen.add(t.lower()); titles.append(t)
    desc = _clean_text(str(data.get("description", "")))
    if footer and footer.strip():
        desc += "\n\n" + _clean_text(footer)
    tags_ai = data.get("tags", [])
    if isinstance(tags_ai, str):
        tags_ai = tags_ai.split(",")
    merged = _parse_tags(", ".join([*(_parse_tags(tags_cur)), *map(str, tags_ai)]))
    hs = []
    for h in data.get("hashtags", [])[:3]:
        h = "#" + re.sub(r"\s+", "", str(h)).lstrip("#")
        if len(h) > 1: hs.append(h)
    if hs:
        desc += "\n\n" + " ".join(hs)
    return (gr.Radio(choices=titles, value=None), desc, ", ".join(merged),
            f"✅ {len(titles)} títulos generados con {model_name or model_id}. Elige uno abajo.")

def _skip():
    return gr.skip() if hasattr(gr, "skip") else gr.update()

def pick_title(t):
    return t if t else _skip()

# ───────────────────────── Vídeos ─────────────────────────
_SKIP_DIRS = {"Wan2GP", "stable-diffusion.cpp", "qwentts.cpp", "ComfyUI", "env_snapshot",
              "__pycache__", "build", "models", "ckpts", "node_modules", "yt_uploader"}
_VID_EXT = (".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v")

def _scan_videos(limit=80):
    roots = []
    for g in ("VIDEOS_DIR", "OUTPUT_DIR"):
        v = globals().get(g)
        if v and os.path.isdir(v): roots.append(v)
    roots.append("/kaggle/working/outputs")
    found = {}
    for root in roots:
        if not os.path.isdir(root): continue
        for dp, dn, fn in os.walk(root):
            dn[:] = [d for d in dn if d not in _SKIP_DIRS and not d.startswith(".")]
            for f in fn:
                if f.lower().endswith(_VID_EXT):
                    full = os.path.join(dp, f)
                    try: found[full] = (os.path.getmtime(full), os.path.getsize(full))
                    except OSError: pass
    items = sorted(found.items(), key=lambda kv: kv[1][0], reverse=True)[:limit]
    return [(f"{os.path.relpath(p, '/kaggle/working')}  ·  {sz/1024**2:.0f} MB  ·  "
             f"{time.strftime('%d/%m %H:%M', time.localtime(mt))}", p) for p, (mt, sz) in items]

def _resolve_video(sel, man, upl):
    upl = getattr(upl, "name", upl)
    for p in (upl, (man or "").strip().strip("\"'"), sel):
        if p and isinstance(p, str) and os.path.isfile(p):
            return p
    return None

def video_info(sel, man, upl):
    p = _resolve_video(sel, man, upl)
    if not p: return "—"
    size = os.path.getsize(p) / 1024**2
    dur = ""
    try:
        r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                            "-of", "default=noprint_wrappers=1:nokey=1", p],
                           capture_output=True, text=True, timeout=30)
        d = float(r.stdout.strip()); dur = f" · {int(d//60)}:{int(d%60):02d} min"
    except Exception: pass
    return f"📁 {p}\n{size:.1f} MB{dur}"

def extract_frame(sel, man, upl, sec):
    p = _resolve_video(sel, man, upl)
    if not p: raise gr.Error("Elige un video primero.")
    out = os.path.join(tempfile.gettempdir(), f"yt_thumb_{int(time.time())}.jpg")
    r = subprocess.run(["ffmpeg", "-y", "-ss", str(float(sec or 0)), "-i", p, "-frames:v", "1",
                        "-vf", "scale=1280:-2", "-q:v", "3", out], capture_output=True, text=True)
    if r.returncode != 0 or not os.path.isfile(out):
        raise gr.Error("No se pudo extraer el frame.")
    return out

def _prep_thumb(path):
    path = getattr(path, "name", path)
    if not path or not os.path.isfile(path): return None
    ok_ext = path.lower().endswith((".jpg", ".jpeg", ".png"))
    if ok_ext and os.path.getsize(path) <= 2 * 1024 * 1024:
        return path
    out = os.path.join(tempfile.gettempdir(), f"yt_thumb_conv_{int(time.time())}.jpg")
    subprocess.run(["ffmpeg", "-y", "-i", path, "-vf", "scale=1280:-2", "-q:v", "4", out],
                   capture_output=True, text=True)
    return out if os.path.isfile(out) and os.path.getsize(out) <= 2 * 1024 * 1024 else None

# ───────────────────────── Subida a YouTube ─────────────────────────
def _resumable_upload(alias, path, body, notify):
    size = os.path.getsize(path)
    if size == 0: raise RuntimeError("El archivo de video está vacío.")
    mime = mimetypes.guess_type(path)[0] or "video/*"
    params = {"uploadType": "resumable", "part": "snippet,status",
              "notifySubscribers": "true" if notify else "false"}
    def _init():
        h = dict(_auth(alias))
        h.update({"Content-Type": "application/json; charset=UTF-8",
                  "X-Upload-Content-Length": str(size), "X-Upload-Content-Type": mime})
        return requests.post("https://www.googleapis.com/upload/youtube/v3/videos",
                             params=params, headers=h, data=json.dumps(body), timeout=60)
    r = _init()
    if r.status_code == 400 and "containsSyntheticMedia" in r.text:
        body["status"].pop("containsSyntheticMedia", None)
        r = _init()
    if r.status_code != 200:
        raise RuntimeError(_yt_err(r))
    loc = r.headers["Location"]
    offset, fails = 0, 0
    with open(path, "rb") as f:
        while offset < size:
            f.seek(offset)
            chunk = f.read(CHUNK)
            end = offset + len(chunk) - 1
            h = dict(_auth(alias))
            h.update({"Content-Length": str(len(chunk)),
                      "Content-Range": f"bytes {offset}-{end}/{size}", "Content-Type": mime})
            rr = None
            try:
                rr = requests.put(loc, headers=h, data=chunk, timeout=300)
            except requests.RequestException:
                pass
            if rr is not None and rr.status_code in (200, 201):
                yield 100, rr.json(); return
            if rr is not None and rr.status_code == 308:
                rng = rr.headers.get("Range")
                offset = int(rng.split("-")[-1]) + 1 if rng else 0
                fails = 0
                yield int(offset * 100 / size), None
                continue
            if rr is not None and rr.status_code == 401:
                _TOK.pop(alias, None)
            fails += 1
            if fails > 6 or (rr is not None and rr.status_code in (400, 403, 404)):
                raise RuntimeError(_yt_err(rr) if rr is not None else "Error de red persistente")
            time.sleep(min(30, 2 ** fails))
            try:   # preguntar hasta dónde llegó el servidor
                h2 = dict(_auth(alias))
                h2.update({"Content-Length": "0", "Content-Range": f"bytes */{size}"})
                q = requests.put(loc, headers=h2, timeout=60)
                if q.status_code in (200, 201):
                    yield 100, q.json(); return
                if q.status_code == 308:
                    rng = q.headers.get("Range")
                    offset = int(rng.split("-")[-1]) + 1 if rng else 0
            except requests.RequestException:
                pass
    raise RuntimeError("La subida terminó sin respuesta final de YouTube.")

def _parse_sched(s, tz):
    s = (s or "").strip()
    if not s: return None
    from zoneinfo import ZoneInfo
    try: z = ZoneInfo((tz or "UTC").strip() or "UTC")
    except Exception: raise ValueError(f"Zona horaria inválida: {tz}")
    dt = None
    for fmt in ("%Y-%m-%d %H:%M", "%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"):
        try: dt = datetime.strptime(s, fmt); break
        except ValueError: pass
    if not dt: raise ValueError("Fecha programada inválida. Formato: 2026-10-15 18:30")
    dt = dt.replace(tzinfo=z).astimezone(timezone.utc)
    if dt <= datetime.now(timezone.utc): raise ValueError("La fecha programada debe estar en el futuro.")
    return dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")

def _history_md():
    h = _load_cfg()["history"][-8:][::-1]
    if not h: return "_Sin subidas registradas._"
    return "\n".join(f"- {x['t']} · **{x['channel']}** · [{x['title']}](https://youtu.be/{x['id']}) · {x['privacy']}" for x in h)

def do_upload(alias, vsel, vman, vupl, title, desc, tags, privacy, cat, lang, kids, synth,
              emb, stats, lic, notify, playlist, sched, tz, thumb):
    log = []
    def out(): return "\n".join(log), _history_md()
    try:
        if not alias or alias not in _load_cfg()["channels"]:
            raise ValueError("Elige un canal (o conecta uno en la pestaña 🔑).")
        path = _resolve_video(vsel, vman, vupl)
        if not path: raise ValueError("Elige un video (lista, ruta manual o subida).")
        title = _clean_text(title)
        if not title: raise ValueError("Falta el título.")
        if len(title) > 100: raise ValueError(f"El título tiene {len(title)} caracteres (máx. 100).")
        desc = _clean_text(desc)
        if len(desc.encode("utf-8")) > 5000: raise ValueError("La descripción supera 5000 bytes.")
        tag_list = _parse_tags(tags)
        publish_at = _parse_sched(sched, tz)
        if publish_at: privacy = "private"
        body = {
            "snippet": {"title": title, "description": desc, "tags": tag_list,
                        "categoryId": str(cat), "defaultLanguage": lang, "defaultAudioLanguage": lang},
            "status": {"privacyStatus": privacy, "selfDeclaredMadeForKids": bool(kids),
                       "embeddable": bool(emb), "publicStatsViewable": bool(stats),
                       "license": lic, "containsSyntheticMedia": bool(synth)},
        }
        if publish_at: body["status"]["publishAt"] = publish_at
        log.append(f"🚀 Subiendo a «{alias}»: {os.path.basename(path)}"
                   + (f"\n🕒 Programado (UTC): {publish_at}" if publish_at else ""))
        pi = len(log); log.append("⬆️ 0%"); yield out()
        last, resp = -1, None
        for pct, r in _resumable_upload(alias, path, body, notify):
            if pct != last:
                log[pi] = f"⬆️ {pct}%"; last = pct; yield out()
            if r: resp = r
        vid = resp["id"]
        log.append(f"✅ Video subido: https://youtu.be/{vid}")
        tp = _prep_thumb(thumb)
        if thumb and not tp:
            log.append("⚠️ Miniatura omitida (debe ser JPG/PNG ≤ 2 MB).")
        elif tp:
            try:
                h = dict(_auth(alias)); h["Content-Type"] = mimetypes.guess_type(tp)[0] or "image/jpeg"
                with open(tp, "rb") as fh:
                    rt = requests.post("https://www.googleapis.com/upload/youtube/v3/thumbnails/set",
                                       params={"videoId": vid, "uploadType": "media"},
                                       headers=h, data=fh.read(), timeout=120)
                log.append("🖼️ Miniatura aplicada." if rt.status_code == 200
                           else f"⚠️ Miniatura no aplicada: {_yt_err(rt)}")
            except Exception as e:
                log.append(f"⚠️ Miniatura no aplicada: {e}")
        pl = (playlist or "").strip()
        if pl:
            try:
                rp = requests.post(f"{YT_API}/playlistItems", params={"part": "snippet"},
                                   headers=_auth(alias), timeout=60,
                                   json={"snippet": {"playlistId": pl,
                                                     "resourceId": {"kind": "youtube#video", "videoId": vid}}})
                log.append("📚 Añadido a la lista de reproducción." if rp.status_code == 200
                           else f"⚠️ No se añadió a la lista: {_yt_err(rp)}")
            except Exception as e:
                log.append(f"⚠️ No se añadió a la lista: {e}")
        st = resp.get("status", {})
        if st.get("privacyStatus") == "private" and privacy != "private":
            log.append("⚠️ YouTube dejó el video en PRIVADO: el proyecto de API aún no está auditado "
                       "(restricción de YouTube para proyectos nuevos). Ver notas.")
        cfg = _load_cfg()
        cfg["history"].append({"t": time.strftime("%Y-%m-%d %H:%M"), "channel": alias,
                               "title": title, "id": vid, "privacy": st.get("privacyStatus", privacy)})
        cfg["history"] = cfg["history"][-30:]
        _save_cfg(cfg)
        yield out()
    except Exception as e:
        log.append(f"❌ {e}")
        yield out()

# ───────────────────────── Canales (OAuth dispositivo) ─────────────────────────
def dev_start(alias, cid, csec, state):
    alias, cid, csec = (alias or "").strip(), (cid or "").strip(), (csec or "").strip()
    if not (alias and cid and csec):
        raise gr.Error("Rellena alias, Client ID y Client Secret.")
    r = requests.post("https://oauth2.googleapis.com/device/code",
                      data={"client_id": cid, "scope": OAUTH_SCOPE}, timeout=30)
    j = r.json()
    if "device_code" not in j:
        raise gr.Error(f"Google rechazó la solicitud: {j.get('error')} {j.get('error_description', '')}. "
                       "¿El cliente es de tipo «TV y dispositivos de entrada limitada»?")
    state = {"alias": alias, "cid": cid, "csec": csec, "device_code": j["device_code"],
             "interval": int(j.get("interval", 5)), "exp": time.time() + int(j.get("expires_in", 1800))}
    md = (f"### Autoriza el canal «{alias}»\n"
          f"1. Abre **{j.get('verification_url', 'https://www.google.com/device')}**\n"
          f"2. Introduce el código: **`{j['user_code']}`**\n"
          f"3. Elige la cuenta **y el canal correcto** (si tienes canales de marca) y acepta.\n"
          f"4. Vuelve aquí y pulsa **✅ Ya autoricé**.")
    return md, state

def _channels_md():
    cfg = _load_cfg()
    if not cfg["channels"]: return "_Aún no hay canales conectados._"
    return "\n".join(f"- **{a}** → {c.get('channel_title') or 'sin verificar'} "
                     f"`{c.get('channel_id', '')}`" for a, c in cfg["channels"].items())

def dev_finish(state):
    if not state: raise gr.Error("Primero pulsa «Pedir código».")
    interval, tok = state["interval"], None
    while time.time() < state["exp"]:
        j = requests.post("https://oauth2.googleapis.com/token", data={
            "client_id": state["cid"], "client_secret": state["csec"],
            "device_code": state["device_code"],
            "grant_type": "urn:ietf:params:oauth:grant-type:device_code"}, timeout=30).json()
        if "access_token" in j: tok = j; break
        err = j.get("error")
        if err == "authorization_pending": time.sleep(interval); continue
        if err == "slow_down": interval += 5; time.sleep(interval); continue
        raise gr.Error(f"Autorización fallida: {err} {j.get('error_description', '')}")
    if not tok: raise gr.Error("El código expiró. Pide uno nuevo.")
    if not tok.get("refresh_token"):
        raise gr.Error("Google no devolvió refresh_token. Revoca el acceso de la app en tu cuenta y repite.")
    cfg = _load_cfg()
    alias = state["alias"]
    old = cfg["channels"].get(alias, {})
    cfg["channels"][alias] = {"client_id": state["cid"], "client_secret": state["csec"],
                              "refresh_token": tok["refresh_token"],
                              "channel_title": "", "channel_id": "",
                              "defaults": old.get("defaults", {})}
    cfg["last_client"] = {"client_id": state["cid"], "client_secret": state["csec"]}
    cfg["last_channel"] = alias
    _save_cfg(cfg); _TOK.pop(alias, None)
    msg = f"✅ Canal «{alias}» conectado."
    try:
        info = _channel_info(alias)
        if info:
            cfg = _load_cfg()
            cfg["channels"][alias]["channel_title"] = info["title"]
            cfg["channels"][alias]["channel_id"] = info["id"]
            _save_cfg(cfg)
            msg = f"✅ Canal «{alias}» conectado → **{info['title']}**"
        else:
            msg += " ⚠️ La cuenta elegida no tiene canal de YouTube."
    except Exception as e:
        msg += f" ⚠️ No se pudo leer el canal: {e}"
    return (msg, gr.Dropdown(choices=_ch_choices(), value=alias),
            gr.Dropdown(choices=_ch_choices(), value=alias), _channels_md(), None)

def verify_channel(alias):
    if not alias: raise gr.Error("Elige un canal.")
    info = _channel_info(alias)
    if not info: return "⚠️ Token válido pero la cuenta no tiene canal."
    cfg = _load_cfg()
    cfg["channels"][alias]["channel_title"] = info["title"]
    cfg["channels"][alias]["channel_id"] = info["id"]
    _save_cfg(cfg)
    return f"✅ «{alias}» funciona → {info['title']} ({info['id']})"

def delete_channel(alias, confirm):
    if not alias: raise gr.Error("Elige un canal.")
    if not confirm: raise gr.Error("Marca la casilla de confirmación.")
    cfg = _load_cfg()
    cfg["channels"].pop(alias, None)
    if cfg["last_channel"] == alias: cfg["last_channel"] = ""
    _save_cfg(cfg); _TOK.pop(alias, None)
    ch = _ch_choices()
    first = ch[0][1] if ch else None
    return (f"🗑️ Canal «{alias}» eliminado.", gr.Dropdown(choices=ch, value=first),
            gr.Dropdown(choices=ch, value=first), _channels_md())

# ───────────────────────── Perfiles por canal ─────────────────────────
def on_channel_change(ch):
    cfg = _load_cfg()
    if ch and ch in cfg["channels"]:
        cfg["last_channel"] = ch; _save_cfg(cfg)
    p = _profile(ch)
    models = list(_xk_models().keys())
    p["ai_model"] = p["ai_model"] if p["ai_model"] in models else (models[0] if models else None)
    return [p[k] for k in PROF_KEYS]

def save_defaults(ch, *vals):
    if not ch or ch not in _load_cfg()["channels"]:
        raise gr.Error("Elige un canal primero.")
    d = dict(zip(PROF_KEYS, vals))
    d["n_titles"] = int(d.get("n_titles") or 6)
    cfg = _load_cfg()
    cfg["channels"][ch]["defaults"] = d
    _save_cfg(cfg)
    return f"💾 Predeterminados guardados para «{ch}» (se autorrellenarán en los próximos videos)."

def load_playlists(ch, current):
    if not ch: raise gr.Error("Elige un canal.")
    r = requests.get(f"{YT_API}/playlists", params={"part": "snippet", "mine": "true", "maxResults": 50},
                     headers=_auth(ch), timeout=30)
    if r.status_code != 200: raise gr.Error(_yt_err(r))
    items = [("— ninguna —", "")] + [(i["snippet"]["title"], i["id"]) for i in r.json().get("items", [])]
    return gr.Dropdown(choices=items, value=current or "")

# ───────────────────────── UI ─────────────────────────
CATEGORIES = [("Cine y animación", "1"), ("Autos", "2"), ("Música", "10"), ("Mascotas", "15"),
              ("Deportes", "17"), ("Viajes", "19"), ("Videojuegos", "20"), ("Gente y blogs", "22"),
              ("Comedia", "23"), ("Entretenimiento", "24"), ("Noticias", "25"),
              ("Cómo hacer / estilo", "26"), ("Educación", "27"), ("Ciencia y tecnología", "28")]
LANGS = [("Español", "es"), ("English", "en"), ("Português", "pt"), ("Français", "fr"),
         ("Deutsch", "de"), ("Italiano", "it"), ("日本語", "ja"), ("한국어", "ko")]

_cfg0 = _load_cfg()
_ch0 = _ch_choices()
_last0 = _cfg0["last_channel"] if _cfg0["last_channel"] in _cfg0["channels"] else (_ch0[0][1] if _ch0 else None)
_models0 = list(_xk_models().keys())

with gr.Blocks(title="Subida a YouTube") as demo_yt:
    gr.Markdown("# 📺 Subida a YouTube — multi-canal + xKiro")
    with gr.Tabs():
        # ───────── TAB SUBIR ─────────
        with gr.Tab("📤 Subir video"):
            with gr.Row():
                ch_dd = gr.Dropdown(choices=_ch_choices(), value=_last0, label="Canal", scale=3)
            gr.Markdown("### 1) Video")
            with gr.Row():
                vid_dd = gr.Dropdown(choices=_scan_videos(), value=None, label="Videos encontrados en el servidor", scale=4)
                vid_ref = gr.Button("🔄", scale=1)
            with gr.Row():
                vid_man = gr.Textbox(label="…o ruta manual", placeholder="/kaggle/working/.../video.mp4")
                vid_up = gr.File(label="…o subir archivo (lento por el túnel)", type="filepath", file_types=["video"])
            vid_info = gr.Textbox(label="Info del video", interactive=False, lines=2)

            gr.Markdown("### 2) Texto con xKiro")
            with gr.Accordion("🤖 Generar títulos, descripción y tags", open=True):
                prompt_tb = gr.Textbox(label="Prompt / idea del video", lines=4,
                                       placeholder="Ej: Final de Got Talent con cantante tímida que sorprende a los jueces con una balada…")
                ctx_tb = gr.Textbox(label="Contexto del canal (se guarda por canal)", lines=2,
                                    placeholder="Tono, público objetivo, estilo, palabras clave fijas…")
                with gr.Row():
                    model_dd = gr.Dropdown(choices=_models0, value=(_models0[0] if _models0 else None),
                                           label="Modelo xKiro", allow_custom_value=True)
                    ntitles_sl = gr.Slider(3, 10, value=6, step=1, label="Nº de títulos")
                gen_btn = gr.Button("✨ Generar con xKiro", variant="primary")
                ai_status = gr.Markdown()
                titles_rd = gr.Radio(choices=[], label="Títulos sugeridos (elige uno → pasa al campo Título)")
            title_tb = gr.Textbox(label="Título (máx. 100)", max_lines=2)
            desc_tb = gr.Textbox(label="Descripción (máx. 5000 bytes)", lines=12)
            tags_tb = gr.Textbox(label="Etiquetas (separadas por coma)", lines=2)

            gr.Markdown("### 3) Ajustes de publicación")
            with gr.Row():
                privacy_rd = gr.Radio([("Privado", "private"), ("No listado", "unlisted"), ("Público", "public")],
                                      value="private", label="Visibilidad")
                category_dd = gr.Dropdown(CATEGORIES, value="22", label="Categoría")
                lang_dd = gr.Dropdown(LANGS, value="es", label="Idioma", allow_custom_value=True)
            with gr.Row():
                kids_cb = gr.Checkbox(label="Hecho para niños", value=False)
                synth_cb = gr.Checkbox(label="Contenido alterado/sintético (IA)", value=True)
                embed_cb = gr.Checkbox(label="Permitir insertar", value=True)
                stats_cb = gr.Checkbox(label="Estadísticas públicas", value=True)
                notify_cb = gr.Checkbox(label="Notificar a suscriptores", value=False)
            license_rd = gr.Radio([("YouTube estándar", "youtube"), ("Creative Commons", "creativeCommon")],
                                  value="youtube", label="Licencia")
            with gr.Row():
                playlist_dd = gr.Dropdown(choices=[("— ninguna —", "")], value="", allow_custom_value=True,
                                          label="Lista de reproducción (ID)", scale=4)
                pl_btn = gr.Button("🔄 Cargar listas del canal", scale=1)
            with gr.Row():
                sched_tb = gr.Textbox(label="Programar (opcional) AAAA-MM-DD HH:MM", placeholder="2026-10-15 18:30")
                tz_tb = gr.Textbox(label="Zona horaria", value="UTC", placeholder="America/New_York")
            footer_tb = gr.Textbox(label="Pie fijo de descripción (se añade al generar con xKiro)", lines=3,
                                   placeholder="Redes, enlaces, avisos legales…")

            gr.Markdown("### 4) Miniatura (opcional)")
            with gr.Row():
                thumb_img = gr.Image(label="Miniatura (JPG/PNG ≤ 2 MB)", type="filepath", height=220)
                with gr.Column():
                    frame_sec = gr.Number(value=3, label="Segundo del frame a extraer", precision=1)
                    frame_btn = gr.Button("🎞️ Extraer frame del video")

            with gr.Row():
                save_btn = gr.Button("💾 Guardar ajustes como predeterminados del canal")
                up_btn = gr.Button("🚀 SUBIR A YOUTUBE", variant="primary")
            save_msg = gr.Markdown()
            up_status = gr.Textbox(label="Estado de la subida", lines=8, interactive=False)
            gr.Markdown("**Últimas subidas**")
            hist_md = gr.Markdown(_history_md())

        # ───────── TAB CANALES ─────────
        with gr.Tab("🔑 Canales / APIs"):
            with gr.Accordion("📘 Cómo obtener Client ID / Secret (una vez por proyecto)", open=False):
                gr.Markdown(
                    "1. console.cloud.google.com → crea un proyecto → **APIs y servicios → Biblioteca** → activa **YouTube Data API v3**.\n"
                    "2. **Pantalla de consentimiento OAuth** → pásala a **«En producción»** (en «Pruebas» el token caduca a los 7 días).\n"
                    "3. **Credenciales → Crear → ID de cliente OAuth** → tipo **«TV y dispositivos de entrada limitada»**.\n"
                    "4. Copia Client ID y Client Secret aquí. Puedes reutilizar el mismo cliente para varios canales; "
                    "al autorizar, Google te deja elegir el canal.")
            with gr.Row():
                new_alias = gr.Textbox(label="Alias del canal", placeholder="Mi canal principal")
                new_cid = gr.Textbox(label="Client ID", value=_cfg0["last_client"].get("client_id", ""))
                new_csec = gr.Textbox(label="Client Secret", type="password",
                                      value=_cfg0["last_client"].get("client_secret", ""))
            dev_state = gr.State(None)
            start_btn = gr.Button("1️⃣ Pedir código de autorización")
            dev_md = gr.Markdown()
            finish_btn = gr.Button("2️⃣ ✅ Ya autoricé → guardar canal", variant="primary")
            conn_msg = gr.Markdown()
            gr.Markdown("### Canales guardados")
            channels_md = gr.Markdown(_channels_md())
            with gr.Row():
                manage_dd = gr.Dropdown(choices=_ch_choices(), value=_last0, label="Canal")
                verify_btn = gr.Button("🔍 Verificar")
                del_cb = gr.Checkbox(label="Confirmo borrar", value=False)
                del_btn = gr.Button("🗑️ Eliminar canal")
            manage_msg = gr.Markdown()

    # ───────── Cableado ─────────
    PROF_COMPS = [privacy_rd, category_dd, lang_dd, tags_tb, footer_tb, kids_cb, synth_cb, embed_cb,
                  stats_cb, license_rd, notify_cb, playlist_dd, ctx_tb, tz_tb, model_dd, ntitles_sl]

    ch_dd.change(on_channel_change, ch_dd, PROF_COMPS)
    demo_yt.load(on_channel_change, ch_dd, PROF_COMPS)
    vid_ref.click(lambda: gr.Dropdown(choices=_scan_videos(), value=None), None, vid_dd)
    for _c in (vid_dd, vid_man, vid_up):
        _c.change(video_info, [vid_dd, vid_man, vid_up], vid_info)
    gen_btn.click(ai_generate, [prompt_tb, ctx_tb, model_dd, ntitles_sl, tags_tb, footer_tb, lang_dd],
                  [titles_rd, desc_tb, tags_tb, ai_status])
    titles_rd.change(pick_title, titles_rd, title_tb)
    save_btn.click(save_defaults, [ch_dd] + PROF_COMPS, save_msg)
    frame_btn.click(extract_frame, [vid_dd, vid_man, vid_up, frame_sec], thumb_img)
    pl_btn.click(load_playlists, [ch_dd, playlist_dd], playlist_dd)
    up_btn.click(do_upload,
                 [ch_dd, vid_dd, vid_man, vid_up, title_tb, desc_tb, tags_tb, privacy_rd, category_dd,
                  lang_dd, kids_cb, synth_cb, embed_cb, stats_cb, license_rd, notify_cb, playlist_dd,
                  sched_tb, tz_tb, thumb_img],
                 [up_status, hist_md])
    start_btn.click(dev_start, [new_alias, new_cid, new_csec, dev_state], [dev_md, dev_state])
    finish_btn.click(dev_finish, dev_state, [conn_msg, ch_dd, manage_dd, channels_md, dev_state])
    verify_btn.click(verify_channel, manage_dd, manage_msg)
    del_btn.click(delete_channel, [manage_dd, del_cb], [manage_msg, ch_dd, manage_dd, channels_md])

try:
    demo_yt.close()   # si re-ejecutas la celda, cierra solo ESTA UI (no las otras)
except Exception:
    pass
demo_yt.queue().launch(share=True, prevent_thread_lock=True, show_error=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://ef0b9fe5885c65a9ec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA EXTRA C: 📦 EXPORTAR / IMPORTAR PLANTILLAS (todo en una UI con 2 pestañas)
#
# - Exportar: empaqueta una plantilla COMPLETA (imágenes, voces, reacciones, plantilla.json) en un .zip.
#   Opcional: incluye las voces clonadas de los jueces (voice_refs/jueces, archivos pequeños).
# - Importar: desde .zip (subido, detectado o por ruta) o carpeta (p. ej. Dataset de Kaggle).
#   Si ya existe una plantilla con el mismo nombre eliges: no tocar / sobrescribir / fusionar.
# - Exportar solo LEE tu proyecto. Funciones con prefijo _tp_ / tp_ para no pisar otras celdas.
# ═══════════════════════════════════════════════════════════════
import os, re, json, time, shutil, zipfile
import gradio as gr

_TP_OUT  = globals().get("OUTPUT_DIR") or "/kaggle/working/VideoAutomationProject"
_TP_TPL  = os.path.realpath(globals().get("PLANTILLAS_DIR") or f"{_TP_OUT}/plantillas")
_TP_JUECES_DIR = os.path.join(_TP_OUT, "voice_refs", "jueces")
_TP_SUBDIRS = list((globals().get("PLANTILLA_SUBDIRS") or {
    "a": "imagenes/jueces", "b": "imagenes/escenario_lejano", "c": "imagenes/escenario_cercano",
    "d": "voces/jueces", "e": "video/reacciones_primera_mitad", "f": "video/reacciones_segunda_mitad",
}).values())
_TP_MANIFEST = "EXPORT_PLANTILLA_MANIFEST.json"
_TP_DESTINOS = ["/kaggle/tmp/exportaciones", "/kaggle/working/exportaciones"]
_TP_SIN_COMPRIMIR = (".mp4", ".wav", ".mp3", ".png", ".jpg", ".jpeg", ".webp", ".mov", ".zip", ".gz")
_TP_JUEZ_RE = re.compile(r"^juez_\d+_.+\.(wav|txt)$", re.I)
_TP_BUSCAR = [("/kaggle/input", 5), ("/kaggle/tmp/exportaciones", 2),
              ("/kaggle/working/exportaciones", 2), ("/kaggle/working", 0)]
_TP_MODOS = ["No tocar si ya existe", "Sobrescribir (reemplaza toda la plantilla)",
             "Fusionar (añade lo que falte, conserva lo existente)"]
for _d in _TP_DESTINOS:
    os.makedirs(_d, exist_ok=True)

def _tp_fmt(n):
    return f"{n/1024**3:.2f} GB" if n >= 1024**3 else f"{n/1024**2:.1f} MB"

def _tp_nombre_ok(n):
    return bool(n) and n not in (".", "..") and not n.startswith(".") and not re.search(r"[/\\]", n)

def _tp_size(path):
    tot = 0
    for root, _, files in os.walk(path):
        for fn in files:
            fp = os.path.join(root, fn)
            if not os.path.islink(fp):
                try: tot += os.path.getsize(fp)
                except OSError: pass
    return tot

def _tp_copiar(fsrc, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, "wb") as o:
        shutil.copyfileobj(fsrc, o, 8 * 1024 * 1024)

# ───────────────────────── EXPORTAR ─────────────────────────
def _tp_lista():
    out = []
    if os.path.isdir(_TP_TPL):
        for slug in sorted(os.listdir(_TP_TPL)):
            p = os.path.join(_TP_TPL, slug)
            mp = os.path.join(p, "plantilla.json")
            if slug.startswith(".") or not os.path.isfile(mp):
                continue
            try: nombre = json.load(open(mp, encoding="utf-8")).get("nombre", slug)
            except Exception: nombre = slug
            out.append((f"{nombre}  ({slug})  ·  {_tp_fmt(_tp_size(p))}", slug))
    return out

def _tp_plan(slug, incl_jueces):
    base = os.path.realpath(os.path.join(_TP_TPL, slug))
    if os.path.dirname(base) != _TP_TPL or not os.path.isdir(base):
        raise RuntimeError(f"Plantilla no válida: {slug}")
    files = []
    for root, dirs, fs in os.walk(base):
        dirs.sort()
        for fn in sorted(fs):
            fp = os.path.join(root, fn)
            if os.path.islink(fp):
                continue
            rel = os.path.relpath(fp, base).replace(os.sep, "/")
            files.append((fp, f"plantilla/{slug}/{rel}", os.path.getsize(fp)))
    jueces = []
    if incl_jueces and os.path.isdir(_TP_JUECES_DIR):
        for fn in sorted(os.listdir(_TP_JUECES_DIR)):
            fp = os.path.join(_TP_JUECES_DIR, fn)
            if _TP_JUEZ_RE.match(fn) and os.path.isfile(fp):
                jueces.append((fp, f"voces_jueces/{fn}", os.path.getsize(fp)))
    return base, files, jueces

def tp_resumen(slug, incl_jueces):
    if not slug:
        return "Elige una plantilla."
    try:
        base, files, jueces = _tp_plan(slug, incl_jueces)
    except Exception as e:
        return f"❌ {e}"
    tot = sum(s for *_, s in files + jueces)
    t = f"📦 Se exportarán **{len(files)}** archivos de la plantilla (**{_tp_fmt(tot)}**)"
    if jueces: t += f" + {len(jueces)} archivos de voces clonadas de jueces"
    elif incl_jueces: t += "\n\nℹ️ No hay voces clonadas de jueces en voice_refs/jueces."
    return t + f"\n\n💾 Libre en /kaggle/tmp: {_tp_fmt(shutil.disk_usage('/kaggle/tmp').free)}"

def tp_exportar(slug, incl_jueces, destino, progress=gr.Progress()):
    if not slug:
        return "⚠️ Elige una plantilla.", None
    if destino not in _TP_DESTINOS:
        destino = _TP_DESTINOS[0]
    try:
        base, files, jueces = _tp_plan(slug, incl_jueces)
    except Exception as e:
        return f"❌ {e}", None
    todos = files + jueces
    total = sum(s for *_, s in todos) or 1
    os.makedirs(destino, exist_ok=True)
    libre = shutil.disk_usage(destino).free
    if libre < total * 1.02 + 200 * 1024**2:
        return f"⛔ Espacio insuficiente en {destino}: necesitas ~{_tp_fmt(total)} y hay {_tp_fmt(libre)}.", None
    try: nombre = json.load(open(os.path.join(base, "plantilla.json"), encoding="utf-8")).get("nombre", slug)
    except Exception: nombre = slug
    manifest = {"formato": 1, "tipo": "plantilla", "slug": slug, "nombre": nombre, "output_dir": _TP_OUT,
                "exportado_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                "n_archivos": len(files), "bytes": sum(s for *_, s in files),
                "incluye_voces_jueces": bool(jueces)}
    zip_final = os.path.join(destino, f"plantilla_{slug}.zip")
    tmp = zip_final + ".part"
    errores, hecho, t0 = [], 0, time.time()
    try:
        with zipfile.ZipFile(tmp, "w", allowZip64=True) as zf:
            zf.writestr(_TP_MANIFEST, json.dumps(manifest, ensure_ascii=False, indent=2),
                        compress_type=zipfile.ZIP_DEFLATED)
            for fp, arc, size in todos:
                comp = zipfile.ZIP_STORED if fp.lower().endswith(_TP_SIN_COMPRIMIR) else zipfile.ZIP_DEFLATED
                try: zf.write(fp, arc, compress_type=comp)
                except OSError as e: errores.append(f"{arc}: {e}")
                hecho += size
                progress(min(hecho / total, 1.0), desc=f"{_tp_fmt(hecho)} / {_tp_fmt(total)}")
        os.replace(tmp, zip_final)
    except Exception as e:
        try: os.remove(tmp)
        except OSError: pass
        return f"❌ Falló la exportación: {e}", None
    msg = [f"✅ Plantilla exportada en {time.time()-t0:.0f}s: {zip_final}  ({_tp_fmt(os.path.getsize(zip_final))})"]
    if errores:
        msg.append("⚠️ No se pudieron leer:\n  " + "\n  ".join(errores[:10]))
    msg.append("➡️ Descárgalo abajo, o súbelo como Dataset de Kaggle en la otra cuenta "
               "(Kaggle lo descomprime solo; la importación acepta .zip o carpeta).")
    try: log("[EXPORT-PLANTILLA] " + " | ".join(msg))
    except Exception: pass
    return "\n".join(msg), zip_final

def tp_refrescar_export(incl_jueces):
    ch = _tp_lista()
    v = ch[0][1] if ch else None
    return gr.update(choices=ch, value=v), tp_resumen(v, incl_jueces)

# ───────────────────────── IMPORTAR ─────────────────────────
def _tp_escanear():
    out, vistos = [], set()
    for root, maxd in _TP_BUSCAR:
        if not os.path.isdir(root):
            continue
        d0 = root.rstrip("/").count("/")
        for cur, dirs, files in os.walk(root):
            depth = cur.rstrip("/").count("/") - d0
            if _TP_MANIFEST in files:
                if cur not in vistos:
                    vistos.add(cur)
                    try: v = json.load(open(os.path.join(cur, _TP_MANIFEST), encoding="utf-8")).get("slug", "?")
                    except Exception: v = "?"
                    out.append((f"📁 plantilla {v}  ←  {cur}", cur))
                dirs[:] = []
                continue
            for fn in files:
                if fn.lower().endswith(".zip"):
                    p = os.path.join(cur, fn)
                    if p not in vistos:
                        vistos.add(p)
                        try: sz = _tp_fmt(os.path.getsize(p))
                        except OSError: sz = "?"
                        out.append((f"📦 {p}  ·  {sz}", p))
            if depth >= maxd:
                dirs[:] = []
    return out

def _tp_origen(upload, sel, manual):
    p = None
    if upload:
        p = upload if isinstance(upload, str) else getattr(upload, "name", None)
    if not p:
        p = (manual or "").strip().strip('"').strip("'")
    return p or sel or ""

def _tp_abrir(path):
    if not path or not os.path.exists(path):
        raise RuntimeError("Elige, escribe o sube un .zip / carpeta válidos.")
    if os.path.isdir(path):
        raiz = path if os.path.exists(os.path.join(path, _TP_MANIFEST)) else None
        if not raiz:
            d0 = path.rstrip("/").count("/")
            for cur, dirs, files in os.walk(path):
                if _TP_MANIFEST in files:
                    raiz = cur; break
                if cur.rstrip("/").count("/") - d0 >= 3:
                    dirs[:] = []
        if not raiz:
            raise RuntimeError(f"No encuentro {_TP_MANIFEST} en esa carpeta (¿es un export de PLANTILLA? "
                               "los exports de video se importan con la otra celda).")
        return "dir", json.load(open(os.path.join(raiz, _TP_MANIFEST), encoding="utf-8")), raiz
    if not zipfile.is_zipfile(path):
        raise RuntimeError("El archivo no es un .zip válido.")
    zf = zipfile.ZipFile(path)
    cand = [n for n in zf.namelist() if n.endswith(_TP_MANIFEST) and n.count("/") <= 1]
    if not cand:
        zf.close()
        raise RuntimeError(f"El .zip no contiene {_TP_MANIFEST} (¿es un export de PLANTILLA? "
                           "los exports de video se importan con la otra celda).")
    mf = min(cand, key=len)
    return "zip", json.loads(zf.read(mf).decode("utf-8")), (zf, mf[:-len(_TP_MANIFEST)])

def _tp_validar(m):
    if m.get("formato") != 1 or m.get("tipo") != "plantilla":
        raise RuntimeError("Este export no es de una plantilla.")
    if not _tp_nombre_ok(m.get("slug", "")):
        raise RuntimeError("Nombre de plantilla no válido en el manifiesto.")

def _tp_cerrar(h):
    try:
        if isinstance(h, tuple): h[0].close()
    except Exception:
        pass

def tp_inspeccionar(upload, sel, manual):
    h = None
    try:
        tipo, m, h = _tp_abrir(_tp_origen(upload, sel, manual))
        _tp_validar(m)
    except Exception as e:
        return f"❌ {e}"
    finally:
        _tp_cerrar(h)
    ya = os.path.isdir(os.path.join(_TP_TPL, m["slug"]))
    return (f"🎭 Plantilla: {m.get('nombre', m['slug'])}  ({m['slug']})\n"
            f"🗂️ {m.get('n_archivos','?')} archivos · {_tp_fmt(m.get('bytes', 0))} · exportada {m.get('exportado_utc','?')}\n"
            f"🎙️ Voces clonadas de jueces incluidas: {'sí' if m.get('incluye_voces_jueces') else 'no'}\n"
            f"{'⚠️ Ya existe una plantilla con ese nombre aquí: elige qué hacer en «Si ya existe».' if ya else '✅ No existe aquí: se creará nueva.'}")

def _tp_reescribir_json(carpeta, viejo, nuevo):
    if not viejo or viejo == nuevo:
        return 0
    n = 0
    for root, _, files in os.walk(carpeta):
        for fn in files:
            if fn.lower().endswith(".json"):
                p = os.path.join(root, fn)
                try:
                    if os.path.getsize(p) > 64 * 1024**2: continue
                    t = open(p, encoding="utf-8").read()
                    if viejo in t:
                        open(p, "w", encoding="utf-8").write(t.replace(viejo, nuevo)); n += 1
                except Exception:
                    pass
    return n

def tp_importar(upload, sel, manual, modo, restaurar_jueces, progress=gr.Progress()):
    h = None
    stage = None
    try:
        tipo, m, h = _tp_abrir(_tp_origen(upload, sel, manual))
        _tp_validar(m)
        slug = m["slug"]
        dest = os.path.join(_TP_TPL, slug)
        existe = os.path.isdir(dest)
        if existe and modo == _TP_MODOS[0]:
            return f"⛔ La plantilla «{slug}» ya existe aquí y elegiste «No tocar». No se hizo nada."
        fusionar = existe and modo == _TP_MODOS[2]
        os.makedirs(_TP_TPL, exist_ok=True)

        # ── lista de archivos del export ──
        if tipo == "zip":
            zf, pref = h
            pv = f"{pref}plantilla/{slug}/"
            items = [(i.filename[len(pv):], i.file_size, i) for i in zf.infolist()
                     if i.filename.startswith(pv) and not i.is_dir()]
        else:
            bt = os.path.join(h, "plantilla", slug)
            if not os.path.isdir(bt):
                raise RuntimeError(f"No encuentro la carpeta plantilla/{slug} en el origen.")
            items = []
            for root, _, files in os.walk(bt):
                for fn in files:
                    fp = os.path.join(root, fn)
                    items.append((os.path.relpath(fp, bt).replace(os.sep, "/"), os.path.getsize(fp), fp))
        if not items:
            raise RuntimeError("El export no contiene archivos de la plantilla.")
        if fusionar:
            items = [it for it in items if not os.path.exists(os.path.join(dest, it[0]))]
        total = sum(s for _, s, _ in items) or 1
        libre = shutil.disk_usage(_TP_TPL).free
        if libre < total * 1.02 + 200 * 1024**2:
            raise RuntimeError(f"Espacio insuficiente: necesitas ~{_tp_fmt(total)} y hay {_tp_fmt(libre)}.")

        # ── destino de copia: directo si fusiona, en staging si es nueva/sobrescritura ──
        if fusionar:
            raiz_copia = os.path.realpath(dest)
        else:
            stage = os.path.join(_TP_TPL, f".import_{slug}")
            if os.path.exists(stage): shutil.rmtree(stage)
            os.makedirs(stage)
            raiz_copia = os.path.realpath(stage)
        hecho = 0
        for rel, size, src in items:
            dst = os.path.realpath(os.path.join(raiz_copia, rel))
            if not dst.startswith(raiz_copia + os.sep):
                continue
            if tipo == "zip":
                with h[0].open(src) as fs: _tp_copiar(fs, dst)
            else:
                with open(src, "rb") as fs: _tp_copiar(fs, dst)
            hecho += size
            progress(min(hecho / total, 1.0) * 0.95, desc=f"{_tp_fmt(hecho)} / {_tp_fmt(total)}")
        for sub in _TP_SUBDIRS:
            os.makedirs(os.path.join(raiz_copia, sub), exist_ok=True)
        reesc = 0 if fusionar else _tp_reescribir_json(raiz_copia, m.get("output_dir", ""), _TP_OUT)

        if not fusionar:
            if existe:
                shutil.rmtree(dest)
            os.replace(raiz_copia, dest)
            stage = None
        accion = ("fusionada (solo se añadió lo que faltaba)" if fusionar
                  else "sobrescrita" if existe else "creada")
        msg = [f"✅ Plantilla «{slug}» {accion}: {dest}  ({len(items)} archivos, {_tp_fmt(total)})"]
        if fusionar and not items:
            msg.append("ℹ️ La plantilla existente ya tenía todos los archivos del export.")
        if reesc:
            msg.append(f"🔧 Rutas ajustadas en {reesc} archivo(s) .json.")

        # ── voces clonadas de jueces (globales: voice_refs/jueces) ──
        if m.get("incluye_voces_jueces"):
            if restaurar_jueces:
                os.makedirs(_TP_JUECES_DIR, exist_ok=True)
                n = 0
                if tipo == "zip":
                    pj = f"{h[1]}voces_jueces/"
                    fuentes = [(i.filename[len(pj):], i) for i in h[0].infolist()
                               if i.filename.startswith(pj) and not i.is_dir()]
                else:
                    dj = os.path.join(h, "voces_jueces")
                    fuentes = [(fn, os.path.join(dj, fn)) for fn in (os.listdir(dj) if os.path.isdir(dj) else [])]
                for fn, src in fuentes:
                    if not _TP_JUEZ_RE.match(fn):
                        continue
                    dst = os.path.join(_TP_JUECES_DIR, fn)
                    if tipo == "zip":
                        with h[0].open(src) as fs: _tp_copiar(fs, dst)
                    else:
                        with open(src, "rb") as fs: _tp_copiar(fs, dst)
                    n += 1
                msg.append(f"🎙️ Voces clonadas de jueces restauradas ({n} archivos; sobrescriben las actuales).")
            else:
                msg.append("🎙️ El export incluía voces clonadas de jueces pero no las restauraste.")
        try: log("[IMPORT-PLANTILLA] " + " | ".join(msg))
        except Exception: pass
        progress(1.0, desc="Listo")
        return "\n".join(msg)
    except Exception as e:
        return f"❌ No se importó nada: {e}"
    finally:
        _tp_cerrar(h)
        if stage and os.path.isdir(stage):
            shutil.rmtree(stage, ignore_errors=True)

def tp_refrescar_import():
    return gr.update(choices=_tp_escanear(), value=None)

# ───────────────────────── UI ─────────────────────────
_c0 = _tp_lista()
_s0 = _c0[0][1] if _c0 else None
with gr.Blocks(title="📦 Plantillas: exportar / importar") as demo_plantillas:
    gr.Markdown("## 📦 Plantillas: exportar / importar entre cuentas de Kaggle")
    with gr.Tab("📤 Exportar plantilla"):
        with gr.Row():
            dd_tpl = gr.Dropdown(choices=_c0, value=_s0, label="Plantilla a exportar", scale=4)
            btn_ref = gr.Button("🔄 Refrescar", scale=1)
        chk_jue = gr.Checkbox(value=True, label="Incluir voces clonadas de los jueces (voice_refs/jueces, archivos pequeños)")
        dd_dest = gr.Dropdown(choices=_TP_DESTINOS, value=_TP_DESTINOS[0],
                              label="Dónde guardar el .zip (/kaggle/tmp = más espacio, no persiste · /kaggle/working = persiste como output)")
        resumen = gr.Markdown(tp_resumen(_s0, True))
        btn_exp = gr.Button("📤 Exportar plantilla", variant="primary")
        out_exp = gr.Textbox(label="Resultado", interactive=False, lines=5)
        f_out = gr.File(label="Descargar .zip", interactive=False)
        for comp in (dd_tpl, chk_jue):
            comp.change(tp_resumen, [dd_tpl, chk_jue], [resumen])
        btn_ref.click(tp_refrescar_export, [chk_jue], [dd_tpl, resumen])
        btn_exp.click(tp_exportar, [dd_tpl, chk_jue, dd_dest], [out_exp, f_out])
    with gr.Tab("📥 Importar plantilla"):
        gr.Markdown("Indica el origen de **una** forma (si hay varias manda: subida → ruta escrita → lista).")
        with gr.Row():
            dd_org = gr.Dropdown(choices=_tp_escanear(), value=None, allow_custom_value=True, scale=4,
                                 label="Detectado en /kaggle/input y rutas habituales (.zip o carpeta)")
            btn_bus = gr.Button("🔄 Buscar de nuevo", scale=1)
        tb_man = gr.Textbox(label="…o escribe la ruta de un .zip o de una carpeta", placeholder="/kaggle/input/mi-dataset/...")
        f_sub = gr.File(label="…o sube aquí el .zip (para archivos grandes es más rápido usar un Dataset)",
                        file_types=[".zip"], type="filepath")
        rd_modo = gr.Radio(choices=_TP_MODOS, value=_TP_MODOS[0], label="Si ya existe una plantilla con el mismo nombre")
        chk_rest = gr.Checkbox(value=False,
                               label="Restaurar voces clonadas de jueces (son globales: sobrescriben las que tengas ahora)")
        with gr.Row():
            btn_ver = gr.Button("🔍 Ver qué contiene", variant="secondary")
            btn_imp = gr.Button("📥 Importar plantilla", variant="primary")
        out_imp = gr.Textbox(label="Resultado", interactive=False, lines=8)
        btn_bus.click(tp_refrescar_import, [], [dd_org])
        btn_ver.click(tp_inspeccionar, [f_sub, dd_org, tb_man], [out_imp])
        btn_imp.click(tp_importar, [f_sub, dd_org, tb_man, rd_modo, chk_rest], [out_imp])

demo_plantillas.queue().launch(share=True, prevent_thread_lock=True, show_error=True, allowed_paths=_TP_DESTINOS)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA EXTRA A: 📤 EXPORTAR UN VIDEO (para continuarlo en otra cuenta de Kaggle)
#
# - Empaqueta UN video (carpeta completa en /videos) en un único .zip.
# - Los assets de la PLANTILLA no se exportan: solo va la estructura vacía + plantilla.json
#   + la lista de archivos que debes volver a subir en la otra cuenta.
# - Opcional: voz del concursante (pequeña) y excluir la carpeta tmp.
# - Solo LEE tu proyecto. No modifica nada. Todas las funciones llevan prefijo _xp_ / xp_
#   para no pisar nombres de otras celdas.
# ═══════════════════════════════════════════════════════════════
import os, json, time, shutil, zipfile
import gradio as gr

_XP_OUT   = globals().get("OUTPUT_DIR") or "/kaggle/working/VideoAutomationProject"
_XP_VID   = os.path.realpath(globals().get("VIDEOS_DIR") or f"{_XP_OUT}/videos")
_XP_TPL   = os.path.realpath(globals().get("PLANTILLAS_DIR") or f"{_XP_OUT}/plantillas")
_XP_UIST  = os.path.join(_XP_OUT, "ui_state.json")
_XP_VOICE_DIR   = os.path.join(_XP_OUT, "voice_refs")
_XP_VOICE_STATE = os.path.join(_XP_OUT, "contestant_voice_state.json")
_XP_VOICE_FILES = ["contestant_ref.wav", "contestant_ref.txt", "contestant_preview.wav",
                   "design_as_ref.wav", "design_as_ref.txt"]
_XP_SUBDIRS = list((globals().get("PLANTILLA_SUBDIRS") or {
    "a": "imagenes/jueces", "b": "imagenes/escenario_lejano", "c": "imagenes/escenario_cercano",
    "d": "voces/jueces", "e": "video/reacciones_primera_mitad", "f": "video/reacciones_segunda_mitad",
}).values())
_XP_DESTINOS = ["/kaggle/tmp/exportaciones", "/kaggle/working/exportaciones"]
_XP_SIN_COMPRIMIR = (".mp4", ".wav", ".mp3", ".png", ".jpg", ".jpeg", ".webp", ".mov", ".zip", ".gz", ".safetensors")
for _d in _XP_DESTINOS:
    os.makedirs(_d, exist_ok=True)

def _xp_fmt(n):
    return f"{n/1024**3:.2f} GB" if n >= 1024**3 else f"{n/1024**2:.1f} MB"

def _xp_json(p, default=None):
    try:
        with open(p, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default

def _xp_dir_size(path):
    tot = 0
    for root, _, files in os.walk(path):
        for fn in files:
            fp = os.path.join(root, fn)
            if not os.path.islink(fp):
                try: tot += os.path.getsize(fp)
                except OSError: pass
    return tot

def _xp_videos():
    st = _xp_json(_XP_UIST, {}) or {}
    activo = os.path.realpath(st.get("current_video_dir", "")) if st.get("current_video_dir") else ""
    out = []
    if os.path.isdir(_XP_VID):
        for d in sorted(os.listdir(_XP_VID)):
            p = os.path.join(_XP_VID, d)
            if d.startswith(".") or not os.path.isdir(p):
                continue
            est = _xp_json(os.path.join(p, "estado.json"), {}) or {}
            out.append({"nombre": d, "ruta": p, "bytes": _xp_dir_size(p), "mtime": os.path.getmtime(p),
                        "tpl": est.get("plantilla_slug", ""), "activo": os.path.realpath(p) == activo})
    out.sort(key=lambda x: -x["mtime"])
    return out

def _xp_choices():
    ch = []
    for v in _xp_videos():
        f = time.strftime("%Y-%m-%d %H:%M", time.localtime(v["mtime"]))
        ch.append((f"{v['nombre']}  ·  {_xp_fmt(v['bytes'])}  ·  {f}" + ("  🟢 ACTIVO" if v["activo"] else ""), v["nombre"]))
    return ch

def _xp_plan(nombre, excluir_tmp, incl_voz):
    vdir = os.path.realpath(os.path.join(_XP_VID, nombre))
    if os.path.dirname(vdir) != _XP_VID or not os.path.isdir(vdir):
        raise RuntimeError(f"Video no válido: {nombre}")
    skip = {"tmp"} if excluir_tmp else set()
    archivos = []
    for root, dirs, files in os.walk(vdir):
        if root == vdir:
            dirs[:] = [d for d in dirs if d not in skip]
        dirs.sort()
        for fn in sorted(files):
            fp = os.path.join(root, fn)
            if os.path.islink(fp):
                continue
            rel = os.path.relpath(fp, vdir).replace(os.sep, "/")
            archivos.append((fp, f"video/{nombre}/{rel}", os.path.getsize(fp)))
    voz = []
    if incl_voz:
        if os.path.isfile(_XP_VOICE_STATE):
            voz.append((_XP_VOICE_STATE, "voz_concursante/contestant_voice_state.json", os.path.getsize(_XP_VOICE_STATE)))
        for fn in _XP_VOICE_FILES:
            p = os.path.join(_XP_VOICE_DIR, fn)
            if os.path.isfile(p):
                voz.append((p, f"voz_concursante/voice_refs/{fn}", os.path.getsize(p)))
    est = _xp_json(os.path.join(vdir, "estado.json"), {}) or {}
    slug = est.get("plantilla_slug", "") or ""
    plantilla = None
    if slug:
        base = os.path.realpath(os.path.join(_XP_TPL, slug))
        if os.path.dirname(base) == _XP_TPL and os.path.isdir(base):
            omit = []
            for root, _, files in os.walk(base):
                for fn in sorted(files):
                    fp = os.path.join(root, fn)
                    rel = os.path.relpath(fp, base).replace(os.sep, "/")
                    if rel == "plantilla.json" or os.path.islink(fp):
                        continue
                    omit.append({"ruta": rel, "bytes": os.path.getsize(fp)})
            plantilla = {"slug": slug,
                         "meta": _xp_json(os.path.join(base, "plantilla.json"), {}) or {},
                         "subdirs": _XP_SUBDIRS, "archivos_omitidos": omit}
    return vdir, archivos, voz, est, slug, plantilla

def xp_resumen(nombre, excluir_tmp, incl_voz):
    if not nombre:
        return "Elige un video."
    try:
        vdir, archivos, voz, est, slug, pl = _xp_plan(nombre, excluir_tmp, incl_voz)
    except Exception as e:
        return f"❌ {e}"
    tot = sum(s for *_, s in archivos + voz)
    txt = f"📦 Se exportarán **{len(archivos)}** archivos del video (**{_xp_fmt(tot)}** en total)"
    if voz: txt += f" + voz del concursante ({len(voz)} archivos)"
    if pl:
        om = sum(o["bytes"] for o in pl["archivos_omitidos"])
        txt += (f"\n\n📁 Plantilla **{slug}**: va VACÍA (solo estructura + plantilla.json). "
                f"No se incluyen {len(pl['archivos_omitidos'])} archivos ({_xp_fmt(om)}).")
    elif slug:
        txt += f"\n\n⚠️ La plantilla «{slug}» de este video ya no existe aquí; no se puede incluir su estructura."
    else:
        txt += "\n\n⚠️ Este video no tiene plantilla registrada en estado.json."
    libre = shutil.disk_usage("/kaggle/tmp").free
    txt += f"\n\n💾 Libre en /kaggle/tmp: {_xp_fmt(libre)}"
    return txt

def xp_exportar(nombre, excluir_tmp, incl_voz, destino, progress=gr.Progress()):
    if not nombre:
        return "⚠️ Elige un video.", None
    if destino not in _XP_DESTINOS:
        destino = _XP_DESTINOS[0]
    try:
        vdir, archivos, voz, est, slug, plantilla = _xp_plan(nombre, excluir_tmp, incl_voz)
    except Exception as e:
        return f"❌ {e}", None
    todos = archivos + voz
    total = sum(s for *_, s in todos) or 1
    os.makedirs(destino, exist_ok=True)
    libre = shutil.disk_usage(destino).free
    if libre < total * 1.02 + 200 * 1024**2:
        return (f"⛔ Espacio insuficiente en {destino}: necesitas ~{_xp_fmt(total)} y hay {_xp_fmt(libre)} libres. "
                "Prueba con el otro destino o excluye tmp."), None
    manifest = {
        "formato": 1, "video": nombre, "plantilla_slug": slug, "personaje": est.get("personaje", ""),
        "output_dir": _XP_OUT, "exportado_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "excluyo_tmp": bool(excluir_tmp), "incluye_voz": bool(voz), "n_archivos": len(archivos),
        "plantilla": plantilla,
    }
    zip_final = os.path.join(destino, f"{nombre}.zip")
    tmp = zip_final + ".part"
    errores, hecho, t0 = [], 0, time.time()
    try:
        with zipfile.ZipFile(tmp, "w", allowZip64=True) as zf:
            zf.writestr("EXPORT_MANIFEST.json", json.dumps(manifest, ensure_ascii=False, indent=2),
                        compress_type=zipfile.ZIP_DEFLATED)
            for fp, arc, size in todos:
                comp = zipfile.ZIP_STORED if fp.lower().endswith(_XP_SIN_COMPRIMIR) else zipfile.ZIP_DEFLATED
                try:
                    zf.write(fp, arc, compress_type=comp)
                except OSError as e:
                    errores.append(f"{arc}: {e}")
                hecho += size
                progress(min(hecho / total, 1.0), desc=f"{_xp_fmt(hecho)} / {_xp_fmt(total)}")
        os.replace(tmp, zip_final)
    except Exception as e:
        try: os.remove(tmp)
        except OSError: pass
        return f"❌ Falló la exportación: {e}", None
    msg = [f"✅ Exportado en {time.time()-t0:.0f}s: {zip_final}  ({_xp_fmt(os.path.getsize(zip_final))})"]
    if plantilla and plantilla["archivos_omitidos"]:
        msg.append(f"📁 Plantilla «{slug}» sin assets: en la otra cuenta tendrás que volver a subir "
                   f"{len(plantilla['archivos_omitidos'])} archivos (la importación te los listará).")
    if errores:
        msg.append("⚠️ Archivos que no se pudieron leer (¿se estaban escribiendo?):\n  " + "\n  ".join(errores[:10]))
    msg.append("➡️ Descárgalo con el componente de abajo, o súbelo como Dataset de Kaggle en la otra cuenta "
               "(Kaggle descomprime los .zip solos; la celda de importación entiende carpeta o .zip).")
    try: log("[EXPORT] " + " | ".join(msg))
    except Exception: pass
    return "\n".join(msg), zip_final

def xp_refrescar(excluir_tmp, incl_voz):
    ch = _xp_choices()
    val = ch[0][1] if ch else None
    return gr.update(choices=ch, value=val), xp_resumen(val, excluir_tmp, incl_voz)

_ch0 = _xp_choices()
_v0 = _ch0[0][1] if _ch0 else None
with gr.Blocks(title="📤 Exportar video") as demo_exportar:
    gr.Markdown("## 📤 Exportar un video para continuarlo en otra cuenta\n"
                "Los assets de la plantilla **no** se exportan (los vuelves a subir en la otra cuenta).")
    with gr.Row():
        dd_video = gr.Dropdown(choices=_ch0, value=_v0, label="Video a exportar", scale=4)
        btn_ref = gr.Button("🔄 Refrescar", scale=1)
    with gr.Row():
        chk_tmp = gr.Checkbox(value=True, label="Excluir carpeta tmp (temporales, se regeneran)")
        chk_voz = gr.Checkbox(value=True, label="Incluir voz del concursante (pequeña)")
    dd_dest = gr.Dropdown(choices=_XP_DESTINOS, value=_XP_DESTINOS[0],
                          label="Dónde guardar el .zip (/kaggle/tmp = más espacio, no persiste · /kaggle/working = persiste como output)")
    resumen = gr.Markdown(xp_resumen(_v0, True, True))
    btn_exp = gr.Button("📤 Exportar", variant="primary")
    salida = gr.Textbox(label="Resultado", interactive=False, lines=6)
    archivo = gr.File(label="Descargar .zip", interactive=False)

    for comp in (dd_video, chk_tmp, chk_voz):
        comp.change(xp_resumen, [dd_video, chk_tmp, chk_voz], [resumen])
    btn_ref.click(xp_refrescar, [chk_tmp, chk_voz], [dd_video, resumen])
    btn_exp.click(xp_exportar, [dd_video, chk_tmp, chk_voz, dd_dest], [salida, archivo])

demo_exportar.queue().launch(share=True, prevent_thread_lock=True, show_error=True, allowed_paths=_XP_DESTINOS)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA EXTRA B: 📥 IMPORTAR UN VIDEO exportado desde otra cuenta de Kaggle
#
# - Acepta: un .zip (subido aquí, o detectado en /kaggle/input y otras rutas) o una CARPETA
#   (por ejemplo un Dataset de Kaggle donde subiste el .zip, que Kaggle descomprime solo).
# - Restaura el video en /videos, recrea la estructura VACÍA de la plantilla si no existe aquí
#   (nunca toca una plantilla existente) y te lista los assets que debes volver a subir.
# - Opcional: restaura la voz del concursante y deja el video como activo.
# - No modifica ninguna celda del proyecto. Funciones con prefijo _ip_ / ip_.
# ═══════════════════════════════════════════════════════════════
import os, re, json, time, shutil, zipfile
import gradio as gr

_IP_OUT   = globals().get("OUTPUT_DIR") or "/kaggle/working/VideoAutomationProject"
_IP_VID   = os.path.realpath(globals().get("VIDEOS_DIR") or f"{_IP_OUT}/videos")
_IP_TPL   = os.path.realpath(globals().get("PLANTILLAS_DIR") or f"{_IP_OUT}/plantillas")
_IP_UIST  = os.path.join(_IP_OUT, "ui_state.json")
_IP_VOICE_DIR   = os.path.join(_IP_OUT, "voice_refs")
_IP_VOICE_STATE = os.path.join(_IP_OUT, "contestant_voice_state.json")
_IP_VOICE_FILES = ["contestant_ref.wav", "contestant_ref.txt", "contestant_preview.wav",
                   "design_as_ref.wav", "design_as_ref.txt"]
_IP_VIDEO_SUBDIRS = ["etapa1_imagenes", "etapa2_dialogos", "etapa3_cancion", "etapa4_final",
                     "tmp", "comfy_input", "comfy_output"]
_IP_MANIFEST = "EXPORT_MANIFEST.json"
_IP_BUSCAR = [("/kaggle/input", 5), ("/kaggle/tmp/exportaciones", 2),
              ("/kaggle/working/exportaciones", 2), ("/kaggle/working", 0)]

def _ip_fmt(n):
    return f"{n/1024**3:.2f} GB" if n >= 1024**3 else f"{n/1024**2:.1f} MB"

def _ip_nombre_ok(n):
    return bool(n) and n not in (".", "..") and not n.startswith(".") and not re.search(r"[/\\]", n)

def _ip_escanear():
    out, vistos = [], set()
    for root, maxd in _IP_BUSCAR:
        if not os.path.isdir(root):
            continue
        d0 = root.rstrip("/").count("/")
        for cur, dirs, files in os.walk(root):
            depth = cur.rstrip("/").count("/") - d0
            if _IP_MANIFEST in files:
                if cur not in vistos:
                    vistos.add(cur)
                    try: v = json.load(open(os.path.join(cur, _IP_MANIFEST), encoding="utf-8")).get("video", "?")
                    except Exception: v = "?"
                    out.append((f"📁 {v}  ←  {cur}", cur))
                dirs[:] = []
                continue
            for fn in files:
                if fn.lower().endswith(".zip"):
                    p = os.path.join(cur, fn)
                    if p not in vistos:
                        vistos.add(p)
                        try: sz = _ip_fmt(os.path.getsize(p))
                        except OSError: sz = "?"
                        out.append((f"📦 {p}  ·  {sz}", p))
            if depth >= maxd:
                dirs[:] = []
    return out

def _ip_origen(upload, sel, manual):
    p = None
    if upload:
        p = upload if isinstance(upload, str) else getattr(upload, "name", None)
    if not p:
        p = (manual or "").strip().strip('"').strip("'")
    if not p:
        p = sel or ""
    return p

def _ip_abrir(path):
    # → (tipo, manifest, handle). tipo: "zip" (handle=(ZipFile, prefijo)) | "dir" (handle=raíz)
    if not path or not os.path.exists(path):
        raise RuntimeError("Elige, escribe o sube un .zip / carpeta válidos.")
    if os.path.isdir(path):
        raiz = None
        if os.path.exists(os.path.join(path, _IP_MANIFEST)):
            raiz = path
        else:
            d0 = path.rstrip("/").count("/")
            for cur, dirs, files in os.walk(path):
                if _IP_MANIFEST in files:
                    raiz = cur; break
                if cur.rstrip("/").count("/") - d0 >= 3:
                    dirs[:] = []
        if not raiz:
            raise RuntimeError(f"No encuentro {_IP_MANIFEST} dentro de esa carpeta (¿es un export de esta celda?).")
        return "dir", json.load(open(os.path.join(raiz, _IP_MANIFEST), encoding="utf-8")), raiz
    if not zipfile.is_zipfile(path):
        raise RuntimeError("El archivo no es un .zip válido.")
    zf = zipfile.ZipFile(path)
    cand = [n for n in zf.namelist() if n.endswith(_IP_MANIFEST) and n.count("/") <= 1]
    if not cand:
        zf.close()
        raise RuntimeError(f"El .zip no contiene {_IP_MANIFEST} (¿es un export de esta celda?).")
    mf = min(cand, key=len)
    return "zip", json.loads(zf.read(mf).decode("utf-8")), (zf, mf[:-len(_IP_MANIFEST)])

def _ip_validar(m):
    if m.get("formato") != 1:
        raise RuntimeError("Formato de export desconocido.")
    if not _ip_nombre_ok(m.get("video", "")):
        raise RuntimeError("Nombre de video no válido en el manifiesto.")
    if m.get("plantilla_slug") and not _ip_nombre_ok(m["plantilla_slug"]):
        raise RuntimeError("Nombre de plantilla no válido en el manifiesto.")

def ip_inspeccionar(upload, sel, manual):
    h = None
    try:
        tipo, m, h = _ip_abrir(_ip_origen(upload, sel, manual))
        _ip_validar(m)
    except Exception as e:
        return f"❌ {e}"
    finally:
        try:
            if isinstance(h, tuple): h[0].close()
        except Exception:
            pass
    pl = m.get("plantilla") or {}
    om = pl.get("archivos_omitidos", [])
    ya = os.path.isdir(os.path.join(_IP_VID, m["video"]))
    return (f"🎬 Video: {m['video']}   ·   🎭 Personaje: {m.get('personaje','?')}\n"
            f"📁 Plantilla: {m.get('plantilla_slug') or '—'}"
            f"{'  (ya existe aquí ✅)' if m.get('plantilla_slug') and os.path.isdir(os.path.join(_IP_TPL, m['plantilla_slug'])) else '  (no existe aquí: se creará vacía)'}\n"
            f"🗂️ {m.get('n_archivos','?')} archivos · exportado {m.get('exportado_utc','?')} · "
            f"voz incluida: {'sí' if m.get('incluye_voz') else 'no'}\n"
            f"📋 Assets de plantilla que tendrás que subir: {len(om)}\n"
            f"{'⚠️ Ya existe un video con ese nombre aquí (marca «Reemplazar» para sobrescribirlo).' if ya else ''}")

def _ip_reescribir_json(carpeta, viejo, nuevo):
    if not viejo or viejo == nuevo:
        return 0
    n = 0
    for root, _, files in os.walk(carpeta):
        for fn in files:
            if fn.lower().endswith(".json"):
                p = os.path.join(root, fn)
                try:
                    if os.path.getsize(p) > 64 * 1024**2: continue
                    t = open(p, encoding="utf-8").read()
                    if viejo in t:
                        open(p, "w", encoding="utf-8").write(t.replace(viejo, nuevo)); n += 1
                except Exception:
                    pass
    return n

def _ip_copiar(fsrc, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, "wb") as o:
        shutil.copyfileobj(fsrc, o, 8 * 1024 * 1024)

def ip_importar(upload, sel, manual, reemplazar, restaurar_voz, activar, progress=gr.Progress()):
    h = None
    stage = None
    try:
        tipo, m, h = _ip_abrir(_ip_origen(upload, sel, manual))
        _ip_validar(m)
        nombre, slug = m["video"], m.get("plantilla_slug", "") or ""
        dest = os.path.join(_IP_VID, nombre)
        if os.path.exists(dest) and not reemplazar:
            return f"⛔ Ya existe el video «{nombre}» aquí. Marca «Reemplazar si ya existe» si quieres sobrescribirlo. No se hizo nada."
        os.makedirs(_IP_VID, exist_ok=True)
        stage = os.path.join(_IP_VID, f".import_{nombre}")
        if os.path.exists(stage):
            shutil.rmtree(stage)
        os.makedirs(stage)
        stage_real = os.path.realpath(stage)

        # ── lista de archivos a copiar ──
        if tipo == "zip":
            zf, pref = h
            pv = f"{pref}video/{nombre}/"
            items = [(i.filename[len(pv):], i.file_size, i) for i in zf.infolist()
                     if i.filename.startswith(pv) and not i.is_dir()]
        else:
            base_v = os.path.join(h, "video", nombre)
            if not os.path.isdir(base_v):
                raise RuntimeError(f"No encuentro la carpeta video/{nombre} en el origen.")
            items = []
            for root, _, files in os.walk(base_v):
                for fn in files:
                    fp = os.path.join(root, fn)
                    items.append((os.path.relpath(fp, base_v).replace(os.sep, "/"), os.path.getsize(fp), fp))
        if not items:
            raise RuntimeError("El export no contiene archivos del video.")
        total = sum(s for _, s, _ in items) or 1
        libre = shutil.disk_usage(_IP_VID).free
        if libre < total * 1.02 + 200 * 1024**2:
            raise RuntimeError(f"Espacio insuficiente en /kaggle/working: necesitas ~{_ip_fmt(total)} y hay {_ip_fmt(libre)}.")

        # ── copiar con protección de rutas ──
        hecho = 0
        for rel, size, src in items:
            dst = os.path.realpath(os.path.join(stage_real, rel))
            if not dst.startswith(stage_real + os.sep):
                continue
            if tipo == "zip":
                with h[0].open(src) as fs:
                    _ip_copiar(fs, dst)
            else:
                with open(src, "rb") as fs:
                    _ip_copiar(fs, dst)
            hecho += size
            progress(min(hecho / total, 1.0) * 0.95, desc=f"{_ip_fmt(hecho)} / {_ip_fmt(total)}")
        for sd in _IP_VIDEO_SUBDIRS:
            os.makedirs(os.path.join(stage_real, sd), exist_ok=True)
        viejo_out = m.get("output_dir", "")
        reesc = _ip_reescribir_json(stage_real, viejo_out, _IP_OUT)

        # ── intercambio atómico ──
        if os.path.exists(dest):
            shutil.rmtree(dest)
        os.replace(stage_real, dest)
        stage = None
        msg = [f"✅ Video importado: {dest}  ({_ip_fmt(total)})"]
        if reesc:
            msg.append(f"🔧 Rutas ajustadas en {reesc} archivo(s) .json.")

        # ── plantilla: solo se crea vacía si NO existe; nunca se toca una existente ──
        pl = m.get("plantilla") or {}
        if slug:
            base_t = os.path.join(_IP_TPL, slug)
            omit = pl.get("archivos_omitidos", [])
            if not os.path.isdir(base_t):
                for sub in (pl.get("subdirs") or []):
                    if _ip_nombre_ok(sub.split("/")[0]) and ".." not in sub.split("/"):
                        os.makedirs(os.path.join(base_t, sub), exist_ok=True)
                os.makedirs(base_t, exist_ok=True)
                meta = dict(pl.get("meta") or {})
                meta.setdefault("nombre", slug); meta["slug"] = slug
                with open(os.path.join(base_t, "plantilla.json"), "w", encoding="utf-8") as f:
                    json.dump(meta, f, ensure_ascii=False, indent=2)
                msg.append(f"📁 Plantilla «{slug}» creada VACÍA (estructura + plantilla.json).")
                faltan = omit
            else:
                msg.append(f"📁 Plantilla «{slug}» ya existía: no se tocó.")
                faltan = [o for o in omit if not os.path.exists(os.path.join(base_t, o["ruta"]))]
            if faltan:
                msg.append(f"📋 Vuelve a subir estos {len(faltan)} assets a la plantilla «{slug}» "
                           f"(carpeta {base_t}):")
                msg += [f"   · {o['ruta']}  ({_ip_fmt(o['bytes'])})" for o in faltan[:80]]
                if len(faltan) > 80: msg.append(f"   … y {len(faltan)-80} más.")
            else:
                msg.append("📋 La plantilla ya tiene todos los assets del original ✅")
        else:
            msg.append("ℹ️ El export no tenía plantilla asociada.")

        # ── voz del concursante ──
        if restaurar_voz and m.get("incluye_voz"):
            n_voz = 0
            os.makedirs(_IP_VOICE_DIR, exist_ok=True)
            destinos = {"contestant_voice_state.json": _IP_VOICE_STATE}
            destinos.update({f"voice_refs/{fn}": os.path.join(_IP_VOICE_DIR, fn) for fn in _IP_VOICE_FILES})
            for rel, dst in destinos.items():
                try:
                    if tipo == "zip":
                        nm = f"{h[1]}voz_concursante/{rel}"
                        if nm not in h[0].namelist(): continue
                        with h[0].open(nm) as fs: _ip_copiar(fs, dst)
                    else:
                        sp = os.path.join(h, "voz_concursante", rel)
                        if not os.path.isfile(sp): continue
                        with open(sp, "rb") as fs: _ip_copiar(fs, dst)
                    n_voz += 1
                except Exception as e:
                    msg.append(f"⚠️ Voz: no pude restaurar {rel}: {e}")
            if viejo_out and viejo_out != _IP_OUT and os.path.isfile(_IP_VOICE_STATE):
                t = open(_IP_VOICE_STATE, encoding="utf-8").read()
                open(_IP_VOICE_STATE, "w", encoding="utf-8").write(t.replace(viejo_out, _IP_OUT))
            try:  # sincroniza el estado en memoria sin reasignar la variable
                st = json.load(open(_IP_VOICE_STATE, encoding="utf-8"))
                cur = globals().get("CONTESTANT_VOICE_STATE")
                if isinstance(cur, dict):
                    cur.clear(); cur.update(st)
            except Exception:
                pass
            msg.append(f"🎙️ Voz del concursante restaurada ({n_voz} archivos; sobrescribe la voz que hubiera aquí).")
        elif m.get("incluye_voz"):
            msg.append("🎙️ El export incluía la voz del concursante pero no la restauraste.")

        # ── dejar como video activo ──
        if activar:
            try:
                st = json.load(open(_IP_UIST, encoding="utf-8")) if os.path.exists(_IP_UIST) else {}
            except Exception:
                st = {}
            st["current_video_dir"] = dest
            if slug: st["current_template_slug"] = slug
            json.dump(st, open(_IP_UIST, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            for k, v in (("_CUR_VP_DIR", dest), ("_CUR_TPL", slug)):
                if k in globals() and v: globals()[k] = v
            ui = globals().get("_UI_STATE")
            if isinstance(ui, dict):
                ui["current_video_dir"] = dest
                if slug: ui["current_template_slug"] = slug
            msg.append("🟢 Marcado como video activo. En la UI principal pulsa «📋 Listar» para verlo.")
        progress(1.0, desc="Listo")
        try: log("[IMPORT] " + " | ".join(x for x in msg if x))
        except Exception: pass
        return "\n".join(msg)
    except Exception as e:
        return f"❌ No se importó nada: {e}"
    finally:
        try:
            if h is not None and isinstance(h, tuple): h[0].close()
        except Exception:
            pass
        if stage and os.path.isdir(stage):
            shutil.rmtree(stage, ignore_errors=True)

def ip_refrescar():
    return gr.update(choices=_ip_escanear(), value=None)

with gr.Blocks(title="📥 Importar video") as demo_importar:
    gr.Markdown("## 📥 Importar un video exportado desde otra cuenta\n"
                "Elige **una** de las tres formas de indicar el origen (si hay varias, manda: subida → ruta escrita → lista).")
    with gr.Row():
        dd_origen = gr.Dropdown(choices=_ip_escanear(), value=None, allow_custom_value=True, scale=4,
                                label="Detectado en /kaggle/input y rutas habituales (.zip o carpeta)")
        btn_buscar = gr.Button("🔄 Buscar de nuevo", scale=1)
    tb_manual = gr.Textbox(label="…o escribe la ruta de un .zip o de una carpeta", placeholder="/kaggle/input/mi-dataset/...")
    f_subida = gr.File(label="…o sube aquí el .zip (para archivos muy grandes es más rápido usar un Dataset)",
                       file_types=[".zip"], type="filepath")
    with gr.Row():
        chk_reemp = gr.Checkbox(value=False, label="Reemplazar si ya existe un video con ese nombre (borra el existente)")
        chk_voz = gr.Checkbox(value=True, label="Restaurar voz del concursante (sobrescribe la actual)")
        chk_act = gr.Checkbox(value=True, label="Dejarlo como video activo")
    with gr.Row():
        btn_ver = gr.Button("🔍 Ver qué contiene", variant="secondary")
        btn_imp = gr.Button("📥 Importar", variant="primary")
    salida = gr.Textbox(label="Resultado", interactive=False, lines=14)

    btn_buscar.click(ip_refrescar, [], [dd_origen])
    btn_ver.click(ip_inspeccionar, [f_subida, dd_origen, tb_manual], [salida])
    btn_imp.click(ip_importar, [f_subida, dd_origen, tb_manual, chk_reemp, chk_voz, chk_act], [salida])

demo_importar.queue().launch(share=True, prevent_thread_lock=True, show_error=True)

In [ ]:
# ═══ 🛑 DETENER LTX — mata el subproceso de generación (no toca nada más) ═══
import os, time, signal, psutil, json

_yo = os.getpid()
_objetivo = []
for p in psutil.process_iter(["pid", "cmdline"]):
    try:
        cmd = " ".join(p.info["cmdline"] or [])
        if "run_ltx25_generate.py" in cmd and p.info["pid"] != _yo:
            _objetivo.append(p)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

if not _objetivo:
    print("ℹ️ No hay ningún subproceso LTX en ejecución.")
else:
    # incluir hijos (por si lanzó procesos auxiliares)
    todos = {}
    for p in _objetivo:
        todos[p.pid] = p
        try:
            for c in p.children(recursive=True):
                todos[c.pid] = c
        except psutil.NoSuchProcess:
            pass

    print(f"🛑 Deteniendo {len(todos)} proceso(s): {sorted(todos)}")
    for p in todos.values():
        try: p.send_signal(signal.SIGTERM)
        except psutil.NoSuchProcess: pass
    _, vivos = psutil.wait_procs(list(todos.values()), timeout=8)
    for p in vivos:
        try: p.kill()
        except psutil.NoSuchProcess: pass
    psutil.wait_procs(vivos, timeout=5)

    # dejar el archivo de progreso coherente para la UI
    try:
        with open("/tmp/ltx25_progress.json", "w") as f:
            json.dump({"status": "🛑 Detenido manualmente"}, f)
    except Exception:
        pass

    time.sleep(1)
    try:
        import subprocess
        print(subprocess.run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader",
                             shell=True, capture_output=True, text=True).stdout.strip(), "(VRAM en uso)")
    except Exception:
        pass
    print("✅ LTX detenido.")

In [ ]:
import requests, time, os

def probar(base, mb=5):
    base = base.rstrip("/")
    try:
        t = time.time(); requests.get(f"{base}/config", timeout=60); rtt = time.time() - t
        data = os.urandom(mb * 1024 * 1024)
        t = time.time()
        r = requests.post(f"{base}/gradio_api/upload", files=[("files", ("test.bin", data))], timeout=300)
        up = time.time() - t
        path = r.json()[0]
        t = time.time()
        d = requests.get(f"{base}/gradio_api/file={path}", timeout=300)
        down = time.time() - t
        print(f"{base[:48]:<49} latencia {rtt:5.2f}s | subida {mb/up:6.2f} MB/s | bajada {len(d.content)/1024**2/down:6.2f} MB/s")
    except Exception as e:
        print(f"{base[:48]:<49} ❌ {type(e).__name__}: {str(e)[:120]}")

probar(demo.local_url)
if getattr(demo, "share_url", None):
    probar(demo.share_url)

In [ ]:
import os, subprocess, shutil, psutil, importlib.metadata as md, socket, glob

def hdr(t): print(f"\n── {t} " + "─"*(50-len(t)))

hdr("RAM / CPU")
vm = psutil.virtual_memory()
print(f"RAM: {vm.used/1024**3:.1f} / {vm.total/1024**3:.1f} GB ({vm.percent}%)  |  CPU: {psutil.cpu_percent(interval=1)}%")

hdr("DISCO")
for p in ["/kaggle/working", "/tmp"]:
    u = shutil.disk_usage(p)
    print(f"{p}: libre {u.free/1024**3:.1f} GB de {u.total/1024**3:.1f} GB")

hdr("TOP PROCESOS (RAM)")
procs = []
for p in psutil.process_iter(["pid", "name", "cmdline", "memory_info", "cpu_percent"]):
    try:
        procs.append((p.info["memory_info"].rss/1024**3, p.info["pid"], " ".join(p.info["cmdline"] or [])[:110]))
    except Exception:
        pass
for rss, pid, cmd in sorted(procs, reverse=True)[:8]:
    print(f"{rss:5.1f} GB  pid {pid:<7} {cmd}")

hdr("PROCESOS QUE PUEDEN BLOQUEAR")
out = subprocess.run("ps aux | grep -E 'run_ltx25|qwen3_tts_worker|hf_transfer|huggingface|git clone|pip ' | grep -v grep",
                     shell=True, capture_output=True, text=True).stdout
print(out.strip() or "ninguno")

hdr("GRADIO / TÚNEL")
print("gradio:", md.version("gradio"))
frpc = glob.glob(os.path.expanduser("~/.cache/huggingface/gradio/frpc/*")) + glob.glob("/usr/local/lib/python3*/dist-packages/gradio/frpc_*")
print("frpc (túnel share=True):", frpc or "NO descargado todavía")
for host in ["huggingface.co", "api.gradio.app", "cdn-media.huggingface.co"]:
    try:
        socket.create_connection((host, 443), timeout=5).close(); print(f"✅ {host}")
    except Exception as e:
        print(f"❌ {host}: {e}")

hdr("LOG")
lg = glob.glob("/kaggle/working/**/run.log", recursive=True)
for f in lg[:3]:
    print(f"{f}: {os.path.getsize(f)/1024**2:.2f} MB")

hdr("HF_TRANSFER")
print("HF_HUB_ENABLE_HF_TRANSFER =", os.environ.get("HF_HUB_ENABLE_HF_TRANSFER"))
try: print("hf_transfer instalado:", md.version("hf_transfer"))
except Exception: print("hf_transfer instalado: no")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA EXTRA: 🧹 LIMPIADOR DE MODELOS QWEN-IMAGE (control total del disco)
#
# - Lista cada modelo Qwen-Image descargado en local_models: diffusion (GGUF), encoder
#   Qwen3-VL (GGUF), mmproj y VAE, con su tamaño real.
# - TÚ marcas cuáles borrar; nada se borra sin tu confirmación.
# - Solo toca archivos dentro de LOCAL_MODELS_DIR. NO toca: TTS, LTX, plantillas, videos,
#   sd-cli ni nada de /kaggle/input (que es de solo lectura).
# - Es independiente: no necesita que la UI principal esté abierta (abre su propio enlace).
# - Si borras un modelo que luego se necesita, la Etapa 1 lo vuelve a descargar sola.
# ═══════════════════════════════════════════════════════════════
import os, re, json, shutil
import gradio as gr

# ── Rutas (usa las del notebook si existen; si no, las de siempre) ──
_LM_DIR = globals().get("LOCAL_MODELS_DIR") or "/kaggle/working/VideoAutomationProject/local_models"
_LM_DIR = os.path.realpath(_LM_DIR)
_QDIFF  = globals().get("QWEN_IMAGE_DIFFUSION_QUANTS") or {
    "Q2_K": "qwen_image_2.1-Q2_K.gguf", "Q3_K": "qwen_image_2.1-Q3_K.gguf",
    "Q4_0": "qwen_image_2.1-Q4_0.gguf", "Q4_K": "qwen_image_2.1-Q4_K.gguf",
    "Q5_0": "qwen_image_2.1-Q5_0.gguf", "Q6_K": "qwen_image_2.1-Q6_K.gguf",
    "Q8_0": "qwen_image_2.1-Q8_0.gguf"}
_QENC   = globals().get("QWEN_VL_ENCODERS") or {
    "Q4_K_M": "Qwen3VL-8B-Instruct-Q4_K_M.gguf", "Q8_0": "Qwen3VL-8B-Instruct-Q8_0.gguf",
    "F16": "Qwen3VL-8B-Instruct-F16.gguf"}
_QMM    = globals().get("QWEN_VL_MMPROJ") or {
    "F16": "mmproj-Qwen3VL-8B-Instruct-F16.gguf", "Q8_0": "mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf"}

_CATS = [  # (clave, título, descripción)
    ("diff", "🎨 Modelo de imagen (diffusion)", "Es el pesado: genera las imágenes A→G."),
    ("enc",  "🧠 Encoder de texto/visión (Qwen3-VL)", "Entiende el prompt y las imágenes de referencia."),
    ("mm",   "👁️ mmproj (visión del encoder)", "Necesario para las ediciones con imágenes de referencia (B→G)."),
    ("vae",  "🖼️ VAE", "Pequeño, pero imprescindible si vas a generar imágenes."),
    ("inc",  "⚠️ Descargas incompletas (.incomplete)", "Restos de descargas cortadas: se pueden borrar sin riesgo."),
]

def _fmt(n: int) -> str:
    return f"{n/1024**3:.2f} GB" if n >= 1024**3 else f"{n/1024**2:.0f} MB"

def _free(path: str) -> float:
    try:
        return shutil.disk_usage(path).free / 1024**3
    except Exception:
        return float("nan")

def _en_uso() -> set:
    # Nombres de archivo configurados en el proyecto activo (solo para avisarte; tú decides).
    usados = set()
    try:
        vd = globals().get("_CUR_VP_DIR") or ""
        pj = os.path.join(vd, "params.json")
        if vd and os.path.exists(pj):
            p = (json.load(open(pj, encoding="utf-8")).get("etapa1") or {})
            for key, tabla in (("diffusion_quant", _QDIFF), ("encoder_quant", _QENC), ("mmproj_quant", _QMM)):
                f = tabla.get(p.get(key, ""))
                if f: usados.add(f)
    except Exception:
        pass
    return usados

def _clasificar(nombre: str):
    n = nombre.lower()
    if n.endswith(".incomplete"):                                   return "inc"
    if n.startswith("qwen_image") and "vae" in n:                   return "vae"
    if n.startswith("mmproj-qwen3vl") and n.endswith(".gguf"):      return "mm"
    if n.startswith("qwen3vl") and n.endswith(".gguf"):             return "enc"
    if n.startswith("qwen_image") and n.endswith(".gguf"):          return "diff"
    return None

def _escanear():
    # → ({cat: [(relpath, nombre, bytes)]}, bytes_otros, bytes_total_dir)
    cat = {k: [] for k, _, _ in _CATS}
    otros = total = 0
    if not os.path.isdir(_LM_DIR):
        return cat, 0, 0
    for root, _, files in os.walk(_LM_DIR):
        for fn in files:
            p = os.path.join(root, fn)
            if os.path.islink(p):
                continue
            try:
                sz = os.path.getsize(p)
            except OSError:
                continue
            total += sz
            c = _clasificar(fn)
            if c:
                cat[c].append((os.path.relpath(p, _LM_DIR), fn, sz))
            else:
                otros += sz
    for k in cat:
        cat[k].sort(key=lambda x: -x[2])
    return cat, otros, total

def _choices(items, usados):
    out = []
    for rel, fn, sz in items:
        tag = "   ⚙️ configurado en el proyecto activo" if fn in usados else ""
        out.append((f"{fn}  ·  {_fmt(sz)}{tag}", rel))
    return out

def _disk_html():
    cat, otros, total = _escanear()
    qsz = sum(sz for k in ("diff", "enc", "mm", "vae") for _, _, sz in cat[k])
    inc = sum(sz for _, _, sz in cat["inc"])
    return (
        "<div style='background:#10192b;border:1px solid #2a3a5c;border-radius:12px;padding:14px;"
        "color:#dbe4ff;font-size:0.95em;line-height:1.7'>"
        f"💾 <b>Libre en /kaggle/working:</b> {_free('/kaggle/working'):.2f} GB &nbsp;·&nbsp; "
        f"<b>/kaggle/tmp:</b> {_free('/kaggle/tmp'):.2f} GB<br>"
        f"📦 <b>local_models total:</b> {_fmt(total)} &nbsp;·&nbsp; "
        f"<b>Qwen-Image (diffusion+encoder+mmproj+VAE):</b> {_fmt(qsz)} &nbsp;·&nbsp; "
        f"<b>Incompletas:</b> {_fmt(inc)}<br>"
        f"🔒 <b>Otros archivos (TTS, etc.) — no se tocan aquí:</b> {_fmt(otros)}"
        "</div>")

def _updates():
    cat, _, _ = _escanear()
    usados = _en_uso()
    return [gr.update(choices=_choices(cat[k], usados), value=[]) for k, _, _ in _CATS]

def _sel_a_rutas(*sels):
    # Revalida CADA ruta contra un escaneo nuevo: solo se puede borrar lo que este escáner reconoce.
    cat, _, _ = _escanear()
    validas = {rel: (fn, sz) for k in cat for rel, fn, sz in cat[k]}
    out = []
    for grupo in sels:
        for rel in (grupo or []):
            if rel in validas:
                out.append((rel, validas[rel][0], validas[rel][1]))
    return out

def ui_calcular(*sels):
    items = _sel_a_rutas(*sels)
    if not items:
        return "Nada marcado."
    return f"Marcados: {len(items)} archivo(s) → se liberarían **{_fmt(sum(i[2] for i in items))}**."

def ui_pedir_confirmacion(*sels):
    items = _sel_a_rutas(*sels)
    if not items:
        return gr.update(visible=False), "ℹ️ No has marcado ningún archivo."
    usados = _en_uso()
    lineas = [f"- `{fn}` — {_fmt(sz)}" + ("  ⚠️ **está configurado en el proyecto activo**" if fn in usados else "")
              for _, fn, sz in items]
    md = ("### ¿Eliminar definitivamente?\n" + "\n".join(lineas) +
          f"\n\n**Total a liberar: {_fmt(sum(i[2] for i in items))}.** Esta acción no se puede deshacer "
          "(para volver a tenerlos habría que descargarlos de nuevo, y eso requiere internet activado).")
    return gr.update(visible=True), md

def ui_cancelar():
    return gr.update(visible=False), "Cancelado. No se borró nada."

def ui_borrar(*sels):
    items = _sel_a_rutas(*sels)
    if not items:
        return ("ℹ️ No había nada válido que borrar.", _disk_html(), gr.update(visible=False), "", *_updates())
    antes = _free("/kaggle/working")
    res, liberado = [], 0
    for rel, fn, sz in items:
        p = os.path.realpath(os.path.join(_LM_DIR, rel))
        if not p.startswith(_LM_DIR + os.sep):             # defensa extra: jamás salir de local_models
            res.append(f"⛔ Ruta fuera de local_models, omitida: {fn}")
            continue
        try:
            os.remove(p)
            liberado += sz
            res.append(f"🗑️ Borrado: {fn}  ({_fmt(sz)})")
            d = os.path.dirname(p)                          # quita la carpeta si quedó vacía (p. ej. vae/)
            while d != _LM_DIR and d.startswith(_LM_DIR + os.sep) and not os.listdir(d):
                os.rmdir(d); d = os.path.dirname(d)
        except Exception as e:
            res.append(f"❌ No se pudo borrar {fn}: {e}")
    res.append("")
    res.append(f"✅ Liberado: {_fmt(liberado)}  ·  Libre ahora en /working: {_free('/kaggle/working'):.2f} GB "
               f"(antes {antes:.2f} GB)")
    try:
        log("[LIMPIADOR QWEN-IMAGE] " + " | ".join(r for r in res if r))
    except Exception:
        pass
    return ("\n".join(res), _disk_html(), gr.update(visible=False), "", *_updates())

def ui_refrescar():
    return (_disk_html(), *_updates(), "Nada marcado.")

# ── Interfaz ────────────────────────────────────────────────
_cat0, _, _ = _escanear()
_usados0 = _en_uso()

with gr.Blocks(title="🧹 Limpiador Qwen-Image") as demo_clean:
    gr.Markdown("## 🧹 Limpiador de modelos Qwen-Image\n"
                "Marca los modelos que quieras eliminar. **Tú decides**: nada se borra hasta que confirmes.")
    disco_html = gr.HTML(_disk_html())
    btn_ref = gr.Button("🔄 Refrescar lista y espacio", variant="secondary")

    grupos = []
    for k, titulo, desc in _CATS:
        with gr.Group():
            gr.Markdown(f"**{titulo}** — {desc}")
            cg = gr.CheckboxGroup(choices=_choices(_cat0[k], _usados0), value=[], label="",
                                  show_label=False)
        grupos.append(cg)

    calc_md = gr.Markdown("Nada marcado.")
    btn_borrar = gr.Button("🗑️ Eliminar los marcados…", variant="stop")
    with gr.Group(visible=False) as confirm_box:
        confirm_md = gr.Markdown("")
        with gr.Row():
            btn_si = gr.Button("✅ Sí, eliminar definitivamente", variant="stop")
            btn_no = gr.Button("↩️ Cancelar", variant="secondary")
    resultado = gr.Textbox(label="Resultado", interactive=False, lines=8)

    for cg in grupos:
        cg.change(ui_calcular, grupos, [calc_md])
    btn_borrar.click(ui_pedir_confirmacion, grupos, [confirm_box, confirm_md])
    btn_no.click(ui_cancelar, [], [confirm_box, confirm_md])
    btn_si.click(ui_borrar, grupos, [resultado, disco_html, confirm_box, confirm_md] + grupos)
    btn_ref.click(ui_refrescar, [], [disco_html] + grupos + [calc_md])

demo_clean.queue().launch(share=True, prevent_thread_lock=True, show_error=True)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 19 (v2 con UI): 🧹 LIMPIEZA DE VIDEOS Y PLANTILLAS — TÚ DECIDES QUÉ BORRAR
#
# - Lista cada VIDEO (proyecto) y cada PLANTILLA con su tamaño real, fecha y estado.
# - Marcas lo que quieras eliminar; nada se borra hasta que confirmes.
# - Solo toca carpetas que cuelgan directamente de /videos y /plantillas.
#   NO toca: sd-cli, qwen-tts, modelos (para eso está el limpiador de Qwen-Image), LTX.
# - ⚠️ Las plantillas guardan TUS imágenes de jueces, voces y reacciones: no se pueden regenerar.
#   Por eso, para borrar una plantilla hay que marcar además una casilla de aviso.
# - Independiente: abre su propio enlace, no necesita la UI principal.
# ═══════════════════════════════════════════════════════════════
import os, re, json, shutil, time
import gradio as gr

_OUT   = globals().get("OUTPUT_DIR") or "/kaggle/working/VideoAutomationProject"
_VID   = os.path.realpath(globals().get("VIDEOS_DIR") or f"{_OUT}/videos")
_TPL   = os.path.realpath(globals().get("PLANTILLAS_DIR") or f"{_OUT}/plantillas")
_UIST  = os.path.join(_OUT, "ui_state.json")

def _fmt(n: int) -> str:
    return f"{n/1024**3:.2f} GB" if n >= 1024**3 else f"{n/1024**2:.0f} MB"

def _free(path: str) -> float:
    try:    return shutil.disk_usage(path).free / 1024**3
    except Exception: return float("nan")

def _dir_size(path: str) -> int:
    tot = 0
    for root, _, files in os.walk(path):
        for fn in files:
            fp = os.path.join(root, fn)
            if not os.path.islink(fp):
                try: tot += os.path.getsize(fp)
                except OSError: pass
    return tot

def _activos():
    # (video_dir_activo, plantilla_activa) del notebook; si no existen, de ui_state.json
    v = globals().get("_CUR_VP_DIR") or ""
    t = globals().get("_CUR_TPL") or ""
    if not (v and t):
        try:
            st = json.load(open(_UIST, encoding="utf-8"))
            v = v or st.get("current_video_dir", "")
            t = t or st.get("current_template_slug", "")
        except Exception:
            pass
    return (os.path.realpath(v) if v else ""), t

def _subdirs(base: str):
    if not os.path.isdir(base):
        return []
    return [d for d in sorted(os.listdir(base)) if os.path.isdir(os.path.join(base, d)) and not d.startswith(".")]

def _escanear():
    # → (videos, plantillas): listas de dict(nombre, ruta, bytes, mtime, extra)
    vids, tpls = [], []
    for d in _subdirs(_VID):
        p = os.path.join(_VID, d)
        vids.append({"nombre": d, "ruta": p, "bytes": _dir_size(p), "mtime": os.path.getmtime(p),
                     "final": os.path.exists(os.path.join(p, "VIDEO_FINAL.mp4"))})
    for d in _subdirs(_TPL):
        p = os.path.join(_TPL, d)
        nombre = d
        try:
            nombre = json.load(open(os.path.join(p, "plantilla.json"), encoding="utf-8")).get("nombre", d)
        except Exception:
            pass
        tpls.append({"nombre": d, "visible": nombre, "ruta": p, "bytes": _dir_size(p), "mtime": os.path.getmtime(p)})
    vids.sort(key=lambda x: -x["mtime"])
    tpls.sort(key=lambda x: -x["bytes"])
    return vids, tpls

def _choices_v(vids):
    av, _ = _activos()
    out = []
    for v in vids:
        f = time.strftime("%Y-%m-%d %H:%M", time.localtime(v["mtime"]))
        tags = ("  🎞️ con VIDEO_FINAL" if v["final"] else "") + ("  🟢 VIDEO ACTIVO" if os.path.realpath(v["ruta"]) == av else "")
        out.append((f"{v['nombre']}  ·  {_fmt(v['bytes'])}  ·  {f}{tags}", v["nombre"]))
    return out

def _choices_t(tpls):
    _, at = _activos()
    out = []
    for t in tpls:
        tag = "  🟢 PLANTILLA ACTIVA" if t["nombre"] == at else ""
        etiqueta = t["nombre"] if t["visible"] == t["nombre"] else f"{t['visible']} ({t['nombre']})"
        out.append((f"{etiqueta}  ·  {_fmt(t['bytes'])}{tag}", t["nombre"]))
    return out

def _disk_html():
    vids, tpls = _escanear()
    return ("<div style='background:#10192b;border:1px solid #2a3a5c;border-radius:12px;padding:14px;"
            "color:#dbe4ff;font-size:0.95em;line-height:1.7'>"
            f"💾 <b>Libre en /kaggle/working:</b> {_free('/kaggle/working'):.2f} GB &nbsp;·&nbsp; "
            f"<b>/kaggle/tmp:</b> {_free('/kaggle/tmp'):.2f} GB<br>"
            f"🎬 <b>Videos:</b> {len(vids)} ({_fmt(sum(v['bytes'] for v in vids))}) &nbsp;·&nbsp; "
            f"📁 <b>Plantillas:</b> {len(tpls)} ({_fmt(sum(t['bytes'] for t in tpls))})"
            "</div>")

def _updates():
    vids, tpls = _escanear()
    return gr.update(choices=_choices_v(vids), value=[]), gr.update(choices=_choices_t(tpls), value=[])

def _validar(sel_v, sel_t):
    # Revalida contra un escaneo nuevo: solo se borra lo que este escáner reconoce bajo /videos y /plantillas.
    vids, tpls = _escanear()
    mv = {v["nombre"]: v for v in vids}
    mt = {t["nombre"]: t for t in tpls}
    return ([("video", mv[n]) for n in (sel_v or []) if n in mv],
            [("plantilla", mt[n]) for n in (sel_t or []) if n in mt])

def ui_calcular(sel_v, sel_t):
    v, t = _validar(sel_v, sel_t)
    if not (v or t):
        return "Nada marcado."
    tot = sum(x["bytes"] for _, x in v + t)
    return f"Marcados: {len(v)} video(s) y {len(t)} plantilla(s) → se liberarían **{_fmt(tot)}**."

def ui_pedir_confirmacion(sel_v, sel_t):
    v, t = _validar(sel_v, sel_t)
    if not (v or t):
        return gr.update(visible=False), "ℹ️ No has marcado nada.", gr.update(visible=False, value=False)
    av, at = _activos()
    lineas = []
    for _, x in v:
        act = "  ⚠️ **es el video activo**" if os.path.realpath(x["ruta"]) == av else ""
        lineas.append(f"- 🎬 `{x['nombre']}` — {_fmt(x['bytes'])}{act}")
    for _, x in t:
        act = "  ⚠️ **es la plantilla activa**" if x["nombre"] == at else ""
        lineas.append(f"- 📁 `{x['nombre']}` — {_fmt(x['bytes'])}{act}")
    md = ("### ¿Eliminar definitivamente?\n" + "\n".join(lineas) +
          f"\n\n**Total a liberar: {_fmt(sum(x['bytes'] for _, x in v + t))}.** No se puede deshacer.")
    if t:
        md += ("\n\n🚨 **Las plantillas contienen tus imágenes de jueces, voces y reacciones y NO se pueden "
               "regenerar.** Marca la casilla de abajo para confirmar que quieres borrarlas.")
    return gr.update(visible=True), md, gr.update(visible=bool(t), value=False)

def ui_cancelar():
    return gr.update(visible=False), "Cancelado. No se borró nada.", gr.update(visible=False, value=False)

def _limpiar_ui_state(borrados_v, borrados_t):
    try:
        st = json.load(open(_UIST, encoding="utf-8"))
    except Exception:
        return
    ch = False
    if st.get("current_video_dir") and os.path.realpath(st["current_video_dir"]) in borrados_v:
        st["current_video_dir"] = ""; ch = True
    if st.get("current_template_slug") in borrados_t:
        st["current_template_slug"] = ""; ch = True
    if ch:
        json.dump(st, open(_UIST, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

def ui_borrar(sel_v, sel_t, ack_plantillas):
    v, t = _validar(sel_v, sel_t)
    if t and not ack_plantillas:
        return ("⛔ Para borrar plantillas marca la casilla de aviso. No se borró nada.",
                _disk_html(), gr.update(visible=True), gr.update(), *[gr.update(), gr.update()],
                gr.update(visible=True))
    if not (v or t):
        return ("ℹ️ No había nada válido que borrar.", _disk_html(), gr.update(visible=False), "",
                *_updates(), gr.update(visible=False, value=False))
    antes = _free("/kaggle/working")
    res, liberado, bv, bt = [], 0, set(), set()
    for tipo, x in v + t:
        base = _VID if tipo == "video" else _TPL
        p = os.path.realpath(x["ruta"])
        if os.path.dirname(p) != base:                      # defensa: solo carpetas hijas directas
            res.append(f"⛔ Ruta no permitida, omitida: {x['nombre']}")
            continue
        try:
            shutil.rmtree(p)
            liberado += x["bytes"]
            (bv if tipo == "video" else bt).add(p if tipo == "video" else x["nombre"])
            res.append(f"🗑️ Borrado ({tipo}): {x['nombre']}  ({_fmt(x['bytes'])})")
        except Exception as e:
            res.append(f"❌ No se pudo borrar {x['nombre']}: {e}")
    _limpiar_ui_state(bv, bt)
    res += ["", f"✅ Liberado: {_fmt(liberado)}  ·  Libre ahora en /working: {_free('/kaggle/working'):.2f} GB "
                f"(antes {antes:.2f} GB)"]
    if bv or bt:
        res.append("ℹ️ Si borraste el video/plantilla activo, en la UI principal pulsa «📋 Listar» y elige otro (o crea uno nuevo).")
    try: log("[LIMPIEZA UI] " + " | ".join(r for r in res if r))
    except Exception: pass
    return ("\n".join(res), _disk_html(), gr.update(visible=False), "", *_updates(),
            gr.update(visible=False, value=False))

def ui_refrescar():
    return (_disk_html(), *_updates(), "Nada marcado.")

# ── Interfaz ────────────────────────────────────────────────
_v0, _t0 = _escanear()
with gr.Blocks(title="🧹 Limpieza de videos y plantillas") as demo_limpieza:
    gr.Markdown("## 🧹 Limpieza de videos y plantillas\n"
                "Marca lo que quieras eliminar. **Tú decides**: nada se borra hasta que confirmes.")
    disco_html = gr.HTML(_disk_html())
    btn_ref = gr.Button("🔄 Refrescar lista y espacio", variant="secondary")

    with gr.Group():
        gr.Markdown("**🎬 Videos (proyectos)** — se borra la carpeta completa del video: imágenes, audios, clips y VIDEO_FINAL.")
        cg_v = gr.CheckboxGroup(choices=_choices_v(_v0), value=[], label="", show_label=False)
    with gr.Group():
        gr.Markdown("**📁 Plantillas** — ⚠️ contienen tus imágenes de jueces, voces y reacciones (no se regeneran).")
        cg_t = gr.CheckboxGroup(choices=_choices_t(_t0), value=[], label="", show_label=False)

    calc_md = gr.Markdown("Nada marcado.")
    btn_borrar = gr.Button("🗑️ Eliminar los marcados…", variant="stop")
    with gr.Group(visible=False) as confirm_box:
        confirm_md = gr.Markdown("")
        ack = gr.Checkbox(label="Entiendo que las plantillas marcadas se borran para siempre", value=False, visible=False)
        with gr.Row():
            btn_si = gr.Button("✅ Sí, eliminar definitivamente", variant="stop")
            btn_no = gr.Button("↩️ Cancelar", variant="secondary")
    resultado = gr.Textbox(label="Resultado", interactive=False, lines=8)

    cg_v.change(ui_calcular, [cg_v, cg_t], [calc_md])
    cg_t.change(ui_calcular, [cg_v, cg_t], [calc_md])
    btn_borrar.click(ui_pedir_confirmacion, [cg_v, cg_t], [confirm_box, confirm_md, ack])
    btn_no.click(ui_cancelar, [], [confirm_box, confirm_md, ack])
    btn_si.click(ui_borrar, [cg_v, cg_t, ack], [resultado, disco_html, confirm_box, confirm_md, cg_v, cg_t, ack])
    btn_ref.click(ui_refrescar, [], [disco_html, cg_v, cg_t, calc_md])

demo_limpieza.queue().launch(share=True, prevent_thread_lock=True, show_error=True)


In [ ]:

#Eliminar escebas de jueses en los dialogos

import glob, os
base = "/kaggle/working/VideoAutomationProject/videos/20261002_023053_mi_plantilla_1_niña"
for etapa in ("etapa2_dialogos", "etapa4_dialogos"):
    for d in sorted(glob.glob(f"{base}/{etapa}/frag_*_juez_*")):
        for f in ("clip.mp4", "clip.sig", "clip.fallback"):
            p = os.path.join(d, f)
            if os.path.exists(p):
                print("borrando", p, os.path.getsize(p)//1024, "KB")
                os.remove(p)

# 🎤 Got Talent — Registro de versiones estables

> **Estado:** ✅ LTX-2.5 funcionando &nbsp;·&nbsp; **Snapshot:** 7-oct-2026 17:22 &nbsp;·&nbsp; **Python:** 3.13.15 &nbsp;·&nbsp; **Ubuntu 24.04.1 LTS**
> Equivale a un `requirements.txt` del proyecto: si algo se rompe, **vuelve a estas versiones**.
> Los `pip freeze` completos están en `/kaggle/working/env_snapshot/` (generados por la celda *Snapshot*).

---

## 🧭 1. Arquitectura de entornos (4 entornos aislados)

| Entorno | Ubicación | Para qué sirve |
|---|---|---|
| 🐍 **Kernel principal** | `/usr/local/lib/python3.13/dist-packages` | UI Gradio, faster-whisper, pipeline |
| 🎬 **Aislado LTX** | `/kaggle/tmp/ltx_hf_compat` (vía `PYTHONPATH`, solo del subproceso) | `transformers` + `huggingface_hub` fijados para LTX-2.5 |
| 🗣️ **Venv Qwen3-TTS** | `/tmp/qwen3_tts_env` (Python 3.13.15) | Worker de voz (Base / CustomVoice / VoiceDesign) |
| ⚙️ **Binarios C++** | `/kaggle/working/*/build/bin` | `sd-cli` (imágenes) y `qwen-tts` (voz GGUF) |

---

## 🖥️ 2. Sistema y hardware

| Item | Valor |
|---|---|
| GPU | **2 × Tesla T4** · 15 360 MiB c/u · compute capability **7.5** · driver **580.178.04** |
| Uso real de GPU | Solo **`cuda:0`**: ni este pipeline ni el notebook de referencia reparten el modelo entre las dos |
| CUDA | nvcc **12.8 (V12.8.93)** · torch `cu128` · `CUDA_ARCH=75` |
| Herramientas | cmake **3.31.10** · ffmpeg **6.1.1** · git **2.43.0** |
| RAM / disco | 31.3 GB RAM · 19.5 GB `/kaggle/working` · `/tmp` amplio |
| Imagen Docker | `gcr.io/kaggle-gpu-images/python@sha256:2757e0c7d1e0a9cb43da657b97e223c321a98f5014bdf64f44f2f6b083ad2b2f` |
| `HF_HUB_ENABLE_HF_TRANSFER` | `0` (kernel) |

---

## 🐍 3. Kernel principal — verificado ✅

| Paquete | Versión | Paquete | Versión |
|---|---|---|---|
| torch | **2.11.0+cu128** | gradio | **6.29.1** |
| torchvision | **0.26.0+cu128** | mmgp | **3.8.2** |
| torchaudio | **2.11.0+cu128** | faster-whisper | **1.2.1** |
| transformers ⚠️ | **5.16.1** | ctranslate2 | **4.8.2** |
| huggingface_hub | **0.36.2** | av | **18.1.0** |
| tokenizers | **0.23.1** | pyannote.audio | **4.0.7** |
| safetensors | **0.8.0** | torchcodec | **0.11.0+cu128** |
| accelerate | **1.14.0** | whisperx | **3.8.7rc1** · commit `771b4a14a9` |
| numpy | **2.5.3** | librosa | **0.11.0** |
| scipy | **1.16.3** | soundfile | **0.14.0** |
| numba | **0.61.2** | sox | **1.5.0** |
| pandas | **3.0.6** | omegaconf | **2.3.1** |
| pillow | **12.3.0** | nltk | **3.9.1** |
| opencv-python | **4.14.0.94** | openai | **2.54.0** |
| imageio | **2.37.4** | edge-tts | **7.2.8** |
| requests | **2.34.2** | psutil | **5.9.5** |
| sentencepiece | **0.2.2** | kaggle · langdetect | **2.0.2** · **1.0.9** |

> ⚠️ `transformers 5.16.1` del kernel **no sirve para LTX** (ver §4). `hf_transfer 0.1.9` está instalado.

---

## 🎬 4. Entorno aislado LTX-2.5 — verificado ✅

| Paquete | Versión | Por qué |
|---|---|---|
| **transformers** | **5.14.1** ⭐ | La **5.15+ rompe Gemma 4** (`AmbiguousGlobalPerLayerAttributeError: 'head_dim'`) |
| huggingface_hub | **1.33.0** | `transformers 5.x` exige `>=1.5,<2.0` |
| tokenizers | **0.22.2** | El que requiere `transformers 5.14.1` (distinto del kernel) |
| safetensors · numpy | 0.8.0 · 2.5.3 | |
| otros | hf_xet 1.7.0 · httpx 0.28.1 · httpcore 1.0.9 · anyio 4.15.1 · typer 0.27.3 · rich 15.0.0 · tqdm 4.70.1 · pyyaml 6.0.3 · regex 2026.9.29 · filelock 4.0.12 · fsspec 2026.9.0 · packaging 26.3 · click 8.5.0 | Dependencias arrastradas por `pip --target` |

**Preset en uso:** ⚡ RÁPIDO — LTX-2.5 Distilled INT8 · 8 steps · cfg 1.0 · 24 fps

---

## 🗣️ 5. Venv Qwen3-TTS (aislado) — verificado ✅

| Paquete | Versión | Paquete | Versión |
|---|---|---|---|
| qwen-tts | **0.0.5** (`--no-deps`) | torch | **2.11.0+cu128** |
| transformers | **4.57.3** | numpy | **2.2.6** |
| huggingface-hub | **0.36.0** | librosa · soundfile | **0.11.0** · **0.14.0** |
| accelerate | **1.12.0** | onnxruntime | **1.30.0** |
| gradio | **6.4.0** | einops | **0.8.2** |
| hf-transfer | **0.1.9** ✅ (instalado a mano) | | |

**Modelos:** `Qwen/Qwen3-TTS-12Hz-1.7B-Base` · `…-CustomVoice` · `…-VoiceDesign`
**Avisos normales (no son errores):** `flash-attn is not installed` · `torch_dtype is deprecated`

---

## 🧱 6. Repos clonados y binarios — verificado ✅

| Componente | Origen | Commit | Fecha | md5 binario |
|---|---|---|---|---|
| **Wan2GP** (WanGP **v17.01**) | `github.com/deepbeepmeep/Wan2GP` | **`0e58385fbd`** | 2026-10-05 | — |
| stable-diffusion.cpp (CUDA, arch 75) | `github.com/leejet/stable-diffusion.cpp` (recursive) | **`3f8527a46c`** | 2026-09-27 | `27ee8e4fc8` |
| qwentts.cpp (CUDA, arch 75) | `github.com/ServeurpersoCom/qwentts.cpp` | **`6fae929140`** | 2026-09-28 | `f12a8d4470` |
| whisperx | `github.com/m-bain/whisperx` (`--no-deps`) | **`771b4a14a9`** | — | — |

**Huellas de Wan2GP (md5[:10]) con las que LTX funciona:**

| Archivo | Huella |
|---|---|
| `wgp.py` | `a33acb7b9a` |
| `requirements.txt` | `23a261e975` |
| `models/ltx2/ltx2_handler.py` | `f65f7785f3` |
| `…/gemma/gemma4_unified.py` | `dfdea959e2` |
| `run_ltx25_generate.py` (script del proyecto) | `4c0fe521dd` |

> `requirements.txt` de Wan2GP pide `transformers==4.54.0` y `numpy==2.1.2`. **Se ignoran a propósito**: instalarlos pisaría el stack ya resuelto.

---

## 🤖 7. Modelos de IA

**Revisión HEAD de cada repo el 7-oct-2026** (referencia; no es necesariamente la descargada):

| Función | Repo HF | Revisión |
|---|---|---|
| Video LTX-2.5 | `DeepBeepMeep/LTX-2` | `3a5be42565` |
| Imágenes | `leejet/Qwen-Image-2.1-GGUF` | `cc11433936` |
| Visión-lenguaje | `Qwen/Qwen3-VL-8B-Instruct-GGUF` | `f982a07559` |
| VAE imagen | `Comfy-Org/Qwen-Image-2.1` | `cb504a4090` |
| TTS GGUF | `Serveurperso/Qwen3-TTS-GGUF` | `b7ee2e8c74` |
| TTS Base | `Qwen/Qwen3-TTS-12Hz-1.7B-Base` | `fd4b254389` |
| TTS CustomVoice | `Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice` | `0c0e3051f1` |
| TTS VoiceDesign | `Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign` | `5ecdb67327` |

**Archivos LTX-2.5 en `/tmp/models`** (tamaños en GiB, total ≈ **37.1 GiB**):

| Archivo | Tamaño |
|---|---|
| `ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors` | 18.15 |
| `gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors` | 12.04 |
| `ltx-2.5-22b_text_embedding_projection_bf16.safetensors` | 2.15 |
| `ltx-2.5-22b_video_embeddings_connector_int8_convrot.safetensors` | 1.50 |
| `ltx-2.5-22b_video_vae_bf16.safetensors` | 1.35 |
| `ltx-2.5-spatial-upscaler-x2-1.0_bf16.safetensors` | 0.93 |
| `ltx-2.5-22b_audio_embeddings_connector_int8_convrot.safetensors` | 0.38 |
| `ltx-2.5-22b_vocoder_bf16.safetensors` | 0.24 |
| `ltx-2.5-temporal-upscaler-x2-1.0_bf16.safetensors` | 0.24 |
| `ltx-2.5-22b_audio_vae_bf16.safetensors` | 0.10 |

> 💡 Los upscalers **espacial x2** y **temporal x2** ya están descargados.

---

## 🚨 8. Problemas conocidos y su causa

| # | Síntoma | Causa real | Solución fijada |
|---|---|---|---|
| 1 | `No module named 'models.ltx2'` | El notebook **nunca clonaba Wan2GP**; funcionaba por una carpeta que persistía en `/kaggle/working` | Bloque de clonado en la Celda 1 (antes de `# Permisos`) |
| 2 | `hf_transfer is enabled but not available` | La imagen Python 3.13 ya no trae `hf_transfer` | `pip install hf_transfer` en el venv de Qwen |
| 3 | `AmbiguousGlobalPerLayerAttributeError: 'head_dim'` | `transformers ≥ 5.15` cambió cómo se leen configs heterogéneas (Gemma 4) | `transformers==5.14.1` aislado en `ltx_hf_compat` |
| 4 | UI lentísima al subir/ver archivos | Túnel `share=True`: ~2 MB/s y 0.58 s de latencia, frente a 155 MB/s en local | Túnel Cloudflare alternativo |
| 5 | Varias UIs/túneles `frpc` abiertos | Re-ejecutar la celda de la UI no cierra la anterior | `gr.close_all()` + `pkill -f frpc` antes de relanzar |
| 6 | `ClientDisconnect` en Gradio | El navegador corta una subida | Inofensivo; reintentar |

---

## 📐 9. Notas de resolución y GPU

| Tema | Estado |
|---|---|
| **Dos GPUs** | No se usan: `mmgp` es de una sola GPU y la referencia tampoco las reparte. LTX oficial solo ofrece multi-GPU como reducción de latencia con el transformer replicado en cada GPU |
| **Resolución** | Usar lados **múltiplos de 32** (p. ej. **1280×704**, 1024×576, 1152×640, 1536×864). 1280×720 no lo es |
| **Techo real en T4** | ⏳ pendiente de medir con la celda de pruebas (25 frames) |
| **Modelo más pequeño** | No aumenta la resolución: el transformer usa un presupuesto fijo de 6000 MB de VRAM |
| **Palancas posibles** | `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,garbage_collection_threshold:0.6` · menos frames · upscaler espacial |

---

## ♻️ 10. Cómo volver a esta configuración

**Wan2GP (commit exacto):**
```bash
git clone https://github.com/deepbeepmeep/Wan2GP /kaggle/working/Wan2GP
git -C /kaggle/working/Wan2GP checkout 0e58385fbd
```

**Binarios C++ (commits exactos):**
```bash
git -C /kaggle/working/stable-diffusion.cpp checkout 3f8527a46c && \
git -C /kaggle/working/stable-diffusion.cpp submodule update --init --recursive
git -C /kaggle/working/qwentts.cpp checkout 6fae929140
```

**whisperx (commit exacto):**
```bash
pip install --no-deps "git+https://github.com/m-bain/whisperx.git@771b4a14a9"
```

**`transformers` aislado para LTX** (o `-r requirements-ltx-isolated.txt` con el archivo del snapshot):
```bash
pip install --target /kaggle/tmp/ltx_hf_compat --upgrade transformers==5.14.1 "huggingface_hub>=1.5.0,<2.0.0"
```

**Venv de Qwen3-TTS:** borrar `/tmp/qwen3_tts_env` y ejecutar la celda de Qwen. Pines: `qwen-tts==0.0.5 --no-deps` · `transformers==4.57.3` · `huggingface-hub==0.36.0` · `accelerate==1.12.0` · `gradio==6.4.0`.

**Cerrar instancias antes de relanzar la UI:**
```python
import gradio as gr, subprocess; gr.close_all(); subprocess.run("pkill -f frpc_linux_amd64", shell=True)
```

**Ajustes de la cuenta:** Internet **ON** (cuenta verificada por teléfono) · GPU **T4 ×2** · Python 3.13 / imagen Docker de §2 (*Settings → Environment → Pin to original environment* si está disponible) · persistencia de archivos si quieres conservar `/kaggle/working`.

---

## 📦 11. Archivos de respaldo

`/kaggle/working/env_snapshot/` → `ENV_SNAPSHOT.md` · `requirements-kernel.txt` · `requirements-ltx-isolated.txt` · `requirements-qwen-venv.txt`
**Descárgalos y guárdalos junto al notebook**: se pierden si cierras la sesión sin guardar versión.